
**Design Campaign 2** \
Author: Seth Woodbury \
Contact: woodbuse@uw.edu 

Start Jupyter from inside the cloned repository, so the notebook can locate the
repository root on its own:
```bash
conda activate zinc_hydro
jupyter lab
```

# **Organization & Notebook Setup**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  NOTEBOOK INITIALIZATION — run this cell at the start of every session
# ══════════════════════════════════════════════════════════════════════════════

PROJECT_NAME = 'phosphotriesterase_rfd3'

# > USER CONFIGURATION
# >> repository root -- auto-detected, nothing to edit. Override with $ZINC_HYDRO_REPO.
def _find_repo_root():
    import os as _os
    from pathlib import Path as _Path
    _markers = ('Scripts', 'Software', 'Environment', 'LICENSE')
    _start = _Path(_os.environ.get('ZINC_HYDRO_REPO') or _Path.cwd()).resolve()
    for _cand in (_start, *_start.parents):
        if all((_cand / _m).exists() for _m in _markers):
            return str(_cand)
    raise RuntimeError(
        f"Could not find the repository root above {_start}. "
        f"Start Jupyter from inside the cloned repository, or set ZINC_HYDRO_REPO."
    )

REPO_DIR      = _find_repo_root()
CAMPAIGN_DIR  = f'{REPO_DIR}/Design_Pipelines/Phosphotriesterase_RFdiffusion3/'
THEOZYME_DIR  = f'{CAMPAIGN_DIR}inputs/theozymes/'

# >> manual overrides (set to None to use defaults)
# Outputs are large. Point _OUTPUT_DIR_OVERRIDE at scratch for a real run;
# leave it as-is to keep everything inside the campaign directory.
_WORKING_DIR_OVERRIDE = CAMPAIGN_DIR
_OUTPUT_DIR_OVERRIDE  = f'{CAMPAIGN_DIR}outputs/'

# >> subdirectories to create
WORKING_SUBDIRS = ['cmds', 'submit', 'logs', 'graphs', 'rfd3_json', 'af3_json', 'important_dfs', 'fixed_residue_jsonls']
OUTPUT_SUBDIRS  = ['rfdiffusion3_out', 'predesign_out', 'af3_out', 'mpnn_out', 'af2_out', 'redesign_out']

# > IMPORTS
# >> standard library
import concurrent.futures, copy, glob, itertools, json, math, multiprocessing, os, operator
import random, re, shlex, shutil, statistics, string, subprocess, sys, textwrap, time, warnings
from collections import Counter, defaultdict, OrderedDict, deque
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed, FIRST_COMPLETED, ThreadPoolExecutor, wait
from datetime import datetime
from itertools import permutations, product, islice
from math import log10, floor
from pathlib import Path
from pprint import pprint
# >> third-party: data & math
import numpy as np, pandas as pd
from scipy.stats import gaussian_kde
# >> third-party: plotting
import matplotlib, matplotlib.pyplot as plt, seaborn as sns
from matplotlib.colors import to_rgba, ListedColormap, LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
# >> third-party: structural biology
import pyrosetta, pyrosetta.distributed.tasks.rosetta_scripts as rosetta_scripts
from Bio import PDB
from Bio.PDB import PDBParser, PPBuilder
from Bio.SeqUtils import seq1
# >> third-party: notebook & misc
from difflib import SequenceMatcher
from IPython.display import display, HTML
# >> custom modules
_NB_FUNCS_PATH = f'{REPO_DIR}/Scripts/notebook_functions'
if _NB_FUNCS_PATH not in sys.path:
    sys.path.insert(0, _NB_FUNCS_PATH)
import notebook_core as nb  # colors, setup_directories, SLURM helpers

# > PATHS
# >> working & output
WORKING_DIR = nb.resolve_working_dir(override=_WORKING_DIR_OVERRIDE, strip_mnt=True)             
OUTPUT_DIR  = nb.resolve_output_dir(WORKING_DIR, override=_OUTPUT_DIR_OVERRIDE, strip_mnt=True)       

# >> theozymes & ligand files (OK if these don't exist yet)
PARAMS_DIR = f'{THEOZYME_DIR}params/'
CST_DIR    = f'{THEOZYME_DIR}cst_files/'

# >> tools & software (all inside this repository)
SPECIAL_SCRIPTS_DIR = f'{REPO_DIR}/Scripts/'
SOFTWARE_DIR        = f'{REPO_DIR}/Software/'

# >> external tools not bundled here. Each falls back to an environment
#    variable; see the campaign README for what each one needs.
GIT_DIR            = os.environ.get('GIT_DIR', f'{REPO_DIR}/Software/')
AF3_DIR            = os.environ.get('AF3_DIR', f'{SOFTWARE_DIR}alphafold3')
AF3_RUNNER         = os.environ.get('AF3_RUNNER', f'{AF3_DIR}/run_alphafold.py')
AF3_SIF            = os.environ.get('AF3_SIF', '')      # container, if you use one

# >> resolve interpreter / container / obabel portably
if SPECIAL_SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SPECIAL_SCRIPTS_DIR)
import env_config
APPTAINER = os.environ.get('ZINC_HYDRO_SIF', '')   # '' -> run with the active python
MAXIT_SIF = os.environ.get('MAXIT_SIF', '')   # RCSB maxit container for CIF->PDB, if you use one

# > SETUP
for p in [WORKING_DIR, OUTPUT_DIR]:
    Path(p).mkdir(parents=True, exist_ok=True)
nb.setup_directories(WORKING_DIR, WORKING_SUBDIRS, export_globals=True, globals_dict=globals())
nb.setup_directories(OUTPUT_DIR,  OUTPUT_SUBDIRS,  export_globals=True, globals_dict=globals())
nb.set_pandas_display(all_on=True)

# > INITIALIZE
os.chdir(WORKING_DIR)
# nb.print_initialization(WORKING_DIR, OUTPUT_DIR, project_name=PROJECT_NAME, obabel_path=OBABEL_PATH, globals_dict=globals(), preview=True)
nb.print_initialization(WORKING_DIR, OUTPUT_DIR, project_name=PROJECT_NAME, globals_dict=globals(), preview=True)

In [ ]:
#####################################################
### Wait for Slurm job, then run local command(s) ###
#####################################################

import json
import shlex
import subprocess

### WAIT INPUTS ###
slurm_job_id  = "14563458"
mode          = "afterany"  # "afterany" runs regardless of final state; "afterok" requires all COMPLETED.

### COMMANDS TO RUN AFTER SLURM JOB FINISHES ###
# Paste any already-built command string here. Multiple commands run sequentially. For maximum robustness, use an argv list instead of a shell string: ["python", "/path/script.py", "--arg", "value with spaces and 'quotes'"]
commands = [
    r"""{APPTAINER} {SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/make_af3_json_input.py --pdb_path {OUTPUT_DIR}redesign_out/i1__ref_pdbs/ --pdb_chain A --json_path {WORKING_DIR}af3_json/i2__holo --num_input_per_run 50 --output_path {OUTPUT_DIR}af3_out/i2__holo/ --ligand_chain B C D --ligand_type ccdCodes smiles smiles --ligand_id "['ZN'] [Zn+2].[OH-] CCOP(=O)(OCC)OC1=CC=C(C=C1)[N+](=O)[O-]" --ptm_from_remark666 "A/LYS/3:KCX" --output_suffix _af3i2 --check_made_output --cleanup_incomplete_outputs --recursive --specific_depth 1""",
    #r""" """, # more commands
]

### EXECUTION OPTIONS ###
continue_on_error = False
dry_run           = False
optional_log_file = None  # None = no log file; watcher prints directly to the terminal/notebook.
poll_interval     = 60    # seconds

### CONSTANTS ###
watcher_script = f"{SPECIAL_SCRIPTS_DIR}automation_tools/wait_for_slurm_then_run.py"

### NORMALIZE COMMANDS ###
commands_argv = []
for i, command in enumerate(commands, 1):
    if isinstance(command, str):
        commands_argv.append(shlex.split(command))
    elif isinstance(command, (list, tuple)) and all(isinstance(part, str) for part in command):
        commands_argv.append(list(command))
    else:
        raise TypeError(f"commands[{i}] must be a shell string or a list/tuple of strings")

### BUILD WATCHER COMMAND ###
watcher_argv = [
    watcher_script,
    "--slurm-job-id", str(slurm_job_id),
    "--mode", mode,
    "--poll-interval", str(poll_interval),
]
if continue_on_error:
    watcher_argv.append("--continue-on-error")
if dry_run:
    watcher_argv.append("--dry-run")
if optional_log_file:
    watcher_argv.extend(["--log-file", optional_log_file])

commands_json = json.dumps(commands_argv, separators=(",", ":"))
watcher_argv.extend(["--commands-json", commands_json])
watcher_command = shlex.join(watcher_argv)

print("### COMMAND PREVIEW ###")
for i, argv in enumerate(commands_argv, 1):
    print(f"\n### Command {i} ###")
    print(shlex.join(argv))

print("\n### WATCHER COMMAND (RUN IN TERMINAL) ###")
print(watcher_command)

# **I. Theozyme Post-Processing for Rosetta**

**GENERAL REMINDER:** Rosetta does not play nice if you name your ligand something that pre-exists in its database, especially something like a sugar (e.g., LAT). You must avoid this now or it will cause you great pain and suffering later. \
\
These steps will likely be user-dependent and decently manual. 

## I.A Identify Active Site(s) of Interest from MLFF Calculations & Create Initial Theozymes (Assuming MLFF Came from Labeled Input)

**NOTE:** For the cropping make sure you take the whole residue, not just the sidechain. \
\
**NOTE:** Do not forget to add hydrogens to your ligand! Sadly, this must typically be done in pymol or by hand. I would also just get all of your different structures aligned at this step too for exporting the new structures (if working in pymol).\
\
STEP 1: "Clean" the PDB files such that we will make REMARK 666 lines, remove protein hydrogens, and combine all separate ligand(s)/cofactor(s) into one ligand complex with a name of your choice.

Note we manually removed the extra REMARK 666 lines

In [ ]:
####################################
### CLEAN PDB CRYSTAL STRUCTURES ###
####################################

### INPUTS ###
input_pdb   = f"{THEOZYME_DIR}p1D1_mlff/uncropped_theozyme/ZAPP_p1D1_active_site_NEB_TS.pdb"

ligand_code = "YYE"
output_pdb  = f"{THEOZYME_DIR}p1D1_mlff/ZAPP_p1D1_active_site_NEB_TS.pdb"

### OPTIONAL ORDERING FOR REMARK 666 LINES (FRONT/BACK) ###
remark_front = ["A93", "A89", "A16", "A170", "A133", "A92"]   # e.g. ["A244","A199"] or []
remark_back  = []                               # e.g. ["A207","A143"] or []

### ncAA FRAGMENTATION & HANDLING ###
frag_ncAA_targets               = []            # None if wanted false | e.g. ["A169", "A72"]; empty list + enable_frag_ncAA=True => fragment ALL ncAAs
disable_intelligent_hstrip      = False         # True -> turn OFF H-stripping logic around ncAA fragmentation

protect_ncAA_from_ligandization = False         # True -> temp HETATM->ATOM for ncAA, then revert | cannot be combined with enable_frag_ncAA
leave_ncAA_as_ATOM              = False         # True -> HETATM->ATOM for ncAA, never revert (mutually exclusive with protect) | cannot be combined with enable_frag_ncAA

### OPTIONAL FLAGS ###
protect_sidechain_polarH_targets = []           # None -> feature OFF; [] -> all protein sidechains; ["A32","A25"] -> specific residues
keep_precleaned_pdb              = False        # True -> keep __precleaned_INPUT.pdb for debugging
disable_resseq_suffix_cleanup    = False        # True -> keep 132N, 132A, etc. as-is
add_CA_to_labeled_frag           = True         # NEW: True -> enable CA construction from CB-H for missing CA residues (in the process of generalizing to any frag)

### CONSTANTS ###
main_script = f"{SPECIAL_SCRIPTS_DIR}theozyme_and_ligand_handling/prepare_PDB_structure_into_theozyme__MAIN.py"

### BUILD COMMAND ###
if protect_ncAA_from_ligandization and leave_ncAA_as_ATOM: # Sanity check on mutually exclusive options
    raise ValueError("protect_ncAA_from_ligandization and leave_ncAA_as_ATOM cannot both be True.")
if frag_ncAA_targets is not None and (protect_ncAA_from_ligandization or leave_ncAA_as_ATOM):
    raise ValueError("frag_ncAA_targets (i.e., fragmentation) cannot be combined with protect_ncAA_from_ligandization or leave_ncAA_as_ATOM (wrapper forbids this).")

cmd = ["python", main_script, "--input_pdb", input_pdb, "--output_pdb_path", output_pdb, "--ligand_complex_3_letter_name", ligand_code,]

if remark_front: cmd += ["--remark666_residue_front_order"] + remark_front # REMARK 666 ordering
if remark_back:  cmd += ["--remark666_residue_back_order"] + remark_back   # REMARK 666 ordering

if disable_resseq_suffix_cleanup:     cmd.append("--disable_resseq_suffix_cleanup")       # Pre-cleaning / ncAA behavior
if protect_ncAA_from_ligandization:   cmd.append("--protect_ncAA_from_ligandization")     # Pre-cleaning / ncAA behavior
if leave_ncAA_as_ATOM:                cmd.append("--leave_ncAA_as_ATOM")                  # Pre-cleaning / ncAA behavior

if frag_ncAA_targets is not None: # ncAA fragmentation into cAA + ligand
    cmd.append("--frag_ncAA_into_cAA_plus_lig")
    if frag_ncAA_targets: cmd += frag_ncAA_targets # [] => "all ncAAs"; non-empty list => specific residues

if disable_intelligent_hstrip: cmd.append("--disable_intelligent_hstrip")  # Intelligent H stripping toggle
if keep_precleaned_pdb:        cmd.append("--keep_precleaned_pdb")         # Debugging: keep intermediate precleaned file
if add_CA_to_labeled_frag:     cmd.append("--add_CA_to_labeled_frag")      # NEW: CA construction flag

if protect_sidechain_polarH_targets is not None: # Sidechain polar H protection
    cmd.append("--protect_sidechain_polarH")
    if protect_sidechain_polarH_targets: cmd += protect_sidechain_polarH_targets  # [] => apply to all protein residues

### CREATE OUTPUT DIR ###
os.makedirs(os.path.dirname(output_pdb), exist_ok=True)

### PRINT COMMAND ###
command_str = " ".join(cmd)
print(command_str)

## I.B Standardize Ligand Naming & Create Ligand Params Files from PDBs

Step 2: Standardize ligand naming for ligand subsets (if applicable). **SKIP FOR THIS DESIGN CAMPAIGN SINCE ONLY ONE LIGAND**

In [ ]:
###########################################################
### BUILD COMMAND TO STANDARDIZE LIGAND ATOM NAMES      ###
###########################################################

### INPUTS ###
ref_pdb   = f"{THEOZYME_DIR}mlff_TSopt/cleaned_theozymes/standardized_ligs/PTE_wGLU_pt9_set1_lig_YZW.pdb"
pdb_glob  = f"{THEOZYME_DIR}mlff_TSopt/cleaned_theozymes/PTE_wGLU_pt9_set*_lig_YZW*.pdb"

### LIGAND IDS (OPTIONAL) ###
ref_ligand    = None     # e.g. "YYE" or None to auto-detect if only one ligand present
target_ligand = None     # e.g. "YYE" or None to auto-detect per PDB

### ANCHOR MAP (REF:TARGET) ###
# These must be atom-name pairs with matching elements.
anchor_map = [
    "P1:P1",
    "ZN1:ZN1",
    "N1:N1",
]

### OUTPUT / BEHAVIOR OPTIONS ###
output_dir = f"{THEOZYME_DIR}mlff_TSopt/cleaned_theozymes/standardized_ligs"  # or None for in-place
dry_run    = False   # True -> do not write files, just print what would happen
verbose    = True    # True -> extra debug printouts

### CONSTANTS ###
script_path = f"{SPECIAL_SCRIPTS_DIR}theozyme_and_ligand_handling/standardize_ligand_atom_names_for_ligand_sets__MAIN.py"

### BUILD COMMAND ###
cmd_parts = ["python", script_path, "--ref_pdb", ref_pdb, "--pdbs_to_standardize_ligs", f"'{pdb_glob}'","--anchor_map",]
cmd_parts += anchor_map

if ref_ligand:
    cmd_parts += ["--ref_ligand", ref_ligand]

if target_ligand:
    cmd_parts += ["--target_ligand", target_ligand]

if output_dir:
    cmd_parts += ["--output_dir", output_dir]

if dry_run:
    cmd_parts.append("--dry_run")

if verbose:
    cmd_parts.append("--verbose")

### PRINT COMMAND ###
command_str = " ".join(cmd_parts)
print(command_str)

**NOTE:** This only works for single conformers unless you hault after XYZ creation. Separate scripts will be needed if you are working with conformers. \
\
Step 3: Specify your input PDBs (one at a time) and make params for them.

In [ ]:
#####################################################
### GENERATE SINGLE COMMAND FOR LIGAND EXTRACTION ###
#####################################################

lig = "YYE"

### REQUIRED PARAMETERS ###
input_single_pdb  = f"{THEOZYME_DIR}p1D1_mlff/ZAPP_p1D1_active_site_NEB_TS.pdb"
output_params_dir = f"{THEOZYME_DIR}params"
desired_code      = f"{lig}"

### OPTIONAL PARAMETERS ###
ligand_filters     = []        # e.g. ["ZN1", "LIG"], leave empty list to skip | automatically takes all ligands
stop_after_xyz     = False     # legacy behavior only: set True to only write XYZ and print downstream commands
stop_after_mol2    = False     # NEW: stop after mol2 (and optional bond-fix), do not run Rosetta
skip_bond_fix      = False     # NEW: skip MOL2 bond/connectivity fixing
preserve_pdb_order = True     # NEW: keep PDB ligand atom naming/order better (PDB->MOL2). False => legacy PDB->XYZ->MOL2

verbose           = True     # NEW: extra debug prints

### CONSTANTS ###
script_path       = f"{SPECIAL_SCRIPTS_DIR}theozyme_and_ligand_handling/extract_ligands_from_SINGLE_pdb_and_create_PARAMS__MODERN.py" # extract_ligands_from_SINGLE_pdb_and_create_PARAMS.py (old)

### GENERATE THE COMMAND ###
cmd = (f"python {script_path} --input_single_pdb {input_single_pdb} --output_dir_for_params_stuff {output_params_dir} --desired_ligand_3letter_code {desired_code}")

if ligand_filters:
    cmd += " --ligands_to_extract_via_3letter_code " + " ".join(ligand_filters)
if preserve_pdb_order:
    cmd += " --preserve_pdb_ligand_atom_order"
if stop_after_xyz:
    cmd += " --stop_after_XYZ_is_made"
if stop_after_mol2:
    cmd += " --stop_after_MOL2_is_made"
if skip_bond_fix:
    cmd += " --skip_bond_fix"
if verbose:
    cmd += " --verbose"

### CREATE OUTPUT DIR ###
os.makedirs(output_params_dir, exist_ok=True)
    
### PRINT THE COMMAND ###
print(cmd)

### *OPTIONAL:* Check if your Ligand Code Already Exists in Rosetta or the CCD to Avoid Downstream Problems

In [ ]:
####################################################
### BUILD COMMAND TO CHECK IF LIGAND CODE EXISTS ###
####################################################

### INPUT ###
ligand_code = "YYE"

### OPTIONAL ###
rosetta_txt_file = None # Default: resolved from $ROSETTA / $ROSETTA_RESIDUE_TYPES (see Scripts/repo_paths.py)

### CONSTANTS ###
script_path = f"{SPECIAL_SCRIPTS_DIR}theozyme_and_ligand_handling/check_if_ligand_3string_code_exists_in_rosetta.py"

### GENERATE THE COMMAND ###
cmd = (f"python {script_path} --alphanumerical_code {ligand_code}")

if rosetta_txt_file:
    cmd += " --stop_after_XYZ_is_made"

### PRINT THE COMMAND ###
print(cmd)

In [ ]:
##############################################################
### BUILD COMMAND TO SEARCH FOR NON-CCD LIGAND 3-LETTER CODES
##############################################################

### INPUT ###
# Define your classes and patterns here; ? = wildcard
classes = {
    "CLASS_0": "W?W",   # e.g. X0*
    "CLASS_A": "X?W",   # e.g. X0*
    "CLASS_B": "Y?W",   # e.g. X1*
    "CLASS_C": "T?W",   # e.g. T0*
    "CLASS_D": "Z?W",   # e.g. T0*
    "CLASS_E": "P?W",   # e.g. T0*
    "CLASS_F": "Q?W",   # e.g. T0*
    "CLASS_G": "E?W",   # e.g. T0*
    "CLASS_G": "G?W",   # e.g. T0*
}

### OPTIONAL ###
letters           = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"  # characters to use for '?'
target_per_class  = 1000        # number of unused codes to find per class
max_workers       = 64       # parallel HTTP workers
batch_size        = 128      # codes per batch
timeout           = 4.0      # HTTP timeout (s)
sleep_between     = 0.0      # sleep between classes
log_every         = 500      # progress print frequency
no_verify_ssl     = False    # set True if you ever need --no-verify-ssl

### CONSTANTS ###
apptainer_sif = APPTAINER
script_path   = f"{SPECIAL_SCRIPTS_DIR}theozyme_and_ligand_handling/find_nonCCD_lig_codes.py"

### BUILD COMMAND (AS LIST FIRST) ###
cmd_parts = ["apptainer", "exec", apptainer_sif, "python", script_path,]

# Add class specs
for name, pattern in classes.items():
    cmd_parts += ["--class", f"{name}:{pattern}"]

# Optional args (only add if not None / not default-ish)
if letters:                  cmd_parts += ["--letters", letters]
if target_per_class is not None: cmd_parts += ["--target-per-class", str(target_per_class)]
if max_workers is not None:  cmd_parts += ["--max-workers", str(max_workers)]
if batch_size is not None:   cmd_parts += ["--batch-size", str(batch_size)]
if timeout is not None:      cmd_parts += ["--timeout", str(timeout)]
if sleep_between:            cmd_parts += ["--sleep-between-classes", str(sleep_between)]
if log_every:                cmd_parts += ["--log-every", str(log_every)]
if no_verify_ssl:            cmd_parts.append("--no-verify-ssl")

### PRINT THE COMMAND ###
command_str = " ".join(cmd_parts)
print(command_str)

# **II. Theozyme Preparation for Scaffold Generation**

## II.A Enumeratively Sample Different His Tautomers/Ring Flips & GLU/ASP & Other Rotations

In [ ]:
###########################
### ENUMERATIVE SAMPLER ###
###########################

print_commands = True

### INPUT PDB LIST OR GLOB PATTERN ###
input_pdbs = [
    f"{THEOZYME_DIR}p1D1_mlff/ZAPP_p1D1_active_site_NEB_TS.pdb"
]

### PDB OUTPUT DIR (IF DIFFERENT FROM INPUT PDB DIR) ###
output_dir = f"{THEOZYME_DIR}step1_theozyme_prep" #f"{THEOZYME_DIR}combined_theozymes/theozyme_ORI/enumerate_confs/ed_group3_group4/"

### STEP 1: HISTIDINE FLIPPING PARAMETERS (IF DESIRED) ###
run_step1     = False
histidine_cfg = {
    "B2": ["0E","1E","0D"],
    "C3": ["0E","1E","1D"],
    "D4": ["0E","1E","0D"],
    "E5": ["0E","1E"],
}
allowed_tautomers = "0,1,2"       # e.g. up to 2 D‐type switches
classic_suffix = False            # use `_0E_1E_0D` style
verbose = False                   # print invoked STEP1/STEP2/STEP3 commands

### STEP 2: GLU / ASP TRANFORMATION (IF DESIRED) ###
run_step2     = False
gluE_aspD_map = {
    "F6": "ED",
    #"A1": "ED"
}

### STEP 3: CUSTOM ROTATION TRANSFORMATION (IF DESIRED) ###
## Carbamylated Lysine, C belongs to the ligand (B chain)
run_step3      = True
rotation_specs = [
    {
        "residue":   "A16", 
        "pivot":     "NZ",
        "axis":      ["B199:C1","A16:NZ"],
        "degrees":   180,
        "periodicity": 2
    }
]

### CONSTANTS ###
nprocs       = 20
main_script  = f"{SPECIAL_SCRIPTS_DIR}theozyme_and_ligand_handling/theozyme_cat_residue_enumerative_sampler__MAIN.py"
step1_script = None # default path
step2_script = None # default path
step3_script = None # default path 

### BUILD THE COMMAND ###
histidine_config = json.dumps(histidine_cfg)
gluE_aspD_json   = json.dumps(gluE_aspD_map)
rotation_config  = json.dumps(rotation_specs)
# common flags
cmd = ["python", main_script]
cmd += ["--input_pdbs"] + input_pdbs
cmd += ["--nprocs", str(nprocs)]
if output_dir:
    cmd += ["--output_dir", output_dir]

# STEP1 Flags
if run_step1:
    if histidine_config:
        cmd += ["--histidine_config", f"'{histidine_config}'"]
        if allowed_tautomers:
            cmd += ["--allowed_tautomer_swaps_at_once", allowed_tautomers]
        if classic_suffix:
            cmd.append("--classic_suffix")
        if verbose:
            cmd.append("--verbose_pdbs")
        if step1_script:
            cmd += ["--step1_script", step1_script]
# STEP2 Flags
if run_step2:
    if gluE_aspD_json:
        cmd += ["--gluE_aspD_json", f"'{gluE_aspD_json}'"]
        if step2_script:
            cmd += ["--step2_script", step2_script]
# STEP3 flags
if run_step3:
    if rotation_config:
        cmd += ["--rotation_config", f"'{rotation_config}'"]
        if step3_script:
            cmd += ["--step3_script", step3_script]

### PRINT COMMAND ###
if print_commands:
    print("### CONSTRUCTED COMMAND ###\n")
    print(" ".join(cmd))

## II.B Add ORI Tokens within a Sphere of a Pre-Determined Coordinate

**NOTE:** Only relevant if using the ORI-token flag in RFdiffusion2 at some point. Otherwise this can be ignored.

Donghyo Kim has some anecdotal empirical evidence showing that some ORI-tokens may work better than others, even if they are in really close proximity to each other. For this reason, it may be wise to pick a file (or files) to generate a bunch of random ORI-tokens within a sphere and do a few RFdiffusion2 runs for each to evaluate. 

My conclusion from running this ORI token analysis previously is that it is probably more worth one's time to just run RFdiffusion2 with all the initial ORI tokens - as this analysis is decently cost/time inefficient and it really doesn't filter the tokens down much unless you are more stringent. For me, I'd rather be liberal and allow for more tokens to get an increased diversity. 

Regardless, we need to add ORI-tokens if you want to use them.

STEP 1: To execute this, we need to take an example PDB file and then install a pseudo ORI token into it with this command in the pymol terminal:

*cmd.delete("molecule1");cmd.pseudoatom(object="molecule1", pos=[-1,3,2], elem="ORI", name="ORI", vdw=1.5, hetatm=True, chain='z', segi='z', resn="ORI"); cmd.show("sphere", "molecule1");*

STEP 2: Then you can click the 'Action Button' on molecule1 -> drag coordinates and drag the ORI token to a candidate spot for the center of a sampling sphere. To get the coordinates of the new ORI token for the input below, simply run this command in the pymol terminal:

*iterate_state 1, molecule1, print(name, x, y, z)* 
- **NOTE:** You must do this in the 'drag coords' mode do not hit 'done'

In [ ]:
############################
### ORI TOKEN GENERATION ###
############################

### INPUT PARAMETERS ###
input_pdb    = f'{THEOZYME_DIR}step1_theozyme_prep/*.pdb' # You can also use a glob pattern here, e.g.: input_pdb = f'{THEOZYME_DIR}mlff_TSopt/cleaned_theozymes/standardized_ligs/*.pdb'
output_dir   = f'{THEOZYME_DIR}step2_theozyme_prep_ori/'
center_coord = '0.8690689206123352 -0.5488795042037964 0.03344640135765076'

# Optional alt example:
# center_coord = '-5.401254653930664 -0.7388721704483032 16.641462326049805'

### SPHERE OPTIONS ###
generate_sphere = True           # False → single ORI at center
sphere_mode     = "volume"        # "surface" or "volume" (only used if generate_sphere=True)
radius          = 2               # Å (only used if generate_sphere=True)
num_ORI         = 15              # total ORIs if sphere=True (exact count, includes center)

### OTHER OPTIONAL FLAGS (ADVANCED CONTROLS) ###
specify_ori_token_specific_pdb_properties = False
chain        = "X"                # chain to contain ORI tokens | default = chain X
serial_start = 1                  # indexing to start ORI token atom numbering | default = 999 (NOTE: This must always be greater than the last HETATM atom numbering)
resseq_start = 1                  # indexing to start ORI token residue numbering | default = 1
verbose = True

### CONSTANTS ###
apptainer = APPTAINER  # keep your original runner
script = f"{SPECIAL_SCRIPTS_DIR}theozyme_and_ligand_handling/add_ORI_token_to_PDB.py"

### FIND INPUT PDB FILES ###
if any(ch in input_pdb for ch in "*?[]"):
    pdb_list = sorted(glob.glob(input_pdb))
else:
    pdb_list = [input_pdb]
if not pdb_list:
    raise FileNotFoundError(f"No PDB files matched pattern: {input_pdb}")

### GENERATE COMMAND ###
# NOTE: If your environment requires `apptainer exec`, replace the next line with: command = (f"apptainer exec {apptainer} python {script} ")
commands = []
for pdb_path in pdb_list:
    command = (f"{apptainer} {script} "
               f"--input_pdb {pdb_path} "
               f"--output_dir {output_dir} "
               f'--center "{center_coord}" ')

    if specify_ori_token_specific_pdb_properties:
        command += (f"--chain {chain} "
                    f"--serial_start {serial_start} "
                    f"--resseq_start {resseq_start} ")
    if generate_sphere:
        command += (f"--sphere "
                    f"--mode {sphere_mode} "
                    f"--radius {radius} "
                    f"--sampling_size {num_ORI} ")
    if verbose:
        command += "--verbose "

    commands.append(command)

### PRINT COMMAND ###
for cmd in commands:
    print(cmd)

**NOTE:** When viewing files, you can use the following command to make the spheres larger or smaller:

*set sphere_scale, num_value*       |||||||       where *num_value* = any positive real number (I like 0.5) ||||||| e.g., *set sphere_scale, 0.5* 

## II.C Create CST & Scoring Files for the Groups

Below I have made a script (for which you will generate commands for) that prepares the CST files for you. The script has specifications for "specs" which is detailed below: \
\
Each dict must include: \
    - residue_1_identifier__ChainResidue (e.g. "Z9") \
    - residue_1_atoms (list of 3 atom names) \
    - residue_2_identifier__ChainResidue \
    - residue_2_atoms \
    - primary (bool | this is if it is a primary sphere residue interacting w ligand) \
    - cst_block_of_upstream_protein_res (ONLY IF primary = False | this specifies which prior cst block/REMARK666 res to constrain to) \
    - covalent (bool) \
  Optional per-spec: \
    - name (string) \
    - dist_tol (float, overrides default 0.02) \
    - ang_tol  (float, overrides default 5.0) \
    - residue2_atom_type (True/False -> False default, True makes it atom_type which is good for GLU/TYR/HIS)... can also do for residue1 \
    - residue2_protonation ("HIS_D" for hydrogen on ND1, "HIS_E" for hydrogen on NE2). Works for residue1 too. Otherwise, this will automatically parse and assume that whatever atom you specified first (ie NE2 or ND1) is Nhis. Very specific for atom_type of HIS residue. \
    - dummy_block True or False
\
**NOTE:** Order should not matter except in niche cases like you have 2 constraints on one residue. \
\
**NOTE:** Trying to use hydrogens in CST files will create extreme pain down the line, I'm not sure why. Just constrain the nearest HETATM.

In [ ]:
#############################
### CST COMMAND GENERATOR ###
#############################

name = None #"group1_pte_pxon_ChenJACS_exact"

### IF NOT NAME ###
input_pdb    = f"{THEOZYME_DIR}p1D1_mlff/ZAPP_p1D1_active_site_NEB_TS.pdb"  #group2_pte_pxon_ChenJACS_smwTSopt_v0__lig_YYF.pdb" #group1_pte_pxon_ChenJACS_exact_v0__lig_YYE
output_cst   = f"{THEOZYME_DIR}cst_files/ZAPP_p1D1.cst"  #group2_pte_pxon_ChenJACS_smwTSopt.cst"   #group1_pte_pxon_ChenJACS_exact

### IF NAME ###
if name:
    input_pdb    = f"{THEOZYME_DIR}combined_theozymes/theozymes_with_crucial_hydrogen/{name}.pdb"
    output_cst   = f"{THEOZYME_DIR}combined_theozymes/cst_files/{name}.cst"
    
### CONSTRAINT SPECIFICATION (list of dicts) ###
specs = [
    ##### FRAG 1 #####
    # BLOCK 1
    {"residue_1_identifier__ChainResidue": "B199", "residue_1_atoms": ["ZN1","O3","P1"], 
     "residue_2_identifier__ChainResidue": "A93", "residue_2_atoms": ["NE2","CE1","ND1"],
     "primary": True, "covalent": True, "residue2_atom_type": True, "ang_tol": 2.5},
    
    # BLOCK 2
    {"residue_1_identifier__ChainResidue": "B199", "residue_1_atoms": ["ZN1","O3","P1"], 
     "residue_2_identifier__ChainResidue": "A89", "residue_2_atoms": ["NE2","CE1","ND1"],
     "primary": True, "covalent": True, "residue2_atom_type": True, "ang_tol": 2.5},
    
    # BLOCK 3
    {"residue_1_identifier__ChainResidue": "B199", "residue_1_atoms": ["C1","O1","ZN1"], 
     "residue_2_identifier__ChainResidue": "A16", "residue_2_atoms": ["NZ","CE","CD"], 
     "primary": True, "covalent": True, "residue2_atom_type": False, "ang_tol": 2.5},
    
    # BLOCK 4
    {"residue_1_identifier__ChainResidue": "B199", "residue_1_atoms": ["ZN2","O3","P1"], 
     "residue_2_identifier__ChainResidue": "A170", "residue_2_atoms": ["NE2","CE1","ND1"],
     "primary": True, "covalent": True, "residue2_atom_type": True, "ang_tol": 2.5},

    # BLOCK 5
    {"residue_1_identifier__ChainResidue": "B199", "residue_1_atoms": ["ZN2","O3","P1"], 
     "residue_2_identifier__ChainResidue": "A133", "residue_2_atoms": ["NE2","CE1","ND1"],
     "primary": True, "covalent": True, "residue2_atom_type": True, "ang_tol": 2.5},
    
    # BLOCK 6
    {"residue_1_identifier__ChainResidue": "B199", "residue_1_atoms": ["O3","P1","O7"], 
     "residue_2_identifier__ChainResidue": "A92", "residue_2_atoms": ["OE2","CD","CG"],
     "primary": True, "covalent": True, "residue2_atom_type": True, "ang_tol": 2.5},
    
]

### OPTIONAL FLAGS ###
verbose_flag = True  # Set to False to omit --verbose
keep_multiple_atom_types_flag = True # this keeps all atom_type (eg Nhis aroC Ntrp instead of just Nhis)

### CONSTANTS ###
script_path  = f"{SPECIAL_SCRIPTS_DIR}theozyme_and_ligand_handling/make_cst_file_from_pdb__MAIN.py"

### GENERATE COMMAND ###
specs_json = json.dumps(specs)
specs_quoted = shlex.quote(specs_json)

cmd = (
    f"python {shlex.quote(script_path)} "
    f"--input_pdb {shlex.quote(input_pdb)} "
    f"--output_cst {shlex.quote(output_cst)} "
    f"--specs {specs_quoted}"
)

if verbose_flag:
    cmd += " --verbose"
    
if keep_multiple_atom_types_flag:
    cmd += " --keep_multiple_atom_types"
    
### PRINT ###
print(cmd)

### *OPTIONAL:* CREATE CST FILES MANUALLY

Created CST files manually -> 1 for each input theozyme \
**NOTE:** DO NOT USE HYDROGEN IN YOUR CST FILES \
A worked example ships with this campaign:
`inputs/theozymes/cst_files/ZAPP_p1D1.cst`

In [ ]:
########################
### CREATE CST FILES ###
########################

### MAKE CST FILES DIRECTORY ###
# CST_DIR is defined in the initialization cell; this campaign ships one worked
# example there, ZAPP_p1D1.cst. Put one .cst per input theozyme alongside it.
cst_files_dir = CST_DIR
os.makedirs(cst_files_dir, exist_ok=True)

print(f"Put one .cst file per input theozyme here:\n{cst_files_dir}")

## II.F Rename a Ligand or Some Other Substring?

You can use a command like the following (it will snake through all directories changing all instances of YYK to YYN):

# **III. Generate Backbones at Scale**

## III.A Setup RFdiffusion3 Inference

In [ ]:
#####################################################################
### CELL TO HELP GENERATE THE CONTIG ATOMS DEFS FOR THE NEXT CELL ###
#####################################################################

### SPECIFY ATOMS FOR FIXED RESIDUES ###
residue_atoms_map = {
    ### SET 1 ###
    "A16":   ["NZ", "CE"],  # Lys **
    "A89":   ["NE2", "CE1", "ND1", "CG", "CD2", "CB"],  # His **
    "A92":   ["ALL"],  # Glu **
    "A93":   ["ALL"],  # His **
    "A133":   ["NE2", "CE1", "ND1", "CG", "CD2", "CB"],  # His **
    "A170":   ["NE2", "CE1", "ND1", "CG", "CD2", "CB"],  # His **
}

### BUILD IT ###
residue_list_str = ",".join(residue_atoms_map.keys())
atom_str_map = {res: ",".join(atoms) for res, atoms in residue_atoms_map.items()}
contig_atoms_defs = {1: json.dumps(atom_str_map)}

### PRINT ###
print(f"### FIXED ATOM DEFINITIONS ###", f"\n{contig_atoms_defs[1]}")
print(f"\n### RESIDUES LIST ###", f"\n\"{residue_list_str}\"")

### Generate JSON Inputs

**Generate JSON Inputs for Input PDBs**

This cell automates the creation of RFdiffusion3‐compatible JSON configuration files from a directory of PDBs.  Each JSON can encode:

- **Multiple parameter combinations** (“combos”) per PDB, if specified. Will create all possible combos based on lists given below.  
- **Multiple ORI tokens** per PDB, if present.  

---

**A) Grouping Definitions**

You define one or more **groups** keyed by a file‐name prefix.  Each group dictionary can include:

```python
grouping_info_by_prefix_andOR_substring = {
    "group1": {
        # ── Optional behavior ────────────────────────────────────────────────────────
        "suffix_to_add":          "SUFFIX",     
          • Appended to every JSON key *and* filename, e.g. `pdb1_combo1SUFFIX.json`
        "pdb_substring_grouping": "0_1_1",      
          • Further restricts matching to files containing this substring:
            `group1*0_1_1*.pdb`

        # ── Optional ligan inputs ───────────────────────────────────────────────────
        "ligand": "SZA",                        
          • Ligand identifier(s).  Must be a single string but can be a list (eg., "SZA,ZN").

        # ── Optional “set_id” parameters ────────────────────────────────────────────
        #   each may be an `int` or list `[int,...]` to generate multiple combos:
        "fixed_atoms_set_id":    [1,2,3],       
          • Picks which atom‐fixing definition to use (see `fixed_atoms_defs`)
        "unindexed_set_id":      1,             
          • Picks which residue‐unindexing string to use (see `unindexed_defs`)
        "contig_set_id":         1,             
          • Picks which contiguous segment definitions to use (see `contig_defs`)
        "length_set_id":         1,             
          • Picks which sequence‐length window to use (see `length_defs`)
        "unfix_sequence_set_id": 1,             
          • Picks which side‐chains to remove from motif (see `unfix_sequence_defs`)
        "atomwise_rasa_set_id":  1,             
          • Picks which per‐atom RASA map to include (see `atomwise_rasa_defs`)
        "atomwise_hbond_set_id": 1,             
          • Picks which per‐atom H‐bond map to include (see `atomwise_hbond_defs`)

        # ── Optional boolean flags ──────────────────────────────────────────────────
        "redesign_motif_sidechains": [True, False],
          • Whether to allow redesign of the motif side‐chains, this shows you would do combinations w True & False, a json for each
        # ── Overrides ───────────────────────────────────────────────────────────────
        "out_path":        "{base}_combo{combo_idx}.json",
          • Custom filename template; available formatting keys:
              {base}, {combo_idx}, {suffix}
    },
}


In [ ]:
##############################################
### 1) GENERATE JSON INPUTS FOR INPUT PDBS ###
##############################################

lig = "YYE" 

print_pdbs_processed = False

### INPUT PDB DIRECTORY (WITH ORI TOKENS IF YOU WANT THEM) ###
input_root    = f"{THEOZYME_DIR}step2_theozyme_prep_ori/"

### INFERENCE CONTROLS VIA GROUPINGS ###
grouping_info_by_prefix_andOR_substring = {
    "ZAPP": {
        # optional parsing & naming suffix behavior:
        "pdb_substring_grouping":   f"",
        "suffix_to_add":            "",
        # overrides:
        "dialect":                   2,
        "redesign_motif_sidechains": False,
        # motif scaffolding
        "ligand": f"{lig}",
        "length_set_id":             [1,2],
        "contig_set_id":             None,
        "unindexed_set_id":          1,
        "unfix_sequence_set_id":     None,
        "fixed_atoms_set_id":        [1,2],
        "ori_set_id":                None,
        # conditioning
        "atomwise_rasa_set_id":      1,#[1,2,3],
        "atomwise_hbond_set_id":     [1,2,3],
    },
}

### CONFIGURATION SET IDs ###
fixed_atoms_defs = {
    ### SET 1 ###
    1: {"A16": "NZ,CE", "A89": "NE2,CE1,ND1,CG,CD2,CB", "A92": "ALL", "A93": "ALL", "A133": "NE2,CE1,ND1,CG,CD2,CB", "A170": "NE2,CE1,ND1,CG,CD2,CB"},
    2: {"A16": "NZ,CE", "A89": "ALL", "A92": "ALL", "A93": "ALL", "A133": "NE2,CE1,ND1,CG,CD2,CB", "A170": "NE2,CE1,ND1,CG,CD2,CB"},
}

unindexed_defs = {
    1: "A16,A89,A92,A93,A133,A170", 
}

unfix_sequence_defs = {
    # 1: "F6,A1",
}

contig_defs = {
    # 1: "A1-3,147-247",
}

### GLOBAL PROPERTY SET IDs ###
length_defs = {
    1: "180-210",
    2: "210-240",
}

ori_defs = {
    1: [[-0.435, -0.012, -2.212], [1.234, 2.345, 3.456]],
}

### ATOM CONDITIONING SET IDs ###
atomwise_rasa_defs = {
    1: {"select_buried": {f"{lig}": "C1,O1,O2,ZN1,ZN2,P1,O3,O4,O5,O6,O7,C8,C9,C10,C11"},
    "select_partially_buried": {f"{lig}": "C5,C6,C7"},
    "select_exposed": {f"{lig}": "N1,O8,O9"}
    },
}


atomwise_hbond_defs = {
    1: {
        # "select_hbond_donor":   {"A1": {"N": 1}},
        "select_hbond_acceptor":{f"{lig}": "O5"}
    },
    2: {
        # "select_hbond_donor":   {"A1": {"N": 1}},
        "select_hbond_acceptor":{f"{lig}": "O7"}
    },
    3: {
        # "select_hbond_donor":   {"A1": {"N": 1}},
        "select_hbond_acceptor":{f"{lig}": "O5,O7"}
    },
}

### WHERE TO DUMP JSONS ###
RFD3_JSON_DIR = RFD3_JSON_DIR
os.makedirs(RFD3_JSON_DIR, exist_ok=True)

### FUNCTION ###
def to_list(x):
    return x if isinstance(x, (list, tuple)) else [x]

### MAKE THE JSON FILES ###
print(f"{RFD3_JSON_DIR}")
overall_total_pdbs = 0
overall_matched    = 0
overall_jsons      = 0

for prefix, cfg in grouping_info_by_prefix_andOR_substring.items():
    print(f"\n=== Group '{prefix}' ===")
    substr    = cfg.get("pdb_substring_grouping", "")
    pattern   = f"{prefix}*{substr}*.pdb" if substr else f"{prefix}*.pdb" # pattern   = f"{prefix}*{substr}" if substr else f"{prefix}*.pdb"
    pdb_paths = sorted(glob.glob(os.path.join(input_root, pattern)))
    group_total = len(pdb_paths)
    overall_total_pdbs += group_total
    print(f"Found {group_total} PDBs matching '{pattern}'")

    # build combos once --> build it dynamically:
    #  1) collect all the “*_set_id” keys
    set_id_keys = sorted(k for k in cfg.keys() if k.endswith("_set_id"))
    #  2) collect the boolean flags (if present)
    flag_order = ["redesign_motif_sidechains"]
    flag_keys  = [k for k in flag_order if k in cfg]
    #  3) concatenate
    param_names = set_id_keys + flag_keys
    param_values = [to_list(cfg.get(name)) for name in param_names]
    combos       = list(itertools.product(*param_values))
    combo_count  = len(combos)
    print(f"Will generate {combo_count} JSON combos per PDB")

    group_matched = 0
    group_jsons   = 0

    for pdb_path in pdb_paths:
        base = os.path.splitext(os.path.basename(pdb_path))[0]

        # determine ORI_tokens either from cfg or by parsing the PDB
        ori_id = cfg.get("ori_set_id")
        if ori_id is not None:
            ORI_tokens = ori_defs[ori_id]
        else:
            # fall back to parsing every HETATM … ORI line
            ORI_tokens = []
            with open(pdb_path) as f:
                for line in f:
                    if line.startswith("HETATM") and " ORI " in line:
                        cols = line.split()
                        ORI_tokens.append([float(cols[6]), float(cols[7]), float(cols[8])])
                        
        multi_ori = len(ORI_tokens) > 1
        group_matched += 1
        overall_matched += 1
        if print_pdbs_processed:
            print(f" - '{base}': {len(ORI_tokens)} ORI token(s)")

        suffix = cfg.get("suffix_to_add", "")
        for combo_idx, combo in enumerate(combos, start=1):
            combo_dict = dict(zip(param_names, combo))
            json_dict  = {}
            # if there are no ORI tokens, we still want one pass, but without adding the field
            ori_iter = ORI_tokens if ORI_tokens else [None]

            for idx, ori in enumerate(ori_iter, start=1):
                entry = {"input": pdb_path}
                ### OVERRIDES ###
                if "dialect" in cfg:
                    entry["dialect"] = cfg["dialect"]
                # boolean flags
                for flag in ("redesign_motif_sidechains",): # ADD MORE BOOLEAN FLAGS HERE
                    val = combo_dict[flag]
                    if val is not None:
                        entry[flag] = val
                ### MOTIF SCAFFOLDING ###
                if "ligand" in cfg:
                    entry["ligand"] = to_list(cfg["ligand"])[0]
                if combo_dict["length_set_id"] is not None:
                    entry["length"] = length_defs[combo_dict["length_set_id"]]
                if combo_dict["contig_set_id"] is not None:
                    entry["contig"] = contig_defs[combo_dict["contig_set_id"]]
                if combo_dict["unindexed_set_id"] is not None:
                    entry["unindex"] = unindexed_defs[combo_dict["unindexed_set_id"]]
                if combo_dict["unfix_sequence_set_id"] is not None:
                    entry["select_unfix_sequence"] = unfix_sequence_defs[combo_dict["unfix_sequence_set_id"]]
                if combo_dict["fixed_atoms_set_id"] is not None:
                    entry["select_fixed_atoms"] = fixed_atoms_defs[combo_dict["fixed_atoms_set_id"]]
                if combo_dict["atomwise_rasa_set_id"] is not None:
                    entry.update(atomwise_rasa_defs[combo_dict["atomwise_rasa_set_id"]])
                if combo_dict["atomwise_hbond_set_id"] is not None:
                    entry.update(atomwise_hbond_defs[combo_dict["atomwise_hbond_set_id"]])

                ### ORI TOKEN INSERTION ###
                if ori is not None:
                    entry["ori_token"] = ori
                ### NAMING ###
                if suffix:
                    suffix_key = suffix + "_"
                else:
                    suffix_key = ""
                if multi_ori:
                    key = f"ORI_{idx}_i"
                else:
                    key = f"i"
                json_dict[key] = entry

            # write JSON
            if multi_ori:
                default_out = os.path.join(RFD3_JSON_DIR, f"{base}_C{combo_idx}{suffix_key}.json")
            else:
                default_out = os.path.join(RFD3_JSON_DIR, f"{base}_C{combo_idx}{suffix_key}.json")
            json_file = os.path.join(default_out)
            with open(json_file, "w") as fo:
                json.dump(json_dict, fo, indent=2) 
            group_jsons += 1
            overall_jsons += 1

    ### PRINT GROUP SUMMARY ###
    print(f"Group '{prefix}' summary:")
    print(f"  PDBs processed:     {group_matched}/{group_total}")
    print(f"  Combos per PDB:     {combo_count}")
    print(f"  JSON files created: {group_jsons}")
    print("-" * 60)

### PRINT OVERALL SUMMARY ###
print("\n=== Overall Summary ===")
print(f"Total PDBs found:           {overall_total_pdbs}")
print(f"Total PDBs with ORI tokens: {overall_matched}")
print(f"Total JSON files generated: {overall_jsons}")

### Production Run (with Optional Hyperparameter Sweeping)

In [ ]:
##############################################
### BUILD COMMAND LINES FOR RFD3 INFERENCE ###
##############################################

json_filters = [
    "ZAPP",
]  # e.g. ["group1","group2"], or [] to include all

### INPUTS ###
base_output_dir      = f"{RFDIFFUSION3_OUT_DIR}i1/"

### INFERENCE PARAMETERS (single–value defaults, but we’ll override) ###
diffusion_batch_size         = 8
n_batches                    = 40
dump_trajectories            = False
cleanup_virtual_atoms        = True

# ── 1) the lists you want to sweep ─────────────────────────────────────────
use_classifier_free_guidance_list = [True]
cfg_scale_list                    = [1.5]
step_scale_list                   = [1.5]
gamma_0_list                      = [0.6]
gamma_min_list                    = [0.1]
s_jitter_origin_list              = [1.5]

### DIRECTORY TO PARSE FOR JSONS & COMMANDS NAMING ###
json_dir      = f"{RFD3_JSON_DIR}"
commands_name = "RFdiffusion3_production"

### RFD3 INFERENCE CLI ###
# Commands invoke the `rfd3 design` CLI from the foundry package
# (Software/foundry, branch=production) -- the same open-source entry point the
# RFdiffusion3 tutorial uses. Set up once before running:
#
#   # Editable install of the bundled submodule (recommended -- pinned to the tested commit):
#   pip install -e "Software/foundry[rfd3]"
#   # OR install from PyPI:
#   pip install "rc-foundry[rfd3]"
#
#   # Download the checkpoint (defaults to ~/.foundry/checkpoints):
#   foundry install rfd3 --checkpoint-dir <path/to/ckpt/dir>
#
# Foundry auto-discovers checkpoints from ~/.foundry/checkpoints plus any
# colon-separated directories in $FOUNDRY_CHECKPOINT_DIRS, so an explicit
# ckpt_path is usually unnecessary. See Software/foundry/models/rfd3/README.md.
checkpoint_path = None    # e.g. os.path.expanduser("~/.foundry/checkpoints/rfd3_latest.ckpt")
os.makedirs(base_output_dir, exist_ok=True)
if json_filters:
    json_files = sorted(set(m for substr in json_filters for m in glob.glob(os.path.join(json_dir, f"*{substr}*.json"))))
else:
    json_files = sorted(glob.glob(os.path.join(json_dir, "*.json")))
print(f"Found {len(json_files)} JSON file(s) matching filters: {json_filters}")

commands_file = os.path.join(CMDS_DIR, commands_name); command_count = 0
with open(commands_file, "w") as fh:
    for cfg_path in sorted(json_files):
        for use_classifier_free_guidance in use_classifier_free_guidance_list:
            # only sweep cfg_scale when guidance is on
            cfg_scales = cfg_scale_list if use_classifier_free_guidance else [None]
            for cfg_scale in cfg_scales:
                for step_scale, gamma_0, gamma_min, s_jitter_origin in product(step_scale_list, gamma_0_list, gamma_min_list, s_jitter_origin_list):
                    seed = random.randint(0, 2**32 - 1)
                    # build a unique out_dir
                    out_dir = f"{base_output_dir}cfg_{'T' if use_classifier_free_guidance else 'F'}__cfgsc_{format(cfg_scale, '.2f').replace('.', '_') if cfg_scale is not None else 'NA'}__step_{format(step_scale, '.2f').replace('.', '_')}__gam0_{format(gamma_0, '.2f').replace('.', '_')}__gamMIN_{format(gamma_min, '.2f').replace('.', '_')}__jit_{format(s_jitter_origin, '.2f').replace('.', '_')}"
                    os.makedirs(out_dir, exist_ok=True)
                    cmd = (
                        f"rfd3 design inputs={cfg_path} out_dir={out_dir} "
                        + (f"ckpt_path={checkpoint_path} " if checkpoint_path is not None else "")
                        + f"dump_trajectories={dump_trajectories} diffusion_batch_size={diffusion_batch_size} n_batches={n_batches} cleanup_virtual_atoms={cleanup_virtual_atoms} "
                        + f"inference_sampler.s_jitter_origin={s_jitter_origin} inference_sampler.use_classifier_free_guidance={use_classifier_free_guidance} "
                        + (f"inference_sampler.cfg_scale={cfg_scale} " if cfg_scale is not None else "")
                        + f"inference_sampler.step_scale={step_scale} inference_sampler.gamma_0={gamma_0} inference_sampler.gamma_min={gamma_min} seed={seed} "
                        + f"skip_existing=False prevalidate_inputs=True"
                    )
                    fh.write(cmd + "\n")
                    command_count += 1
            
### SETUP BATCH JOBS ###
qtime        = '08:00:00'
cmds_per_job = 3
cores        = '1'
memory       = '16g'
queue        = 'gpu'
job_name     = os.path.basename(commands_file)
submit_file  = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs     = math.ceil(command_count / cmds_per_job)

print(f"\nStructures per Cmd =", diffusion_batch_size*n_batches, f"\nTotal Structures to Generate =", diffusion_batch_size*n_batches*command_count)
print(f"\nNumber of Cmds =", command_count, f"\nNumber of Jobs =", num_jobs)
print("Job Name =", job_name)
print("\nCommands File for Testing:", f"\n{commands_file}")
print(f"\nNavigate to Output Directory:", f"\ncd {base_output_dir}\n")
nb.submit_array_job(commands_file, qtime, cores, job_name,memory, submit_file, LOGS_DIR,num_jobs+1, cmds_per_job, queue)

## III.B Rapid Output Quality Filtering (Using RFd3 Jsons)

### Parse JSONs into Combined File

In [ ]:
############################################
### COMBINE JSON FILES INTO SINGLE JSON  ###
############################################

### INPUT DIRECTORY TO PARSE ###
RFD3_OUTPUT_DIR_TO_PARSE = f"{RFDIFFUSION3_OUT_DIR}i1/"

### OUTPUT JSON PATH ###
OUTPUT_COMBINED_JSON_PATH = f"{RFDIFFUSION3_OUT_DIR}i1/combined_stats.json"

### PARSING & SORTING LOGIC ###
json_files = glob.glob(f'{RFD3_OUTPUT_DIR_TO_PARSE}**/*.json', recursive=True) # sort alphanumerically by absolute path
json_files = sorted(json_files, key=lambda p: os.path.abspath(p)) # skip our own combined_stats.json if it lives in the same folder
json_files = [ p for p in json_files # skip our own combined_stats.json if it lives in the same folder
    if os.path.abspath(p) != os.path.abspath(OUTPUT_COMBINED_JSON_PATH)
]
print(f"Found {len(json_files)} JSON files to parse")

all_records = []
for path in json_files:
    # load the JSON
    with open(path, 'r') as f:
        rec = json.load(f)

    # compute our metadata
    ABS_PATH   = os.path.abspath(path)
    CIFGZ_PATH = ABS_PATH.replace('.json', '.cif.gz')
    SUBDIR     = os.path.basename(os.path.dirname(path))

    # build a new dict so that our three keys come first
    ordered_rec = {
        'rfd3_json_path':  ABS_PATH,
        'rfd3_cifgz_path': CIFGZ_PATH,
        'subdirectory':    SUBDIR,
    }

    # ensure rec is a dict before merging
    if isinstance(rec, dict):
        ordered_rec.update(rec)
        ### ADD: MULTIPLE NORMALIZED Rg METRICS USING NESTED METRICS #
        metrics_dict = ordered_rec.get("metrics")
        if isinstance(metrics_dict, dict):
            Rg = metrics_dict.get("radius_of_gyration", None)
            N  = metrics_dict.get("num_residues", None)
            try:
                Rg = float(Rg)
                N  = float(N)
            except (TypeError, ValueError):
                Rg = None
                N  = None
            if Rg is not None and N is not None and N > 0:
                # 1) Globular-like scaling: Rg / N^(1/3)
                metrics_dict["radius_of_gyration.norm_by_globularity"] = Rg / (N ** (1.0 / 3.0))

                # 2) Coil-like scaling: Rg / N^0.58  (approx. random-coil exponent)
                metrics_dict["radius_of_gyration.norm_by_coil"] = Rg / (N ** 0.58)

                # 3) Ideal-sphere normalization (dimensionless sphericity factor)
                v_res = 110.0      # Å^3 per amino acid (average)
                conv  = 1.212e-3   # Å^3 -> nm^3 (incl. packing/hydration fudge)
                V_nm3 = N * v_res * conv
                R_sphere_nm = (3.0 * V_nm3 / (4.0 * np.pi)) ** (1.0 / 3.0)
                R_sphere_A  = R_sphere_nm * 10.0  # back to Å
                #metrics_dict["radius_of_gyration.ideal_sphere"] = R_sphere_A
                metrics_dict["radius_of_gyration.norm_by_ideal_sphere"] = (Rg / R_sphere_A if R_sphere_A > 0.0 else None)
            # write back (not strictly necessary since it's same object, but explicit)
            ordered_rec["metrics"] = metrics_dict
        ##############################################################
        all_records.append(ordered_rec)
    else:
        print(f"⚠️  Skipping {path!r}: top‐level JSON is not an object")

# write combined JSON
with open(OUTPUT_COMBINED_JSON_PATH, 'w') as out:
    json.dump(all_records, out, indent=2)
print(f"Saved {len(all_records)} records → {OUTPUT_COMBINED_JSON_PATH}")

### OPTIONAL: Filter & Investigate Hyperparameter Sweep

In [ ]:
###########################################
### LOAD COMBINED JSON & FILTER METRICS ###
###########################################

### COMBINED JSON PATH FROM ABOVE ###
OUTPUT_COMBINED_JSON_PATH = f"{RFDIFFUSION3_OUT_DIR}i1/combined_stats.json"

# Written by RFdiffusion3 inference plus the JSON-merge cell above. Run those
# first; this cell reads their output.
if not os.path.exists(OUTPUT_COMBINED_JSON_PATH) or not json.load(open(OUTPUT_COMBINED_JSON_PATH)):
    raise FileNotFoundError(
        f"No RFdiffusion3 statistics found at {OUTPUT_COMBINED_JSON_PATH}.\n"
        f"  Run the inference commands emitted above, then the JSON-merge cell,\n"
        f"  before running this filtering step."
    )


### HELPER TO BUILD EXTREME FUNCTIONS ###
def take_json_nest_min_or_max(prefix: str, agg: str = 'max'):
    def fn(df: pd.DataFrame) -> pd.Series:
        cols = df.filter(regex=rf'^{prefix}\.').columns
        if agg == 'max':
            return df[cols].max(axis=1)
        elif agg == 'min':
            return df[cols].min(axis=1)
        else:
            raise ValueError("agg must be 'max' or 'min'")
    fn.__name__ = f"{agg}_{prefix.split('.')[-1]}"
    return fn

### FILTERS ###
conditions = [
    ### sidechain quality ###
    (take_json_nest_min_or_max('metrics.join_point_rmsd_by_token', 'max'), '<', 0.8),
    #('metrics.insertion.mae', '<', 0.6), # MAE = (no alignment) | really just for debugging
    ('metrics.insertion.rmcd', '<', 0.6), # RMCD = (centers but no rotation) - should agree w RMSD
    ('metrics.insertion_rmsd', '<', 0.6), # RMSD = (full optimal alignment) - best metric
    ('metrics.join_point_rmsd', '<', 0.6),
    ('metrics.n_conjoined_residues', '<=', 0), # | NOT SURE WHAT THIS MEANS
    ### diversity content ###
    ('metrics.alanine_content', '<', 0.4),
    ('metrics.glycine_content', '<', 0.3),
    #('metrics.num_ss_elements', '<', 9),
    ('metrics.non_loop_fraction', '>', 0.4),
    ('metrics.loop_fraction', '<', 0.6),
    ('metrics.helix_fraction', '>', 0.05),
    ('metrics.sheet_fraction', '>', 0.000),

    ### backbone quality ###
    ('metrics.max_ca_deviation', '<', 4.0), # 3.8 Å between consecutive Cα atoms is ideal
    ('metrics.n_chainbreaks', '<=', 1),
    ('metrics.n_clashing.interresidue_clashes_w_sidechain', '<=', 3), # includes any sc-sc clash, usually resolved by seq design
    ('metrics.n_clashing.interresidue_clashes_w_backbone', '<=', 0),
   # ('metrics.n_clashing.ligand_min_distance', '>', 2), # backbone nearest dist with ligand
   # ('metrics.radius_of_gyration', '<', 18), # NON-NORMALIZED by number of residues (N)
    ('metrics.radius_of_gyration.norm_by_globularity', '<', 2.75), # NORMALIZED (N^(1/3))
    #('metrics.radius_of_gyration.norm_by_coil', '<', 18), # NORMALIZED (N^(0.58))
    ('metrics.radius_of_gyration.norm_by_ideal_sphere', '<', 0.86), # NORMALIZED by ideal protein sphere | Values ~~1 = very compact / sphere-like &&& Values ≫1 = elongated / dumbbell / multi-lobed
]

### SORT ###
number_of_sorted_to_print = 20
metric_to_sort            = "metrics.loop_fraction" # OR FUNCTION --> take_json_nest_min_or_max('metrics.join_point_rmsd_by_token','max')    # e.g. 'metrics.insertion_rmsd', 'metrics.join_point_rmsd', etc.
sort_by_highest_values    = True                         # True → top N highest values; False → bottom N lowest values

### FILTERING LOGIC ###
with open(OUTPUT_COMBINED_JSON_PATH, 'r') as f:
    records = json.load(f)
df    = pd.json_normalize(records, sep='.')
total = len(df)
print(f"Total records: {total}\n")

# prepare aligned printing
labels      = [(col.__name__ if callable(col) else col) for col, *_ in conditions]
label_width = max(len(str(l)) for l in labels)
opval_strs  = [f"{op} {val}" for *_, op, val in conditions]
opval_width = max(len(s) for s in opval_strs)
num_width   = len(str(total))
op_funcs = {'<': operator.lt, '<=': operator.le, '>': operator.gt, '>=': operator.ge}

# Print each filter’s pass count, aligned
for (col, op, val), label, opval in zip(conditions, labels, opval_strs):
    series = col(df) if callable(col) else df[col]
    mask   = op_funcs[op](series, val)
    count  = mask.sum()
    pct    = count / total * 100
    print(f"[{label:<{label_width}} {opval:<{opval_width}} ]:  "
          f"{count:>{num_width}} / {total:<{num_width}}  ({pct:6.3f}%)")

# Combined filter
combined_mask = pd.Series(True, index=df.index)
for col, op, val in conditions:
    series        = col(df) if callable(col) else df[col]
    combined_mask &= op_funcs[op](series, val)

combined_count = combined_mask.sum()
combined_pct   = combined_count / total * 100
print(f"\nPassed All Conditions → {combined_count} / {total} ({combined_pct:.3f}%)")

### GROUP BY SUBDIRECTORY AND COMPUTE PASS PERCENTAGES ###
# Mark which rows passed
df['passed'] = combined_mask
sub_stats = (df.groupby('subdirectory')['passed'].agg(total='size', passed_count='sum').assign(pass_pct=lambda d: d['passed_count'] / d['total'] * 100).sort_values('pass_pct', ascending=False))
print("Subdirectories ranked by filter pass % (top 10):")
print(sub_stats.head(10)[['total', 'passed_count', 'pass_pct']])
rfd3_json_df_filtered = df[combined_mask]

### SORT PASSING STRUCTURES BY A VALUE & PRINT THEIR PATHS ###
df_tmp = (rfd3_json_df_filtered.assign(_sv=lambda d: metric_to_sort(d)).sort_values('_sv', ascending=not sort_by_highest_values)) if callable(metric_to_sort) else rfd3_json_df_filtered.sort_values(metric_to_sort, ascending=not sort_by_highest_values)
col = '_sv' if callable(metric_to_sort) else metric_to_sort

sel = df_tmp.head(number_of_sorted_to_print)
names = [os.path.basename(p) for p in sel['rfd3_cifgz_path']]
svals = [f"{v:.4f}" for v in sel[col]]
w1, w2 = max(map(len, names)), max(map(len, svals))

print(f"\n{'TOP' if sort_by_highest_values else 'BOTTOM'} {number_of_sorted_to_print} "
      f"{'highest' if sort_by_highest_values else 'lowest'} {col}:")
for n, s in zip(names, svals): print(f"  {s:<{w2}} ----> {n:<{w1}}")
print("\npymol", *sel['rfd3_cifgz_path'])

### Filter

In [ ]:
#############################################
### LOAD COMBINED JSON & FILTER METRICS   ###
#############################################

### COMBINED JSON PATH FROM ABOVE ###
OUTPUT_COMBINED_JSON_PATH = f"{RFDIFFUSION3_OUT_DIR}i1/combined_stats.json"

# Written by RFdiffusion3 inference plus the JSON-merge cell above. Run those
# first; this cell reads their output.
if not os.path.exists(OUTPUT_COMBINED_JSON_PATH) or not json.load(open(OUTPUT_COMBINED_JSON_PATH)):
    raise FileNotFoundError(
        f"No RFdiffusion3 statistics found at {OUTPUT_COMBINED_JSON_PATH}.\n"
        f"  Run the inference commands emitted above, then the JSON-merge cell,\n"
        f"  before running this filtering step."
    )


### HELPER TO BUILD EXTREME FUNCTIONS ###
def take_json_nest_min_or_max(prefix: str, agg: str = 'max'):
    def fn(df: pd.DataFrame) -> pd.Series:
        cols = df.filter(regex=rf'^{prefix}\.').columns
        if agg == 'max':
            return df[cols].max(axis=1)
        elif agg == 'min':
            return df[cols].min(axis=1)
        else:
            raise ValueError("agg must be 'max' or 'min'")
    fn.__name__ = f"{agg}_{prefix.split('.')[-1]}"
    return fn

### FILTERS ###
conditions = [
    ### sidechain quality ###
    (take_json_nest_min_or_max('metrics.join_point_rmsd_by_token', 'max'), '<', 0.6),
   # ('metrics.insertion.mae', '<', 0.6), # MAE = (no alignment) | really just for debugging
    ('metrics.insertion.rmcd', '<', 0.45), # RMCD = (centers but no rotation) - should agree w RMSD
    ('metrics.insertion_rmsd', '<', 0.45), # RMSD = (full optimal alignment) - best metric
    ('metrics.join_point_rmsd', '<',0.45),
    ('metrics.n_conjoined_residues', '<=', 0), # | NOT SURE WHAT THIS MEANS
    ### diversity content ###
    ('metrics.alanine_content', '<', 0.4),
    ('metrics.glycine_content', '<', 0.3),
    #('metrics.num_ss_elements', '<', 9),
    ('metrics.non_loop_fraction', '>', 0.5),
    #('metrics.loop_fraction', '<', 0.6),
    #('metrics.helix_fraction', '>', 0.05),
    #('metrics.sheet_fraction', '>', 0.000),

    ### backbone quality ###
    ('metrics.max_ca_deviation', '<', 4.0), # 3.8 Å between consecutive Cα atoms is ideal
    ('metrics.n_chainbreaks', '<=', 0),
    ('metrics.n_clashing.interresidue_clashes_w_sidechain', '<=', 1), # includes any sc-sc clash, usually resolved by seq design
    ('metrics.n_clashing.interresidue_clashes_w_backbone', '<=', 0),
    ('metrics.n_clashing.ligand_min_distance', '>', 2.5), # backbone nearest dist with ligand
   # ('metrics.radius_of_gyration', '<', 18), # NON-NORMALIZED by number of residues (N)
    ('metrics.radius_of_gyration.norm_by_globularity', '<', 2.75), # NORMALIZED (N^(1/3))
    #('metrics.radius_of_gyration.norm_by_coil', '<', 18), # NORMALIZED (N^(0.58))
   # ('metrics.radius_of_gyration.norm_by_ideal_sphere', '<', 0.86), # NORMALIZED by ideal protein sphere | Values ~~1 = very compact / sphere-like &&& Values ≫1 = elongated / dumbbell / multi-lobed
]

### SORT ###
number_of_sorted_to_print = 10
metric_to_sort            = "metrics.radius_of_gyration.norm_by_ideal_sphere" # OR FUNCTION --> take_json_nest_min_or_max('metrics.join_point_rmsd_by_token','max')    # e.g. 'metrics.insertion_rmsd', 'metrics.join_point_rmsd', etc.
sort_by_highest_values    = True                         # True → top N highest values; False → bottom N lowest values

### FILTERING LOGIC ###
with open(OUTPUT_COMBINED_JSON_PATH, 'r') as f:
    records = json.load(f)
df    = pd.json_normalize(records, sep='.')
total = len(df)
print(f"Total records: {total}\n")

# prepare aligned printing
labels      = [(col.__name__ if callable(col) else col) for col, *_ in conditions]
label_width = max(len(str(l)) for l in labels)
opval_strs  = [f"{op} {val}" for *_, op, val in conditions]
opval_width = max(len(s) for s in opval_strs)
num_width   = len(str(total))
op_funcs = {'<': operator.lt, '<=': operator.le, '>': operator.gt, '>=': operator.ge}

# Print each filter’s pass count, aligned
for (col, op, val), label, opval in zip(conditions, labels, opval_strs):
    series = col(df) if callable(col) else df[col]
    mask   = op_funcs[op](series, val)
    count  = mask.sum()
    pct    = count / total * 100
    print(f"[{label:<{label_width}} {opval:<{opval_width}} ]:  "
          f"{count:>{num_width}} / {total:<{num_width}}  ({pct:6.3f}%)")

# Combined filter
combined_mask = pd.Series(True, index=df.index)
for col, op, val in conditions:
    series        = col(df) if callable(col) else df[col]
    combined_mask &= op_funcs[op](series, val)

combined_count = combined_mask.sum()
combined_pct   = combined_count / total * 100
print(f"\nPassed All Conditions → {combined_count} / {total} ({combined_pct:.3f}%)")

# keep the filtered DataFrame
rfd3_json_df_filtered = df[combined_mask]

### SORT PASSING STRUCTURES BY A VALUE & PRINT THEIR PATHS ###
df_tmp = (rfd3_json_df_filtered.assign(_sv=lambda d: metric_to_sort(d)).sort_values('_sv', ascending=not sort_by_highest_values)) if callable(metric_to_sort) else rfd3_json_df_filtered.sort_values(metric_to_sort, ascending=not sort_by_highest_values)
col = '_sv' if callable(metric_to_sort) else metric_to_sort

sel = df_tmp.head(number_of_sorted_to_print)
names = [os.path.basename(p) for p in sel['rfd3_cifgz_path']]
svals = [f"{v:.4f}" for v in sel[col]]
w1, w2 = max(map(len, names)), max(map(len, svals))

print(f"\n{'TOP' if sort_by_highest_values else 'BOTTOM'} {number_of_sorted_to_print} "
      f"{'highest' if sort_by_highest_values else 'lowest'} {col}:")
for n, s in zip(names, svals): print(f"  {s:<{w2}} ----> {n:<{w1}}")
print("\npymol", *sel['rfd3_cifgz_path'])

### Copy Filtered Files 

In [ ]:
#########################################
### COPY FILTERED JSON & CIF.GZ FILES ###
#########################################

### WHERE TO COPY FILES ###
COPY_DEST_DIR = f"{RFDIFFUSION3_OUT_DIR}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/"
os.makedirs(COPY_DEST_DIR, exist_ok=True)

### COPYING LOGIC ###
n_rows = len(rfd3_json_df_filtered)
needed = n_rows * 2
print(f"Need to copy {needed} files ({n_rows} JSON + {n_rows} CIFs)")

copied = 0
for _, row in rfd3_json_df_filtered.iterrows():
    SUBDIR = row['subdirectory']
    for key, is_cif in [('rfd3_json_path', False), ('rfd3_cifgz_path', True)]:
        src = row[key]
        base = os.path.basename(src)
        if is_cif and base.lower().endswith('.cif.gz'):
            name = base[:-7]
            ext  = '.cif.gz'
        else:
            name, ext = os.path.splitext(base)
        dest_name = f"{name}_{SUBDIR}{ext}"
        dest_path = os.path.join(COPY_DEST_DIR, dest_name)
        shutil.copy(src, dest_path)
        copied += 1

print(f"Copied {copied}/{needed} files to {COPY_DEST_DIR}")

## III.C Additional Scaffold Filters

### Run Analysis

**NOTE:** Cannot have the reference pdb contain an ORI Token!

In [ ]:
#####################################################################
### RUN PROCESS RF-FLOW OUTPUTS (PARAMETERIZED PER-LIGAND CONFIG) ###
#####################################################################

### CONFIGURATIONS ###
#   - ref_catres and ligand_exposed_atoms are now OPTIONAL. - You can give them as a space-separated string ("A94 A96") or as a list (["A94", "A96"]).
run_configs = [
    {"ligand": "YYE", "pre_lig_frag1": "ZAPP", "post_lig_frag1": "model_0", "ref_catres": "", "ligand_exposed_atoms": "N1 O8 O9"},
    {"ligand": "YYE", "pre_lig_frag1": "ZAPP", "post_lig_frag1": "model_1", "ref_catres": "", "ligand_exposed_atoms": "N1 O8 O9"},
    {"ligand": "YYE", "pre_lig_frag1": "ZAPP", "post_lig_frag1": "model_2", "ref_catres": "", "ligand_exposed_atoms": "N1 O8 O9"},
    {"ligand": "YYE", "pre_lig_frag1": "ZAPP", "post_lig_frag1": "model_3", "ref_catres": "", "ligand_exposed_atoms": "N1 O8 O9"},
    {"ligand": "YYE", "pre_lig_frag1": "ZAPP", "post_lig_frag1": "model_4", "ref_catres": "", "ligand_exposed_atoms": "N1 O8 O9"},
    {"ligand": "YYE", "pre_lig_frag1": "ZAPP", "post_lig_frag1": "model_5", "ref_catres": "", "ligand_exposed_atoms": "N1 O8 O9"},
    {"ligand": "YYE", "pre_lig_frag1": "ZAPP", "post_lig_frag1": "model_6", "ref_catres": "", "ligand_exposed_atoms": "N1 O8 O9"},
    {"ligand": "YYE", "pre_lig_frag1": "ZAPP", "post_lig_frag1": "model_7", "ref_catres": "", "ligand_exposed_atoms": "N1 O8 O9"},


]

### VARIABLES ###
scaffolds_to_analyze_DIR   = f"{RFDIFFUSION3_OUT_DIR}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/"
combined_input_ligands_DIR = f"{THEOZYME_DIR}step2_theozyme_prep_ori/"
params_path_DIR            = PARAMS_DIR

# Optional numeric thresholds: set to None to skip passing that flag
exposed_atom_SASA = 1.0     # None → do not pass --exposed_atom_SASA
cart_bonded       = 1.0     # None → do not pass --cart_bonded
fa_dun            = 1.0     # None → do not pass --fa_dun

# Optional boolean flags controlling script behavior
analyze_only                       = True   # True  → add --analyze  (no filtering/moving)
apply_loop_catres_filter           = False  # True  → use default behavior (filter) | False → pass --loop_catres to DISABLE filter
fix_unmatched_remark_lines_to_lig  = True   # True  → add --fix_unmatched_remark_lines_to_lig

# Other optional flags
partial_diffusion = False          # True  → add --partial
nproc             = None           # e.g. 8; None → let script auto-detect

### CONSTANTS ###
apptainer = APPTAINER
script    = "{SPECIAL_SCRIPTS_DIR}process_diffusion3_outputs.py"
file_extn = ".cif.gz"

### GENERATE & PRINT COMMANDS ###
for cfg in run_configs:
    ligand = cfg["ligand"]
    frag   = cfg["pre_lig_frag1"]
    post   = cfg.get("post_lig_frag1", "")

    label = f"{ligand}_{frag}" + (f"_{post}" if post else "")     # build label, avoid trailing underscore if post is empty
    post_pattern = f"*{post}" if post else ""     # build wildcard patterns, avoid double-asterisk if post is empty

    #pdb_path    = f"{scaffolds_to_analyze_DIR}*{frag}*{ligand}{post_pattern}*{file_extn}"
    pdb_path    = f"{scaffolds_to_analyze_DIR}*{frag}*{post_pattern}*{file_extn}"
    ref_path    = f"{combined_input_ligands_DIR}*{frag}*{ligand}{post_pattern}*.pdb"
    params_path = f"{params_path_DIR}{ligand}.params"

    cmd_parts = [apptainer, script,
        "--pdb", pdb_path,
        "--params", params_path,
    ]

    # Optional numeric thresholds
    if cart_bonded is not None:
        cmd_parts += ["--cart_bonded", str(cart_bonded)]
    if fa_dun is not None:
        cmd_parts += ["--fa_dun", str(fa_dun)]

    # Optional ligand-exposed atoms + SASA (per-config)
    lig_exp_atoms = cfg.get("ligand_exposed_atoms", None)
    if lig_exp_atoms:
        if isinstance(lig_exp_atoms, str):
            lig_exp_atoms_tokens = lig_exp_atoms.split()
        else:
            lig_exp_atoms_tokens = list(lig_exp_atoms)
        cmd_parts += ["--ligand_exposed_atoms", *lig_exp_atoms_tokens]

        if exposed_atom_SASA is not None:
            cmd_parts += ["--exposed_atom_SASA", str(exposed_atom_SASA)]

    # Optional ref_catres (per-config)
    ref_catres_cfg = cfg.get("ref_catres", None)
    if ref_catres_cfg:
        if isinstance(ref_catres_cfg, str):
            ref_catres_tokens = ref_catres_cfg.split()
        else:
            ref_catres_tokens = list(ref_catres_cfg)
        cmd_parts += ["--ref_catres", *ref_catres_tokens]

    if not apply_loop_catres_filter:     #   - apply_loop_catres_filter == True  → do nothing (keep default behavior) | apply_loop_catres_filter == False → pass --loop_catres to disable filter
        cmd_parts.append("--loop_catres")
    if analyze_only:
        cmd_parts.append("--analyze")
    if fix_unmatched_remark_lines_to_lig:     # Fix REMARK 666 unmatched targets
        cmd_parts.append("--fix_unmatched_remark_lines_to_lig")
    if partial_diffusion:     # Partial diffusion flag
        cmd_parts.append("--partial")
    if nproc is not None:     # nproc: let the script auto-detect if None
        cmd_parts += ["--nproc", str(nproc)]
    cmd = " ".join(cmd_parts) # Final command string

    print(cmd)
    print(f"mv diffusion_analysis.sc rfdiffusion3_analysis_{label}.sc", f"\n")

Remember, if you are seeing bugs - it is probably because your reference PDBs have ORI tokens in them. I also had a problem with the ligand naming, you need to stop that problem early in this stage.

In [ ]:
############################################################################
### COMBINE ALL VALID RFdiffusion3 .SC FILES (PYTHON EXECUTION; NO BASH) ###
############################################################################

### INPUTS ###
ANALYSIS_DIR = f'{RFDIFFUSION3_OUT_DIR}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/'  # change as needed

### FIND FILES ###
sc_pattern = os.path.join(ANALYSIS_DIR, "rfdiffusion3_analysis_*.sc")
all_files = sorted(glob.glob(sc_pattern))

included_files = []
excluded_files = []

# Decide which files have any valid rows (fewer than 9 NaNs per row)
for file in all_files:
    try:
        df = pd.read_csv(file, sep=r'\s+', engine='python')
        df_clean = df[df.isna().sum(axis=1) < 9]
        if len(df_clean) > 0:
            included_files.append(file)
        else:
            excluded_files.append(file)
    except Exception:
        excluded_files.append(file)

# Report what we found
print(f"# Total .sc files found: {len(all_files)}")
print(f"# Excluding {len(excluded_files)} files with no valid rows:")
for f in excluded_files:
    print(f"  - {os.path.basename(f)}")
print(f"# Including {len(included_files)} files:")
for f in included_files:
    print(f"  - {os.path.basename(f)}")

if not included_files:
    print("# No files to combine.")
else:
    ### OUTPUT ###
    combined_file = os.path.join(ANALYSIS_DIR, "combined_RFdiffusion3_analysis.sc")

    # ---------- helpers ----------
    def count_rows(path: str) -> int:
        # counts total lines (including header)
        with open(path, "r") as fh:
            return sum(1 for _ in fh)

    def num_fields(line: str) -> int:
        return len(line.strip().split())

    def print_dimensions(path: str):
        rows = count_rows(path)
        with open(path, "r") as fh:
            header = fh.readline()
        cols = num_fields(header)
        print(f"{path} - Rows: {rows}, Columns: {cols}")

    # ---------- combine ----------
    with open(included_files[0], "r") as f0:
        header = f0.readline()

    with open(combined_file, "w") as out:
        out.write(header)
        # append all data lines (skip each file header)
        for fp in included_files:
            with open(fp, "r") as fin:
                _ = fin.readline()  # skip header
                for line in fin:
                    out.write(line)

    print(f"\n[INFO] Wrote combined file:\n  {combined_file}\n")

    # ---------- print dimensions ----------
    for fp in included_files:
        print_dimensions(fp)
    print_dimensions(combined_file)

    # ---------- verify row counts ----------
    expected_rows = sum(count_rows(fp) for fp in included_files) - (len(included_files) - 1)  # remove duplicate headers
    actual_rows = count_rows(combined_file)
    if expected_rows == actual_rows:
        print("\n[CHECK] Row count matches expectation.")
    else:
        print(f"\n[CHECK] Mismatch in row count! Expected: {expected_rows}, Found: {actual_rows}")

    # ---------- verify column count consistency ----------
    expected_cols = num_fields(header)
    bad_lines = 0
    with open(combined_file, "r") as fh:
        for i, line in enumerate(fh, start=1):
            if not line.strip():
                continue
            nf = num_fields(line)
            if nf != expected_cols:
                bad_lines += 1
                if bad_lines <= 10:
                    print(f"[WARN] Column mismatch at line {i}: NF={nf} (expected {expected_cols})")
                elif bad_lines == 11:
                    print("[WARN] (More column mismatches exist; suppressing further warnings.)")

    if bad_lines == 0:
        print("[CHECK] Column count is consistent.")
    else:
        print(f"[CHECK] Column count mismatch! Total mismatching lines: {bad_lines}")


### Plot the Distributions & Look at Filtering Criteria/Pass Rates

These steps are technically optional but highly recommended - it is a manual way for you to get a sense of what values to use for when you run the actual filtering script without the `--analyze` flag. 

In [ ]:
##############################################
### PLOT THE DISTRIBUTION OF ALL THE STATS ###
##############################################

# INPUT COMBINED .SC FILE TO LOAD
# FILE_PATH = f'{RFDIFFUSION3_OUT_DIR}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/combined_RFdiffusion3_analysis.sc' #change to RFdiffusion3 in future
FILE_PATH = f'{RFDIFFUSION3_OUT_DIR}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/combined_RFdiffusion3_analysis_temp_copy.sc' #change to RFdiffusion3 in future


# Load the data
df = pd.read_csv(FILE_PATH, sep=r'\s+', engine='python')

##################
### PLOT STUFF ###
##################

# Define numerical columns, excluding 'loop_at_motif' if it exists
numerical_cols = df.select_dtypes(include=[np.number]).columns
numerical_cols = numerical_cols[numerical_cols != 'loop_at_motif']

quartile_colors = ['skyblue', 'mediumseagreen', 'gold', 'salmon']

# -------------------------- #
#  ADJUSTABLE FONT SCALES    #
# -------------------------- #
title_fontsize = 20
label_fontsize = 18
tick_fontsize  = 16

# Set up the grid for the subplots
num_cols_per_row = 6
num_rows = int(np.ceil(len(numerical_cols) / num_cols_per_row))
fig, axes = plt.subplots(num_rows, num_cols_per_row, figsize=(6 * num_cols_per_row, 6 * num_rows))

# Flatten axes for easier iteration; remove excess axes if needed
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    ax = axes[idx]
    
    # 1) Draw the KDE line with a non-zero linewidth
    sns.kdeplot(
        data=df, x=col, ax=ax, fill=False, 
        linewidth=2, color='black'
    )
    
    # 2) Calculate quartiles
    q1, q2, q3 = df[col].quantile([0.25, 0.50, 0.75])
    
    # 3) Fill the area under the KDE curve in quartiles (if a line was created)
    if ax.lines:
        line = ax.lines[-1]
        x_data, y_data = line.get_data()
        
        ax.fill_between(
            x_data, y_data,
            where=(x_data < q1),
            color=quartile_colors[0], alpha=0.5
        )
        ax.fill_between(
            x_data, y_data,
            where=((x_data >= q1) & (x_data < q2)),
            color=quartile_colors[1], alpha=0.5
        )
        ax.fill_between(
            x_data, y_data,
            where=((x_data >= q2) & (x_data < q3)),
            color=quartile_colors[2], alpha=0.5
        )
        ax.fill_between(
            x_data, y_data,
            where=(x_data >= q3),
            color=quartile_colors[3], alpha=0.5
        )
    
    # Set title and labels with adjustable font sizes
    ax.set_title(f'Distribution of {col}', fontsize=title_fontsize, weight='semibold')
    ax.set_xlabel(f'{col}', fontsize=label_fontsize, weight='semibold')
    ax.set_ylabel('Density', fontsize=label_fontsize, weight='semibold')
    
    # Tick parameters: size, direction, etc.
    ax.tick_params(
        axis='both', which='major', 
        labelsize=tick_fontsize, 
        direction='in', length=5, width=1.5,
        top=True, right=True  # turn on top & right ticks
    )
    
    # Make spines visible on all four sides and set their width/color
    for side in ['left', 'right', 'top', 'bottom']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(2)
        ax.spines[side].set_edgecolor('black')

# Hide any excess axes
for ax in axes[len(numerical_cols):]:
    ax.remove()

plt.tight_layout()

# Save the figure to the specified directory
SAVE_PATH = f'{GRAPHS_DIR}RFdiffusion3_stats_analysis_PRODUCTION.png'
#plt.savefig(SAVE_PATH, dpi=300)
plt.show()

The next cell is setup quite ideally so that you can look at the extrema for certain filtering criteria. Just choose a column to sort by and if you want it ascending or descending. Then it will print out the n_example of pdb files that passed all the filters and are the extrema for the sorting column you picked - you can use this to adjust cutoffs and inform the values you want to use. Just copy the list of pdb files and then do `pymol {paste copied list of pdb files}` in your terminal and you will see the pdb files ranked in order of the filtering column.

**NOTE:** I will probably go fairly light with this filtering, especially SASA, and really focus in after predesign.

In [ ]:
#########################################
### INVESTIGATE IDEAL FILTER CRITERIA ###
#########################################

### SHOW THE FIRST FEW ROWS OF THE FILTERED DF? ###
show_df_head = True

### SORTING PARAMETERS FOR FILTERED DF ###
sorting_col = "SASA_rel"
sorting_ascending = False # smallest first = true
n_example = 10  # How many top examples to display

### INPUT DATAFRAME .SC FROM ANALYSIS FILTERING STEP ###
# FILE_PATH = f'{RFDIFFUSION3_OUT_DIR}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/combined_RFdiffusion3_analysis.sc' 
FILE_PATH = f'{RFDIFFUSION3_OUT_DIR}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/combined_RFdiffusion3_analysis_temp_copy.sc' 


### LOGIC ###
df = pd.read_csv(FILE_PATH, sep=r'\s+', engine='python') # Meant for .sc files
initial_count = len(df)
df_clean = df[df.isna().sum(axis=1) < 9].copy()
removed_count = initial_count - len(df_clean)
print(f"Removed {removed_count} rows with 9 or more NaN values.")
print()
df = df_clean

### (OPTIONAL) CONSIDER ONLY ROWS WITH A CERTAIN SUBSTRING ###
description_filter = ""  # e.g. "frag1"
if description_filter:
    before_desc = len(df)
    # df = df[df['description'].str.contains(description_filter, na=False)]
    df = df[df['design_path'].str.contains(description_filter, na=False)]
    desc_count = len(df)
    print(f"Filtered by description ‘{description_filter}’: removed {before_desc - desc_count} rows.")

### MANDATORY (HARD-CODED) FILTERS ###
chainbreak = 4.5     # CA-distance threshold
lig_dist   = 2.0     # Clash distance with ligand
rCA_nonadj = 3.0     # Non-adjacent residues cannot be closer than this
bondlen_dev = 0.05  # Allowed bond-length deviation

### OPTIONAL FILTERS ###
loop_frac       = 0.30   # Usually 0.3
longest_helix   = 46     # Max helix length
rog             = 17.5   # Radius of gyration
term_mindist    = 5      # Terminus distance from ligand?
cart_bonded_avg = 12.5     # sidechain quality | smaller = better | recommended ~5
fa_dun_avg      = 10     # sidechain quality | smaller = better | recommended ~15

### SASA UPPER + LOWER FILTERS ###
SASA_rel_lower = 0.01   # Scale 0-1
SASA_rel_upper = 0.15    # Scale 0-1

### SASA EXPOSED ATOM FILTERS ###
SASA_exposed_atoms_lower = 25
SASA_exposed_atoms_upper = 66

### DEFINE FILTERS (adjust or comment out as desired) ###
filters = {
    ("chainbreak", "<="): chainbreak,
    ("lig_dist", ">="): lig_dist,
    ("rCA_nonadj", ">="): rCA_nonadj,
    ("bondlen_dev", "<="): bondlen_dev,
    ("loop_frac", "<="): loop_frac,
    ("longest_helix", "<="): longest_helix,
    #("rog", "<="): rog,
    ("SASA_rel", ">="): SASA_rel_lower,
    ("SASA_rel", "<="): SASA_rel_upper,
    ("SASA_exposed_atoms", ">="): SASA_exposed_atoms_lower,
    #("SASA_exposed_atoms", "<="): SASA_exposed_atoms_upper,
    ("term_mindist", ">="): term_mindist,
    
    ("cart_bonded_avg", "<="): cart_bonded_avg,
    ("fa_dun_avg", "<="): fa_dun_avg,
}

def filtering_df(dataframe_name, filters, target_df, print_stat=True):
    """
    Apply a set of filters to a DataFrame and print how many rows pass each one.
    Returns the filtered DataFrame.
    """
    if print_stat:
        print(f"[{dataframe_name}] ({len(target_df)} designs)")
    
    converted_filters = []
    for (key, operator_str), cutoff_value in filters.items():
        # Build a condition string, e.g. "target_df['chainbreak'] <= 4.5"
        filter_str = f"target_df['{key}'] {operator_str} {cutoff_value}"
        converted_filters.append(filter_str)
        
        if print_stat:
            pass_mask = eval(filter_str)
            count_passed = pass_mask.sum()
            pct_passed = (count_passed / len(target_df)) * 100
            print(
                f"# [ {key.ljust(19)} {operator_str.ljust(2)} "
                f"{str(cutoff_value).ljust(6)}]: "
                f"{str(count_passed).ljust(6)} ({pct_passed:.1f}%)"
            )
    
    # Combine all filters with logical AND
    final_mask = eval(" & ".join([f"({condition})" for condition in converted_filters]))
    filtered_df = target_df[final_mask]
    
    if print_stat:
        total_passed = len(filtered_df)
        pct_passed = (total_passed / len(target_df)) * 100
        print(f"# [             ALL              ]: {str(total_passed).ljust(6)} ({pct_passed:.1f}%)")
    
    return filtered_df

### APPLY FILTERS ###
filtered_RFdiffusion3_df = filtering_df("RFdiffusion3", filters, df)
filter_rate = (len(filtered_RFdiffusion3_df) / len(df)) * 100.0

print(f"\n{len(filtered_RFdiffusion3_df)} of {len(df)} designs ({round(filter_rate, 3)}%) passed\n")

### OPTIONALLY SHOW FIRST n ROWS SORTED BY A SPECIFIC COLUMN ###
if show_df_head:
    print(f"# SORTED BY [{sorting_col}] IN [{'Descending' if not sorting_ascending else 'Ascending'}] ORDER:")
    filtered_RFdiffusion3_df.sort_values(by=sorting_col, ascending=sorting_ascending, inplace=True)
    # top_descriptions = filtered_RFdiffusion3_df['description'].head(n_example).tolist()
    top_descriptions = filtered_RFdiffusion3_df['design_path'].head(n_example).tolist()
    print("pymol " + ' '.join(top_descriptions))
    print('')
    #print(filtered_RFdiffusion3_df.head(n_example))
    # print(filtered_RFdiffusion3_df[[sorting_col, "description"]].head(n_example))    # Only show the sorting column in the head
    print(filtered_RFdiffusion3_df[[sorting_col, "design_path"]].head(n_example))    # Only show the sorting column in the head

### Execute the Filtering

In [ ]:
#####################################################################
### RUN PROCESS RF-FLOW OUTPUTS (PARAMETERIZED PER-LIGAND CONFIG) ###
#####################################################################

### CONFIGURATIONS ###
#   - ref_catres and ligand_exposed_atoms are OPTIONAL.
#     Give as space-separated string ("A94 A96") or list (["A94", "A96"]).
run_configs = [
  {"ligand": "YYE", "pre_lig_frag1": "ZAPP", "post_lig_frag1": "model", "ref_catres": "", "ligand_exposed_atoms": "N1 O8 O9"},
]

### VARIABLES ###
scaffolds_to_analyze_DIR   = f"{RFDIFFUSION3_OUT_DIR}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/"
combined_input_ligands_DIR = f"{THEOZYME_DIR}step2_theozyme_prep_ori/"
params_path_DIR            = PARAMS_DIR

### PROTEIN QUALITY FILTERS ###
rog           = None
longest_helix = 46
loop_limit    = 0.3
lig_dist      = 2.0

### LIGAND-PROTEIN QUALITY FILTERS ###
SASA_limit          = 0.15  # upper limit
terminus_dist_limit = 5.0   # lower limit

### SIDE CHAIN QUALITY FILTERS ###
bondlen_dev = 0.05

# Optional numeric thresholds: set to None to skip passing that flag
exposed_atom_SASA = 25
cart_bonded       = 12.5
fa_dun            = 10

# Optional boolean flags controlling script behavior
analyze_only                       = False
apply_loop_catres_filter           = False  # True -> default (filter) | False -> pass --loop_catres to DISABLE filter
fix_unmatched_remark_lines_to_lig  = True

# Other optional flags
partial_diffusion = False
nproc             = None   # None -> orchestrator will pass --nproc = sbatch_cores automatically

# Orchestrator-specific
chunk_threshold   = 4000
scorefile_prefix  = "rfdiffusion3_analysis"

### SBATCH TOGGLE ###
use_sbatch          = False
sbatch_cores        = 16
sbatch_mem          = "32G"
sbatch_time         = "01:00:00"
sbatch_queue        = "cpu"
sbatch_cmds_per_job = 1
sbatch_no_wait      = False

### CONSTANTS ###
apptainer    = APPTAINER
orchestrator = "{SPECIAL_SCRIPTS_DIR}process_diffusion3_outputs__ORCHESTRATOR.py"
file_extn    = ".cif.gz"

### GENERATE & PRINT COMMANDS ###
for cfg in run_configs:
  ligand = cfg["ligand"]
  frag   = cfg["pre_lig_frag1"]
  post   = cfg.get("post_lig_frag1", "")

  label = f"{ligand}_{frag}" + (f"_{post}" if post else "")
  post_pattern = f"*{post}" if post else ""

  pdb_path      = f"{scaffolds_to_analyze_DIR}*{frag}*{post_pattern}*{file_extn}"
  ref_path      = f"{combined_input_ligands_DIR}*{frag}*{ligand}{post_pattern}*.pdb"
  params_path   = f"{params_path_DIR}{ligand}.params"
  scorefile_out = f"{scorefile_prefix}_{label}.sc"

  # Always quote the glob so the orchestrator (not the shell) expands it -> bypasses ARG_MAX
  pdb_path_quoted = shlex.quote(pdb_path)

  # Launcher: apptainer for local, plain python3 for sbatch (sbatch isn't in the SIF)
  if use_sbatch:
      cmd_parts = ["python3", orchestrator]
  else:
      cmd_parts = [apptainer, orchestrator]

  cmd_parts += [
      "--pdb", pdb_path_quoted,
      "--params", params_path,
      "--scorefile_out", scorefile_out,
      "--chunk_threshold", str(chunk_threshold),
  ]

  # Protein quality filters
  if rog is not None:
      cmd_parts += ["--rog", str(rog)]
  if longest_helix is not None:
      cmd_parts += ["--longest_helix", str(longest_helix)]
  if loop_limit is not None:
      cmd_parts += ["--loop_limit", str(loop_limit)]
  if lig_dist is not None:
      cmd_parts += ["--lig_dist", str(lig_dist)]

  # Ligand-protein quality filters
  if SASA_limit is not None:
      cmd_parts += ["--SASA_limit", str(SASA_limit)]
  if terminus_dist_limit is not None:
      cmd_parts += ["--term_limit", str(terminus_dist_limit)]

  # Side chain quality filters
  if bondlen_dev is not None:
      cmd_parts += ["--bondlen_dev", str(bondlen_dev)]

  # Optional numeric thresholds
  if cart_bonded is not None:
      cmd_parts += ["--cart_bonded", str(cart_bonded)]
  if fa_dun is not None:
      cmd_parts += ["--fa_dun", str(fa_dun)]

  # Optional ligand-exposed atoms + SASA
  lig_exp_atoms = cfg.get("ligand_exposed_atoms", None)
  if lig_exp_atoms:
      lig_exp_atoms_tokens = lig_exp_atoms.split() if isinstance(lig_exp_atoms, str) else list(lig_exp_atoms)
      cmd_parts += ["--ligand_exposed_atoms", *lig_exp_atoms_tokens]
      if exposed_atom_SASA is not None:
          cmd_parts += ["--exposed_atom_SASA", str(exposed_atom_SASA)]

  # Optional ref_catres
  ref_catres_cfg = cfg.get("ref_catres", None)
  if ref_catres_cfg:
      ref_catres_tokens = ref_catres_cfg.split() if isinstance(ref_catres_cfg, str) else list(ref_catres_cfg)
      cmd_parts += ["--ref_catres", *ref_catres_tokens]

  # Boolean pass-through flags
  if not apply_loop_catres_filter:
      cmd_parts.append("--loop_catres")
  if analyze_only:
      cmd_parts.append("--analyze")
  if fix_unmatched_remark_lines_to_lig:
      cmd_parts.append("--fix_unmatched_remark_lines_to_lig")
  if partial_diffusion:
      cmd_parts.append("--partial")
  if nproc is not None:
      cmd_parts += ["--nproc", str(nproc)]

  # Sbatch-specific flags
  if use_sbatch:
      cmd_parts += [
          "--sbatch",
          "--sbatch_queue", sbatch_queue,
          "--sbatch_cores", str(sbatch_cores),
          "--sbatch_mem", sbatch_mem,
          "--sbatch_time", sbatch_time,
          "--sbatch_cmds_per_job", str(sbatch_cmds_per_job),
          "--sbatch_apptainer", apptainer,
      ]
      if sbatch_no_wait:
          cmd_parts.append("--sbatch_no_wait")

  cmd = " ".join(cmd_parts)
  print(cmd, "\n")

### OPTIONAL: Filter by Catalytic Residue Distance

Now I decided to do some filtering in the sequence space. Turns out, there are pretty distinct motifs for zinc binding: https://pmc.ncbi.nlm.nih.gov/articles/PMC3268031/ \
There is also a super amazing table here: https://febs.onlinelibrary.wiley.com/doi/epdf/10.1016/0014-5793%2894%2901079-X \
**HExxH** motif forming an α-helix \
**HExxHxxGFxHExxRxDR** furthermore... \
**HxxE(D)-aan-H** in carboxypeptidase family \
**HxD-aa12-H-aa12-H** matrix metalloprotease \
**HELLGH** in dipeptidyl peptidase & three kinds of monooxygenases

In [ ]:
###############################################################
### FILTER PDB FILES BY CATALYTIC RESIDUE SEQUENCE DISTANCE ###
###############################################################

### INPUT VARIABLES ###
input_dir = f"{RFDIFFUSION3_OUT_DIR}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/filtered_structures"
catalytic_residue_pair_types = ["HIS GLU"]#["HIS HIS", "HIS GLU"]  # Residue pairs to check (comma list)
max_sequence_distances = [1]  # Max sequence distance thresholds for each pair (comma list)
min_sequence_distances = [1]  # Min sequence distance thresholds for each pair (comma list)

### CONSTANTS ###
script_path = f"{SPECIAL_SCRIPTS_DIR}scaffold_handling/filter_pdbsDIR_by_catres_sequence_distance.py"

### INITIALIZE COMMAND ###
command = (f"python {script_path} "
           f"--input_dir_of_pdbs_with_remark666_lines {input_dir} "
           f"--catalytic_residue_pair_types " + " ".join(f'"{pair}"' for pair in catalytic_residue_pair_types) + " ")

# Add optional distance thresholds
if max_sequence_distances:
    command += "--max_sequence_distances " + " ".join(map(str, max_sequence_distances)) + " "
if min_sequence_distances:
    command += "--min_sequence_distances " + " ".join(map(str, min_sequence_distances)) + " "

### PRINT COMMAND ###
print("#" * 49)
print("### GENERATED COMMAND FOR FILTERING PDB FILES ###")
print("#" * 49)
print("")
print(command)
print()

# **IV. Predesign Scaffolds** 

## IV.A Execute Predesign (Cartesian Relax for Geometry Idealization)

### Run Predesign

In [ ]:
#################################################################                                                                                                                       
### STAGE 1: GEOMETRY IDEALIZATION (RFdiffusion3 → Idealized) ###
#################################################################                                                                                                                       
                                                                    
### INPUTS ###                                                                                                                                                                          
input_pdb_structures_dir = f"{RFDIFFUSION3_OUT_DIR}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/filtered_structures"
# Optional: if JSONs are in a different directory than PDBs, specify here                                                                                                               
corresponding_json_dir = f"{RFDIFFUSION3_OUT_DIR}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/"                                                                                                  
# Optional: override per-structure JSON name (rare). Default is auto-detect from PDB basename.                                                                                          
json_file = None                                                                                                                                                                        
                                                                                                                                                                                        
### GROUP → LIGAND MAP & FRAGS ###                                                                                                                                                      
groups_to_ligands = {"ZAPP": ["YYE"]}                               
frags = [""]
                                                                                                                                                                                        
### OUTPUTS ###
specific_idealization_output_dir = f"{PREDESIGN_OUT_DIR}i1_stage1/"                                                                                                                     
output_pdb = None                                                   
                                                                                                                                                                                        
### PARAMS FILE LOCATION ###
params_files_dir = PARAMS_DIR                                                                                                                                                           
                                                                    
### CONSTANTS ###
apptainer_path = APPTAINER
script_path    = f"{SPECIAL_SCRIPTS_DIR}scaffold_handling/idealize_rfdiffusion3_geometry__MAIN.py"  # NOTE: script filename is __MAIN.py (the overhauled version with MPNN + new metrics)                                                                                     
                                                                                                                                                                                        
### GEOMETRY IDEALIZATION PARAMETERS (script defaults in comments) ###                                                                                                                  
mobile_radius              = None     # Optional; default: None = whole protein mobile. Set e.g. 12.5 to restrict.                                                                      
coord_cst_weight           = 750.0    # Optional; script default: 100.0 (pass to keep tight constraints on fixed atoms)                                                                 
coord_cst_stdev            = 0.02     # Optional; script default: 0.01 Å (looser tolerance keeps fixed atoms rigid but forgiving)                                                       
cart_bonded_weight         = 4.0      # Optional; script default: 4.0 (sweep sweet spot; 10 = max idealization, 1 = balanced score)                                                     
covalent_contact_threshold = 2.5      # Optional; script default: 2.5 Å (catches covalent + metal-ligand + salt-bridge; H-bonds excluded)                                               
fastrelax_cycles           = 3        # Optional; script default: 3 (more = better idealization at higher cost)                                                                         
min_tolerance              = 5e-5     # Optional; script default: 5e-5 (tight final-minimization tolerance)                                                                             
                                                                                                                                                                                        
### MPNN PRE-DESIGN PARAMETERS ###                                                                                                                                                      
skip_mpnn         = False   # Optional flag; script default: False (MPNN is ON by default)                                                                                              
mpnn_num_designs  = 10      # Optional; script default: 10                                                                                                                              
mpnn_temperature  = 0.1     # Optional; script default: 0.1                                                                                                                             
mpnn_omit_aa      = ""      # Optional; script default: "" (allow all 20 AAs; set e.g. "CM" to omit Cys+Met)                                                                            
                                                                                                                                                                                        
### PROTOCOL FLAGS ###                                                                                                                                                                  
idealize_ss    = False  # Optional flag; default: False                                                                                                                                 
skip_fastrelax = False  # Optional flag; default: False                                                                                                                                 
skip_minimize  = False  # Optional flag; default: False             
no_ca_rmsd     = False  # Optional flag; default: False (CA-RMSD computed by default; set True to skip one pose clone)                                                                  
debug_metrics  = False  # Optional flag; default: False (True emits verbose per-atom displacement & timing data)                                                                        
                                                                                                                                                                                        
### QUICK LOGIC ###                                                                                                                                                                     
all_ligands  = sorted({lig for ligs in groups_to_ligands.values() for lig in ligs})                                                                                                     
def _find_params(lig):
    """Locate a ligand's .params, with an error that says how to make one."""
    matches = sorted(glob.glob(os.path.join(params_files_dir, f"*{lig}*.params")))
    if not matches:
        raise FileNotFoundError(
            f"No .params file for ligand '{lig}' in {params_files_dir}\n"
            f"  Run the ligand .params generation cell earlier in this notebook,\n"
            f"  then run its printed command in a terminal."
        )
    return matches[0]

params_files = {lig: _find_params(lig) for lig in all_ligands}                                                           
os.makedirs(specific_idealization_output_dir, exist_ok=True)          

### GENERATE COMMANDS ###                                                                                                                                                               
commands_name = "predesign_i1_stage1"                               
commands = []                                                                                                                                                                           
for g, lig_list in groups_to_ligands.items():
    for lig in lig_list:                                                                                                                                                                
        for f in frags:                                             
            pattern = f"*{g}*{f}*.pdb"  # <---- CHECK THAT THIS MATCHES YOUR FILE BASENAME STRUCTURE
            for pdb_file in sorted(glob.glob(os.path.join(input_pdb_structures_dir, pattern))):                                                                                         
                params_file = params_files[lig]                                                                                                                                         
                # Required: --pdb                                                                                                                                                       
                cmd = f"{apptainer_path} {script_path} --pdb {pdb_file} "                                                                                                               
                # Practically required (recommended): params                                                                                                                            
                if params_file is not None:                                                                                                                                             
                    cmd += f"--params {params_file} "                                                                                                                                   
                # Auto-detected JSON (only pass if overriding)                                                                                                                          
                if json_file is not None:                                                                                                                                               
                    cmd += f"--json {json_file} "
                if corresponding_json_dir is not None:                                                                                                                                  
                    cmd += f"--corresponding_json_dir {corresponding_json_dir} "
                # Output controls (only pass if overriding; output_dir is recommended)                                                                                                  
                if output_pdb is not None:                                                                                                                                              
                    cmd += f"--output {output_pdb} "                                                                                                                                    
                if specific_idealization_output_dir is not None:                                                                                                                        
                    cmd += f"--output_dir {specific_idealization_output_dir} "                                                                                                          
                # Geometry parameters (only pass if overriding script defaults)                                                                                                         
                if mobile_radius is not None:
                    cmd += f"--mobile_radius {mobile_radius} "                                                                                                                          
                if coord_cst_weight != 100.0:                                                                                                                                           
                    cmd += f"--coord_cst_weight {coord_cst_weight} "
                if coord_cst_stdev != 0.01:                                                                                                                                             
                    cmd += f"--coord_cst_stdev {coord_cst_stdev} "                                                                                                                      
                if cart_bonded_weight != 4.0:
                    cmd += f"--cart_bonded_weight {cart_bonded_weight} "                                                                                                                
                if covalent_contact_threshold != 2.5:                                                                                                                                   
                    cmd += f"--covalent_contact_threshold {covalent_contact_threshold} "
                if fastrelax_cycles != 3:                                                                                                                                               
                    cmd += f"--fastrelax_cycles {fastrelax_cycles} "
                if min_tolerance != 5e-5:                                                                                                                                               
                    cmd += f"--min_tolerance {min_tolerance} "                                                                                                                          
                # MPNN parameters (only pass if overriding)
                if skip_mpnn:                                                                                                                                                           
                    cmd += "--skip_mpnn "                           
                if mpnn_num_designs != 10:                                                                                                                                              
                    cmd += f"--mpnn_num_designs {mpnn_num_designs} "
                if mpnn_temperature != 0.1:                                                                                                                                             
                    cmd += f"--mpnn_temperature {mpnn_temperature} "
                if mpnn_omit_aa != "":                                                                                                                                                  
                    cmd += f"--mpnn_omit_aa {mpnn_omit_aa} "                                                                                                                            
                # Protocol flags (only pass when True)
                if idealize_ss:                                                                                                                                                         
                    cmd += "--idealize_ss "                         
                if skip_fastrelax:                                                                                                                                                      
                    cmd += "--skip_fastrelax "                      
                if skip_minimize:                                                                                                                                                       
                    cmd += "--skip_minimize "
                if no_ca_rmsd:                                                                                                                                                          
                    cmd += "--no_ca_rmsd "                          
                if debug_metrics:
                    cmd += "--debug "                                                                                                                                                   
                commands.append(cmd.strip())
commands.sort()                                                                                                                                                                         
                                                                    
### OUTPUT & WRITE COMMANDS FILE ###
commands_file_path = os.path.join(CMDS_DIR, f"{commands_name}")
with open(commands_file_path, "w") as f:                                                                                                                                                
    f.write("\n".join(commands) + "\n")
print("### COMMANDS FILE ###")                                                                                                                                                          
print(commands_file_path)                                           
print("\nNumber of Commands =", len(commands))                                                                                                                                          
                                                                    
### SETUP BATCH JOBS ###                                                                                                                                                                
# Runtime note: whole-protein relax + 3 FastRelax cycles + MPNN on CPU ≈ 20–30 min per structure. Memory note: MPNN on CPU + PyRosetta ≈ 6–10 GB peak; 12g gives headroom.                                                                                                              
qtime, cores, memory, queue = "02:00:00", "1", "8g", "cpu"                                                                                                                             
cmds_per_job = 2                                                                                                                                                                        
job_name = os.path.basename(commands_file_path)                                                                                                                                         
submit_file = f"{SUBMIT_DIR}{job_name}.sh"                          
num_jobs = int(len(commands) / cmds_per_job)                                                                                                                                            
                                                                                                                                                                                        
print("Number of Jobs =", num_jobs)
print("Job Name =", job_name)                                                                                                                                                           
print("\nNavigate here:")                                           
print("cd", specific_idealization_output_dir, "\n")
                                                                                                                                                                                        
# make submit script
nb.submit_array_job(commands_file_path, qtime, cores, job_name, memory, submit_file, LOGS_DIR, num_jobs + 1, cmds_per_job, queue)      
                                                                                                                                                                                        

### Parse JSONs

In [ ]:
############################################                       
### COMBINE JSON FILES INTO SINGLE JSON  ###                        
############################################

### INPUT DIRECTORY TO PARSE ###                                                                                                                                                        
rfd3_output_dir_to_parse = f"{PREDESIGN_OUT_DIR}i1_stage1/"
                                                                                                                                                                                       
### OUTPUT JSON PATH ###                                                                                                                                                                
output_combined_json_path = f"{PREDESIGN_OUT_DIR}i1_stage1/combined_stats.json"
                                                                                                                                                                                       
### PARSING LOGIC ###                                                                                                                                                                   
# The new idealizer precomputes every aggregate we need in global_metrics, so
# this cell just walks the JSONs, drops heavy per-atom sections, and writes.                                                                                                            
json_files = sorted(glob.glob(f"{rfd3_output_dir_to_parse}**/*_metrics.json", recursive=True), key=lambda p: os.path.abspath(p),)               
# don't re-ingest the combined file itself                          
json_files = [p for p in json_files if os.path.abspath(p) != os.path.abspath(output_combined_json_path)]                                                                                
                                                                                                                                                                                       
print(f"Found {len(json_files)} JSON files to parse")                                                                                                                                   
                                                                                                                                                                                       
all_records = []                                                    
skipped = 0

for path in json_files:
   try:
       with open(path, "r") as f:
           rec = json.load(f)                                                                                                                                                          
   except Exception as e:
       print(f"⚠️   Skipping {path!r}: failed to read JSON ({e})")                                                                                                                      
       skipped += 1                                                
       continue
                                                                                                                                                                                       
   if not isinstance(rec, dict):
       print(f"⚠️   Skipping {path!r}: top-level JSON is not an object")                                                                                                                
       skipped += 1                                                
       continue

   # Drop heavy sections — all filter-relevant aggregates are already in global_metrics.                                                                                               
   rec.pop("catalytic_residues", None)        # per-residue bond/angle entries (large)
   rec.pop("declared_covalent_contacts", None)  # per-pair list (optional — keep if needed)                                                                                            
   # rec.pop("mpnn", None)                    # keep if you want candidate-level scoring info                                                                                          
                                                                                                                                                                                       
   all_records.append(rec)                                                                                                                                                             
                                                                                                                                                                                       
with open(output_combined_json_path, "w") as out:                                                                                                                                       
   json.dump(all_records, out, indent=2)
                                                                                                                                                                                       
print(f"Saved {len(all_records)} records → {output_combined_json_path} (skipped {skipped})")     

### Filter

In [ ]:
                                                                    
###############################################
### LOAD COMBINED JSON & FILTER (PREDESIGN) ###
###############################################
                                                                                                                                                                                        
### COMBINED JSON PATH FROM ABOVE ###
output_combined_json_path = f"{PREDESIGN_OUT_DIR}i1_stage1/combined_stats.json"                                                                                                         
                                                                                                                                                                                        
### FILTERS (EDIT THESE THRESHOLDS FREELY) ###                                                                                                                                          
# Semantic key:                                                                                                                                                                         
#   *_effective      = clashes after subtracting chemistry-declared contacts (LYS-ligand                                                                                                
#                      PTM, His-metal bonds, etc.). Mobile↔fixed clashes still count.                                                                                                   
#   *_actionable_*   = bond/angle deviations restricted to fixed_mobile + all_mobile                                                                                                    
#                      classifications — excludes intentional QM-fixed intra-residue                                                                                                    
#                      geometry.                                                                                                                                                        
#   *_excluding_fixed_only = pseudo-cart_bonded restricted to actionable entries.                                                                                                       
conditions = [                                                                                                                                                                          
    # Core structural sanity                                        
    ("global_metrics.num_chain_breaks",                          "<=", 0),                                                                                                              
    ("global_metrics.ca_rmsd_overall",                           "<",  1.0),
                                                                                                                                                                                        
    # Clashes (effective, post-exclusion)                                                                                                                                               
    ("global_metrics.num_clashing_residues_effective",           "<",  6),                                                                                                              
    ("global_metrics.num_catalytic_clashing_effective",          "<",  1),                                                                                                              
                                                                                                                                                                                        
    # Catalytic actionable geometry — the main quality signal the relax could fix                                                                                                       
    ("global_metrics.catalytic_max_actionable_bond_deviation",   "<=", 0.04),                                                                                                           
    ("global_metrics.catalytic_mean_actionable_bond_deviation",  "<=", 0.01),                                                                                                           
    ("global_metrics.catalytic_max_actionable_angle_deviation",  "<=", 4.5),
    ("global_metrics.catalytic_mean_actionable_angle_deviation", "<=", 1.5),                                                                                                            
    ("global_metrics.mean_catalytic_cart_bonded_excluding_fixed_only", "<=", 1.0),                                                                                                      
                                                                                                                                                                                        
    # Fixed atoms stayed pinned (coord csts held)                                                                                                                                       
    ("global_metrics.mean_fixed_atom_displacement",              "<=", 0.025),
    ("global_metrics.max_fixed_atom_displacement",               "<",  0.05),                                                                                                           
                                                                                                                                                                                        
    # Score sanity (more negative is better; omit if you want all-pass on score)                                                                                                        
    # ("global_metrics.total_score",                             "<",  0.0),                                                                                                            
]                                                                                                                                                                                       
                                                                    
### SORT ###                                                                                                                                                                            
number_of_sorted_to_print = 10                                      
metric_to_sort            = "global_metrics.mean_catalytic_cart_bonded_excluding_fixed_only"  # lower = better idealized
sort_by_highest_values    = True   # False -> lowest (best) first; True -> reverse                                                                                                     
                                                                                                                                                                                        
### LOAD ###                                                                                                                                                                            
with open(output_combined_json_path, "r") as f:                                                                                                                                         
    records = json.load(f)                                          
                                                                                                                                                                                        
df = pd.json_normalize(records, sep=".")
total = len(df)                                                                                                                                                                         
print(f"Total records: {total}\n")                                  

### FILTERING LOGIC ###
op_funcs = {"<": operator.lt, "<=": operator.le, ">": operator.gt, ">=": operator.ge}
                                                                                                                                                                                        
labels      = [col for col, *_ in conditions]
label_width = max(len(str(l)) for l in labels) if labels else 10                                                                                                                        
opval_strs  = [f"{op} {val}" for *_, op, val in conditions]                                                                                                                             
opval_width = max(len(s) for s in opval_strs) if opval_strs else 10
num_width   = len(str(total))                                                                                                                                                           
                                                                    
# per-filter pass counts                                                                                                                                                                
for (col, op, val), opval in zip(conditions, opval_strs):           
    series  = df[col] if col in df.columns else pd.Series([None] * total, index=df.index)                                                                                               
    mask    = op_funcs[op](series, val)
    count   = int(mask.sum())                                                                                                                                                           
    pct     = (count / total * 100) if total else 0.0                                                                                                                                   
    missing = int(series.isna().sum())
    print(f"[{col:<{label_width}} {opval:<{opval_width}} ]:  "                                                                                                                          
        f"{count:>{num_width}} / {total:<{num_width}}  ({pct:6.3f}%)   | missing={missing}")                                                                                          
                                                                                                                                                                                        
# combined                                                                                                                                                                              
combined_mask = pd.Series(True, index=df.index)                                                                                                                                         
for col, op, val in conditions:                                     
    series = df[col] if col in df.columns else pd.Series([None] * total, index=df.index)
    combined_mask &= op_funcs[op](series, val)                                                                                                                                          

combined_count = int(combined_mask.sum())                                                                                                                                               
combined_pct   = (combined_count / total * 100) if total else 0.0   
print(f"\nPassed All Conditions → {combined_count} / {total} ({combined_pct:.3f}%)")                                                                                                    

df_filt = df[combined_mask].copy()                                                                                                                                                      
                                                                    
### SORT + PRINT ###                                                                                                                                                                    
if combined_count == 0:                                             
    print("\nNo structures passed. Loosen thresholds or check which columns are missing.")
else:
    df_tmp = df_filt.sort_values(metric_to_sort, ascending=not sort_by_highest_values)                                                                                                  
    sel = df_tmp.head(number_of_sorted_to_print)
                                                                                                                                                                                        
    name_col = "metadata.structure_name"                            
    pdb_col  = "metadata.pdb_path"                                                                                                                                                      
                                                                    
    names = sel[name_col].tolist() if name_col in sel.columns else ["(missing)"] * len(sel)                                                                                             
    paths = sel[pdb_col].tolist()  if pdb_col  in sel.columns else ["(missing)"] * len(sel)
                                                                                                                                                                                        
    col = metric_to_sort                                            
    svals = []                                                                                                                                                                          
    for v in sel[col].tolist():                                     
        try:
            svals.append(f"{float(v):.4f}")
        except Exception:
            svals.append(str(v))                                                                                                                                                        

    w1 = max(map(len, names)) if names else 10                                                                                                                                          
    w2 = max(map(len, svals)) if svals else 10                      

    print(f"\n{'TOP' if sort_by_highest_values else 'BOTTOM'} {min(number_of_sorted_to_print, len(sel))} "                                                                              
        f"{'highest' if sort_by_highest_values else 'lowest'} {col}:")
    for s, n, p in zip(svals, names, paths):                                                                                                                                            
        print(f"  {s:<{w2}} ----> {n:<{w1}}   |   {p}")             
                                                                                                                                                                                        
    print("\n# quick copy/paste PyMOL command")                     
    print("pymol", *paths)     

### Copy Filtered Files

In [ ]:
#########################################
### COPY FILTERED PDB STRUCTURES ###
#########################################

### WHERE TO COPY FILES ###
copy_dest_dir = f"{PREDESIGN_OUT_DIR}i1_stage1/ZZZ_FILTERED_STRUCTURES_ZZZ/"
os.makedirs(copy_dest_dir, exist_ok=True)

### EXPECTED COLUMNS ###
pdb_col   = "metadata.pdb_path"
name_col  = "metadata.structure_name"

### COPYING LOGIC ###
n_rows = len(df_filt)
print(f"Need to copy {n_rows} PDB files")

copied = 0
missing = 0

for _, row in df_filt.iterrows():
    src = row.get(pdb_col, None)
    if not isinstance(src, str) or not os.path.exists(src):
        missing += 1
        continue

    base = os.path.basename(src)
    name = row.get(name_col, os.path.splitext(base)[0])

    # preserve uniqueness + readability
    dest_name = f"{name}.pdb"
    dest_path = os.path.join(copy_dest_dir, dest_name)

    shutil.copy(src, dest_path)
    copied += 1

print(f"Copied {copied}/{n_rows} PDBs to {copy_dest_dir}")
if missing:
    print(f"⚠️  Skipped {missing} entries due to missing PDB paths")

## IV.B Execute Metric Monster

In [ ]:
##############################
### RUN THE METRIC MONSTER ###
##############################

### INPUT ###
pdb_or_dir = f"{PREDESIGN_OUT_DIR}i1_stage1/ZZZ_FILTERED_STRUCTURES_ZZZ"  # either a .pdb file or a directory

### OPTIONAL INPUT FOR MAIN SCRIPT ###
nproc            = None   # number of parallel jobs when directory; set to None for auto
keep_temp_csvs   = False  # do not delete intermediate CSVs
quiet            = False  # suppress sub-script output

### OPTIONAL INPUTS FOR THE SUBSCRIPTS ###
add_subflags = False   # set to False to skip adding any cc/pss/fpock flags
cc_args      = ["--ligands", "LIG1", "--cutoffs", "4", "5", "6"] if add_subflags else []  # contact_counter__MAIN.py flags
pss_args     = ["--sphere-samples", "1000"]                      if add_subflags else []  # protein_size_shape_metrics__MAIN.py flags
fpock_args   = ["--min_spheres_per_pocket", "20"]                if add_subflags else []  # execute_fpocket_on_holo_structure__MAIN.py flags

### SCRIPT PATH(S) ###
metric_monster_script_path = os.path.join(SPECIAL_SCRIPTS_DIR, "design_filtering/metric_monster__MAIN.py")

### DETECT MODE ###
run_parallel = os.path.isdir(pdb_or_dir)

### BUILD THE COMMAND ###
cmd = ["python", metric_monster_script_path]

if run_parallel:
    cmd += ["--run_parallel_in_terminal_on_pdb_dir", pdb_or_dir]
    if nproc is not None:
        cmd += ["--nproc", str(nproc)]
else:
    cmd += [pdb_or_dir]

### ADD CONDITIONALS ###
if keep_temp_csvs:
    cmd.append("--keep-temp-csvs")
if quiet:
    cmd.append("--quiet")

if add_subflags:
    if cc_args:
        cmd += ["--cc"] + cc_args
    if pss_args:
        cmd += ["--pss"] + pss_args
    if fpock_args:
        cmd += ["--fpock"] + fpock_args

### PRINT THE FULL SHELL COMMAND ###
print("### GENERATED COMMAND ###")
print(" ".join(shlex.quote(x) for x in cmd))

### PREDICTED OUTPUT ### 
if run_parallel:
    print()
    print("### EXPECTED CSV FILE ###")
    # avoid nested double quotes inside an f-string (would terminate the outer string)
    print(os.path.join(pdb_or_dir, "Zzz___METRIC_MONSTER_DF___zzZ.csv"))

## IV.C Filter on Comprehensive Metrics

In [ ]:
################################
### QUICK METRIC EXPLORATION ###
################################

### INPUTS ###
file_path = f'{PREDESIGN_OUT_DIR}i1_stage1/ZZZ_FILTERED_STRUCTURES_ZZZ/Zzz___METRIC_MONSTER_DF___zzZ.csv'

### --- METRIC OF INTEREST --- ###
metric = "contact_counter__all_hetatm_contacts__CUTOFF_6A"

### LOAD & PREP ###
df   = pd.read_csv(file_path, engine='python')
data = df[metric]
total, n_na = len(df), data.isna().sum()
print(f"[METRIC EVAL]: {metric}")
print(f"[INFORMATION]: ROWS = {total} ---- NaNs = {n_na}")

### STATS ###
data = data.dropna()
q   = data.quantile([0,.25,.5,.75,1]); iqr = q.loc[0.75] - q.loc[0.25]
stats = {
    "min": q.loc[0.0], "Q1": q.loc[0.25], "med": q.loc[0.5], "Q3": q.loc[0.75],
    "max": q.loc[1.0], "mean": data.mean(), "std": data.std(), "IQR": iqr
}
print()
print("  |  ".join(f"{k}={v:.4g}" for k,v in stats.items()))

### OUTLIERS ###
lower, upper = stats["Q1"] - 1.5*iqr, stats["Q3"] + 1.5*iqr
below, above = data[data<lower], data[data>upper]
print()
print(f"LOWER OUTLIERS: <{lower:.4g} = {len(below)}")
print(f"UPPER OUTLIERS: >{upper:.4g} = {len(above)}")

### HISTOGRAM ###
plt.figure(figsize=(8,4))
plt.hist(data, bins=30, edgecolor='black')
plt.title(f"Histogram of {metric}")
plt.xlabel(metric); plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
##################################################################
### FILTER THE METRIC MONSTER COMBINED DATAFRAME ###############
##################################################################

### INPUT COMBINED CSV ###
file_path = f'{PREDESIGN_OUT_DIR}i1_stage1/ZZZ_FILTERED_STRUCTURES_ZZZ/Zzz___METRIC_MONSTER_DF___zzZ.csv'
df_mm = pd.read_csv(file_path, engine='python')

### NORMALIZE PDB PATHS ###
# If a path ends with "_out.pdb", convert to ".pdb"
if "pdb_path" in df_mm.columns:
    df_mm["pdb_path"] = (df_mm["pdb_path"].astype(str).str.replace(r"_out\.pdb$", ".pdb", regex=True))
else:
    print("⚠️  Column 'pdb_path' not found in df_mm — skipping _out.pdb → .pdb normalization")


### OPTIONAL CONFIGURATIONS ###
allow_universal_nans_to_pass  = True   # If True, NaN in any filter column counts as pass
show_df_head                  = True    # Show the first rows of the filtered DataFrame
show_nans                     = False

### SORTING PRINTS ###
sorting_col       = "protein_SizeShape__max_dist"  # Column to sort by
sorting_ascending = False   # False = descending, True = ascending
n_example         = 20      # Number of examples to display

### FILTER CRITERIA ###
filters = [
     #("contact_counter__all_hetatm_contacts__CUTOFF_4A",    (">=",  5)),
#     ("contact_counter__all_hetatm_contacts__CUTOFF_5A",    ("<=", 10000)),
      ("contact_counter__all_hetatm_contacts__CUTOFF_6A",    (">=",  0)),
     ##("contact_counter__all_hetatm_contacts__CUTOFF_6A",    ("<=", 145)),
#     ("contact_counter__aromatic_sc_contacts__CUTOFF_4A",   ("<=",   1000)),
#     ("contact_counter__aromatic_sc_contacts__CUTOFF_5A",   (">=",   3)),
#     ("contact_counter__aromatic_sc_contacts__CUTOFF_6A",   (">=",   4)),
     ##("contact_counter__backbone_contacts__CUTOFF_4A",      ("<=",   17)),
     ##("contact_counter__backbone_contacts__CUTOFF_5A",      ("<=",   35)),
     ##("contact_counter__backbone_contacts__CUTOFF_6A",      ("<=",   72)),
#     ("contact_counter__protein_nonpolar_contacts__CUTOFF_4A", (">",   0)),
#     ("contact_counter__protein_nonpolar_contacts__CUTOFF_5A", (">=",  1)),
#     ("contact_counter__protein_nonpolar_contacts__CUTOFF_6A", (">=",  2)),
#     ("contact_counter__protein_polar_contacts__CUTOFF_4A",   (">=",  1)),
#     ("contact_counter__protein_polar_contacts__CUTOFF_5A",   (">=",  1)),
#     ("contact_counter__protein_polar_contacts__CUTOFF_6A",   (">=",  1)),
     #("contact_counter__sidechain_contacts__CUTOFF_4A",       (">=",  20)),
#     ("contact_counter__sidechain_contacts__CUTOFF_5A",       (">=",  3)),
#     ("contact_counter__sidechain_contacts__CUTOFF_6A",       (">=",  4)),

#     ("fpocket__cumulative_apolar_sasa",                     ("<=", 100.0)),
#     ("fpocket__cumulative_cent_of_mass_alpha_sphere_max_dist",("<=", 20.0)),
#     ("fpocket__cumulative_charge_score",                    (">=",   0.0)),
#     ("fpocket__cumulative_flexibility",                     (">=",   0.0)),
#     ("fpocket__cumulative_number_of_alpha_spheres",         (">=",  10)),
#     ("fpocket__cumulative_polar_sasa",                      ("<=",  50.0)),
#     ("fpocket__cumulative_polarity_score",                  (">=",   0.0)),
#     ("fpocket__cumulative_total_sasa",                      ("<=", 200.0)),
#     ("fpocket__cumulative_volume",                          ("<=", 500.0)),
#     ("fpocket__cumulative_volume_score",                    (">=",   0.0)),
    # ("fpocket__max_druggability_score",                     (">=",   0.5)),
#     ("fpocket__max_raw_score",                              (">=",   0.5)),
#     ("fpocket__weighted_alpha_sphere_density",              (">=",   0.0)),
#     ("fpocket__weighted_apolar_alpha_sphere_proportion",    (">=",   0.0)),
#     ("fpocket__weighted_charge_score",                      (">=",   0.0)),
    # ("fpocket__weighted_druggability_score",                (">=",   4.525)),
#     ("fpocket__weighted_flexibility",                       (">=",   0.0)),
#     ("fpocket__weighted_hydrophobicity_score",              (">=",   0.0)),
#     ("fpocket__weighted_mean_alp_sph_solvent_access",       (">=",   0.0)),
#     ("fpocket__weighted_mean_alpha_sphere_radius",          (">=",   0.0)),
#     ("fpocket__weighted_mean_local_hydrophobic_density",    (">=",   0.0)),
#     ("fpocket__weighted_polarity_score",                    (">=",   0.0)),
#     ("fpocket__weighted_score",                             (">=",   0.0)),
#     ("fpocket__weighted_volume_score",                      (">=",   0.0)),

#     ("protein_SizeShape__residue_length",                   ("<=", 200)),
#     ("protein_SizeShape__Rg",                               ("<=",  25.0)),
    # ("protein_SizeShape__Rg_norm_by_ideal_sphere",          ("<=",   0.9)),
    # ("protein_SizeShape__Rg_norm",                          ("<=",   2.8)),
#     ("protein_SizeShape__eig1",                             (">=",  10.0)), # PROPORTIONAL TO LENGTH
    # ("protein_SizeShape__eig2",                             ("<=",   100)), # PROPORTIONAL TO SOME ORTHOGONAL LENGTH 
#     ("protein_SizeShape__eig3",                             (">=",   1.0)), # PROPORTIONAL TO THICKNESS
   #  ("protein_SizeShape__shape_anisotropy",                 ("<=",   0.16052)),
     ##("protein_SizeShape__shape_anisotropy",                 (">=",   0.009)),
   # ("protein_SizeShape__max_dist",                         ("<=",   52.6)),
   #  ("protein_SizeShape__dist_COM_lig",                     ("<=",  12)),
#     ("protein_SizeShape__nearest_prot_dist_lig",            ("<=",   5.0)),
]

# pull out just the column names
filter_cols = [col for col, spec in filters]

### CALCULATE WIDTHS FOR ALIGNMENT ###
label_width = max(max(len(col) for col in filter_cols),
                  len("ALL FILTERS APPLIED")) + 2
thr_width   = max(len(str(spec[1])) for _,spec in filters) + 1
count_width = len(str(len(df_mm)))

### REPORT NaNs (only if >0) ###
total = len(df_mm)
nan_counts = df_mm[filter_cols].isna().sum()
print(f"[MetricMonster] ({total} designs)")
print("### NaN counts per filter column (only >0) ###")
for col, cnt in nan_counts.items():
    if cnt > 0:
        print(f"  {col.ljust(label_width)} : {cnt}")
print()

### APPLY FILTERS INDIVIDUALLY ###
mask_list = []
for col, spec in filters:
    op, thr = spec[0], spec[1]
    # if you supported per-filter allow_nan:
    allow_nan = (len(spec) == 3 and spec[2]) or allow_universal_nans_to_pass

    series = df_mm[col]
    if   op == "<=": cond = series <= thr
    elif op == ">=": cond = series >= thr
    else: raise ValueError(f"Unsupported operator {op}")

    if allow_nan:
        cond |= series.isna()

    count = int(cond.sum())
    pct   = count / total * 100
    print(f"# [ {col.ljust(label_width)} {op.ljust(2)} {str(thr).ljust(thr_width)} ]: "
          f"{str(count).rjust(count_width)}   ({pct:5.1f}%)")

    mask_list.append(cond)


### COMBINE FILTERS ###
combined_mask = mask_list[0]
for m in mask_list[1:]:
    combined_mask &= m

passed = int(combined_mask.sum())
pct_passed = passed / total * 100
print()
print(f"# [ {'ALL FILTERS APPLIED'} ]: "
      f"{str(passed).rjust(count_width)} ({pct_passed:5.1f}%)")
print(f"{passed} of {total} ({pct_passed:.3f}%) passed\n")

### SORT AND DISPLAY RESULTS ###
# Build the sorted view once, before either display block: show_nans reads it,
# and the predesign cell below takes it as its input, so it has to exist
# whatever the two display toggles are set to.
subset = df_mm[combined_mask].sort_values(by=sorting_col, ascending=sorting_ascending)
cols = list(subset.columns)
cols.insert(0, cols.pop(cols.index(sorting_col)))   # sorting column first
subset = subset[cols]

if show_nans:
    # show passed rows with NaN in sorting_col
    nan_passed = subset[subset[sorting_col].isna()]
    if len(nan_passed):
        print(f"\n[Warning] {len(nan_passed)} designs passed all filters but have NaN in '{sorting_col}'. Showing up to 20:")
        print("pymol " + " ".join(nan_passed['pdb_path'].head(20).tolist()))
    print()

if show_df_head:
    print(f"# SORTED BY [{sorting_col}] IN "
          f"[{'Ascending' if sorting_ascending else 'Descending'}]:")
    top_paths = subset['pdb_path'].head(n_example).tolist()
    print("pymol " + " ".join(top_paths))
    print()
    display(subset.head(n_example))

## IV.D Copy More Filtered Files into New Folder 

In [ ]:
###########################################
### COPY FILTERED PREDESIGNED BACKBONES ###
###########################################

filtered_predesign_df = subset.copy()

### PATH FOR FILTERED STRUCTURES TO GO ###
filtered_predesign_path = f"{PREDESIGN_OUT_DIR}filtered_RFdiffusion3/filtered_structures/further_filtered/"

### EXECUTE ###
if not os.path.isdir(filtered_predesign_path):
    os.mkdir(filtered_predesign_path)

print (f"Number of PDB files to copy: {len(filtered_predesign_df)}")
for i, desc in enumerate(filtered_predesign_df["pdb_path"]):
    shutil.copy(desc, os.path.join(filtered_predesign_path, os.path.basename(desc)))
    #shutil.copy(os.path.join(trb_file_path, os.path.basename(desc)+".trb"), os.path.join(filtered_predesign_path, os.path.basename(desc)+".trb"))
    if i % 1000 == 0:
        print (f"[{time.ctime()}] {i}th PDB file was copied...")
print ("Done")

# **V. LigandMPNN Sequence Design**

**NOTE:** Renaming files with this command:
```
for f in *.pdb; do
  new=$(echo "$f" | sed -e 's/ZAPP_p1D1_active_site_NEB_TS_/ZAPP_p1D1_/' \
                             -e 's/cfg_T__cfgsc_1_50__step_1_50__gam0_0_60__gamMIN_0_10__jit_1_50//')
  mv -v "$f" "$new"
done
```

## V.A Generate Fixed Residues json for MPNN

**NOTE:** At this point I copied all the filtered inpainting and RFd2 structures into a new predesign directory.

In [ ]:
##################################################################
### GRAB CATALYTIC RESIDUES AND MAKE FIXED POS .JSONL FOR MPNN ###
##################################################################

### INPUTS FOR THE FUNCTION ###
input_pdb_directory = f"{PREDESIGN_OUT_DIR}i1_stage1/ZZZ_FILTERED_STRUCTURES_ZZZ/rename/"
output_jsonl = f"{FIXED_RESIDUE_JSONLS_DIR}scaffold_fixed_residues.json"

### OPTIONAL PARAMETERS ###
use_legacy = False

### CONSTANTS ###
apptainer = APPTAINER
script_path = f"{SPECIAL_SCRIPTS_DIR}mpnn_relevant_utils/grab_cat_residues_from_pdb_and_write_fixedAA_JSON.py"

### CONSTRUCT COMMAND ###
command = (f"singularity exec {apptainer} python {script_path} --input_dir {input_pdb_directory} --output_json {output_jsonl}")
if use_legacy:
    command += " --use_legacy"
    
### PRINT THE COMMAND FOR TERMINAL EXECUTION ###
print("### EXECUTE THIS COMMAND ###")
print(command)

## V.B Generate & Execute LigandMPNN Commands

**NOTE:** At this point, we will combine the inpainted & non-inpainted directories together via merging in the same output directory. \

In [ ]:
############################################################
### LIGANDMPNN SEQUENCE DESIGN (orchestrated, one cmd/PDB) ###
############################################################
# One self-contained command per input PDB. Each one:
#   PRE  : hold the REMARK 666 catalytic residues fixed; omit Met at residue 1;
#          optionally conserve designable sidechains that H-bond the active site
#          (rolled per combination, so the second shell varies between designs).
#   MPNN : run LigandMPNN once per temperature/batch combination, via
#          run_ligandmpnn.py -- so fixed residues keep their exact input
#          coordinates instead of being rebuilt at idealized geometry.
#   POST : flatten to one directory of packed PDBs; protonate (holo with a
#          .params, else apo with the ligand copied back), restoring the input's
#          catalytic tautomers; restore REMARK 666, write REMARK 668 (+ PTM) and
#          DESIGN_PATH; copy the input alongside the designs.

### INPUTS ###
input_pdb_directory  = f"{PREDESIGN_OUT_DIR}i1_stage1/ZZZ_FILTERED_STRUCTURES_ZZZ/rename/"
pdb_glob             = "*.pdb"
fixed_residues_jsonl = ""     # "" -> auto-fix from REMARK 666; else a fixed-residues JSON

### OPTIONAL LIGAND / PTM ###
lig_params = f"{PARAMS_DIR}YYE.params"   # None -> apo protonation (ligand copied back from input)
ptm_spec   = "A/LYS/3:KCX"               # REMARK 668 PTM annotation; "" -> none

### OUTPUTS ###
mpnn_output_dir = f"{MPNN_OUT_DIR}i1"    # each input PDB lands in <mpnn_output_dir>/<stem>/

### MPNN PARAMETERS (universal; a combo below overrides only the keys it names) ###
model_type             = "ligand_mpnn"   # protein_mpnn | soluble_mpnn | ligand_mpnn
omit_aa                = "CX"
bias_AA                = "K:-0.5,R:-0.75,E:0.75,D:0.75"
mpnn_batch_size        = 1               # sequences per pass
pack                   = 1               # 1 = pack side chains
repack_everything      = 0               # 0 = only repack redesigned residues
sc_num_denoising_steps = 3
use_side_chain_context = 1

### PER-COMBO SWEEP (each dict overrides only its keys) ###
# Keys: number_of_batches, temperature, batch_size, omit_AA, bias_AA, tag.
combos = [
    {"number_of_batches": 15, "temperature": 0.1},
    {"number_of_batches": 15, "temperature": 0.2},
    {"number_of_batches": 10, "temperature": 0.3},
]

### PRE-PROCESSING TOGGLES ###
fix_remark666_catres = True   # hold the REMARK 666 catalytic residues fixed
omit_nterm_met       = True   # omit Met at residue 1 (the expression tag supplies it)

### H-BOND SIDECHAIN CONSERVATION ###
conserve_hbonds          = True   # pin designable sidechains that H-bond the active site
conserve_prob            = 0.8    # per-residue probability (0-1, or a percentage)
conserve_all_or_none     = False  # True = one roll for every combo; else independent
conserve_seed            = None   # int to replay a roll; None -> generated and printed
conserve_anchors         = "ligand,catalytic,user_fixed"
conserve_hbond_max_dist  = 3.9    # heavy-atom donor...acceptor cutoff (A)
conserve_hbond_max_angle = 90     # antecedent-D-A gate; larger is more permissive
conserve_keep_clashing   = False  # keep candidates clashing with fixed backbone/ligand

### PROTONATION ###
protonate = True   # LigandMPNN emits zero protein hydrogens; this adds them

### OUTPUT KNOBS ###
copy_input         = True
transfer_remarks   = True
design_path_remark = True
keep_intermediates = False

### RE-RUN SAFETY ###
force_overwrite = False
DRY_RUN         = False   # True -> commands are printed but MPNN does not run

### MODEL WEIGHTS ###
# Not bundled. Download once, then export LIGANDMPNN_WEIGHTS:
#   bash Software/fastmpnndesign/lib/LigandMPNN/get_model_params.sh model_weights/
ligandmpnn_weights_dir   = os.environ.get("LIGANDMPNN_WEIGHTS", "")
ligandmpnn_checkpoint    = "ligandmpnn_v_32_010_25.pt"      # matches model_type above
ligandmpnn_sc_checkpoint = "ligandmpnn_sc_v_32_002_16.pt"   # side-chain packing model

### CONSTANTS ###
orchestrator = f"{SPECIAL_SCRIPTS_DIR}mpnn_relevant_utils/design_orchestrator.py"

### SANITY CHECKS ###
if not Path(orchestrator).is_file():
    raise FileNotFoundError(f"orchestrator not found: {orchestrator}")
if not Path(input_pdb_directory).is_dir():
    raise FileNotFoundError(f"input_pdb_directory not found: {input_pdb_directory}")
if lig_params and not Path(lig_params).is_file():
    raise FileNotFoundError(f"lig_params not found: {lig_params}\n"
                            f"  Run the ligand .params cell in Section II first.")
Path(mpnn_output_dir).mkdir(parents=True, exist_ok=True)

### QUICK LOGIC ###
input_pdbs = sorted(glob.glob(os.path.join(input_pdb_directory, pdb_glob)))
if not input_pdbs:
    raise FileNotFoundError(f"No PDBs matched {os.path.join(input_pdb_directory, pdb_glob)}")
designs_per_input = sum(int(c.get("number_of_batches", 1)) * int(c.get("batch_size", mpnn_batch_size))
                        for c in combos)

### BUILD COMMANDS (one self-contained line per input PDB) ###
def combo_to_run(combo: dict) -> str:
    return ";".join(f"{k}={v}" for k, v in combo.items())

commands, skipped_existing = [], []
for pdb_file in input_pdbs:
    stem = Path(pdb_file).stem
    out_dir = os.path.join(mpnn_output_dir, stem)
    if not force_overwrite and Path(out_dir).is_dir() and list(Path(out_dir).glob("*.pdb")):
        skipped_existing.append(stem)
        continue

    parts = ["python", orchestrator,
             "--pdb_path", pdb_file,
             "--out_folder", out_dir,
             "--model_type", model_type,
             "--batch_size", str(mpnn_batch_size),
             "--pack_side_chains", str(pack),
             "--repack_everything", str(repack_everything),
             "--sc_num_denoising_steps", str(sc_num_denoising_steps),
             "--ligand_mpnn_use_side_chain_context", str(use_side_chain_context),
             "--omit_AA", omit_aa,
             "--bias_AA", bias_AA]

    # ---- model weights ----
    if ligandmpnn_weights_dir:
        parts += [f"--checkpoint_{model_type}", f"{ligandmpnn_weights_dir}/{ligandmpnn_checkpoint}"]
        if pack:
            parts += ["--checkpoint_path_sc", f"{ligandmpnn_weights_dir}/{ligandmpnn_sc_checkpoint}"]

    # ---- fixed residues / pre-processing ----
    if fixed_residues_jsonl:
        parts += ["--fixed_residues_multi", fixed_residues_jsonl]
    if not fix_remark666_catres:
        parts += ["--no_fix_remark666_catres"]
    if not omit_nterm_met:
        parts += ["--no_omit_nterm_met"]

    # ---- H-bond conservation ----
    if conserve_hbonds:
        parts += ["--conserve_hbonds",
                  "--conserve_hbond_prob", str(conserve_prob),
                  "--conserve_anchors", conserve_anchors,
                  "--conserve_hbond_max_dist", str(conserve_hbond_max_dist),
                  "--conserve_hbond_max_angle", str(conserve_hbond_max_angle)]
        if conserve_all_or_none:
            parts += ["--conserve_hbond_all_or_none"]
        if conserve_seed is not None:
            parts += ["--conserve_seed", str(conserve_seed)]
        if conserve_keep_clashing:
            parts += ["--conserve_keep_clashing"]

    # ---- protonation ----
    if not protonate:
        parts += ["--no_protonate"]
    elif lig_params:
        parts += ["--ligand_params", lig_params]
    if ptm_spec:
        parts += ["--ptm", ptm_spec]

    # ---- output knobs ----
    if not copy_input:
        parts += ["--no_copy_input_structure"]
    if not transfer_remarks:
        parts += ["--no_transfer_remarks"]
    if not design_path_remark:
        parts += ["--no_design_path_remark"]
    if keep_intermediates:
        parts += ["--keep_intermediates"]
    if DRY_RUN:
        parts += ["--dry_run"]

    # ---- combos (one --run per combination) ----
    for c in combos:
        parts += ["--run", combo_to_run(c)]

    commands.append(" ".join(shlex.quote(x) for x in parts))

if skipped_existing:
    print(f"Skipped {len(skipped_existing)} PDB(s) with existing outputs "
          f"(force_overwrite = True to re-run), e.g. {skipped_existing[:3]}")
if not commands:
    raise RuntimeError("No commands generated; every input was skipped by re-run protection.")

### WRITE COMMANDS FILE ###
commands_file = f"{CMDS_DIR}ligandmpnn_orchestrated_i1"
with open(commands_file, "w") as f:
    f.write("\n".join(commands) + "\n")
command_count = len(commands)

print(f"Inputs              = {command_count} PDB(s)")
print(f"Designs per input   = {designs_per_input} across {len(combos)} combo(s)")
print(f"Total designs       = {command_count * designs_per_input}")
print(f"Fixed residues      =", "user jsonl" if fixed_residues_jsonl else
      ("REMARK 666 auto-fix" if fix_remark666_catres else "NONE (full redesign)"))
print(f"Conserve H-bonds    =", (f"ON (p={conserve_prob}, "
      f"{'all-or-none' if conserve_all_or_none else 'per-combo'})") if conserve_hbonds else "off")
print(f"Protonation         =", ("holo" if lig_params else "apo") if protonate else "off")
print(f"\nCommands File:\n{commands_file}")
print(f"\n# example command:\n{commands[0]}")

### SETUP BATCH JOBS ###
qtime        = '00:45:00'
cmds_per_job = 1
cores        = '1'
memory       = '8g'
queue        = 'cpu'
job_name     = os.path.basename(commands_file)
submit_file  = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs     = math.ceil(command_count / cmds_per_job)
print(f"\nNumber of Jobs = {num_jobs}   Job Name = {job_name}")
print(f"\nNavigate here:\ncd {mpnn_output_dir}\n")
nb.submit_array_job(commands_file, qtime, cores, job_name, memory, submit_file,
                    LOGS_DIR, num_jobs, cmds_per_job, queue)


## V.C Paste Remark 666 Lines & Hydrogens into Packed MPNN Structures for Reference

In [ ]:
######################################################################################
### TRANSFER REMARK 666 + ADD ROSETTA HYDROGENS FROM REFERENCE PDBS TO OUTPUT PDBS ###
######################################################################################
"""
IMPORTANT NOTES:
1. "directory_batch" mode: Runs the optimized script directly with multiprocessing
2. "directory_slurm" mode: Generates per-ref-PDB commands for SLURM array jobs
   - Requires --single_ref_pdb flag in script (check if implemented)
3. "single_file" mode: Process one ref→output pair (1:1, no suffix logic)
   - Requires --ref_pdb and --output_pdb flags in script (check if implemented)

PERFORMANCE TIPS:
- For 100k+ files: Set manual_suffixes to skip auto-detection (~45s savings)
- For debugging: Set single_thread=True and debug=True
- For cluster: Match n_workers to allocated cores

EXPECTED RATES:
- ~6-10 PDBs/sec with 20 workers (PyRosetta bottleneck)
- 100k files ≈ 3-4 hours total runtime
"""

### MODE SELECTION ###
mode = "directory_slurm"  # Options: "directory_batch", "directory_slurm", "single_file"

### DIRECTORY MODE INPUTS ###
ref_pdb_dir      = f"{PREDESIGN_OUT_DIR}i1_stage1/ZZZ_FILTERED_STRUCTURES_ZZZ/rename/"
output_pdb_dir   = f"{MPNN_OUT_DIR}i1/packed/"
final_output_dir = f"{MPNN_OUT_DIR}i1/packed/outputs/"  # None = overwrite output_pdb_dir

### SINGLE FILE MODE INPUTS ###
ref_pdb    = None  # e.g., "/path/to/reference.pdb"
output_pdb = None  # e.g., "/path/to/output.pdb"

### SUFFIX SPECIFICATION ###
# None = auto-detect suffixes from random sample OR paste list directly from script output, e.g.: ['', '_eV1_T0_10__1_1', '_eV1_T0_10__2_1', '_eV1_T0_15__1_1']
manual_suffixes = ['_eV2_T0_10__10_1', '_eV2_T0_10__11_1', '_eV2_T0_10__12_1', '_eV2_T0_10__13_1', '_eV2_T0_10__14_1', '_eV2_T0_10__15_1', '_eV2_T0_10__1_1', '_eV2_T0_10__2_1', '_eV2_T0_10__3_1', '_eV2_T0_10__4_1', '_eV2_T0_10__5_1', '_eV2_T0_10__6_1', '_eV2_T0_10__7_1', '_eV2_T0_10__8_1', '_eV2_T0_10__9_1', '_eV2_T0_20__10_1', '_eV2_T0_20__11_1', '_eV2_T0_20__12_1', '_eV2_T0_20__13_1', '_eV2_T0_20__14_1', '_eV2_T0_20__15_1', '_eV2_T0_20__1_1', '_eV2_T0_20__2_1', '_eV2_T0_20__3_1', '_eV2_T0_20__4_1', '_eV2_T0_20__5_1', '_eV2_T0_20__6_1', '_eV2_T0_20__7_1', '_eV2_T0_20__8_1', '_eV2_T0_20__9_1', '_eV2_T0_30__10_1', '_eV2_T0_30__1_1', '_eV2_T0_30__2_1', '_eV2_T0_30__3_1', '_eV2_T0_30__4_1', '_eV2_T0_30__5_1', '_eV2_T0_30__6_1', '_eV2_T0_30__7_1', '_eV2_T0_30__8_1', '_eV2_T0_30__9_1'] # NOTE: I would run the script once to get the possible suffixes then put them here to help performance
sample_size     = 1     # For auto-detection: number of random ref PDBs to sample

### PERFORMANCE SETTINGS ###
n_workers     = None        # Number of CPU workers (None = auto-detect all cores)
single_thread = True       # True = disable multiprocessing (for debugging)
debug         = False       # True = enable detailed debug output

### FILE HANDLING ###
clobber = True  # Overwrite existing output files

### SLURM PARAMETERS (only used in directory_slurm mode) ###
qtime = '00:15:00'
cores = '1'
memory = '2g'
queue = 'cpu'
cmds_per_job = 5
unique_cmds_suffix = "I1_PACKED"

### CONSTANTS ###
apptainer = APPTAINER
script = f'{SPECIAL_SCRIPTS_DIR}general_utils/add_remark666_lines_AND_rosetta_hydrogens_to_pdb_OPTIMIZED.py'

### VALIDATION ###
if mode in ("directory_batch", "directory_slurm"):
    assert ref_pdb_dir is not None, "ref_pdb_dir must be set for directory modes"
    assert output_pdb_dir is not None, "output_pdb_dir must be set for directory modes"
elif mode == "single_file":
    assert ref_pdb is not None, "ref_pdb must be set for single_file mode"
    assert output_pdb is not None, "output_pdb must be set for single_file mode"
else:
    raise ValueError(f"Invalid mode: {mode}. Choose 'directory_batch', 'directory_slurm', or 'single_file'")

### HELPER: BUILD SUFFIX ARGS ###
def build_suffix_args(manual_suffixes, sample_size):
    """Convert manual_suffixes list to command-line format, or use sample_size for auto-detect."""
    if manual_suffixes is not None:
        # Convert Python list to space-separated quoted args
        suffix_args = " ".join([f'"{s}"' for s in manual_suffixes])
        return f"--suffixes {suffix_args} "
    elif sample_size:
        return f"--find_suffixes_from_random_sample {sample_size} "
    return ""

### HELPER: BUILD COMMON FLAGS ###
def build_common_flags():
    """Build flag string for performance/debug/clobber settings."""
    flags = ""
    if n_workers is not None:
        flags += f"--n_workers {n_workers} "
    if single_thread:
        flags += "--single_thread "
    if debug:
        flags += "--debug "
    if clobber:
        flags += "--clobber "
    return flags

### AUTOMATED COMMAND GENERATION & EXECUTION ###
if mode == "directory_batch":
    # DIRECTORY BATCH MODE - Direct Execution
    command = f"{apptainer} {script} "
    command += f"--ref_pdb_dir {ref_pdb_dir} "
    command += f"--output_pdb_dir {output_pdb_dir} " 
    if final_output_dir:
        command += f"--final_output_dir {final_output_dir} "
    command += build_suffix_args(manual_suffixes, sample_size)
    command += build_common_flags()
    
    # Print summary
    print("="*80)
    print("MODE: DIRECTORY BATCH (Direct Execution)")
    print("="*80)
    print(f"  Reference PDB dir:  {ref_pdb_dir}")
    print(f"  Output PDB dir:     {output_pdb_dir}")
    print(f"  Final output dir:   {final_output_dir if final_output_dir else '(overwrite output_pdb_dir)'}")
    print(f"  Suffixes:           {'manual (' + str(len(manual_suffixes)) + ' specified)' if manual_suffixes else 'auto-detect (sample=' + str(sample_size) + ')'}")
    print(f"  Workers:            {n_workers if n_workers else 'auto-detect'}")
    print(f"  Single thread:      {single_thread}")
    print(f"  Debug:              {debug}")
    print(f"  Clobber:            {clobber}")
    print("="*80)
    print("\n### COMMAND TO EXECUTE ###")
    print(command)

elif mode == "directory_slurm":
    # DIRECTORY SLURM MODE - Batch Command Generation
    print("[INFO] Scanning for reference PDBs...")
    ref_pdb_list = sorted(glob.glob(os.path.join(ref_pdb_dir, "*.pdb")))
    if not ref_pdb_list:
        raise FileNotFoundError(f"No PDB files found in {ref_pdb_dir}")
    print(f"[INFO] Found {len(ref_pdb_list)} reference PDBs")
    
    # Build one command per reference PDB
    all_cmds = []
    suffix_args = build_suffix_args(manual_suffixes, sample_size)
    common_flags = build_common_flags()
    
    for ref_pdb_path in ref_pdb_list:
        cmd = f"{apptainer} {script} "
        cmd += f"--single_ref_pdb {ref_pdb_path} "  # NOTE: Requires script support
        cmd += f"--output_pdb_dir {output_pdb_dir} "
        if final_output_dir:
            cmd += f"--final_output_dir {final_output_dir} "
        cmd += suffix_args
        cmd += common_flags
        all_cmds.append(cmd.strip())
    all_cmds_sorted = sorted(all_cmds)     # Sort commands lexically
    # Write to commands file
    commands_file = f"{CMDS_DIR}add_remark666_hydrogens_{unique_cmds_suffix}"
    with open(commands_file, 'w') as f:
        for cmd in all_cmds_sorted:
            f.write(cmd + "\n")
    # Calculate job counts
    total_cmds = len(all_cmds_sorted)
    num_jobs = (total_cmds + cmds_per_job - 1) // cmds_per_job  # Ceiling division
    # SLURM submission setup
    job_name = f"remark666_H_{unique_cmds_suffix}"
    submit_file = f"{SUBMIT_DIR}{job_name}.sh"
    
    # Print summary
    print("="*80)
    print("MODE: DIRECTORY SLURM (Batch Command Generation)")
    print("="*80)
    print(f"  Reference PDB dir:  {ref_pdb_dir}")
    print(f"  Output PDB dir:     {output_pdb_dir}")
    print(f"  Final output dir:   {final_output_dir if final_output_dir else '(overwrite output_pdb_dir)'}")
    print(f"  Suffixes:           {'manual (' + str(len(manual_suffixes)) + ' specified)' if manual_suffixes else 'auto-detect (sample=' + str(sample_size) + ')'}")
    print("-"*80)
    print(f"  Total ref PDBs:     {len(ref_pdb_list)}")
    print(f"  Total commands:     {total_cmds}")
    print(f"  Commands per job:   {cmds_per_job}")
    print(f"  Number of jobs:     {num_jobs}")
    print("-"*80)
    print(f"  Queue:              {queue}")
    print(f"  Time limit:         {qtime}")
    print(f"  Cores per job:      {cores}")
    print(f"  Memory per job:     {memory}")
    print("="*80)
    print(f"\n### COMMANDS FILE ###")
    print(commands_file)
    print(f"\n### SAMPLE COMMANDS (first 3) ###")
    for cmd in all_cmds_sorted[:3]:
        print(cmd)
    if len(all_cmds_sorted) > 3:
        print(f"... and {len(all_cmds_sorted) - 3} more")
    # Submit array job
    print("\n### SUBMITTING SLURM ARRAY JOB ###")
    nb.submit_array_job(commands_file, qtime, cores, job_name, memory,submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue)

elif mode == "single_file":
    # SINGLE FILE MODE - 1:1 Processing
    command = f"{apptainer} {script} "
    command += f"--ref_pdb {ref_pdb} "
    command += f"--output_pdb {output_pdb} "
    if final_output_dir:
        command += f"--final_output_dir {final_output_dir} "
    if clobber:
        command += "--clobber "
    if debug:
        command += "--debug "
    # Print summary
    print("="*80)
    print("MODE: SINGLE FILE (1:1 Processing)")
    print("="*80)
    print(f"  Reference PDB:      {ref_pdb}")
    print(f"  Output PDB:         {output_pdb}")
    print(f"  Final output dir:   {final_output_dir if final_output_dir else '(overwrite output_pdb)'}")
    print(f"  Debug:              {debug}")
    print(f"  Clobber:            {clobber}")
    print("="*80)
    print("\n[WARN] This mode requires --ref_pdb and --output_pdb flags in the script.")
    print("       If not implemented, the script will fail. Check script capabilities first.")
    print("\n### COMMAND TO EXECUTE ###")
    print(command)


In [ ]:
############################################################
### CHECK PDBs FOR REMARK 666 LINES (MIN/MAX THRESHOLDS) ###
############################################################

### ARGUMENTS ###
pdb_dir_to_check = f"{MPNN_OUT_DIR}i1/packed/"
pdb_file_glob_pattern = None # e.g., "/other/path/to_pdbs/*group1*.pdb"

### OPTIONAL THRESHOLDS ###
min_remark666 = 9
max_remark666 = 19

### CONSTANTS ###
check_script = f"{SPECIAL_SCRIPTS_DIR}general_utils/check_pdbs_for_remark666_lines.py"

### CONSTRUCT COMMAND ###
cmd = (f"python {check_script} ")

if pdb_dir_to_check is not None:
    cmd += f"--pdb_dir_input {pdb_dir_to_check} "

if pdb_file_glob_pattern is not None:
    cmd += f'--pdb_input "{pdb_file_glob_pattern}" '  # in quotes if it might contain wildcards

if min_remark666 is not None:
    cmd += f"--number_of_minimal_remark666_lines_to_expect {min_remark666} "

if max_remark666 is not None:
    cmd += f"--number_of_maximal_remark666_lines_to_expect {max_remark666} "

print("### COMMAND TO EXECUTE ###")
print(cmd)

# **VI. Apo-AF3 (without tags)**

## VI.A.1 Execute AF3 Commands

In [ ]:
################################                                                                                                                                               
### Make AF3 input json file ###                                                                                                                                               
################################                                                                                                                                               

### SHARED INPUTS ###                                                                                                                                                          
input_pdb_path          = f"{MPNN_OUT_DIR}i1/packed/outputs/"        
input_pdb_protein_chain = ["A"]                                                                                                                                                

af3_output_dir    = f"{AF3_OUT_DIR}i1__apo/"                                                                                                                                         
num_input_per_run = 45                                          
                                                                                                                                                                                
### SHARED OPTIONAL LIGAND FLAG ###                                                                                                                                            
include_ligands_if_present = False
input_ligand_dic           = {                                                                                                                                                 
  #  "B": {"ccdCodes": "['ZN']"},                                                                                                                                               
  #  "C": {"smiles": "[Zn+2].[OH-]"},                                                                                                                                               
  #  "D": {"smiles": "CCOP(=O)(OCC)OC1=CC=C(C=C1)[N+](=O)[O-]"},                                                                                                          
}   # leave empty {} or set to None to omit ligand args                                                                                                                        
                                                                                                                                                                                
### SHARED OPTIONAL PTM FLAG ###                                                                                                                                               
include_post_translational_mods_if_present = False                                                                                                                              
ptm_specs = ["A/LYS/3:KCX"]                                     
                                                                                                                                                                                
### SHARED OPTIONAL TERMINUS TAG FLAGS ###                                                                                                                                     
n_terminus_tag = ""                                                                                                                                                         
c_terminus_tag = ""                                                                                                                                                 
                                                                                                                                                                                
### SHARED OPTIONAL FLAGS ###                                                                                                                                                  
output_suffix              = "_af3i1"                                                                                                                                          
check_made_output          = True                               
cleanup_incomplete_outputs = True                                                                                                                                              

### SHARED OPTIONAL RECURSIVE SEARCH FLAGS ###                                                                                                                                 
recursive      = False                                           
max_depth      = None                                                                                                                                                          
specific_depth = None                                              
                                                                                                                                                                                
### SHARED SEED OPTIONS ###
no_random_seed = False                                                                                                                                                         
base_seed      = None

### GROUP DEFINITIONS (set to None for single-group behavior) ###
# Each group can override: pdb_prefix, input_ligand_dic, ptm_specs, output_suffix. Any key omitted inherits the shared default above                                                                                                                            
groups = [                                                                                                                                                                     
]                                                                                                                                                                              

### CONSTANTS ###                                                                                                                                                              
apptainer     = APPTAINER
script        = f"{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/make_af3_json_input.py"                                                                             
af3_json_path = os.path.join(AF3_JSON_DIR, os.path.relpath(af3_output_dir, AF3_OUT_DIR))                                                                                       
os.makedirs(af3_output_dir, exist_ok=True)                                                                                                                                     
os.makedirs(af3_json_path, exist_ok=True)                                                                                                                                      
                                                                                                                                                                                
### BUILD COMMAND(S) DYNAMICALLY ###                                                                                                                                           
def build_af3_command(ligand_dic, ptm_list, suffix, pdb_prefix_list=None):
    """Build a single make_af3_json_input.py command string."""                                                                                                                
    cmd_parts = [                                                                                                                                                              
        f"{apptainer} {script}",                                                                                                                                               
        f"--pdb_path {input_pdb_path}",                                                                                                                                        
        f"--pdb_chain {' '.join(input_pdb_protein_chain)}",                                                                                                                    
        f"--json_path {af3_json_path}",                         
        f"--num_input_per_run {num_input_per_run}",                                                                                                                            
        f"--output_path {af3_output_dir}",                                                                                                                                     
    ]
    if ligand_dic:                                                                                                                                                             
        ligand_chains = list(ligand_dic.keys())                                                                                                                                
        ligand_types  = []
        ligand_ids    = []                                                                                                                                                     
        for v in ligand_dic.values():                           
            if len(v) != 1:                                                                                                                                                    
                raise ValueError("Each ligand spec should have exactly one key indicating type (e.g., 'smiles').")                                                             
            ltype, lid = next(iter(v.items()))                                                                                                                                 
            ligand_types.append(ltype)                                                                                                                                         
            ligand_ids.append(lid)                                                                                                                                             
        cmd_parts.append(f"--ligand_chain {' '.join(ligand_chains)}")                                                                                                          
        cmd_parts.append(f"--ligand_type {' '.join(ligand_types)}")
        cmd_parts.append(f'--ligand_id "{" ".join(ligand_ids)}"')                                                                                                              
    if ptm_list:                                                
        cmd_parts.append("--ptm_from_remark666 " + " ".join([f'"{s}"' for s in ptm_list]))                                                                                     
    if n_terminus_tag:                                                                                                                                                         
        cmd_parts.append(f"--n_terminus_tag {n_terminus_tag}")
    if c_terminus_tag:                                                                                                                                                         
        cmd_parts.append(f"--c_terminus_tag {c_terminus_tag}")  
    if suffix:                                                                                                                                                                 
        cmd_parts.append(f"--output_suffix {suffix}")
    if check_made_output:                                                                                                                                                      
        cmd_parts.append("--check_made_output")                 
    if check_made_output and cleanup_incomplete_outputs:
        cmd_parts.append("--cleanup_incomplete_outputs")                                                                                                                       
    if recursive:
        cmd_parts.append("--recursive")                                                                                                                                        
    if max_depth is not None:                                   
        cmd_parts.append(f"--max_depth {max_depth}")
    if specific_depth is not None:                                                                                                                                             
        cmd_parts.append(f"--specific_depth {specific_depth}")
    if no_random_seed:                                                                                                                                                         
        cmd_parts.append("--no_random_seed")                    
    if base_seed is not None:                                                                                                                                                  
        cmd_parts.append(f"--base_seed {base_seed}")
    if pdb_prefix_list:                                                                                                                                                        
        cmd_parts.append(f"--pdb_prefix {' '.join(pdb_prefix_list)}")
    return " ".join(cmd_parts)                                                                                                                                                 

# Resolve shared defaults for ligands/PTMs                                                                                                                                     
shared_ligand_dic = input_ligand_dic if (include_ligands_if_present and input_ligand_dic) else {}
shared_ptm_specs  = ptm_specs if (include_post_translational_mods_if_present and ptm_specs) else []                                                                            
                                                                                                                                                                                
if groups:                                                                                                                                                                     
    # Multi-group mode: each group inherits shared defaults unless overridden                                                                                                  
    print(f"Generating {len(groups)} group commands:\n")                                                                                                                       
    for i, grp in enumerate(groups, 1):                                                                                                                                        
        cmd = build_af3_command(                                                                                                                                               
            ligand_dic      = grp.get("input_ligand_dic", shared_ligand_dic),                                                                                                  
            ptm_list        = grp.get("ptm_specs", shared_ptm_specs),
            suffix          = grp.get("output_suffix", output_suffix),                                                                                                         
            pdb_prefix_list = grp.get("pdb_prefix", None),      
        )                                                                                                                                                                      
        print(f"### Group {i}: {grp.get('name', f'group_{i}')} ###")
        print(cmd)                                                                                                                                                             
        print()                                                                                                                                                                
else:                                                                                                                                                                          
    # Single-group fallback (original behavior)                                                                                                                                
    cmd = build_af3_command(ligand_dic = shared_ligand_dic, ptm_list   = shared_ptm_specs, suffix     = output_suffix,                                                                                                                                            )
    print(cmd) 

In [ ]:
###############                                                                                                                                                                          
### Run AF3 ###                                                                                                                                                                          
###############                                                                                                                                                                          
                                                                                                                                                                                        
### INPUTS ###
af3_output_dir = f"{AF3_OUT_DIR}i1__apo/"
cmds_base_name = f"AF3_i1_apo"

### PARAMETERS ###
N_total_structures = 5  # Total number of structures you want per target (N)
mode = "one_seed"        # "one_seed" -> N diffusion samples from 1 seed | "n_seeds" -> 1 sample from each of N seeds

### CONSTANTS ###
apptainer       = AF3_SIF
script          = AF3_RUNNER
apptainer_b4000 = AF3_SIF
af3_json_path   = os.path.join(AF3_JSON_DIR, os.path.relpath(af3_output_dir, AF3_OUT_DIR))
commands_base   = os.path.join(CMDS_DIR, cmds_base_name)

### QUEUE CONFIGURATION ###
b4000_fraction     = 0.75   # Fraction of commands to B4000 GPUs (0.0 = none, 1.0 = all)
gpu_fraction       = 0.25   # Within regular: fraction to priority queue (gpu vs gpu-bf)
gpu_fraction_b4000 = 1.00   # Within B4000:  fraction to priority queue (gpu-b4000 vs gpu-bf-b4000)

queue_config = {
    "gpu":          {"qtime": "06:30:00", "cmds_per_job": 6, "memory": "12g", "cores": "1"},
    "gpu-bf":       {"qtime": "07:00:00", "cmds_per_job": 8, "memory": "12g", "cores": "1"},

    "gpu-b4000":    {"qtime": "06:30:00", "cmds_per_job": 16, "memory": "12g", "cores": "1"},
    "gpu-bf-b4000": {"qtime": "06:30:00", "cmds_per_job": 16, "memory": "12g", "cores": "1"},
}

### DERIVED FLAGS ###
if mode not in {"one_seed", "n_seeds"}:
    raise ValueError(f"mode must be 'one_seed' or 'n_seeds'. Got: {mode}")
if mode == "one_seed":
    num_seeds = None
    num_diffusion_samples = N_total_structures
else:
    num_seeds = N_total_structures
    num_diffusion_samples = 1

### BUILD COMMANDS ###
# Build commands for both container types from the same JSON list
af3_jsons = sorted(glob.glob(os.path.join(af3_json_path, "*.json")), key=str.lower)

b4000_split = round(len(af3_jsons) * b4000_fraction)
jsons_regular = af3_jsons[b4000_split:]
jsons_b4000   = af3_jsons[:b4000_split]

cmds_regular = []
for af3_json in jsons_regular:
    cmd = f"{apptainer} python {script} --json_path={af3_json} --output_dir={af3_output_dir} --num_diffusion_samples={num_diffusion_samples}"
    if num_seeds is not None:
        cmd += f" --num_seeds={num_seeds}"
    cmds_regular.append(cmd)

cmds_b4000 = []
for af3_json in jsons_b4000:
    cmd = f"{apptainer_b4000} af3 --json_path={af3_json} --output_dir={af3_output_dir} --num_diffusion_samples={num_diffusion_samples}"
    if num_seeds is not None:
        cmd += f" --num_seeds={num_seeds}"
    cmds_b4000.append(cmd)

### SPLIT AND SUBMIT ###
for frac in [gpu_fraction, gpu_fraction_b4000, b4000_fraction]:
    if not (0.0 <= frac <= 1.0):
        raise ValueError(f"Fractions must be between 0.0 and 1.0. Got: {frac}")
queue_splits = {}

if cmds_regular: # Regular GPU split
    pri_idx = round(len(cmds_regular) * gpu_fraction)
    if pri_idx > 0:
        queue_splits["gpu"] = cmds_regular[:pri_idx]
    if pri_idx < len(cmds_regular):
        queue_splits["gpu-bf"] = cmds_regular[pri_idx:]

if cmds_b4000: # B4000 GPU split
    pri_idx = round(len(cmds_b4000) * gpu_fraction_b4000)
    if pri_idx > 0:
        queue_splits["gpu-b4000"] = cmds_b4000[:pri_idx]
    if pri_idx < len(cmds_b4000):
        queue_splits["gpu-bf-b4000"] = cmds_b4000[pri_idx:]

total_cmds = len(cmds_regular) + len(cmds_b4000)
print(f"{'='*60}")
print(f"  AF3 Job Summary │ {total_cmds} total cmds │ Mode: {mode} │ N: {N_total_structures}")
print(f"  Regular: {len(cmds_regular)} cmds │ B4000: {len(cmds_b4000)} cmds")
print(f"{'='*60}")

for queue, queue_cmds in queue_splits.items():
    cfg = queue_config[queue]
    suffix = f"_{queue.replace('-', '')}" if len(queue_splits) > 1 else ""
    commands_file = f"{commands_base}{suffix}"
    job_name = os.path.basename(commands_file)
    submit_file = f"{SUBMIT_DIR}{job_name}.sh"
    num_jobs = math.ceil(len(queue_cmds) / cfg["cmds_per_job"])
    with open(commands_file, 'w') as fo:
        print("\n".join(queue_cmds), file=fo)
    nb.submit_array_job(commands_file, cfg["qtime"], cfg["cores"], job_name, cfg["memory"], submit_file, LOGS_DIR, num_jobs, cfg["cmds_per_job"], queue)

## VI.A.2 Execute AF3 Failed Commands

In [ ]:
##########################################
### Re-run only unfinished AF3 targets ###
##########################################                                                                                                                                

### SHARED INPUTS ###                                                                                                                                                          
input_pdb_path          = f"{MPNN_OUT_DIR}i1/packed/outputs/"        
input_pdb_protein_chain = ["A"]                                                                                                                                                

af3_output_dir    = f"{AF3_OUT_DIR}i1__apo/"                                                                                                                                         
num_input_per_run = 45                                          
                                                                                                                                                                                
### SHARED OPTIONAL LIGAND FLAG ###                                                                                                                                            
include_ligands_if_present = False
input_ligand_dic           = {                                                                                                                                                 
  #  "B": {"ccdCodes": "['ZN']"},                                                                                                                                               
  #  "C": {"smiles": "[Zn+2].[OH-]"},                                                                                                                                               
  #  "D": {"smiles": "CCOP(=O)(OCC)OC1=CC=C(C=C1)[N+](=O)[O-]"},                                                                                                          
}   # leave empty {} or set to None to omit ligand args                                                                                                                        
                                                                                                                                                                                
### SHARED OPTIONAL PTM FLAG ###                                                                                                                                               
include_post_translational_mods_if_present = False                                                                                                                              
ptm_specs = ["A/LYS/3:KCX"]                                     
                                                                                                                                                                                
### SHARED OPTIONAL TERMINUS TAG FLAGS ###                                                                                                                                     
n_terminus_tag = ""                                                                                                                                                         
c_terminus_tag = ""                                                                                                                                                 
                                                                                                                                                                                
### SHARED OPTIONAL FLAGS ###                                                                                                                                                  
output_suffix              = "_af3i1"                                                                                                                                          
check_made_output          = True                               
cleanup_incomplete_outputs = True                                                                                                                                              

### SHARED OPTIONAL RECURSIVE SEARCH FLAGS ###                                                                                                                                 
recursive      = False                                           
max_depth      = None                                                                                                                                                          
specific_depth = None                                              
                                                                                                                                                                                
### SHARED SEED OPTIONS ###
no_random_seed = False                                                                                                                                                         
base_seed      = None

### GROUP DEFINITIONS (set to None for single-group behavior) ###
# Each group can override: pdb_prefix, input_ligand_dic, ptm_specs, output_suffix. Any key omitted inherits the shared default above                                                                                                                            
groups = [                                                                                                                                                                     
]                                                                                                                                                                              

### CONSTANTS ###                                                                                                                                                              
apptainer     = APPTAINER
script        = f"{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/make_af3_json_input.py"                                                                             

# IMPORTANT: use a NEW json staging directory for reruns
af3_json_path = os.path.join(AF3_JSON_DIR, "i1__apo_rerun_unfinished")

os.makedirs(af3_output_dir, exist_ok=True)
os.makedirs(af3_json_path, exist_ok=True)
                                                                                                                                                                                
### BUILD COMMAND(S) DYNAMICALLY ###                                                                                                                                           
def build_af3_command(ligand_dic, ptm_list, suffix, pdb_prefix_list=None):
    """Build a single make_af3_json_input.py command string."""                                                                                                                
    cmd_parts = [                                                                                                                                                              
        f"{apptainer} {script}",                                                                                                                                               
        f"--pdb_path {input_pdb_path}",                                                                                                                                        
        f"--pdb_chain {' '.join(input_pdb_protein_chain)}",                                                                                                                    
        f"--json_path {af3_json_path}",                         
        f"--num_input_per_run {num_input_per_run}",                                                                                                                            
        f"--output_path {af3_output_dir}",                                                                                                                                     
    ]
    if ligand_dic:                                                                                                                                                             
        ligand_chains = list(ligand_dic.keys())                                                                                                                                
        ligand_types  = []
        ligand_ids    = []                                                                                                                                                     
        for v in ligand_dic.values():                           
            if len(v) != 1:                                                                                                                                                    
                raise ValueError("Each ligand spec should have exactly one key indicating type (e.g., 'smiles').")                                                             
            ltype, lid = next(iter(v.items()))                                                                                                                                 
            ligand_types.append(ltype)                                                                                                                                         
            ligand_ids.append(lid)                                                                                                                                             
        cmd_parts.append(f"--ligand_chain {' '.join(ligand_chains)}")                                                                                                          
        cmd_parts.append(f"--ligand_type {' '.join(ligand_types)}")
        cmd_parts.append(f'--ligand_id "{" ".join(ligand_ids)}"')                                                                                                              
    if ptm_list:                                                
        cmd_parts.append("--ptm_from_remark666 " + " ".join([f'"{s}"' for s in ptm_list]))                                                                                     
    if n_terminus_tag:                                                                                                                                                         
        cmd_parts.append(f"--n_terminus_tag {n_terminus_tag}")
    if c_terminus_tag:                                                                                                                                                         
        cmd_parts.append(f"--c_terminus_tag {c_terminus_tag}")  
    if suffix:                                                                                                                                                                 
        cmd_parts.append(f"--output_suffix {suffix}")
    if check_made_output:                                                                                                                                                      
        cmd_parts.append("--check_made_output")                 
    if check_made_output and cleanup_incomplete_outputs:
        cmd_parts.append("--cleanup_incomplete_outputs")                                                                                                                       
    if recursive:
        cmd_parts.append("--recursive")                                                                                                                                        
    if max_depth is not None:                                   
        cmd_parts.append(f"--max_depth {max_depth}")
    if specific_depth is not None:                                                                                                                                             
        cmd_parts.append(f"--specific_depth {specific_depth}")
    if no_random_seed:                                                                                                                                                         
        cmd_parts.append("--no_random_seed")                    
    if base_seed is not None:                                                                                                                                                  
        cmd_parts.append(f"--base_seed {base_seed}")
    if pdb_prefix_list:                                                                                                                                                        
        cmd_parts.append(f"--pdb_prefix {' '.join(pdb_prefix_list)}")
    return " ".join(cmd_parts)                                                                                                                                                 

# Resolve shared defaults for ligands/PTMs                                                                                                                                     
shared_ligand_dic = input_ligand_dic if (include_ligands_if_present and input_ligand_dic) else {}
shared_ptm_specs  = ptm_specs if (include_post_translational_mods_if_present and ptm_specs) else []                                                                            
                                                                                                                                                                                
if groups:                                                                                                                                                                     
    # Multi-group mode: each group inherits shared defaults unless overridden                                                                                                  
    print(f"Generating {len(groups)} group commands:\n")                                                                                                                       
    for i, grp in enumerate(groups, 1):                                                                                                                                        
        cmd = build_af3_command(                                                                                                                                               
            ligand_dic      = grp.get("input_ligand_dic", shared_ligand_dic),                                                                                                  
            ptm_list        = grp.get("ptm_specs", shared_ptm_specs),
            suffix          = grp.get("output_suffix", output_suffix),                                                                                                         
            pdb_prefix_list = grp.get("pdb_prefix", None),      
        )                                                                                                                                                                      
        print(f"### Group {i}: {grp.get('name', f'group_{i}')} ###")
        print(cmd)                                                                                                                                                             
        print()                                                                                                                                                                
else:                                                                                                                                                                          
    # Single-group fallback (original behavior)                                                                                                                                
    cmd = build_af3_command(ligand_dic = shared_ligand_dic, ptm_list   = shared_ptm_specs, suffix     = output_suffix,                                                                                                                                            )
    print(cmd) 



In [ ]:
###############                                                                                                                                                                          
### Run AF3 ###                                                                                                                                                                          
###############                                                                                                                                                                          
                                                                                                                                                                                        
### INPUTS ###
af3_output_dir = f"{AF3_OUT_DIR}i1__apo/"
cmds_base_name = f"AF3_i1_apo_rerun_unfinished"

### PARAMETERS ###
N_total_structures = 5  # Total number of structures you want per target (N)
mode = "one_seed"        # "one_seed" -> N diffusion samples from 1 seed | "n_seeds" -> 1 sample from each of N seeds

### CONSTANTS ###
apptainer       = AF3_SIF
script          = AF3_RUNNER
apptainer_b4000 = AF3_SIF

# IMPORTANT: point to the NEW rerun json dir
af3_json_path   = os.path.join(AF3_JSON_DIR, "i1__apo_rerun_unfinished")
commands_base   = os.path.join(CMDS_DIR, cmds_base_name)

### QUEUE CONFIGURATION ###
b4000_fraction     = 1.00   # Fraction of commands to B4000 GPUs (0.0 = none, 1.0 = all)
gpu_fraction       = 0.00   # Within regular: fraction to priority queue (gpu vs gpu-bf)
gpu_fraction_b4000 = 1.00   # Within B4000:  fraction to priority queue (gpu-b4000 vs gpu-bf-b4000)

queue_config = {
    "gpu":          {"qtime": "06:30:00", "cmds_per_job": 6, "memory": "12g", "cores": "1"},
    "gpu-bf":       {"qtime": "07:00:00", "cmds_per_job": 8, "memory": "12g", "cores": "1"},

    "gpu-b4000":    {"qtime": "04:30:00", "cmds_per_job": 8, "memory": "12g", "cores": "1"},
    "gpu-bf-b4000": {"qtime": "06:30:00", "cmds_per_job": 16, "memory": "12g", "cores": "1"},
}

### DERIVED FLAGS ###
if mode not in {"one_seed", "n_seeds"}:
    raise ValueError(f"mode must be 'one_seed' or 'n_seeds'. Got: {mode}")
if mode == "one_seed":
    num_seeds = None
    num_diffusion_samples = N_total_structures
else:
    num_seeds = N_total_structures
    num_diffusion_samples = 1

### BUILD COMMANDS ###
# Build commands for both container types from the same JSON list
af3_jsons = sorted(glob.glob(os.path.join(af3_json_path, "*.json")), key=str.lower)

b4000_split = round(len(af3_jsons) * b4000_fraction)
jsons_regular = af3_jsons[b4000_split:]
jsons_b4000   = af3_jsons[:b4000_split]

cmds_regular = []
for af3_json in jsons_regular:
    cmd = f"{apptainer} python {script} --json_path={af3_json} --output_dir={af3_output_dir} --num_diffusion_samples={num_diffusion_samples}"
    if num_seeds is not None:
        cmd += f" --num_seeds={num_seeds}"
    cmds_regular.append(cmd)

cmds_b4000 = []
for af3_json in jsons_b4000:
    cmd = f"{apptainer_b4000} af3 --json_path={af3_json} --output_dir={af3_output_dir} --num_diffusion_samples={num_diffusion_samples}"
    if num_seeds is not None:
        cmd += f" --num_seeds={num_seeds}"
    cmds_b4000.append(cmd)

### SPLIT AND SUBMIT ###
for frac in [gpu_fraction, gpu_fraction_b4000, b4000_fraction]:
    if not (0.0 <= frac <= 1.0):
        raise ValueError(f"Fractions must be between 0.0 and 1.0. Got: {frac}")
queue_splits = {}

if cmds_regular: # Regular GPU split
    pri_idx = round(len(cmds_regular) * gpu_fraction)
    if pri_idx > 0:
        queue_splits["gpu"] = cmds_regular[:pri_idx]
    if pri_idx < len(cmds_regular):
        queue_splits["gpu-bf"] = cmds_regular[pri_idx:]

if cmds_b4000: # B4000 GPU split
    pri_idx = round(len(cmds_b4000) * gpu_fraction_b4000)
    if pri_idx > 0:
        queue_splits["gpu-b4000"] = cmds_b4000[:pri_idx]
    if pri_idx < len(cmds_b4000):
        queue_splits["gpu-bf-b4000"] = cmds_b4000[pri_idx:]

total_cmds = len(cmds_regular) + len(cmds_b4000)
print(f"{'='*60}")
print(f"  AF3 Job Summary │ {total_cmds} total cmds │ Mode: {mode} │ N: {N_total_structures}")
print(f"  Regular: {len(cmds_regular)} cmds │ B4000: {len(cmds_b4000)} cmds")
print(f"{'='*60}")

for queue, queue_cmds in queue_splits.items():
    cfg = queue_config[queue]
    suffix = f"_{queue.replace('-', '')}" if len(queue_splits) > 1 else ""
    commands_file = f"{commands_base}{suffix}"
    job_name = os.path.basename(commands_file)
    submit_file = f"{SUBMIT_DIR}{job_name}.sh"
    num_jobs = math.ceil(len(queue_cmds) / cfg["cmds_per_job"])
    with open(commands_file, 'w') as fo:
        print("\n".join(queue_cmds), file=fo)
    nb.submit_array_job(commands_file, cfg["qtime"], cfg["cores"], job_name, cfg["memory"], submit_file, LOGS_DIR, num_jobs, cfg["cmds_per_job"], queue)

## VI.B Process AF3 Subdirectories

### VI.B.1 Process AF3 Subdirectories

In [ ]:
################################################################
### Convert AF3 output format and remove useless AF3 outputs ###
################################################################

### INPUTS ###
af3_output_dir = f"{AF3_OUT_DIR}i1__apo/"    

### OPTIONAL SCRIPT ARGS ###
cif_sif_path              = MAXIT_SIF  # or None to omit
disable_deletions         = False   # set False to allow cleanup
verbose_logging           = True    # set False for quieter output
include_nonindexed_cifs   = False   # True to also convert non-indexed CIFs
robust_mode               = False   # should probably turn default as false, but this is good for re-running
af3_subdirs_are_lowercase = False

### CONSTANTS ###
apptainer = "python"   # I don't find a suitable apptainer yet.
script    = f"{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/process_af3_cif.py"

### EXECUTE ###
commands = os.path.join(CMDS_DIR, "AF3_process_iteration3")
af3_subdirs = glob.glob(os.path.join(af3_output_dir, "*"))
with os.scandir(af3_output_dir) as it:
    af3_subdirs = [e.path for e in it if e.is_dir(follow_symlinks=False)]
if af3_subdirs_are_lowercase:
    af3_subdirs = list(filter(lambda fi: os.path.basename(fi).lower() == os.path.basename(fi), af3_subdirs))
af3_subdirs = sorted(af3_subdirs, key=lambda p: os.path.basename(p).lower())
print (f"Number of AF3 subdirectories: {len(af3_subdirs)}")
print ()

# build extra args string once
extra_args = []
if cif_sif_path:
    extra_args += ["--cif_sif", cif_sif_path]
if disable_deletions:
    extra_args.append("--disable-deletions")
if verbose_logging:
    extra_args.append("--verbose")
if include_nonindexed_cifs:
    extra_args.append("--include-nonindexed-cifs")
if robust_mode:
    extra_args.append("--robust-mode")
extra_args_str = " ".join(extra_args)

command_num = 0
with open(commands, 'w') as f_cmd:
    for af3_path in af3_subdirs:
        # Only include if "samples" subdir exists
        if os.path.isdir(os.path.join(af3_path, "samples")):
            f_cmd.write(f"{apptainer} {script} --af3_subdir {af3_path} {extra_args_str}\n")
            command_num += 1

### SETUP SLURM SUBMISSION ###
job_name = os.path.basename(commands)
qtime = '00:25:00'
cores = '1'
memory = '2g'
queue = 'cpu'
cmds_per_job = 50
num_jobs = int(command_num / cmds_per_job) + 1
submit_file = f'{SUBMIT_DIR}{job_name}.sh'
print("Job/Commands Name:", job_name)
print("Number of Jobs to Run:", num_jobs)
print('')
print('### COMMANDS FILE BELOW ###')
print(commands)
print('')

# Submit the job array using your rainier module.
nb.submit_array_job(commands, qtime, cores, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue)

In [ ]:
##############################                                                                                                                                                     
### Process AF3 prediction ###                                                                                                                                                     
##############################                                                                                                                                                     
                                                                                                                                                                                    
### PATHS ###
af3_out_path       = f"{AF3_OUT_DIR}i1__apo/"    
ref_pdb_path       = f"{MPNN_OUT_DIR}i1/packed/outputs/"   
backup_ref_subdir  = "ref_pdbs"                              # Subdirectory in ref_pdb_path for fallback (empty = disabled)
backup_ref_dir     = os.path.join(ref_pdb_path, backup_ref_subdir) if backup_ref_subdir else ""
commands           = f"{CMDS_DIR}AF3_pdb_processing_substrate_zn_zno"

### PARAMETERS ###
af3_N                  = 5
af3_suffix             = "_af3i1"
default_ligand         = "USER_LIG"
use_split_subdirs      = False                            # True: ref PDBs in subdirectories; False: flat directory
subdir_strip_pattern   = r"_design_\d+"                  # Regex removed from name to derive subdirectory name
use_ligand_in_filename = False                            # Search for ligand anywhere in filename (e.g., *PTE_wKCX*XDW*)
use_ligand             = False

### MAP REF PDB PREFIXES TO LIGAND NAMES ###
prefix_ligand_map = {
    "ZAPP": ["YYE"],  # Multiple ligands per prefix is possible
}

### MAP REF PDB PREFIXES TO CATRES SUBSETS ###
prefix_catres_map = {
    "ZAPP": "1,2,3,4,5,6",
}

### ATOM NAMING INPUT TEMPLATE ###
atom_groups_template = [
    {"label": "subst",
     "name3": ["G_D", "USER_LIG"], 
     "chain": ["D", "B"],
     "atoms": [["P1", "N1", "O5", "O6", "O1", "C2", "C1", "O3", "C3", "C4", "O2", "O4", "C5", "C6", "C7", "C8", "C9", "C10"], 
               ["P1", "N1", "O9", "O8", "O6", "C9", "C11", "O4", "C8", "C10", "O5", "O7", "C7", "C5", "C3", "C2", "C4", "C6"]],
     "symmetric_atom_groups": [{
                  "lig1": [["C5", "C6", "C7", "C8", "C9", "C10"], # BENZENE
                           ["N1", "O5", "O6"], # NITRO
                           ["O1", "C2", "C1", "O3", "C3", "C4"]], # ETHYL
         
                  "lig2": [[["C7", "C6", "C4", "C2", "C3", "C5"], ["C7", "C5", "C3", "C2", "C4", "C6"]], 
                           [["N1", "O9", "O8"], ["N1", "O8", "O9"]],
                           [["O6", "C9", "C11", "O4", "C8", "C10"], ["O4", "C8", "C10", "O6", "C9", "C11"]]]
              }],},
    
    {"label": "subst_core",
     "name3": ["G_D", "USER_LIG"], 
     "chain": ["D", "B"],
     "atoms": [["P1", "O1", "O3", "O2", "O4"],
               ["P1", "O6", "O4", "O5", "O7"]],
     "symmetric_atom_groups": [{
                  "lig1": [["O1", "O3"]], # ETHYL
                  "lig2": [[["O4", "O6"], ["O6", "O4"]]]
              }],},

    # Hydroxide ion
    {"label": "hydroxide_ion", "name3": ["G_C", "USER_LIG"], "chain": ["C", "B"], "atoms": [["O1"], ["O3"]]},

    # NCAA
    {"label": "kcx_tips", "name3": ["KCX", "USER_LIG"], "chain": ["A", "B"],
    "atoms": [["CX", "OQ1", "OQ2"], ["C1", "O1", "O2"]],
    "symmetric_atom_groups": [{"lig1": [["CX", "OQ1", "OQ2"]], "lig2": [[["C1", "O1", "O2"], ["C1", "O2", "O1"]]]}]},
    # Zinc ion for ZN1
    {"label": "zinc1_chainB", "name3": ["ZN", "USER_LIG"], "chain": ["B", "B"], "atoms": [["ZN"], ["ZN1"]]},
    {"label": "zinc1_chainC", "name3": ["G_C", "USER_LIG"], "chain": ["C", "B"], "atoms": [["ZN1"], ["ZN1"]]},
    # Zinc ion for ZN2
    {"label": "zinc2_chainB", "name3": ["ZN", "USER_LIG"], "chain": ["B", "B"], "atoms": [["ZN"], ["ZN2"]]},
    {"label": "zinc2_chainC", "name3": ["G_C", "USER_LIG"], "chain": ["C", "B"], "atoms": [["ZN1"], ["ZN2"]]},
]

### OPTIONAL: SPECIFY N & C TERMINUS TAGS TO IGNORE ###
N_tag_ignore = 0    # For example, set to 3 if you want to ignore the first 3 residues (MSG = 3)
C_tag_ignore = 0    # For example, set to 2 if you want to ignore the last 2 residues  (GSAWSHPQFEK = 11)

### FLAGS ###
verbose                     = True   # --verbose
calc_interchain_pae_mean    = True   # --calculate_avg_pae_in_addition_to_pair
rapid_mode_skip_sc_if_found = False  # Skip if .sc is found

### CONSTANTS ###
script    = f'{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/process_af3_pdb.py'
apptainer = APPTAINER

### GENERATE COMMANDS ###

# Pre-compute all AF3 subdirectories that have the correct number of predictions
from concurrent.futures import ThreadPoolExecutor
def _check_af3_dir(d):
    pdb_count = sum(1 for e in os.scandir(d) if e.name.endswith(".pdb") and e.is_file())
    return os.path.basename(d) if pdb_count == af3_N else None
with ThreadPoolExecutor() as executor:
    af3_dirs = glob.glob(os.path.join(af3_out_path, f"*{af3_suffix}"))
    af3_valid_subdirs = {r for r in executor.map(_check_af3_dir, af3_dirs) if r}
print(f"{len(af3_valid_subdirs)} valid AF3 directories detected (with {af3_N} PDBs each)")

# Match AF3 directories to reference PDBs
pairs = []
if use_split_subdirs:
    matched_subdir, matched_backup, unmatched = 0, 0, 0
    for af3_name in af3_valid_subdirs:
        pdb_base   = af3_name[:-len(af3_suffix)]
        parent_dir = re.sub(subdir_strip_pattern, "", pdb_base)
        ref_pdb    = os.path.join(ref_pdb_path, parent_dir, pdb_base + ".pdb")
        if os.path.isfile(ref_pdb):
            pairs.append((af3_name, ref_pdb))
            matched_subdir += 1
        elif backup_ref_dir:
            ref_pdb = os.path.join(backup_ref_dir, pdb_base + ".pdb")
            if os.path.isfile(ref_pdb):
                pairs.append((af3_name, ref_pdb))
                matched_backup += 1
            else:
                unmatched += 1
        else:
            unmatched += 1
    print(f"  Matched from subdirs: {matched_subdir}")
    if backup_ref_dir:
        print(f"  Matched from backup: {matched_backup}")
    print(f"  Unmatched:           {unmatched}")
else:
    ref_pdbs = glob.glob(os.path.join(ref_pdb_path, "*.pdb"))
    print(f"{len(ref_pdbs)} REF PDBs were detected")
    for ref_pdb in ref_pdbs:
        af3_name = os.path.basename(ref_pdb).replace(".pdb", "") + af3_suffix
        if af3_name in af3_valid_subdirs:
            pairs.append((af3_name, ref_pdb))

# Pre-build prefix lookup (uppercase for case-insensitive matching)
prefix_map_upper    = {k.upper(): (v if isinstance(v, list) else [v]) for k, v in prefix_ligand_map.items()}
catres_map_upper    = {k.upper(): v for k, v in prefix_catres_map.items()}
sorted_catres_items = sorted(catres_map_upper.items(), key=lambda x: -len(x[0]))

# Pre-build command parts that don't change
base_cmd   = [apptainer, script]
flag_parts = []
if verbose: flag_parts.append("--verbose")
if calc_interchain_pae_mean: flag_parts.append("--calculate_avg_pae_in_addition_to_pair")
if rapid_mode_skip_sc_if_found: flag_parts.append("--rapid_mode_skip_sc_if_found")

# Cache for ligand-specific atom groups (avoid redundant deep copies)
atom_groups_cache = {}
command_lines = []
for af3_pdb_subdir, ref_pdb in pairs:
    ref_pdb_bn = os.path.basename(ref_pdb)

    # Determine ligand (optimized with cached uppercase prefixes)
    ligand_name = default_ligand
    up = ref_pdb_bn.upper()
    for prefix_up, ligs in prefix_map_upper.items():
        if up.startswith(prefix_up):
            ligand_name = next((lig for lig in ligs if lig.upper() in up), ligs[0]) if use_ligand_in_filename else ligs[0]
            break

    # Determine catres_subset (find longest matching prefix)
    catres_subset = None
    for prefix_up, subset_val in sorted_catres_items:
        if up.startswith(prefix_up):
            catres_subset = subset_val
            break

    # Use cached atom groups for this ligand (avoid redundant processing)
    if ligand_name not in atom_groups_cache:
        atom_groups_cache[ligand_name] = [
            dict(g, name3=[item if item != "USER_LIG" else ligand_name for item in g["name3"]])
            if "symmetric_atom_groups" not in g
            else {**copy.deepcopy(g), "name3": [item if item != "USER_LIG" else ligand_name for item in g["name3"]]}
            for g in atom_groups_template]

    # Build command (reuse pre-built parts)
    af3_dir_path = os.path.join(af3_out_path, af3_pdb_subdir)
    outscr_path  = os.path.join(af3_dir_path, af3_pdb_subdir + ".sc")
    cmd = base_cmd + [f"--af3_dir {af3_dir_path}", f"--ref_pdb {ref_pdb}", f"--outscr {outscr_path}",] + flag_parts
    if use_ligand:
        cmd.append(f"--ligand_groups_json '{json.dumps(atom_groups_cache[ligand_name])}'")
    if catres_subset:
        cmd.append(f"--catres_subset {catres_subset}")
    if N_tag_ignore > 0:
        cmd.append(f"--N_terminus_tag_length_to_ignore {N_tag_ignore}")
    if C_tag_ignore > 0:
        cmd.append(f"--C_terminus_tag_length_to_ignore {C_tag_ignore}")
    command_lines.append(" ".join(cmd))
command_lines.sort()
with open(commands, 'w') as fo:
    fo.write("\n".join(command_lines) + "\n")

### SETUP SLURM SUBMISSION ###
job_name = os.path.basename(commands)
qtime, cores, memory, queue, cmds_per_job = '00:25:00', '1', '2g', 'cpu', 75
num_jobs = (len(command_lines) + cmds_per_job - 1) // cmds_per_job
submit_file = f'{SUBMIT_DIR}{job_name}.sh'

# Submit job array
nb.submit_array_job(commands, qtime, cores, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue)

### VI.B.2 Parse AF3 Stats

In [ ]:
##################################################################
### PARSE & PROCESS AF3 .SC FILES ONLY NEED TO DO THIS ONCE ######
##################################################################

# INPUT DIRECTORY OF SUBDIRECTORIES TO PARSE
af3_dir_of_subdirs = f"{AF3_OUT_DIR}i1__apo/"   # CHANGE EVERY TIME

### OPTIONALS ###
workers                         = 64 # None or integer
optional_path_for_summary_stats = f"{IMPORTANT_DFS_DIR}af3_i1__apo.csv"  # e.g. f"{IMPORTANT_DFS_DIR}af3_stats_iteration2_holo.csv"
optional_chunk_rows             = None  # 20000 # Optional: rows per shard before writing to temp CSV | default = 10000
find_subdirs_without_viable_sc  = False # DRY RUN IF TRUE

### CONSTANTS ###
script = f"{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/concat_af3_sc_dir_of_subdirs.py"
container_path = APPTAINER

### GENERATE COMMAND ###
command = f"singularity exec {container_path} python {script} --af3_dir_of_subdirs {af3_dir_of_subdirs}"
if optional_path_for_summary_stats:
    command += f" --optional_path_for_summary_stats {optional_path_for_summary_stats}"
if optional_chunk_rows:
    command += f" --chunk_rows {optional_chunk_rows}"
if workers:
    command += f" --workers {workers}"
if find_subdirs_without_viable_sc:
    command += f" --find_subdirs_without_viable_sc"

### PRINT COMMAND TO SUBMIT ###
print("SUBMIT THIS:")
print(command)
print('')

### PRINT OUTPUT FILE ###
print("OUTPUT CSV FILE HERE:")
if optional_path_for_summary_stats:
    print(optional_path_for_summary_stats)
else:
    print(af3_dir_of_subdirs + 'zzzzz_af3_analysis_csv_zzzzz.csv')

In [ ]:
########################################################################                                                                      
### APPLY TRANSFORMATIONS & MANIPULATIONS TO THE CONCATENATED AF3 DF ###
########################################################################                                                                      
                                                                
### LOAD DF ###                                                                                                                               
csv_path = f"{IMPORTANT_DFS_DIR}af3_i1__apo.csv"                 
af3_scores_df = pd.read_csv(csv_path)                                                                                                         

### SAVE DF PATH ###                                                                                                                          
out_csv = f"{IMPORTANT_DFS_DIR}modified_af3_i1__apo.csv"         
                                                                                                                                            
### APPLY TRANSFORMATIONS & MANIPULATIONS ###
_desc = af3_scores_df["af3_models_dir"].astype(str)                                                                                           
desc_base = _desc.apply(lambda s: re.split(r"[\\/]", s)[-1])                                                                                  

# Build scaffold_family from the basename (applies to apo + holo)                                                                             
af3_scores_df["scaffold_family"] = (                            
    desc_base.str.split("_inp", n=1).str[0].str.split("_eV2", n=1).str[0]                                                                     
)                                                                                                                                             

# -------------------- ZINC pair assignment (Kabsch) --------------------                                                                     
# Only runs if zinc columns exist (skipped automatically for apo)
pattern = re.compile(r"^zinc[12]_chain[BC]_rmsd_idx_(\d+)$")                                                                                  
all_idxs = sorted({int(m.group(1)) for col in af3_scores_df.columns
                    if (m := pattern.match(col))})                                                                                             
                                                                
if not all_idxs:                                                                                                                              
    print("[skip] no zinc*_chain[BC]_rmsd_idx_* columns found — apo mode or no metals; skipping Kabsch zinc-pair block")
else:                                                                                                                                         
    def compute_pair_for_idx(row, i):                           
        cols = {"z1B": f"zinc1_chainB_rmsd_idx_{i}",                                                                                          
                "z1C": f"zinc1_chainC_rmsd_idx_{i}",            
                "z2B": f"zinc2_chainB_rmsd_idx_{i}",                                                                                          
                "z2C": f"zinc2_chainC_rmsd_idx_{i}"}
        if any(c not in row.index for c in cols.values()):                                                                                    
            return np.nan, None                                 
        v = {k: row[c] for k, c in cols.items()}                                                                                              
        if any(pd.isna(x) for x in v.values()):                                                                                               
            return np.nan, None
        bc_avg = float(np.mean([v["z1B"], v["z2C"]]))                                                                                         
        cb_avg = float(np.mean([v["z1C"], v["z2B"]]))                                                                                         
        if bc_avg <= cb_avg:
            return bc_avg, "BC"                                                                                                               
        return cb_avg, "CB"                                                                                                                   

    for i in all_idxs:                                                                                                                        
        avg_col = f"zinc_pair_avg_rmsd_idx_{i}"                 
        asn_col = f"zinc_pair_assignment_idx_{i}"
        af3_scores_df[avg_col] = np.nan                                                                                                       
        af3_scores_df[asn_col] = None
        results = af3_scores_df.apply(lambda r: compute_pair_for_idx(r, i), axis=1, result_type="expand")                                     
        af3_scores_df[avg_col] = results[0].astype(float)                                                                                     
        af3_scores_df[asn_col] = results[1].astype("object")                                                                                  
    print(f"[ok] Kabsch zinc-pair block ran on idx values: {all_idxs}")                                                                       
                                                                                                                                            
# -------------------- ZINC pair assignment (TM-align) --------------------                                                                   
# Only runs if TMalign zinc columns exist (skipped automatically for apo)                                                                     
pattern_tm = re.compile(r"^zinc[12]_chain[BC]_rmsd_TMalign_idx_(\d+)$")                                                                       
all_idxs_tm = sorted({int(m.group(1)) for col in af3_scores_df.columns                                                                        
                    if (m := pattern_tm.match(col))})                                                                                       
                                                                                                                                            
if not all_idxs_tm:                                                                                                                           
    print("[skip] no zinc*_chain[BC]_rmsd_TMalign_idx_* columns found — apo mode or no metals; skipping TM-align zinc-pair block")
else:                                                                                                                                         
    def compute_pair_for_idx_TMalign(row, i):
        cols = {"z1B": f"zinc1_chainB_rmsd_TMalign_idx_{i}",                                                                                  
                "z1C": f"zinc1_chainC_rmsd_TMalign_idx_{i}",    
                "z2B": f"zinc2_chainB_rmsd_TMalign_idx_{i}",                                                                                  
                "z2C": f"zinc2_chainC_rmsd_TMalign_idx_{i}"}    
        if any(c not in row.index for c in cols.values()):                                                                                    
            return np.nan, None                                 
        v = {k: row[c] for k, c in cols.items()}                                                                                              
        if any(pd.isna(x) for x in v.values()):                                                                                               
            return np.nan, None
        bc_avg = float(np.mean([v["z1B"], v["z2C"]]))                                                                                         
        cb_avg = float(np.mean([v["z1C"], v["z2B"]]))                                                                                         
        if bc_avg <= cb_avg:
            return bc_avg, "BC"                                                                                                               
        return cb_avg, "CB"                                                                                                                   

    for i in all_idxs_tm:                                                                                                                     
        avg_col = f"zinc_pair_avg_rmsd_TMalign_idx_{i}"         
        asn_col = f"zinc_pair_assignment_TMalign_idx_{i}"                                                                                     
        af3_scores_df[avg_col] = np.nan
        af3_scores_df[asn_col] = None                                                                                                         
        results = af3_scores_df.apply(lambda r: compute_pair_for_idx_TMalign(r, i), axis=1, result_type="expand")                             
        af3_scores_df[avg_col] = results[0].astype(float)                                                                                     
        af3_scores_df[asn_col] = results[1].astype("object")                                                                                  
    print(f"[ok] TM-align zinc-pair block ran on idx values: {all_idxs_tm}")                                                                  
                                                                                                                                            
### SAVE DF ###
af3_scores_df.to_csv(out_csv, index=False)                                                                                                    
print(f"Saved: {out_csv}  (rows={len(af3_scores_df)})")         
                                                                                                                                            
### LOOKING ###
_preview_cols = ["af3_models_dir", "scaffold_family"] + [c for c in af3_scores_df.columns if c.startswith("zinc_pair_")][:4]                  
af3_scores_df[_preview_cols].head()   

## VI.C Filter AF3

### VI.C.1 Check Stat Distributions

In [ ]:
############################################################
### AF3 METRICS: 5-NUMBER SUMMARY & PERCENTILE ANALYSIS  ###
############################################################
"""
For each metric, compute:
1. 5-number summary (min, Q1, median, Q3, max)
2. Mean and std
3. Top N% threshold (direction-aware: lower or upper percentile)
4. Suggested filter values based on percentiles
"""

import os
import numpy as np
import pandas as pd
from typing import Dict, List

# ============================================================================
# CONFIGURATION
# ============================================================================

### PATHS ###
af3_scores_df_path = os.path.join(IMPORTANT_DFS_DIR, 'modified_af3_stats_i3_updated.csv')

### NUMBER OF AF3 PREDICTIONS ###
af3_N = 5

### TOP N% FOR FILTER RECOMMENDATIONS ###
# If metric uses <= filter, we take the Nth percentile (lower is better)
# If metric uses >= filter, we take the (100-N)th percentile (higher is better)
top_N_percent = 20  # Top 20%

### ADDITIONAL PERCENTILES TO SHOW ###
percentiles_to_show = [5, 10, 15, 20, 25, 30, 50]

### METRICS CONFIGURATION ###
# Format: base_metric -> (filter_operator, expected_direction)
# "lower" means lower values are better (use <= filter)
# "higher" means higher values are better (use >= filter)

metrics_config = {
    # RMSD metrics (lower is better, use <=)
    "ca_rmsd": ("<=", "lower"),
    "zinc_pair_avg_rmsd": ("<=", "lower"),
    "catres_rmsd": ("<=", "lower"),
    "catres_subset_rmsd": ("<=", "lower"),
    "kcx_tips_rmsd": ("<=", "lower"),
    "hydroxide_ion_rmsd": ("<=", "lower"),
    "subst_rmsd": ("<=", "lower"),
    "subst_core_rmsd": ("<=", "lower"),

    # RMSD metrics (lower is better, use <=)
    "ca_rmsd_TMalign": ("<=", "lower"),
    "zinc_pair_avg_rmsd_TMalign": ("<=", "lower"),
    "catres_rmsd_TMalign": ("<=", "lower"),
    "catres_subset_rmsd_TMalign": ("<=", "lower"),
    "kcx_tips_rmsd_TMalign": ("<=", "lower"),
    "hydroxide_ion_rmsd_TMalign": ("<=", "lower"),
    "subst_rmsd_TMalign": ("<=", "lower"),
    "subst_core_rmsd_TMalign": ("<=", "lower"),

    # TM & lDDT metrics (higher is better, use >=)
    "tm_score": (">=", "higher"),
    "catres_lddt": (">=", "higher"),
    "catres_subset_lddt": (">=", "higher"),

    # pLDDT metrics (higher is better, use >=)
    "chainA_plddt": (">=", "higher"),
    "chainB_plddt": (">=", "higher"),
    "chainC_plddt": (">=", "higher"),
    "chainD_plddt": (">=", "higher"),
    "catres_plddt": (">=", "higher"),
    "KCX3_extra_atoms_plddt": (">=", "higher"),
    "hydroxide_ion_plddt": (">=", "higher"),
    "subst_core_plddt": (">=", "higher"),
    "kcx_tips_plddt": (">=", "higher"),

    # PTM/iPTM metrics (higher is better, use >=)
    "iptm": (">=", "higher"),
    "ptm": (">=", "higher"),

    # PAE min metrics (lower is better, use <=)
    "chainA_chainB_pair_pae_min": ("<=", "lower"),
    "chainA_chainC_pair_pae_min": ("<=", "lower"),
    "chainA_chainD_pair_pae_min": ("<=", "lower"),
    "chainB_chainC_pair_pae_min": ("<=", "lower"),
    "chainB_chainD_pair_pae_min": ("<=", "lower"),
    "chainC_chainD_pair_pae_min": ("<=", "lower"),

    # PAE mean metrics (lower is better, use <=)
    "chainA_chainB_pair_pae_mean": ("<=", "lower"),
    "chainA_chainC_pair_pae_mean": ("<=", "lower"),
    "chainA_chainD_pair_pae_mean": ("<=", "lower"),
    "chainB_chainC_pair_pae_mean": ("<=", "lower"),
    "chainB_chainD_pair_pae_mean": ("<=", "lower"),
    "chainC_chainD_pair_pae_mean": ("<=", "lower"),
}

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def print_header(title: str, char: str = "=", width: int = 90):
    print(f"\n{char * width}")
    print(f" {title}")
    print(f"{char * width}")
def print_subheader(title: str, char: str = "-", width: int = 70):
    print(f"\n{char * width}")
    print(f" {title}")
    print(f"{char * width}")

# ============================================================================
# LOAD DATA
# ============================================================================
print_header("AF3 METRICS: 5-NUMBER SUMMARY & PERCENTILE ANALYSIS")
af3_scores_df = pd.read_csv(af3_scores_df_path)
print(f"\nLoaded {len(af3_scores_df)} designs from:\n  {af3_scores_df_path}")

# ============================================================================
# COMPUTE STATISTICS FOR EACH METRIC
# ============================================================================
print_subheader(f"STATISTICS FOR ALL METRICS (Top {top_N_percent}% thresholds)")
all_stats = []
for base_metric, (operator, direction) in metrics_config.items():
    # Collect all idx values for this metric
    idx_cols = [f"{base_metric}_idx_{i}" for i in range(af3_N)]
    available_cols = [c for c in idx_cols if c in af3_scores_df.columns]
    if not available_cols:
        continue
    # Stack all idx values together
    all_values = af3_scores_df[available_cols].values.flatten()
    all_values = all_values[~np.isnan(all_values)]
    if len(all_values) == 0:
        continue
    # 5-number summary
    min_val = np.min(all_values)
    q1 = np.percentile(all_values, 25)
    median_val = np.median(all_values)
    q3 = np.percentile(all_values, 75)
    max_val = np.max(all_values)
    mean_val = np.mean(all_values)
    std_val = np.std(all_values)
    # Top N% threshold (direction-aware)
    if direction == "lower":
        # Lower is better, so top N% means the Nth percentile
        top_n_threshold = np.percentile(all_values, top_N_percent)
    else:
        # Higher is better, so top N% means the (100-N)th percentile
        top_n_threshold = np.percentile(all_values, 100 - top_N_percent)
    # Compute all requested percentiles
    percentile_values = {}
    for p in percentiles_to_show:
        if direction == "lower":
            percentile_values[p] = np.percentile(all_values, p)
        else:
            percentile_values[p] = np.percentile(all_values, 100 - p)
    all_stats.append({
        "metric": base_metric,
        "operator": operator,
        "direction": direction,
        "n_values": len(all_values),
        "min": min_val,
        "Q1": q1,
        "median": median_val,
        "Q3": q3,
        "max": max_val,
        "mean": mean_val,
        "std": std_val,
        f"top_{top_N_percent}%": top_n_threshold,
        **{f"top_{p}%": percentile_values[p] for p in percentiles_to_show},
    })
stats_df = pd.DataFrame(all_stats)

# ============================================================================
# DISPLAY: 5-NUMBER SUMMARY TABLE
# ============================================================================
print_subheader("5-NUMBER SUMMARY (All idx combined)")
print(f"\n  {'Metric':<35s} {'Op':<4s} {'Min':>8s} {'Q1':>8s} {'Median':>8s} {'Q3':>8s} {'Max':>8s} {'Mean':>8s} {'Std':>8s}")
print(f"  {'-'*107}")
for _, row in stats_df.iterrows():
    print(f"  {row['metric']:<35s} {row['operator']:<4s} "
          f"{row['min']:>8.2f} {row['Q1']:>8.2f} {row['median']:>8.2f} "
          f"{row['Q3']:>8.2f} {row['max']:>8.2f} {row['mean']:>8.2f} {row['std']:>8.2f}")

# ============================================================================
# DISPLAY: TOP N% THRESHOLDS (FILTER RECOMMENDATIONS)
# ============================================================================
print_subheader(f"TOP N% THRESHOLDS (Filter Recommendations)")
print(f"\nFor '<=' filters: shows Nth percentile (lower is better)")
print(f"For '>=' filters: shows (100-N)th percentile (higher is better)")
# Header with all percentiles
pct_header = "  " + f"{'Metric':<35s} {'Op':<4s}"
for p in percentiles_to_show:
    pct_header += f" {'Top '+str(p)+'%':>10s}"
print(f"\n{pct_header}")
print(f"  {'-'*(39 + 11*len(percentiles_to_show))}")
for _, row in stats_df.iterrows():
    line = f"  {row['metric']:<35s} {row['operator']:<4s}"
    for p in percentiles_to_show:
        val = row[f"top_{p}%"]
        line += f" {val:>10.2f}"
    print(line)

# ============================================================================
# DISPLAY: SUGGESTED FILTER VALUES (COPY-PASTE READY)
# ============================================================================
print_subheader(f"SUGGESTED FILTER VALUES (Top {top_N_percent}% - Copy-Paste Ready)")
print(f"\n# Filter thresholds for top {top_N_percent}% of designs")
print(f"# Based on {'percentile ' + str(top_N_percent) if True else ''} analysis\n")
for _, row in stats_df.iterrows():
    threshold = row[f"top_{top_N_percent}%"]
    # Format nicely
    if abs(threshold) < 1:
        formatted = f"{threshold:.3f}"
    elif abs(threshold) < 10:
        formatted = f"{threshold:.2f}"
    else:
        formatted = f"{threshold:.1f}"
    var_name = row['metric'].replace("_", "_")
    print(f"{var_name:<35s} = {formatted}")

# ============================================================================
# DISPLAY: FILTERS DICT FORMAT
# ============================================================================
print_subheader(f"FILTERS DICT FORMAT (Top {top_N_percent}%)")
print("\nfilters = {")
for _, row in stats_df.iterrows():
    threshold = row[f"top_{top_N_percent}%"]
    # Format nicely
    if abs(threshold) < 1:
        formatted = f"{threshold:.3f}"
    elif abs(threshold) < 10:
        formatted = f"{threshold:.2f}"
    else:
        formatted = f"{threshold:.1f}"

    print(f'    ("{row["metric"]}", "{row["operator"]}"): {formatted},')
print("}")

# ============================================================================
# PER-IDX STATISTICS
# ============================================================================
print_header("PER-IDX STATISTICS")
print("Checking if different AF3 seeds have different distributions")
per_idx_stats = []
for base_metric, (operator, direction) in metrics_config.items():
    for idx in range(af3_N):
        col = f"{base_metric}_idx_{idx}"
        if col not in af3_scores_df.columns:
            continue
        values = af3_scores_df[col].dropna().values
        if len(values) == 0:
            continue
        if direction == "lower":
            top_n_val = np.percentile(values, top_N_percent)
        else:
            top_n_val = np.percentile(values, 100 - top_N_percent)
        per_idx_stats.append({
            "metric": base_metric,
            "idx": idx,
            "operator": operator,
            "n": len(values),
            "mean": np.mean(values),
            "std": np.std(values),
            "median": np.median(values),
            f"top_{top_N_percent}%": top_n_val,
        })
per_idx_df = pd.DataFrame(per_idx_stats)

# Show summary: do different idx have different thresholds?
print_subheader("Per-Idx Threshold Variation")
print(f"\n  {'Metric':<35s} ", end="")
for idx in range(af3_N):
    print(f"{'idx_'+str(idx):>10s} ", end="")
print(f"{'Range':>10s}")
print(f"  {'-'*(35 + 11*(af3_N+1))}")
for base_metric in stats_df["metric"].values:
    sub = per_idx_df[per_idx_df["metric"] == base_metric]
    if len(sub) == 0:
        continue
    line = f"  {base_metric:<35s} "
    vals = []
    for idx in range(af3_N):
        row = sub[sub["idx"] == idx]
        if len(row) > 0:
            val = row[f"top_{top_N_percent}%"].values[0]
            vals.append(val)
            line += f"{val:>10.2f} "
        else:
            line += f"{'N/A':>10s} "
    if vals:
        range_val = max(vals) - min(vals)
        line += f"{range_val:>10.2f}"
    print(line)

# ============================================================================
# SPECIAL ANALYSIS: HIT-BASED THRESHOLDS (if hit classification exists)
# ============================================================================
# Check if we have hit classification from the retro analysis
if "hit_class" in af3_scores_df.columns or "is_hit" in af3_scores_df.columns:
    print_header("HIT-BASED THRESHOLD ANALYSIS")
    print("Thresholds based on hit distribution (where hits tend to fall)")
    # Determine hit column
    if "is_any_hit" in af3_scores_df.columns:
        hit_col = "is_any_hit"
    elif "is_hit" in af3_scores_df.columns:
        hit_col = "is_hit"
    else:
        hit_col = None
    if hit_col:
        hits_df = af3_scores_df[af3_scores_df[hit_col]]
        non_hits_df = af3_scores_df[~af3_scores_df[hit_col]]
        print(f"\nHits: {len(hits_df)} | Non-hits: {len(non_hits_df)}")
        print_subheader("Hit Distribution Percentiles")
        print(f"\nShowing where hits fall - use these as filter thresholds to capture most hits")
        print(f"\n  {'Metric':<35s} {'Op':<4s} {'Hit 25%':>10s} {'Hit 50%':>10s} {'Hit 75%':>10s} {'NonHit 50%':>12s}")
        print(f"  {'-'*77}")
        for base_metric, (operator, direction) in metrics_config.items():
            idx_cols = [f"{base_metric}_idx_{i}" for i in range(af3_N)]
            available_cols = [c for c in idx_cols if c in af3_scores_df.columns]
            if not available_cols:
                continue
            hit_vals = hits_df[available_cols].values.flatten()
            hit_vals = hit_vals[~np.isnan(hit_vals)]
            non_hit_vals = non_hits_df[available_cols].values.flatten()
            non_hit_vals = non_hit_vals[~np.isnan(non_hit_vals)]
            if len(hit_vals) < 2:
                continue
            hit_25 = np.percentile(hit_vals, 25)
            hit_50 = np.percentile(hit_vals, 50)
            hit_75 = np.percentile(hit_vals, 75)
            non_hit_50 = np.percentile(non_hit_vals, 50) if len(non_hit_vals) > 0 else np.nan
            print(f"  {base_metric:<35s} {operator:<4s} {hit_25:>10.2f} {hit_50:>10.2f} {hit_75:>10.2f} {non_hit_50:>12.2f}")
        print_subheader("Recommended Thresholds to Capture 75% of Hits")
        print("\n# Use these thresholds to keep ~75% of hits")
        print("# (Based on 75th percentile for '<=' and 25th percentile for '>=')\n")
        for base_metric, (operator, direction) in metrics_config.items():
            idx_cols = [f"{base_metric}_idx_{i}" for i in range(af3_N)]
            available_cols = [c for c in idx_cols if c in af3_scores_df.columns]
            if not available_cols:
                continue
            hit_vals = hits_df[available_cols].values.flatten()
            hit_vals = hit_vals[~np.isnan(hit_vals)]
            if len(hit_vals) < 2:
                continue
            if direction == "lower":
                threshold = np.percentile(hit_vals, 75)  # 75th percentile for lower-is-better
            else:
                threshold = np.percentile(hit_vals, 25)  # 25th percentile for higher-is-better
            if abs(threshold) < 1:
                formatted = f"{threshold:.3f}"
            elif abs(threshold) < 10:
                formatted = f"{threshold:.2f}"
            else:
                formatted = f"{threshold:.1f}"
            print(f'    ("{base_metric}", "{operator}"): {formatted},')

# ============================================================================
# SUMMARY
# ============================================================================
print_header("SUMMARY")
print(f"""
  Total metrics analyzed: {len(stats_df)}
  Total designs: {len(af3_scores_df)}
  AF3 predictions per design: {af3_N}

  Top {top_N_percent}% thresholds are computed as:
    - For '<=' filters (lower is better): {top_N_percent}th percentile
    - For '>=' filters (higher is better): {100-top_N_percent}th percentile

  Use the 'FILTERS DICT FORMAT' section above to copy-paste filter values.
""")
print_header("ANALYSIS COMPLETE")


### VI.C.2 Filter

In [ ]:
############################################################
### COMPUTE AF3 STATISTICS FROM SCORES DF & DO FILTERING ###
############################################################

### LOAD THE DF & CREATE scaffold_family ###
af3_scores_df = pd.read_csv(os.path.join(IMPORTANT_DFS_DIR, 'modified_af3_i1__apo.csv'))

### PATHS & NUMBER OF AF3 STRUCTURES PER PDB ###
af3_N   = 5    # Number of AF3 predictions
af3_path = f"{AF3_OUT_DIR}i1__apo/"

### TOGGLE: APPLY FILTERS ONLY TO OVER-REPRESENTED CLUSTERS? ###
apply_filters_only_to_large_groups = False  # Set True to filter only groups with ≥ min_group_size examples

### GROUP SIZE THRESHOLD (for over-represented clusters) ###
min_group_size = 80

### DISPLAY CONTROLS ###
show_example = True
n_examples  = 10
sort_by     = "catres_rmsd"
ascending   = True
print_stat  = True

### SAVE FILTERED DF CONTROLS ###
save_filtered_df = True
save_filtered_df_path = os.path.join(IMPORTANT_DFS_DIR, "filtered_TEMP_modified_af3_i1.csv")

### AF3 FILTER OPTIONS ###
majority_threshold  = 0.8

ca_rmsd             = 1.1
catres_rmsd         = 1.1
catres_subset_rmsd  = 1.1
kcx_tips_rmsd       = 1.5
zinc_rmsd           = 1.25
hydroxide_rmsd      = 1.5
subst_rmsd          = 5.5
subst_core_rmsd     = 2.5
tm_score            = 0.95

chainA_plddt        = 82.5  # protein
chainB_plddt        = 90  # zinc1
chainC_plddt        = 90  # zinc2
chainD_plddt        = 80  # hydroxide
chainE_plddt        = 80  # substrate
catres_plddt        = 80
catres_subset_plddt = 80
kcx_tips_plddt      = 90
subst_core_plddt    = 80
hydroxide_plddt     = 80

interface_ptm       = 0.9
ptm_of_protein      = 0.8

### PAE FILTER OPTIONS ###
chainA_chainB_pae_avg = 6.5  # protein - zinc
chainA_chainC_pae_avg = 6.5  # protein - zinc
chainA_chainD_pae_avg = 6.5  # protein - hydroxide
chainA_chainE_pae_avg = 6.5  # protein - substrate
chainB_chainC_pae_avg = 6.0  # zinc - zinc
chainB_chainD_pae_avg = 6.0  # zinc - hydroxide
chainB_chainE_pae_avg = 6.0  # zinc - substrate
chainC_chainD_pae_avg = 6.0  # zinc - hydroxide
chainC_chainE_pae_avg = 6.0  # zinc - substrate
chainD_chainE_pae_avg = 6.0  # hydroxide - substrate

chainA_chainB_pae_min = 1.25  # protein - zinc
chainA_chainC_pae_min = 1.25  # protein - zinc
chainA_chainD_pae_min = 1.5  # protein - hydroxide
chainA_chainE_pae_min = 1.5  # protein - substrate
chainB_chainC_pae_min = 1.9  # zinc - zinc
chainB_chainD_pae_min = 1.9  # zinc - hydroxide
chainB_chainE_pae_min = 6.0  # zinc - substrate
chainC_chainD_pae_min = 1.9  # zinc - hydroxide
chainC_chainE_pae_min = 6.0  # zinc - substrate
chainD_chainE_pae_min = 6.0  # hydroxide - substrate


### ADD BACKBONES BACK INTO FILTERED DF? ###
apply_scaffold_selection = False
# If the column string does NOT contain "_idx_" then the median across "<column>_idx_*" will be computed.
scaffold_selection_rules = [
    # {"column": "subst_rmsd", "order": "asc", "N": 1},
    {"column": "catres_rmsd","order": "asc", "N": 1},
]

### FILTER OPTIONS ###
# OR logic: use a tuple of column names as the key, e.g.: (("ca_rmsd_TMalign", "ca_rmsd"), "<="): val -> means: passes if ca_rmsd_TMalign_idx_i <= val  OR  ca_rmsd_idx_i <= val | Standard single-string keys still work exactly as before.
filters = {
    (("ca_rmsd_TMalign", "ca_rmsd"), "<="):                        ca_rmsd,
    (("catres_rmsd_TMalign", "catres_rmsd"), "<="):                catres_rmsd,
    (("catres_subset_rmsd_TMalign", "catres_subset_rmsd"), "<="):  catres_subset_rmsd,
#    (("kcx_tips_rmsd_TMalign", "kcx_tips_rmsd"), "<="):            kcx_tips_rmsd,
#    (("zinc_pair_avg_rmsd_TMalign", "zinc_pair_avg_rmsd"), "<="):  zinc_rmsd,
#    (("hydroxide_ion_rmsd_TMalign", "hydroxide_ion_rmsd"), "<="):  hydroxide_rmsd,
#    (("subst_rmsd_TMalign", "subst_rmsd"), "<="):                  subst_rmsd,
#    (("subst_core_rmsd_TMalign", "subst_core_rmsd"), "<="):        subst_core_rmsd,

    ("tm_score", ">="):        tm_score,
    ("chainA_plddt", ">="):    chainA_plddt,
#    ("chainB_plddt", ">="):    chainB_plddt,
#    ("chainC_plddt", ">="):    chainC_plddt,
#    ("chainD_plddt", ">="):    chainD_plddt,
#    ("chainE_plddt", ">="):    chainE_plddt,
    ("catres_plddt", ">="):    catres_plddt,
    ("catres_subset_plddt", ">="):    catres_subset_plddt,
#    ("KCX6_extra_atoms_plddt", ">="): kcx_tips_plddt,
#    ("hydroxide_ion_plddt", ">="): hydroxide_plddt,

#    ("iptm", ">="):            interface_ptm,
    ("ptm",  ">="):            ptm_of_protein,

#    ("chainA_chainB_pair_pae_mean", "<="): chainA_chainB_pae_avg,
#    ("chainA_chainC_pair_pae_mean", "<="): chainA_chainC_pae_avg,
#    ("chainA_chainD_pair_pae_mean", "<="): chainA_chainD_pae_avg,
#    ("chainA_chainE_pair_pae_mean", "<="): chainA_chainE_pae_avg,
#    ("chainB_chainC_pair_pae_mean", "<="): chainB_chainC_pae_avg,
#    ("chainB_chainD_pair_pae_mean", "<="): chainB_chainD_pae_avg,
#    ("chainB_chainE_pair_pae_mean", "<="): chainB_chainE_pae_avg,
#    ("chainC_chainD_pair_pae_mean", "<="): chainC_chainD_pae_avg,
#    ("chainC_chainE_pair_pae_mean", "<="): chainC_chainE_pae_avg,
#    ("chainD_chainE_pair_pae_mean", "<="): chainD_chainE_pae_avg,

#    ("chainA_chainB_pair_pae_min", "<="): chainA_chainB_pae_min,
#    ("chainA_chainC_pair_pae_min", "<="): chainA_chainC_pae_min,
#    ("chainA_chainD_pair_pae_min", "<="): chainA_chainD_pae_min,
#    ("chainA_chainE_pair_pae_min", "<="): chainA_chainE_pae_min,
#    ("chainB_chainC_pair_pae_min", "<="): chainB_chainC_pae_min,
#    ("chainB_chainD_pair_pae_min", "<="): chainB_chainD_pae_min,
#    ("chainB_chainE_pair_pae_min", "<="): chainB_chainE_pae_min,
#    ("chainC_chainD_pair_pae_min", "<="): chainC_chainD_pae_min,
#    ("chainC_chainE_pair_pae_min", "<="): chainC_chainE_pae_min,
#    ("chainD_chainE_pair_pae_min", "<="): chainD_chainE_pae_min,
}

### HELPERS FOR OR-GROUP FILTER KEYS ###
def _build_filter_expr(col_key, op, val, i, df_name):
    """Build an eval-able expression. col_key is a str or tuple-of-strs (OR group)."""
    if isinstance(col_key, tuple):
        parts = [f"({df_name}['{c}_idx_{i}'] {op} {val})" for c in col_key]
        return "(" + " | ".join(parts) + ")"
    else:
        return f"({df_name}['{col_key}_idx_{i}'] {op} {val})"

def _filter_label(col_key):
    """Human-readable label for printing."""
    if isinstance(col_key, tuple):
        return " OR ".join(col_key)
    return col_key

### FILTERING FUNCTION (compact but identical behavior, now with OR support) ###
def filtering_df(dataframe_name, filters, target_df, print_stat=True):
    if print_stat:
        print(f'[{dataframe_name}] ({len(target_df)} designs)')
    print_rows, filtered_desc = [], []
    for i in range(af3_N):
        row_bits, exprs = [], []
        for (col, op), val in filters.items():
            expr = _build_filter_expr(col, op, val, i, "target_df")
            exprs.append(expr)
            if print_stat and i == 0:
                label = _filter_label(col)
                row_bits.append(f'# [ {label.ljust(50)} {op.ljust(2)} {str(val).ljust(6)}]: ')
        mask = eval(" & ".join(exprs))
        if print_stat:
            for j, e in enumerate(exprs):
                m = eval(e)
                pct = f"({m.to_list().count(True)/len(target_df)*100:.1f} %)"
                if i == 0:
                    print_rows.append(row_bits[j])
                print_rows[j] += f"{str(m.to_list().count(True)).rjust(6)}  {pct.rjust(8)}   "
        filtered_desc.extend(target_df[mask]["af3_models_dir"].to_list())
        if print_stat:
            pct = f"({mask.sum()/len(target_df)*100:.1f} %)"
            if i == 0:
                print_rows.append(f'# [                             ALL                             ]: {str(mask.sum()).rjust(6)}  {pct.rjust(8)}   ')
            else:
                print_rows[-1] += f"{str(mask.sum()).rjust(6)}  {pct.rjust(8)}   "
    keep = set(d for d in set(filtered_desc) if (filtered_desc.count(d)/af3_N) >= majority_threshold)
    filtered_df = target_df[target_df["af3_models_dir"].isin(keep)]
    if print_stat:
        for line in print_rows: print(line)
        pct = f"({len(filtered_df)/len(target_df)*100:.1f} %)"
        print(f'# [                     MAJORITY VOTE ({int(majority_threshold*100)} %)                    ]: {str(len(filtered_df)).rjust(6)}  {pct.rjust(8)}')
    return filtered_df

### ORIGINAL FILTERING ###
total_unique_scaffolds = af3_scores_df["scaffold_family"].nunique()
print(f"Total unique scaffold groups before filtering: {total_unique_scaffolds}\n")

if apply_filters_only_to_large_groups:
    # Identify over-represented groups
    group_counts = af3_scores_df["scaffold_family"].value_counts()
    large_groups = group_counts[group_counts >= min_group_size].index
    # Split into large-group subset and the rest
    df_large = af3_scores_df[af3_scores_df["scaffold_family"].isin(large_groups)].copy()
    df_small = af3_scores_df[~af3_scores_df["scaffold_family"].isin(large_groups)].copy()
    # Apply filtering only to the over-represented clusters
    filtered_large = filtering_df("af3_scores_df (large groups)", filters, df_large, print_stat=print_stat)
    # Keep the small-group data unfiltered; combine
    filtered_af3_score_df = pd.concat([filtered_large, df_small]).reset_index(drop=True)
else:
    filtered_af3_score_df = filtering_df("af3_scores_df", filters, af3_scores_df, print_stat=print_stat)

# Print unique scaffold info & counts at this stage
print(f"\nNumber of unique scaffolds (BB) in filtered designs: {filtered_af3_score_df['scaffold_family'].nunique()}")
print(f"Number of Designs BEFORE Filtering: {len(af3_scores_df)}")
print(f"Number of Designs AFTER  Filtering: {len(filtered_af3_score_df)}")

### ADDITIONAL SCAFFOLD SELECTION OPTIONS ###
if apply_scaffold_selection:
    print("\n### ADDING BACK SCAFFOLDS BASED ON YOUR SELECTION CRITERIA ###")
    extra_selected = []
    present_scaffolds = set(filtered_af3_score_df["scaffold_family"].unique())
    for rule in scaffold_selection_rules:
        base, order, N_keep = rule["column"], rule["order"], rule["N"]
        for scaf, group in af3_scores_df.groupby("scaffold_family"):
            if scaf in present_scaffolds:
                continue
            group = group.copy()
            sort_col = base
            if "_idx_" not in base:
                cols = [c for c in group.columns if c.startswith(base + "_idx_")]
                if not cols:
                    continue
                group["median_" + base] = group[cols].median(axis=1)
                sort_col = "median_" + base
            if len(group) >= N_keep:
                extra_selected.append(group.sort_values(by=sort_col, ascending=(order == "asc")).head(N_keep))
    if extra_selected:
        extra_df = pd.concat(extra_selected).drop_duplicates(subset="af3_models_dir")
        print(f"Unique scaffolds in additional selection: {extra_df['scaffold_family'].nunique()}")
        combined = pd.concat([filtered_af3_score_df, extra_df]).drop_duplicates(subset="af3_models_dir")
        print(f"Designs after additional scaffold selection: {len(combined)}")
        print(f"Unique scaffolds after additional selection: {combined['scaffold_family'].nunique()}")
        filtered_af3_score_df = combined
    else:
        print("No additional designs selected via scaffold rules.")

### SHOW EXAMPLES & SAVE ###
# Build set of passing (af3_models_dir, idx) pairs from the filtering logic
passing_pairs = set()
for i in range(af3_N):
    mask = pd.Series([True] * len(filtered_af3_score_df), index=filtered_af3_score_df.index)
    for (col, op), val in filters.items():
        expr = _build_filter_expr(col, op, val, i, "filtered_af3_score_df")
        mask = mask & eval(expr)
    for desc in filtered_af3_score_df.loc[mask, "af3_models_dir"]:
        passing_pairs.add((desc, i))

# For each design, find the extremum value among passing indices
sort_col_base = sort_by
extremum_values = []
passing_info = {}
for idx, row in filtered_af3_score_df.iterrows():
    desc = row["af3_models_dir"]
    passing_idxs = [i for i in range(af3_N) if (desc, i) in passing_pairs]
    if not passing_idxs:
        extremum_values.append(float('inf') if ascending else float('-inf'))
        passing_info[desc] = ([], None, None)
        continue
    vals = {i: row[f"{sort_col_base}_idx_{i}"] for i in passing_idxs}
    rep_idx = min(vals, key=vals.get) if ascending else max(vals, key=vals.get)
    extremum_values.append(vals[rep_idx])
    passing_info[desc] = (passing_idxs, rep_idx, vals[rep_idx])

# Add passing_idx column (None for designs added via scaffold selection)
if "passing_idx" not in filtered_af3_score_df.columns:
    filtered_af3_score_df = filtered_af3_score_df.copy()
    filtered_af3_score_df["passing_idx"] = filtered_af3_score_df["af3_models_dir"].apply(lambda desc: sorted(passing_info[desc][0]) if desc in passing_info
and passing_info[desc][0] else None)

if show_example:
    filtered_af3_score_df["_sort_extremum"] = extremum_values
    filtered_af3_score_df.sort_values(by="_sort_extremum", ascending=ascending, inplace=True)
    desc_list = filtered_af3_score_df["af3_models_dir"].head(n_examples).to_list()
    cmd = "pymol "
    print(f"\nTop {n_examples} designs sorted by {'min' if ascending else 'max'} {sort_by} among passing indices:\n")
    for desc in desc_list:
        passing_idxs, rep_idx, rep_val = passing_info[desc]
        af3_pdb = os.path.join(desc, "*.pdb")
        ref_pdb = filtered_af3_score_df.loc[filtered_af3_score_df["af3_models_dir"] == desc, "ref_path"].iloc[0]
        cmd += af3_pdb + " " + ref_pdb + " "
        print(f"  passing_idx={passing_idxs}, rep_idx={rep_idx}, {sort_by}={rep_val:.3f} | {os.path.basename(desc)}")

    filtered_af3_score_df.drop(columns=["_sort_extremum"], inplace=True)
    print("\nPyMOL command:")
    print(cmd)
if save_filtered_df:
    filtered_af3_score_df.to_csv(save_filtered_df_path, index=False)
    print("\n")
    print(f"Dataframe was saved at {save_filtered_df_path}")

## VI.D Look at Highly Represented Scaffolds & Optionally Filter

### Optional: Look at Over-Represented Backbones 

In [ ]:
#####################################################################################
### ANALYZE DISTRIBUTION OF HOW MANY ENTRIES (ROWS) PER scaffold_family IN FILTER ###
#####################################################################################

### LOAD DF CSV ###
af3_scores_path = f"{IMPORTANT_DFS_DIR}filtered_TEMP_modified_af3_i1.csv"
filtered_df = pd.read_csv(af3_scores_path)

### GROUP BY SCAFFOLDS & COUNT ###
filtered_df["scaffold_family_basename"] = filtered_df["scaffold_family"].astype(str).apply(os.path.basename)
group_counts = (filtered_df.groupby("scaffold_family").size().reset_index(name="count"))
group_counts["scaffold_family_basename"] = group_counts["scaffold_family"].astype(str).apply(os.path.basename)

### OPTIONAL CONTROLS TO PRINT DESCRIPTIONS FROM GROUPS THAT EXCEED A ROW THRESHOLD ###
# --- Toggle 4A: Print N random descriptions from each group with row_count > row_threshold
show_random_examples = False       # Change to False to disable this feature
row_threshold_for_random = 50      # Only consider groups with > this many rows
random_n = 25                      # Number of random descriptions to print from each group

# --- Toggle 4B: Print top/bottom N from each group by a specific column
show_top_bottom_examples = True    # Change to False to disable this feature
ascending_flag = True              # True => smallest->largest (e.g., "top" = best/lowest)
                                   # False => largest->smallest (e.g., "top" = worst/highest)
top_bottom_n = 1                  # Number to print
sort_column = "ca_rmsd_idx_0"      # Column on which to sort
row_threshold_for_sorted = 5

### SUMMARY STATS ###
num_scaffolds = len(group_counts)
more_than_10 = (group_counts["count"] > 10).sum()
more_than_20 = (group_counts["count"] > 20).sum()
more_than_30 = (group_counts["count"] > 30).sum()

print(f"Total unique scaffolds in filtered set: {num_scaffolds}")
print(f"Scaffolds with more than 10 rows: {more_than_10}")
print(f"Scaffolds with more than 20 rows: {more_than_20}")
print(f"Scaffolds with more than 30 rows: {more_than_30}\n")

### INSPECT & PLOT SCAFFOLDS WITH >1 DESIGN ###
multi_entry_groups = group_counts[group_counts["count"] > 1]
print("Scaffolds with more than 1 row (showing top 20):")
print(
    multi_entry_groups
        .sort_values("count", ascending=False)
        .head(20)[["scaffold_family_basename", "count"]]
        .rename(columns={"scaffold_family_basename":"scaffold_family (basename)"})
)
plt.figure(figsize=(7, 5))
plt.hist(multi_entry_groups["count"], bins=range(1, multi_entry_groups["count"].max() + 2), edgecolor='k')
plt.title("Distribution of scaffold_family Sizes (Groups > 1)")
plt.xlabel("Number of Rows (Designs) in Group")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### EXECUTE ###
group_counts_sorted = group_counts.sort_values("count", ascending=False)

if show_random_examples:
    print("\n##############################################")
    print("### RANDOM EXAMPLES FROM LARGE SCAFFOLDS  ###")
    print("##############################################")
    # Filter to groups that exceed row_threshold_for_random
    big_groups = group_counts_sorted[group_counts_sorted["count"] > row_threshold_for_random]
    if big_groups.empty:
        print(f"(No scaffold families have > {row_threshold_for_random} rows; nothing to sample.)")
    for _, row in big_groups.iterrows():
        group_name_full = row["scaffold_family"]
        group_name_base = row["scaffold_family_basename"]
        group_size = row["count"]
        subset_df = filtered_df[filtered_df["scaffold_family"] == group_name_full]
        sample_size = min(random_n, group_size)
        sampled_paths = subset_df["af3_models_dir"].sample(n=sample_size, random_state=42).tolist()
        print(f"\n### scaffold_family '{group_name_base}' (n={group_size}) ###")
        print("### RANDOM SAMPLES ###")
        print("pymol " + " ".join(sampled_paths))

if show_top_bottom_examples:
    print("\n################################################################")
    print("### TOP/BOTTOM EXAMPLES BY A SPECIFIC COLUMN FROM BIG GROUPS ###")
    print("################################################################")
    big_groups = group_counts_sorted[group_counts_sorted["count"] > row_threshold_for_sorted]
    if big_groups.empty:
        print(f"(No scaffold families have > {row_threshold_for_sorted} rows; adjust 'row_threshold_for_sorted' or inspect 'group_counts_sorted'.)")
    else:
        for _, row in big_groups.iterrows():
            group_name_full = row["scaffold_family"]
            group_name_base = row["scaffold_family_basename"]
            group_size = row["count"]
            
            subset_df = filtered_df[filtered_df["scaffold_family"] == group_name_full]
            sorted_subset = subset_df.sort_values(by=sort_column, ascending=ascending_flag)
            
            selection_df = sorted_subset.head(top_bottom_n)
            print(f"\n### scaffold_family '{group_name_base}' (n={group_size}) ###")
            print(f"### FIRST {top_bottom_n} by '{sort_column}' (ascending_flag={ascending_flag}) ###")
            print("pymol " + " ".join([desc + "/*.pdb" for desc in selection_df["af3_models_dir"].tolist()]))

### Optional: Decide to Keep All Designs for Any Particular Scaffold? Remove All Designs for Any Particular Scaffold?

List of designs to remove:
group1_pte_pxon_ChenJACS_exact_v0__lig_YYE_0000_EEEE_rotP_1_D__ORI_01_combo2___1_model_1_cfg_False__cfgscale_NA__stepscale_1_500__gamma0_0_100__gamma_min_0_500__jitter_1_000


In [ ]:
############################################################################################
### FILTER TO KEEP ONLY SPECIFIED SCAFFOLD_FAMILY BASENAMES & SAVE TO NEW CSV ###
############################################################################################

### LOAD DF CSV ###
unfiltered_af3_scores_path = f"{IMPORTANT_DFS_DIR}af3_stats_iteration1_apo.csv"
af3_scores_path = os.path.join(IMPORTANT_DFS_DIR, f"filtered_{Path(unfiltered_af3_scores_path).stem}.csv")
filtered_df = pd.read_csv(af3_scores_path)

### INPUT: LIST OF BASENAMES TO KEEP ###
scaffold_families_to_keep = [
    "group1_pte_pxon_ChenJACS_exact_v0__lig_YYE_0000_EEEE_rotP_1_D__ORI_01_combo2___3_model_0_cfg_True__cfgscale_1_500__stepscale_1_500__gamma0_0_700__gamma_min_1_000__jitter_3_000",
    "group2_pte_pxon_ChenJACS_smwTSopt_v0__lig_YYF_0000_EEEE_rotP_1_D__ORI_01_combo1___1_model_0_cfg_True__cfgscale_2_000__stepscale_1_500__gamma0_0_700__gamma_min_0_500__jitter_2_000",
    # add more basenames here
]

### MAKE A BASENAME COLUMN JUST FOR FILTERING ###
filtered_df["scaffold_family_basename"] = filtered_df["scaffold_family"].astype(str).apply(os.path.basename)

### FILTER ###
kept_df = filtered_df[filtered_df["scaffold_family_basename"].isin(scaffold_families_to_keep)].copy()

### DROP THE HELPER COLUMN BEFORE SAVING ###
kept_df = kept_df.drop(columns=["scaffold_family_basename"])

### OUTPUT PATH ###
kept_csv_path = os.path.join(IMPORTANT_DFS_DIR, f"filtered_SCAFFOLD_FAMS_MUST_KEEP_{Path(unfiltered_af3_scores_path).stem}.csv")

### SAVE ###
kept_df.to_csv(kept_csv_path, index=False)

print(f"Subset saved to: {kept_csv_path}")
print(f"Rows before: {len(filtered_df)} | Rows after filtering: {len(kept_df)}")
print(f"Scaffold families kept: {scaffold_families_to_keep}")

### Filter Over-Represented AF3 Scaffolds Down

## VI.E Copy AF3 Filtered Structures & Ref PDBs

In [ ]:
###############################################
### STEP 1: LOAD CSV & PICK COLUMNS OF INTEREST
###############################################

### INPUTS ###
unfiltered_af3_scores_path = f"{IMPORTANT_DFS_DIR}modified_af3_i1.csv"
filtered_df_path = os.path.join(IMPORTANT_DFS_DIR, f"filtered_TEMP_modified_af3_i1.csv")

### PASSING_IDX MODE CONTROLS ###
use_passing_idx = True         # Set True to use passing_idx column, False for default behavior
fallback_on_nan = True         # If True, fall back to metric-based selection when passing_idx is NaN | False = design is excluded

### CONTROLS (ONLY FOR FALLBACK OR IF PASSING_IDX MODE OFF) ###
selection_metric = "ca_rmsd"   # which metric to use
af3_N            = 5           # number of AF3 predictions per design

N_pick     = 5           # how many indices per row to keep (used for fallback or default mode)
pick_order = "asc"       # "asc" = lowest values, "desc" = highest values

### LOAD ###
df = pd.read_csv(filtered_df_path)

### BUILD COLUMN LIST ###
metric_cols = [f"{selection_metric}_idx_{i}" for i in range(af3_N)]
columns_of_interest = ["af3_models_dir"] + [c for c in metric_cols if c in df.columns]
if "ref_path" in df.columns and "ref_path" not in columns_of_interest:
    columns_of_interest.append("ref_path")
if "passing_idx" in df.columns and "passing_idx" not in columns_of_interest:
    columns_of_interest.append("passing_idx")

### SLIMMED DF ###
df_slim = df[columns_of_interest].copy()
    
print(f"Loaded {len(df)} rows. Kept columns: {columns_of_interest}")

#############################################
### STEP 2: PICK TOP/BOTTOM-N IDX PER ROW ###
#############################################

# Metric columns we want to look at
metric_cols = [c for c in df_slim.columns if c.startswith(f"{selection_metric}_idx_")]

def pick_indices_by_metric(row, n_pick):
    """Pick indices based on metric values (original behavior)."""
    vals = pd.to_numeric(row[metric_cols], errors="coerce")
    if pick_order == "asc":
        chosen = vals.nsmallest(n_pick)
    else:
        chosen = vals.nlargest(n_pick)
    return [c.split("_")[-1] for c in chosen.index.tolist()]

def parse_passing_idx(val):
    """Parse passing_idx column value (handles string representation of list)."""
    if pd.isna(val):
        return None
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            import ast
            parsed = ast.literal_eval(val)
            return parsed if isinstance(parsed, list) else None
        except:
            return None
    return None

# Track statistics
n_used_passing_idx = 0
n_fallback = 0
n_default = 0
total_idx_count = 0
idx_count_distribution = {}  # Track how many designs have N indices

def pick_indices(row):
    global n_used_passing_idx, n_fallback, n_default, total_idx_count
    
    if use_passing_idx:
        passing = parse_passing_idx(row.get("passing_idx"))
        if passing is not None and len(passing) > 0:
            n_used_passing_idx += 1
            total_idx_count += len(passing)
            return [str(i) for i in passing]
        elif fallback_on_nan:
            n_fallback += 1
            indices = pick_indices_by_metric(row, N_pick)
            total_idx_count += len(indices)
            return indices
        else:
            # No fallback, return empty
            return []
    else:
        n_default += 1
        indices = pick_indices_by_metric(row, N_pick)
        total_idx_count += len(indices)
        return indices

# Apply row by row - get raw indices first
raw_indices = df_slim.apply(pick_indices, axis=1)

# Build distribution of index counts
for indices in raw_indices:
    n = len(indices)
    idx_count_distribution[n] = idx_count_distribution.get(n, 0) + 1

# Determine max number of indices across all rows
max_indices = raw_indices.apply(len).max()
idx_cols = [f"idx_to_copy_{i+1}" for i in range(max_indices)]

# Expand into columns with proper formatting
def format_indices(indices):
    formatted = [f"_idx_{i}_model.pdb" if i is not None else pd.NA for i in indices]
    # Pad with NA if fewer indices than max
    formatted.extend([pd.NA] * (max_indices - len(formatted)))
    return formatted

df_slim[idx_cols] = pd.DataFrame(raw_indices.apply(format_indices).tolist(), index=df_slim.index)

# Add basename of `af3_models_dir` as column "bn"
df_slim["bn"] = df_slim["af3_models_dir"].astype(str).map(lambda p: os.path.basename(os.path.normpath(p)))

### SUMMARY STATISTICS ###
print("\n" + "="*60)
print("SELECTION SUMMARY")
print("="*60)
if use_passing_idx:
    print(f"Mode: passing_idx selection (fallback={'ON' if fallback_on_nan else 'OFF'})")
    print(f"  - Used passing_idx:  {n_used_passing_idx:>6} rows")
    print(f"  - Fallback to metric:{n_fallback:>6} rows")
    if not fallback_on_nan:
        n_skipped = len(df_slim) - n_used_passing_idx
        print(f"  - Skipped (no idx):  {n_skipped:>6} rows")
else:
    print(f"Mode: metric-based selection ({selection_metric}, {pick_order}, N={N_pick})")
    print(f"  - Processed:         {n_default:>6} rows")

print("-"*60)
print("INDEX COUNT DISTRIBUTION:")
for n_idx in sorted(idx_count_distribution.keys()):
    count = idx_count_distribution[n_idx]
    pct = count / len(df_slim) * 100
    bar = "█" * int(pct / 2)  # Simple bar chart
    print(f"  {n_idx} idx pass: {count:>6} designs ({pct:>5.1f}%) {bar}")

print("-"*60)
print(f"TOTAL INDICES TO COPY: {total_idx_count}")
print("="*60)

print("\n")
print(df_slim.head())

In [ ]:
#############################################################
### STEP 3: COPY FILES WITH PROGRESS / WARNINGS / TIMING  ###
#############################################################

### INPUT ###
dest_dir = f"{AF3_OUT_DIR}filtered_i1"
os.makedirs(dest_dir, exist_ok=True)

### CONTROLS ###
preserve_metadata = False  # False = faster, True = keeps original timestamps/permissions

### OPTIONAL: copy reference PDBs listed in 'ref_path' into subdir 'ref_pdbs' ###
copy_ref_pdbs = True
ref_subdir = os.path.join(dest_dir, "ref_pdbs")

def should_log(n):
    """Log at 1, 10, 100, 500, then every 1000 until 10k, then every 10k."""
    if n <= 0:
        return False
    if n < 1000:
        return n in (1, 10, 100, 500)
    elif n < 10000:
        return n % 1000 == 0
    else:
        return n % 10000 == 0

def copy_file(src, dst):
    """Copy file with or without metadata based on setting."""
    if preserve_metadata:
        shutil.copy2(src, dst)
    else:
        shutil.copy(src, dst)

if copy_ref_pdbs and "ref_path" in df_slim.columns:
    ref_list = [str(p) for p in df_slim["ref_path"].tolist() if pd.notna(p) and str(p).strip()]
    all_exist = all(os.path.isfile(p) for p in ref_list)
    if not ref_list:
        print("[WARN] Skipping reference PDB copy: 'ref_path' column is present but empty.")
    elif not all_exist:
        print("[WARN] Skipping reference PDB copy: one or more 'ref_path' paths are missing.")
    else:
        os.makedirs(ref_subdir, exist_ok=True)
        seen = set()
        ref_jobs = []
        for p in ref_list:
            if p not in seen:
                seen.add(p)
                ref_jobs.append((p, os.path.join(ref_subdir, os.path.basename(p))))

        print(f"Copying {len(ref_jobs)} reference PDB(s) into: {ref_subdir}")
        r_start = time.time()
        r_count = 0
        r_errors = 0
        for src, dst in ref_jobs:
            try:
                copy_file(src, dst)
                r_count += 1
            except Exception:
                r_errors += 1
            
            if should_log(r_count) or r_count == len(ref_jobs):
                r_elapsed = time.time() - r_start
                r_avg = r_elapsed / r_count
                r_remaining = r_avg * (len(ref_jobs) - r_count)
                print(f"  [{r_count}/{len(ref_jobs)}] {r_elapsed:.1f}s elapsed | ETA: {r_remaining:.1f}s")
        
        r_elapsed_total = time.time() - r_start
        print(f"  Reference PDBs: {r_count}/{len(ref_jobs)} copied in {r_elapsed_total:.1f}s | Errors: {r_errors}")

### LOGIC ###
if "bn" not in df_slim.columns:
    df_slim["bn"] = df_slim["af3_models_dir"].astype(str).map(lambda p: os.path.basename(os.path.normpath(p)))

copy_jobs = [
    (os.path.join(row["af3_models_dir"], f"{row['bn']}{row[col]}"), 
     os.path.join(dest_dir, f"{row['bn']}{row[col]}"))
    for _, row in df_slim.iterrows()
    for col in idx_cols
    if pd.notna(row[col])
]

total_jobs = len(copy_jobs)
print(f"\nCopying {total_jobs} files into: {dest_dir}")

existing = sum(1 for _, dst in copy_jobs if os.path.exists(dst))
if existing:
    print(f"[WARN] {existing} files already exist (will be overwritten)")

start = time.time()
count = 0
errors = 0
missing = 0

for src, dst in copy_jobs:
    try:
        if not os.path.isfile(src):
            missing += 1
            continue
        copy_file(src, dst)
        count += 1
    except Exception:
        errors += 1

    processed = count + missing + errors
    if should_log(processed):
        elapsed = time.time() - start
        avg = elapsed / processed
        remaining = avg * (total_jobs - processed)
        print(f"  [{processed}/{total_jobs}] {elapsed:.1f}s elapsed | ETA: {remaining:.1f}s")

elapsed_total = time.time() - start

### FINAL SUMMARY ###
print("\n" + "="*60)
print("COPY SUMMARY")
print("="*60)
print(f"  Successfully copied: {count:>6} files")
print(f"  Missing sources:     {missing:>6} files")
print(f"  Copy errors:         {errors:>6} files")
print(f"  Total time:          {elapsed_total:>6.1f} seconds")
print(f"  Average rate:        {count/elapsed_total if elapsed_total > 0 else 0:>6.1f} files/s")
print("="*60)

# **VII. FoldSeek Structural Clustering**

In [ ]:

#########################################################################
### FOLDSEEK STRUCTURE CLUSTERING: NOTEBOOK DRIVER CELL / CMD PRINTER ###
#########################################################################

### RUN ###
inputs = [
    f"{AF3_OUT_DIR}filtered_i1", # INPUT DIRECTORY #1
    f"{AF3_OUT_DIR}filtered_i1/ref_pdbs", # INPUT DIRECTORY #2
]

# Optional. If None, the script derives a timestamped default prefix inside
# the first listed input directory. Example explicit override: output_prefix = f"{AF3_OUT_DIR}filtered_i1/foldseek_i1_plus_ref_pdbs_strict"
output_prefix = None

### SCRIPT / CONTAINER ###
# True = print lab-style outer-Apptainer command. False = print direct host command; the script will decide how to launch Foldseek.
use_outer_apptainer = True
apptainer_path      = APPTAINER
script_path         = "{SPECIAL_SCRIPTS_DIR}general_utils/cluster_pdb_structures_with_foldseek.py"

### OUTPUT / FILE-LABELING MODE ###
# Controls whether files are only clustered, copied elsewhere with cluster suffixes, or renamed in place with cluster suffixes. Choices: "none", "copy", "rename-in-place"
clustered_files_mode = "rename-in-place"
clustered_files_dir  = None    # Used only when clustered_files_mode == "copy"
cluster_label_prefix = "FS"    # Example label: FS001

### COMMON PARAMETERS ###
recursive                = False      # False = only exact directories passed; True = descend into subdirectories
threads                  = 6
strictness               = "very-loose" # very-loose, loose, medium, strict, very-strict
alignment_type           = 1          # 0=3Di local, 1=TM-align global, 2=3Di+AA local
coverage                 = None       # Optional explicit override for -c/--coverage
tmscore_threshold        = None       # Optional explicit override for --tmscore-threshold
tmscore_threshold_mode   = 0          # 0=alignment length, 1=query/rep length, 2=target/member length
cov_mode                 = 0          # 0=both query+target, 1=target only, 2=query only, 3/4/5=length-based
cluster_mode             = 1          # 0=set-cover, 1=connected-component, 2/3=greedy length-based
single_step_clustering   = True
status_interval_seconds  = 60

### EXECUTION MODE ###
# For outer-Apptainer execution, use "direct" so the script does not try to launch
# another apptainer inside the container. Choices: "auto", "apptainer", "direct"
foldseek_execution_mode = "direct" if use_outer_apptainer else "auto"

### ADVANCED / LESS COMMON FOLDSEEK PARAMETERS ###
evalue               = None   # Optional E-value cutoff; smaller is stricter
min_seq_id           = None   # Optional minimum sequence-identity threshold
lddt_threshold       = None   # Optional LDDT cutoff; higher is stricter
exact_tmscore        = False  # If True, use exact TM-score instead of faster approximation
tmalign_hit_order    = None   # TM-align hit ranking mode: 0=avg, 1=qTM, 2=tTM, 3=min, 4=max
tmalign_fast         = None   # TM-align fast-mode override: 1=keep fast mode, 0=disable
sensitivity          = None   # Foldseek -s sensitivity; higher is usually more sensitive but slower
max_seqs             = None   # Max hits per query surviving early filtering
split_memory_limit   = None   # Memory cap like "8G"; lets Foldseek split work if needed
cluster_reassign     = False  # Enable Foldseek cluster reassignment
chain_name_mode      = 1      # 0=auto, 1=always append chain name to parsed structure name
model_name_mode      = 0      # 0=auto, 1=always append model name to parsed structure name

### RETENTION / RERUN CONTROLS ###
keep_raw_foldseek_outputs = False
keep_staging_map          = False
keep_staging              = False
keep_tmp                  = False
force                     = False
dry_run                   = False

### RARE PASSTHROUGH ###
# In outer-Apptainer mode, these become extra outer --bind mounts. In host mode, these are passed through as --extra-bind to the script.
extra_bind          = []  # e.g. ["/my/path:/my/path"]
extra_foldseek_args = []  # e.g. ["--max-accept 123 --max-rejected 456"]

def add_opt(cmd, flag, value):
    if value is not None:
        cmd.extend([flag, str(value)])

def add_flag(cmd, flag, enabled):
    if enabled:
        cmd.append(flag)

if use_outer_apptainer:
    cmd = ["apptainer", "exec", "--bind", "/net:/net", "--bind", "/home:/home",]
    for bind in extra_bind:
        cmd.extend(["--bind", bind])
    cmd.extend([apptainer_path, script_path, "--foldseek-execution-mode", foldseek_execution_mode,])
else:
    cmd = [script_path]
    add_opt(cmd, "--foldseek-execution-mode", foldseek_execution_mode)

if output_prefix is not None:
    cmd.extend(["--output-prefix", output_prefix])

# Common knobs
if recursive:
    cmd.append("--recursive")
add_opt(cmd, "--threads", threads)
add_opt(cmd, "--strictness", strictness)
add_opt(cmd, "--alignment-type", alignment_type)
add_opt(cmd, "-c", coverage)
add_opt(cmd, "--tmscore-threshold", tmscore_threshold)
add_opt(cmd, "--tmscore-threshold-mode", tmscore_threshold_mode)
add_opt(cmd, "--cov-mode", cov_mode)
add_opt(cmd, "--cluster-mode", cluster_mode)
add_opt(cmd, "--status-interval-seconds", status_interval_seconds)

if single_step_clustering:
    cmd.append("--single-step-clustering")
else:
    cmd.append("--no-single-step-clustering")

# File-labeling behavior
add_opt(cmd, "--clustered-files-mode", clustered_files_mode)
if clustered_files_mode == "copy":
    add_opt(cmd, "--clustered-files-dir", clustered_files_dir)
add_opt(cmd, "--cluster-label-prefix", cluster_label_prefix)

# Advanced knobs
add_opt(cmd, "-e", evalue)
add_opt(cmd, "--min-seq-id", min_seq_id)
add_opt(cmd, "--lddt-threshold", lddt_threshold)
add_flag(cmd, "--exact-tmscore", exact_tmscore)
add_opt(cmd, "--tmalign-hit-order", tmalign_hit_order)
add_opt(cmd, "--tmalign-fast", tmalign_fast)
add_opt(cmd, "--sensitivity", sensitivity)
add_opt(cmd, "--max-seqs", max_seqs)
add_opt(cmd, "--split-memory-limit", split_memory_limit)
add_flag(cmd, "--cluster-reassign", cluster_reassign)
add_opt(cmd, "--chain-name-mode", chain_name_mode)
add_opt(cmd, "--model-name-mode", model_name_mode)

# Retention / rerun controls
add_flag(cmd, "--keep-raw-foldseek-outputs", keep_raw_foldseek_outputs)
add_flag(cmd, "--keep-staging-map", keep_staging_map)
add_flag(cmd, "--keep-staging", keep_staging)
add_flag(cmd, "--keep-tmp", keep_tmp)
add_flag(cmd, "--force", force)
add_flag(cmd, "--dry-run", dry_run)

# Rare passthrough
if not use_outer_apptainer:
    for bind in extra_bind:
        cmd.extend(["--extra-bind", bind])
for raw_arg in extra_foldseek_args:
    cmd.append(f"--foldseek-arg={raw_arg}")

# Inputs go last
cmd.extend(inputs)

print(f"### EXECUTE THIS COMMAND ###")
print(shlex.join(cmd))
print()

In [ ]:
#############################################################################
### FOLDSEEK STRUCTURE CLUSTERING: EXPLORATION OF MULTI-BACKBONE CLUSTERS ###
#############################################################################

folder = Path("{OUTPUT_DIR}af3_out/filtered_i1")
fs_re = re.compile(r"_FS(\d+)\.pdb$")

# groups[fs][rfd3_backbone] = [pdb1, pdb2, ...]
groups = defaultdict(lambda: defaultdict(list))

for pdb in sorted(folder.glob("*.pdb")):
    m = fs_re.search(pdb.name)
    if not m:
        continue

    fs = m.group(1)
    rfd3_backbone = pdb.name.split("__", 1)[0]
    groups[fs][rfd3_backbone].append(pdb)

for fs in sorted(groups):
    backbone_map = groups[fs]

    # Only keep FS groups that contain more than one unique backbone
    if len(backbone_map) < 2:
        continue

    # Pick one representative file per backbone: first by sorted filename
    chosen = [
        sorted(pdbs, key=lambda p: p.name)[0]
        for backbone, pdbs in sorted(backbone_map.items())
    ]

    print(f"# FS{fs}: {len(chosen)} representative structures across {len(backbone_map)} RFD3 backbones")
    print("# chosen files:")
    for pdb in chosen:
        print(f"#   {pdb.name}")

    print(
        "pymol \\\n  " +
        " \\\n  ".join(shlex.quote(str(pdb)) for pdb in chosen)
    )
    print()

Potential things to filter on: Caver, MutCompute

# **VIII. LigandMPNN Redesign (RFd3 Designs Only | NOT AF3 Preds)**

In [ ]:
#############################################################
### VIII: LIGANDMPNN REDESIGN (orchestrated, 1 cmd/PDB) ###
#############################################################
# Redesign each input structure with LigandMPNN, holding the catalytic site
# fixed. One self-contained command per input PDB, through
# Scripts/mpnn_relevant_utils/design_orchestrator.py:
#   PRE  : fix the REMARK 666 catalytic residues; omit Met at residue 1;
#          conserve designable sidechains that H-bond the active site
#          (rolled per combination, so the second shell varies between designs).
#   MPNN : one LigandMPNN run per temperature combination, via run_ligandmpnn.py,
#          so fixed residues keep their exact input coordinates.
#   POST : flatten to one directory of packed PDBs; protonate (holo with the
#          ligand .params), restoring the input's catalytic tautomers; restore
#          REMARK 666, write REMARK 668 (+ PTM) and DESIGN_PATH.
# Designs are named <input stem>_<combo tag>_<batch>_<seq>.pdb.

### INPUTS ###
input_pdb_structures_dir = f"{AF3_OUT_DIR}filtered_i1/ref_pdbs/"
pdb_glob                 = "*.pdb"

### LIGAND / PTM ###
lig_params = f"{PARAMS_DIR}YYE.params"   # None -> apo protonation
ptm_spec   = "A/LYS/3:KCX"                # REMARK 668 PTM annotation; "" -> none

### OUTPUTS ###
redesign_output_dir = f"{REDESIGN_OUT_DIR}i1__ref_pdbs/"   # <dir>/<input stem>/

### MPNN PARAMETERS (universal; a combo overrides only the keys it names) ###
model_type             = "ligand_mpnn"
omit_aa                = "CX"
bias_AA                = "K:-0.5,R:-0.75,E:0.75,D:0.75"
mpnn_batch_size        = 1
pack                   = 1
repack_everything      = 0
sc_num_denoising_steps = 3
use_side_chain_context = 1

### PER-COMBO SWEEP (50 designs per input structure) ###
combos = [
    {"number_of_batches": 20, "temperature": 0.1},
    {"number_of_batches": 20, "temperature": 0.2},
    {"number_of_batches": 10, "temperature": 0.3},
]

### PRE-PROCESSING TOGGLES ###
fix_remark666_catres = True
omit_nterm_met       = True

### H-BOND SIDECHAIN CONSERVATION ###
conserve_hbonds          = True
conserve_prob            = 0.8
conserve_all_or_none     = False
conserve_seed            = None    # int to replay a roll; None -> generated and printed
conserve_anchors         = "ligand,catalytic,user_fixed"
conserve_hbond_max_dist  = 3.9
conserve_hbond_max_angle = 90
conserve_keep_clashing   = False

### PROTONATION / OUTPUT KNOBS ###
protonate          = True
copy_input         = True
transfer_remarks   = True
design_path_remark = True
keep_intermediates = False

### RE-RUN SAFETY ###
force_overwrite = False
DRY_RUN         = False

### MODEL WEIGHTS ###
ligandmpnn_weights_dir   = os.environ.get("LIGANDMPNN_WEIGHTS", "")
ligandmpnn_checkpoint    = "ligandmpnn_v_32_010_25.pt"
ligandmpnn_sc_checkpoint = "ligandmpnn_sc_v_32_002_16.pt"

### CONSTANTS ###
orchestrator = f"{SPECIAL_SCRIPTS_DIR}mpnn_relevant_utils/design_orchestrator.py"

### SANITY CHECKS ###
if not Path(orchestrator).is_file():
    raise FileNotFoundError(f"orchestrator not found: {orchestrator}")
if not Path(input_pdb_structures_dir).is_dir():
    raise FileNotFoundError(f"input_pdb_structures_dir not found: {input_pdb_structures_dir}")
if lig_params and not Path(lig_params).is_file():
    raise FileNotFoundError(f"lig_params not found: {lig_params}")
Path(redesign_output_dir).mkdir(parents=True, exist_ok=True)

### QUICK LOGIC ###
input_pdbs = sorted(glob.glob(os.path.join(input_pdb_structures_dir, pdb_glob)))
if not input_pdbs:
    raise FileNotFoundError(f"No PDBs matched {os.path.join(input_pdb_structures_dir, pdb_glob)}")
designs_per_input = sum(int(c["number_of_batches"]) * mpnn_batch_size for c in combos)

### BUILD COMMANDS ###
def combo_to_run(combo: dict) -> str:
    return ";".join(f"{k}={v}" for k, v in combo.items())

commands, skipped_existing = [], []
for pdb_file in input_pdbs:
    stem = Path(pdb_file).stem
    out_dir = os.path.join(redesign_output_dir, stem)
    if not force_overwrite and Path(out_dir).is_dir() and list(Path(out_dir).glob("*.pdb")):
        skipped_existing.append(stem)
        continue

    parts = ["python", orchestrator,
             "--pdb_path", pdb_file,
             "--out_folder", out_dir,
             "--model_type", model_type,
             "--batch_size", str(mpnn_batch_size),
             "--pack_side_chains", str(pack),
             "--repack_everything", str(repack_everything),
             "--sc_num_denoising_steps", str(sc_num_denoising_steps),
             "--ligand_mpnn_use_side_chain_context", str(use_side_chain_context),
             "--omit_AA", omit_aa,
             "--bias_AA", bias_AA]
    if ligandmpnn_weights_dir:
        parts += [f"--checkpoint_{model_type}", f"{ligandmpnn_weights_dir}/{ligandmpnn_checkpoint}"]
        if pack:
            parts += ["--checkpoint_path_sc", f"{ligandmpnn_weights_dir}/{ligandmpnn_sc_checkpoint}"]
    if not fix_remark666_catres:
        parts += ["--no_fix_remark666_catres"]
    if not omit_nterm_met:
        parts += ["--no_omit_nterm_met"]
    if conserve_hbonds:
        parts += ["--conserve_hbonds",
                  "--conserve_hbond_prob", str(conserve_prob),
                  "--conserve_anchors", conserve_anchors,
                  "--conserve_hbond_max_dist", str(conserve_hbond_max_dist),
                  "--conserve_hbond_max_angle", str(conserve_hbond_max_angle)]
        if conserve_all_or_none:
            parts += ["--conserve_hbond_all_or_none"]
        if conserve_seed is not None:
            parts += ["--conserve_seed", str(conserve_seed)]
        if conserve_keep_clashing:
            parts += ["--conserve_keep_clashing"]
    if not protonate:
        parts += ["--no_protonate"]
    elif lig_params:
        parts += ["--ligand_params", lig_params]
    if ptm_spec:
        parts += ["--ptm", ptm_spec]
    if not copy_input:
        parts += ["--no_copy_input_structure"]
    if not transfer_remarks:
        parts += ["--no_transfer_remarks"]
    if not design_path_remark:
        parts += ["--no_design_path_remark"]
    if keep_intermediates:
        parts += ["--keep_intermediates"]
    if DRY_RUN:
        parts += ["--dry_run"]
    for c in combos:
        parts += ["--run", combo_to_run(c)]

    commands.append(" ".join(shlex.quote(x) for x in parts))

if skipped_existing:
    print(f"Skipped {len(skipped_existing)} PDB(s) with existing outputs "
          f"(force_overwrite = True to re-run), e.g. {skipped_existing[:3]}")
if not commands:
    raise RuntimeError("No commands generated; every input was skipped by re-run protection.")

### WRITE COMMANDS FILE ###
commands_file = f"{CMDS_DIR}ligandmpnn_redesign_i1_ref_pdbs"
with open(commands_file, "w") as f:
    f.write("\n".join(commands) + "\n")
command_count = len(commands)

print(f"Inputs            = {command_count} PDB(s)")
print(f"Designs per input = {designs_per_input} across {len(combos)} combo(s)")
print(f"Total designs     = {command_count * designs_per_input}")
print(f"Protonation       =", ("holo" if lig_params else "apo") if protonate else "off")
print(f"\nCommands File:\n{commands_file}")
print(f"\n# example command:\n{commands[0]}")

### SETUP BATCH JOBS ###
qtime        = '01:30:00'
cmds_per_job = 1
cores        = '1'
memory       = '8g'
queue        = 'cpu'
job_name     = os.path.basename(commands_file)
submit_file  = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs     = math.ceil(command_count / cmds_per_job)
print(f"\nNumber of Jobs = {num_jobs}   Job Name = {job_name}")
print(f"\nNavigate here:\ncd {redesign_output_dir}\n")
nb.submit_array_job(commands_file, qtime, cores, job_name, memory, submit_file,
                    LOGS_DIR, num_jobs, cmds_per_job, queue)


# **IX. Holo-AF3 (without tags)**

## IX.A.1 Execute AF3 Commands

**NOTE:** At this point it is paramount that you go and make SMILES files for your ligand(s) and cofactor(s). Be careful, remember that these are ML models so they are biased by things they've seen during training - this means that it may be more comfortable with something like a phosphodiester transition state analog for a prediction than your tetrahedral carbon intermediate. This is worth thinking about. \

You should be able to run a command like this: `obabel -ipdb example_lig.pdb -osmi -O example_lig.smi -d -x Zn`
(Open Babel comes with `Environment/zinc_hydro.yml`; `Scripts/env_config.py` reports where it resolved.) \

This website is super helpful for checking your SMILES file or for editing it or for crafting it from scratch: https://www.cheminfo.org/flavor/malaria/Utilities/SMILES_generator___checker/index.html

In [ ]:
################################                                                                                                                                               
### Make AF3 input json file ###                                                                                                                                               
################################                                                                                                                                               

### SHARED INPUTS ###                                                                                                                                                          
input_pdb_path          = f"{REDESIGN_OUT_DIR}i1__ref_pdbs/"        
input_pdb_protein_chain = ["A"]                                                                                                                                                

af3_output_dir    = f"{AF3_OUT_DIR}i2__holo/"                                                                                                                                         
num_input_per_run = 50                                          
                                                                                                                                                                                
### SHARED OPTIONAL LIGAND FLAG ###                                                                                                                                            
include_ligands_if_present = True
input_ligand_dic           = {                                                                                                                                                 
    "B": {"ccdCodes": "['ZN']"},                                                                                                                                               
    "C": {"smiles": "[Zn+2].[OH-]"},                                                                                                                                               
    "D": {"smiles": "CCOP(=O)(OCC)OC1=CC=C(C=C1)[N+](=O)[O-]"},                                                                                                          
}   # leave empty {} or set to None to omit ligand args                                                                                                                        
                                                                                                                                                                                
### SHARED OPTIONAL PTM FLAG ###                                                                                                                                               
include_post_translational_mods_if_present = True                                                                                                                              
ptm_specs = ["A/LYS/3:KCX"]                                     
                                                                                                                                                                                
### SHARED OPTIONAL TERMINUS TAG FLAGS ###                                                                                                                                     
n_terminus_tag = ""                                                                                                                                                         
c_terminus_tag = ""                                                                                                                                                 
                                                                                                                                                                                
### SHARED OPTIONAL FLAGS ###                                                                                                                                                  
output_suffix              = "_af3i2"                                                                                                                                          
check_made_output          = True                               
cleanup_incomplete_outputs = True                                                                                                                                              

### SHARED OPTIONAL RECURSIVE SEARCH FLAGS ###                                                                                                                                 
recursive      = True # False                                           
max_depth      = None # None                                                                                                                                                          
specific_depth = 1    # None                                              
                                                                                                                                                                                
### SHARED SEED OPTIONS ###
no_random_seed = False                                                                                                                                                         
base_seed      = None

### GROUP DEFINITIONS (set to None for single-group behavior) ###
# Each group can override: pdb_prefix, input_ligand_dic, ptm_specs, output_suffix. Any key omitted inherits the shared default above                                                                                                                            
groups = [                                                                                                                                                                     
]                                                                                                                                                                              

### CONSTANTS ###                                                                                                                                                              
apptainer     = APPTAINER
script        = f"{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/make_af3_json_input.py"                                                                             
af3_json_path = os.path.join(AF3_JSON_DIR, os.path.relpath(af3_output_dir, AF3_OUT_DIR))                                                                                       
os.makedirs(af3_output_dir, exist_ok=True)                                                                                                                                     
os.makedirs(af3_json_path, exist_ok=True)                                                                                                                                      
                                                                                                                                                                                
### BUILD COMMAND(S) DYNAMICALLY ###                                                                                                                                           
def build_af3_command(ligand_dic, ptm_list, suffix, pdb_prefix_list=None):
    """Build a single make_af3_json_input.py command string."""                                                                                                                
    cmd_parts = [                                                                                                                                                              
        f"{apptainer} {script}",                                                                                                                                               
        f"--pdb_path {input_pdb_path}",                                                                                                                                        
        f"--pdb_chain {' '.join(input_pdb_protein_chain)}",                                                                                                                    
        f"--json_path {af3_json_path}",                         
        f"--num_input_per_run {num_input_per_run}",                                                                                                                            
        f"--output_path {af3_output_dir}",                                                                                                                                     
    ]
    if ligand_dic:                                                                                                                                                             
        ligand_chains = list(ligand_dic.keys())                                                                                                                                
        ligand_types  = []
        ligand_ids    = []                                                                                                                                                     
        for v in ligand_dic.values():                           
            if len(v) != 1:                                                                                                                                                    
                raise ValueError("Each ligand spec should have exactly one key indicating type (e.g., 'smiles').")                                                             
            ltype, lid = next(iter(v.items()))                                                                                                                                 
            ligand_types.append(ltype)                                                                                                                                         
            ligand_ids.append(lid)                                                                                                                                             
        cmd_parts.append(f"--ligand_chain {' '.join(ligand_chains)}")                                                                                                          
        cmd_parts.append(f"--ligand_type {' '.join(ligand_types)}")
        cmd_parts.append(f'--ligand_id "{" ".join(ligand_ids)}"')                                                                                                              
    if ptm_list:                                                
        cmd_parts.append("--ptm_from_remark666 " + " ".join([f'"{s}"' for s in ptm_list]))                                                                                     
    if n_terminus_tag:                                                                                                                                                         
        cmd_parts.append(f"--n_terminus_tag {n_terminus_tag}")
    if c_terminus_tag:                                                                                                                                                         
        cmd_parts.append(f"--c_terminus_tag {c_terminus_tag}")  
    if suffix:                                                                                                                                                                 
        cmd_parts.append(f"--output_suffix {suffix}")
    if check_made_output:                                                                                                                                                      
        cmd_parts.append("--check_made_output")                 
    if check_made_output and cleanup_incomplete_outputs:
        cmd_parts.append("--cleanup_incomplete_outputs")                                                                                                                       
    if recursive:
        cmd_parts.append("--recursive")                                                                                                                                        
    if max_depth is not None:                                   
        cmd_parts.append(f"--max_depth {max_depth}")
    if specific_depth is not None:                                                                                                                                             
        cmd_parts.append(f"--specific_depth {specific_depth}")
    if no_random_seed:                                                                                                                                                         
        cmd_parts.append("--no_random_seed")                    
    if base_seed is not None:                                                                                                                                                  
        cmd_parts.append(f"--base_seed {base_seed}")
    if pdb_prefix_list:                                                                                                                                                        
        cmd_parts.append(f"--pdb_prefix {' '.join(pdb_prefix_list)}")
    return " ".join(cmd_parts)                                                                                                                                                 

# Resolve shared defaults for ligands/PTMs                                                                                                                                     
shared_ligand_dic = input_ligand_dic if (include_ligands_if_present and input_ligand_dic) else {}
shared_ptm_specs  = ptm_specs if (include_post_translational_mods_if_present and ptm_specs) else []                                                                            
                                                                                                                                                                                
if groups:                                                                                                                                                                     
    # Multi-group mode: each group inherits shared defaults unless overridden                                                                                                  
    print(f"Generating {len(groups)} group commands:\n")                                                                                                                       
    for i, grp in enumerate(groups, 1):                                                                                                                                        
        cmd = build_af3_command(                                                                                                                                               
            ligand_dic      = grp.get("input_ligand_dic", shared_ligand_dic),                                                                                                  
            ptm_list        = grp.get("ptm_specs", shared_ptm_specs),
            suffix          = grp.get("output_suffix", output_suffix),                                                                                                         
            pdb_prefix_list = grp.get("pdb_prefix", None),      
        )                                                                                                                                                                      
        print(f"### Group {i}: {grp.get('name', f'group_{i}')} ###")
        print(cmd)                                                                                                                                                             
        print()                                                                                                                                                                
else:                                                                                                                                                                          
    # Single-group fallback (original behavior)                                                                                                                                
    cmd = build_af3_command(ligand_dic = shared_ligand_dic, ptm_list   = shared_ptm_specs, suffix     = output_suffix,                                                                                                                                            )
    print(cmd) 

In [ ]:
###############                                                                                                                                                                          
### Run AF3 ###                                                                                                                                                                          
###############                                                                                                                                                                          
                                                                                                                                                                                        
### INPUTS ###
af3_output_dir = f"{AF3_OUT_DIR}i2__holo/"
cmds_base_name = f"AF3_i2_holo"

### PARAMETERS ###
N_total_structures = 5  # Total number of structures you want per target (N)
mode = "one_seed"        # "one_seed" -> N diffusion samples from 1 seed | "n_seeds" -> 1 sample from each of N seeds

### CONSTANTS ###
apptainer       = AF3_SIF
script          = AF3_RUNNER
apptainer_b4000 = AF3_SIF
af3_json_path   = os.path.join(AF3_JSON_DIR, os.path.relpath(af3_output_dir, AF3_OUT_DIR))
commands_base   = os.path.join(CMDS_DIR, cmds_base_name)

### QUEUE CONFIGURATION ###
b4000_fraction     = 0.80   # Fraction of commands to B4000 GPUs (0.0 = none, 1.0 = all)
gpu_fraction       = 0.80   # Within regular: fraction to priority queue (gpu vs gpu-bf)
gpu_fraction_b4000 = 0.80   # Within B4000:  fraction to priority queue (gpu-b4000 vs gpu-bf-b4000)

queue_config = {
    "gpu":          {"qtime": "07:30:00", "cmds_per_job": 6, "memory": "12g", "cores": "1"},
    "gpu-bf":       {"qtime": "03:30:00", "cmds_per_job": 4, "memory": "12g", "cores": "1"},

    "gpu-b4000":    {"qtime": "07:00:00", "cmds_per_job": 16, "memory": "12g", "cores": "1"},
    "gpu-bf-b4000": {"qtime": "07:00:00", "cmds_per_job": 16, "memory": "12g", "cores": "1"},
}

### DERIVED FLAGS ###
if mode not in {"one_seed", "n_seeds"}:
    raise ValueError(f"mode must be 'one_seed' or 'n_seeds'. Got: {mode}")
if mode == "one_seed":
    num_seeds = None
    num_diffusion_samples = N_total_structures
else:
    num_seeds = N_total_structures
    num_diffusion_samples = 1

### BUILD COMMANDS ###
# Build commands for both container types from the same JSON list
af3_jsons = sorted(glob.glob(os.path.join(af3_json_path, "*.json")), key=str.lower)

b4000_split = round(len(af3_jsons) * b4000_fraction)
jsons_regular = af3_jsons[b4000_split:]
jsons_b4000   = af3_jsons[:b4000_split]

cmds_regular = []
for af3_json in jsons_regular:
    cmd = f"{apptainer} python {script} --json_path={af3_json} --output_dir={af3_output_dir} --num_diffusion_samples={num_diffusion_samples}"
    if num_seeds is not None:
        cmd += f" --num_seeds={num_seeds}"
    cmds_regular.append(cmd)

cmds_b4000 = []
for af3_json in jsons_b4000:
    cmd = f"{apptainer_b4000} af3 --json_path={af3_json} --output_dir={af3_output_dir} --num_diffusion_samples={num_diffusion_samples}"
    if num_seeds is not None:
        cmd += f" --num_seeds={num_seeds}"
    cmds_b4000.append(cmd)

### SPLIT AND SUBMIT ###
for frac in [gpu_fraction, gpu_fraction_b4000, b4000_fraction]:
    if not (0.0 <= frac <= 1.0):
        raise ValueError(f"Fractions must be between 0.0 and 1.0. Got: {frac}")
queue_splits = {}

if cmds_regular: # Regular GPU split
    pri_idx = round(len(cmds_regular) * gpu_fraction)
    if pri_idx > 0:
        queue_splits["gpu"] = cmds_regular[:pri_idx]
    if pri_idx < len(cmds_regular):
        queue_splits["gpu-bf"] = cmds_regular[pri_idx:]

if cmds_b4000: # B4000 GPU split
    pri_idx = round(len(cmds_b4000) * gpu_fraction_b4000)
    if pri_idx > 0:
        queue_splits["gpu-b4000"] = cmds_b4000[:pri_idx]
    if pri_idx < len(cmds_b4000):
        queue_splits["gpu-bf-b4000"] = cmds_b4000[pri_idx:]

total_cmds = len(cmds_regular) + len(cmds_b4000)
print(f"{'='*60}")
print(f"  AF3 Job Summary │ {total_cmds} total cmds │ Mode: {mode} │ N: {N_total_structures}")
print(f"  Regular: {len(cmds_regular)} cmds │ B4000: {len(cmds_b4000)} cmds")
print(f"{'='*60}")

for queue, queue_cmds in queue_splits.items():
    cfg = queue_config[queue]
    suffix = f"_{queue.replace('-', '')}" if len(queue_splits) > 1 else ""
    commands_file = f"{commands_base}{suffix}"
    job_name = os.path.basename(commands_file)
    submit_file = f"{SUBMIT_DIR}{job_name}.sh"
    num_jobs = math.ceil(len(queue_cmds) / cfg["cmds_per_job"])
    with open(commands_file, 'w') as fo:
        print("\n".join(queue_cmds), file=fo)
    nb.submit_array_job(commands_file, cfg["qtime"], cfg["cores"], job_name, cfg["memory"], submit_file, LOGS_DIR, num_jobs, cfg["cmds_per_job"], queue)

## IX.A.2 Execute AF3 Failed Commands

In [ ]:
##########################################
### Re-run only unfinished AF3 targets ###
##########################################                                                                                                                                

### SHARED INPUTS ###                                                                                                                                                          
input_pdb_path          = f"{REDESIGN_OUT_DIR}i1__ref_pdbs/"       
input_pdb_protein_chain = ["A"]                                                                                                                                                

af3_output_dir    = f"{AF3_OUT_DIR}i2__holo/"                                                                                                                                         
num_input_per_run = 50                                          
                                                                                                                                                                                
### SHARED OPTIONAL LIGAND FLAG ###                                                                                                                                            
include_ligands_if_present = True
input_ligand_dic           = {                                                                                                                                                 
    "B": {"ccdCodes": "['ZN']"},                                                                                                                                               
    "C": {"smiles": "[Zn+2].[OH-]"},                                                                                                                                               
    "D": {"smiles": "CCOP(=O)(OCC)OC1=CC=C(C=C1)[N+](=O)[O-]"},                                                                                                          
}   # leave empty {} or set to None to omit ligand args                                                                                                                     
                                                                                                                                                                                
### SHARED OPTIONAL PTM FLAG ###                                                                                                                                               
include_post_translational_mods_if_present = True                                                                                                                              
ptm_specs = ["A/LYS/3:KCX"]                                     
                                                                                                                                                                                
### SHARED OPTIONAL TERMINUS TAG FLAGS ###                                                                                                                                     
n_terminus_tag = ""                                                                                                                                                         
c_terminus_tag = ""                                                                                                                                                 
                                                                                                                                                                                
### SHARED OPTIONAL FLAGS ###                                                                                                                                                  
output_suffix              = "_af3i2"                                                                                                                                          
check_made_output          = True                               
cleanup_incomplete_outputs = True                                                                                                                                              

### SHARED OPTIONAL RECURSIVE SEARCH FLAGS ###                                                                                                                                 
recursive      = True                                           
max_depth      = None                                                                                                                                                          
specific_depth = 1    #None                                              
                                                                                                                                                                                
### SHARED SEED OPTIONS ###
no_random_seed = False                                                                                                                                                         
base_seed      = None

### GROUP DEFINITIONS (set to None for single-group behavior) ###
# Each group can override: pdb_prefix, input_ligand_dic, ptm_specs, output_suffix. Any key omitted inherits the shared default above                                                                                                                            
groups = [                                                                                                                                                                     
]                                                                                                                                                                              

### CONSTANTS ###                                                                                                                                                              
apptainer     = APPTAINER
script        = f"{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/make_af3_json_input.py"                                                                             

# IMPORTANT: use a NEW json staging directory for reruns
af3_json_path = os.path.join(AF3_JSON_DIR, "i2__holo_rerun_unfinished")

os.makedirs(af3_output_dir, exist_ok=True)
os.makedirs(af3_json_path, exist_ok=True)
                                                                                                                                                                                
### BUILD COMMAND(S) DYNAMICALLY ###                                                                                                                                           
def build_af3_command(ligand_dic, ptm_list, suffix, pdb_prefix_list=None):
    """Build a single make_af3_json_input.py command string."""                                                                                                                
    cmd_parts = [                                                                                                                                                              
        f"{apptainer} {script}",                                                                                                                                               
        f"--pdb_path {input_pdb_path}",                                                                                                                                        
        f"--pdb_chain {' '.join(input_pdb_protein_chain)}",                                                                                                                    
        f"--json_path {af3_json_path}",                         
        f"--num_input_per_run {num_input_per_run}",                                                                                                                            
        f"--output_path {af3_output_dir}",                                                                                                                                     
    ]
    if ligand_dic:                                                                                                                                                             
        ligand_chains = list(ligand_dic.keys())                                                                                                                                
        ligand_types  = []
        ligand_ids    = []                                                                                                                                                     
        for v in ligand_dic.values():                           
            if len(v) != 1:                                                                                                                                                    
                raise ValueError("Each ligand spec should have exactly one key indicating type (e.g., 'smiles').")                                                             
            ltype, lid = next(iter(v.items()))                                                                                                                                 
            ligand_types.append(ltype)                                                                                                                                         
            ligand_ids.append(lid)                                                                                                                                             
        cmd_parts.append(f"--ligand_chain {' '.join(ligand_chains)}")                                                                                                          
        cmd_parts.append(f"--ligand_type {' '.join(ligand_types)}")
        cmd_parts.append(f'--ligand_id "{" ".join(ligand_ids)}"')                                                                                                              
    if ptm_list:                                                
        cmd_parts.append("--ptm_from_remark666 " + " ".join([f'"{s}"' for s in ptm_list]))                                                                                     
    if n_terminus_tag:                                                                                                                                                         
        cmd_parts.append(f"--n_terminus_tag {n_terminus_tag}")
    if c_terminus_tag:                                                                                                                                                         
        cmd_parts.append(f"--c_terminus_tag {c_terminus_tag}")  
    if suffix:                                                                                                                                                                 
        cmd_parts.append(f"--output_suffix {suffix}")
    if check_made_output:                                                                                                                                                      
        cmd_parts.append("--check_made_output")                 
    if check_made_output and cleanup_incomplete_outputs:
        cmd_parts.append("--cleanup_incomplete_outputs")                                                                                                                       
    if recursive:
        cmd_parts.append("--recursive")                                                                                                                                        
    if max_depth is not None:                                   
        cmd_parts.append(f"--max_depth {max_depth}")
    if specific_depth is not None:                                                                                                                                             
        cmd_parts.append(f"--specific_depth {specific_depth}")
    if no_random_seed:                                                                                                                                                         
        cmd_parts.append("--no_random_seed")                    
    if base_seed is not None:                                                                                                                                                  
        cmd_parts.append(f"--base_seed {base_seed}")
    if pdb_prefix_list:                                                                                                                                                        
        cmd_parts.append(f"--pdb_prefix {' '.join(pdb_prefix_list)}")
    return " ".join(cmd_parts)                                                                                                                                                 

# Resolve shared defaults for ligands/PTMs                                                                                                                                     
shared_ligand_dic = input_ligand_dic if (include_ligands_if_present and input_ligand_dic) else {}
shared_ptm_specs  = ptm_specs if (include_post_translational_mods_if_present and ptm_specs) else []                                                                            
                                                                                                                                                                                
if groups:                                                                                                                                                                     
    # Multi-group mode: each group inherits shared defaults unless overridden                                                                                                  
    print(f"Generating {len(groups)} group commands:\n")                                                                                                                       
    for i, grp in enumerate(groups, 1):                                                                                                                                        
        cmd = build_af3_command(                                                                                                                                               
            ligand_dic      = grp.get("input_ligand_dic", shared_ligand_dic),                                                                                                  
            ptm_list        = grp.get("ptm_specs", shared_ptm_specs),
            suffix          = grp.get("output_suffix", output_suffix),                                                                                                         
            pdb_prefix_list = grp.get("pdb_prefix", None),      
        )                                                                                                                                                                      
        print(f"### Group {i}: {grp.get('name', f'group_{i}')} ###")
        print(cmd)                                                                                                                                                             
        print()                                                                                                                                                                
else:                                                                                                                                                                          
    # Single-group fallback (original behavior)                                                                                                                                
    cmd = build_af3_command(ligand_dic = shared_ligand_dic, ptm_list   = shared_ptm_specs, suffix     = output_suffix,                                                                                                                                            )
    print(cmd) 

In [ ]:
###############                                                                                                                                                                          
### Run AF3 ###                                                                                                                                                                          
###############                                                                                                                                                                          
                                                                                                                                                                                        
### INPUTS ###
af3_output_dir = f"{AF3_OUT_DIR}i2__holo/"
cmds_base_name = f"AF3_i2_holo_rerun_unfinished"

### PARAMETERS ###
N_total_structures = 5  # Total number of structures you want per target (N)
mode = "one_seed"        # "one_seed" -> N diffusion samples from 1 seed | "n_seeds" -> 1 sample from each of N seeds

### CONSTANTS ###
apptainer       = AF3_SIF
script          = AF3_RUNNER
apptainer_b4000 = AF3_SIF

# IMPORTANT: point to the NEW rerun json dir
af3_json_path   = os.path.join(AF3_JSON_DIR, "i2__holo_rerun_unfinished")
commands_base   = os.path.join(CMDS_DIR, cmds_base_name)

### QUEUE CONFIGURATION ###
b4000_fraction     = 0.70   # Fraction of commands to B4000 GPUs (0.0 = none, 1.0 = all)
gpu_fraction       = 0.00   # Within regular: fraction to priority queue (gpu vs gpu-bf)
gpu_fraction_b4000 = 1.00   # Within B4000:  fraction to priority queue (gpu-b4000 vs gpu-bf-b4000)

queue_config = {
    "gpu":          {"qtime": "06:30:00", "cmds_per_job": 6, "memory": "12g", "cores": "1"},
    "gpu-bf":       {"qtime": "02:00:00", "cmds_per_job": 2, "memory": "12g", "cores": "1"},

    "gpu-b4000":    {"qtime": "02:00:00", "cmds_per_job": 2, "memory": "12g", "cores": "1"},
    "gpu-bf-b4000": {"qtime": "06:30:00", "cmds_per_job": 16, "memory": "12g", "cores": "1"},
}

### DERIVED FLAGS ###
if mode not in {"one_seed", "n_seeds"}:
    raise ValueError(f"mode must be 'one_seed' or 'n_seeds'. Got: {mode}")
if mode == "one_seed":
    num_seeds = None
    num_diffusion_samples = N_total_structures
else:
    num_seeds = N_total_structures
    num_diffusion_samples = 1

### BUILD COMMANDS ###
# Build commands for both container types from the same JSON list
af3_jsons = sorted(glob.glob(os.path.join(af3_json_path, "*.json")), key=str.lower)

b4000_split = round(len(af3_jsons) * b4000_fraction)
jsons_regular = af3_jsons[b4000_split:]
jsons_b4000   = af3_jsons[:b4000_split]

cmds_regular = []
for af3_json in jsons_regular:
    cmd = f"{apptainer} python {script} --json_path={af3_json} --output_dir={af3_output_dir} --num_diffusion_samples={num_diffusion_samples}"
    if num_seeds is not None:
        cmd += f" --num_seeds={num_seeds}"
    cmds_regular.append(cmd)

cmds_b4000 = []
for af3_json in jsons_b4000:
    cmd = f"{apptainer_b4000} af3 --json_path={af3_json} --output_dir={af3_output_dir} --num_diffusion_samples={num_diffusion_samples}"
    if num_seeds is not None:
        cmd += f" --num_seeds={num_seeds}"
    cmds_b4000.append(cmd)

### SPLIT AND SUBMIT ###
for frac in [gpu_fraction, gpu_fraction_b4000, b4000_fraction]:
    if not (0.0 <= frac <= 1.0):
        raise ValueError(f"Fractions must be between 0.0 and 1.0. Got: {frac}")
queue_splits = {}

if cmds_regular: # Regular GPU split
    pri_idx = round(len(cmds_regular) * gpu_fraction)
    if pri_idx > 0:
        queue_splits["gpu"] = cmds_regular[:pri_idx]
    if pri_idx < len(cmds_regular):
        queue_splits["gpu-bf"] = cmds_regular[pri_idx:]

if cmds_b4000: # B4000 GPU split
    pri_idx = round(len(cmds_b4000) * gpu_fraction_b4000)
    if pri_idx > 0:
        queue_splits["gpu-b4000"] = cmds_b4000[:pri_idx]
    if pri_idx < len(cmds_b4000):
        queue_splits["gpu-bf-b4000"] = cmds_b4000[pri_idx:]

total_cmds = len(cmds_regular) + len(cmds_b4000)
print(f"{'='*60}")
print(f"  AF3 Job Summary │ {total_cmds} total cmds │ Mode: {mode} │ N: {N_total_structures}")
print(f"  Regular: {len(cmds_regular)} cmds │ B4000: {len(cmds_b4000)} cmds")
print(f"{'='*60}")

for queue, queue_cmds in queue_splits.items():
    cfg = queue_config[queue]
    suffix = f"_{queue.replace('-', '')}" if len(queue_splits) > 1 else ""
    commands_file = f"{commands_base}{suffix}"
    job_name = os.path.basename(commands_file)
    submit_file = f"{SUBMIT_DIR}{job_name}.sh"
    num_jobs = math.ceil(len(queue_cmds) / cfg["cmds_per_job"])
    with open(commands_file, 'w') as fo:
        print("\n".join(queue_cmds), file=fo)
    nb.submit_array_job(commands_file, cfg["qtime"], cfg["cores"], job_name, cfg["memory"], submit_file, LOGS_DIR, num_jobs, cfg["cmds_per_job"], queue)

## IX.B Process AF3 Subdirectories

### IX.B.1 Process AF3 Subdirectories

In [ ]:
################################################################
### Convert AF3 output format and remove useless AF3 outputs ###
################################################################

### INPUTS ###
af3_output_dir = f"{AF3_OUT_DIR}i2__holo/"    

### OPTIONAL SCRIPT ARGS ###
cif_sif_path              = MAXIT_SIF  # or None to omit
disable_deletions         = False   # set False to allow cleanup
verbose_logging           = True    # set False for quieter output
include_nonindexed_cifs   = False   # True to also convert non-indexed CIFs
robust_mode               = False   # should probably turn default as false, but this is good for re-running
af3_subdirs_are_lowercase = False

### CONSTANTS ###
apptainer = "python"   # I don't find a suitable apptainer yet.
script    = f"{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/process_af3_cif.py"

### EXECUTE ###
commands = os.path.join(CMDS_DIR, "AF3_process_iteration3")
af3_subdirs = glob.glob(os.path.join(af3_output_dir, "*"))
with os.scandir(af3_output_dir) as it:
    af3_subdirs = [e.path for e in it if e.is_dir(follow_symlinks=False)]
if af3_subdirs_are_lowercase:
    af3_subdirs = list(filter(lambda fi: os.path.basename(fi).lower() == os.path.basename(fi), af3_subdirs))
af3_subdirs = sorted(af3_subdirs, key=lambda p: os.path.basename(p).lower())
print (f"Number of AF3 subdirectories: {len(af3_subdirs)}")
print ()

# build extra args string once
extra_args = []
if cif_sif_path:
    extra_args += ["--cif_sif", cif_sif_path]
if disable_deletions:
    extra_args.append("--disable-deletions")
if verbose_logging:
    extra_args.append("--verbose")
if include_nonindexed_cifs:
    extra_args.append("--include-nonindexed-cifs")
if robust_mode:
    extra_args.append("--robust-mode")
extra_args_str = " ".join(extra_args)

command_num = 0
with open(commands, 'w') as f_cmd:
    for af3_path in af3_subdirs:
        # Only include if "samples" subdir exists
        if os.path.isdir(os.path.join(af3_path, "samples")):
            f_cmd.write(f"{apptainer} {script} --af3_subdir {af3_path} {extra_args_str}\n")
            command_num += 1

### SETUP SLURM SUBMISSION ###
job_name = os.path.basename(commands)
qtime = '00:30:00'
cores = '1'
memory = '2g'
queue = 'cpu'
cmds_per_job = 50
num_jobs = int(command_num / cmds_per_job) + 1
submit_file = f'{SUBMIT_DIR}{job_name}.sh'
print("Job/Commands Name:", job_name)
print("Number of Jobs to Run:", num_jobs)
print('')
print('### COMMANDS FILE BELOW ###')
print(commands)
print('')

# Submit the job array using your rainier module.
nb.submit_array_job(commands, qtime, cores, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue)

In [ ]:
##############################                                                                                                                                                     
### Process AF3 prediction ###                                                                                                                                                     
##############################                                                                                                                                                     
                                                                                                                                                                                    
### PATHS ###
af3_out_path       = f"{AF3_OUT_DIR}i2__holo/"
ref_pdb_path       = f"{REDESIGN_OUT_DIR}i1__ref_pdbs/"
backup_ref_subdir  = "ref_pdbs"
backup_ref_dir     = os.path.join(ref_pdb_path, backup_ref_subdir) if backup_ref_subdir else ""
commands           = f"{CMDS_DIR}AF3_pdb_processing_substrate_zn_zno"

### PARAMETERS ###
af3_N                  = 5
af3_suffix             = "_af3i2"
default_ligand         = "USER_LIG"
use_split_subdirs      = True                                    # True: ref PDBs in subdirectories; False: flat directory
subdir_strip_pattern   = r"_(?:t[\d_]+|c\d+)_\d+_\d+$"   # strips the redesign suffix to recover the input stem
use_ligand_in_filename = False                                   # Search for ligand anywhere in filename (e.g., *PTE_wKCX*XDW*)
use_ligand             = True

### MAP REF PDB PREFIXES TO LIGAND NAMES ###
prefix_ligand_map = {
    "ZAPP": ["YYE"],  # Multiple ligands per prefix is possible
}

### MAP REF PDB PREFIXES TO CATRES SUBSETS ###
prefix_catres_map = {
    "ZAPP": "1,2,3,4,5,6",
}

### ATOM NAMING INPUT TEMPLATE ###
atom_groups_template = [
    {"label": "subst",
     "name3": ["G_D", "USER_LIG"], 
     "chain": ["D", "B"],
     "atoms": [["P1", "N1", "O5", "O6", "O1", "C2", "C1", "O3", "C3", "C4", "O2", "O4", "C5", "C6", "C7", "C8", "C9", "C10"], 
               ["P1", "N1", "O9", "O8", "O6", "C9", "C11", "O4", "C8", "C10", "O5", "O7", "C7", "C5", "C3", "C2", "C4", "C6"]],
     "symmetric_atom_groups": [{
                  "lig1": [["C5", "C6", "C7", "C8", "C9", "C10"], # BENZENE
                           ["N1", "O5", "O6"], # NITRO
                           ["O1", "C2", "C1", "O3", "C3", "C4"]], # ETHYL
         
                  "lig2": [[["C7", "C6", "C4", "C2", "C3", "C5"], ["C7", "C5", "C3", "C2", "C4", "C6"]], 
                           [["N1", "O9", "O8"], ["N1", "O8", "O9"]],
                           [["O6", "C9", "C11", "O4", "C8", "C10"], ["O4", "C8", "C10", "O6", "C9", "C11"]]]
              }],},
    
    {"label": "subst_core",
     "name3": ["G_D", "USER_LIG"], 
     "chain": ["D", "B"],
     "atoms": [["P1", "O1", "O3", "O2", "O4"],
               ["P1", "O6", "O4", "O5", "O7"]],
     "symmetric_atom_groups": [{
                  "lig1": [["O1", "O3"]], # ETHYL
                  "lig2": [[["O4", "O6"], ["O6", "O4"]]]
              }],},

    # Hydroxide ion
    {"label": "hydroxide_ion", "name3": ["G_C", "USER_LIG"], "chain": ["C", "B"], "atoms": [["O1"], ["O3"]]},

    # NCAA
    {"label": "kcx_tips", "name3": ["KCX", "USER_LIG"], "chain": ["A", "B"],
    "atoms": [["CX", "OQ1", "OQ2"], ["C1", "O1", "O2"]],
    "symmetric_atom_groups": [{"lig1": [["CX", "OQ1", "OQ2"]], "lig2": [[["C1", "O1", "O2"], ["C1", "O2", "O1"]]]}]},
    # Zinc ion for ZN1
    {"label": "zinc1_chainB", "name3": ["ZN", "USER_LIG"], "chain": ["B", "B"], "atoms": [["ZN"], ["ZN1"]]},
    {"label": "zinc1_chainC", "name3": ["G_C", "USER_LIG"], "chain": ["C", "B"], "atoms": [["ZN1"], ["ZN1"]]},
    # Zinc ion for ZN2
    {"label": "zinc2_chainB", "name3": ["ZN", "USER_LIG"], "chain": ["B", "B"], "atoms": [["ZN"], ["ZN2"]]},
    {"label": "zinc2_chainC", "name3": ["G_C", "USER_LIG"], "chain": ["C", "B"], "atoms": [["ZN1"], ["ZN2"]]},
]

### OPTIONAL: SPECIFY N & C TERMINUS TAGS TO IGNORE ###
N_tag_ignore = 0    # For example, set to 3 if you want to ignore the first 3 residues (MSG = 3)
C_tag_ignore = 0    # For example, set to 2 if you want to ignore the last 2 residues  (GSAWSHPQFEK = 11)

### FLAGS ###
verbose                     = True   # --verbose
calc_interchain_pae_mean    = True   # --calculate_avg_pae_in_addition_to_pair
rapid_mode_skip_sc_if_found = False  # Skip if .sc is found

### CONSTANTS ###
script    = f'{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/process_af3_pdb.py'
apptainer = APPTAINER

### GENERATE COMMANDS ###

# Pre-compute all AF3 subdirectories that have the correct number of predictions
from concurrent.futures import ThreadPoolExecutor
def _check_af3_dir(d):
    pdb_count = sum(1 for e in os.scandir(d) if e.name.endswith(".pdb") and e.is_file())
    return os.path.basename(d) if pdb_count == af3_N else None
with ThreadPoolExecutor() as executor:
    af3_dirs = glob.glob(os.path.join(af3_out_path, f"*{af3_suffix}"))
    af3_valid_subdirs = {r for r in executor.map(_check_af3_dir, af3_dirs) if r}
print(f"{len(af3_valid_subdirs)} valid AF3 directories detected (with {af3_N} PDBs each)")

# Match AF3 directories to reference PDBs
pairs = []
if use_split_subdirs:
    matched_subdir, matched_backup, unmatched = 0, 0, 0
    unmatched_examples = []  # NEW: collect examples for debugging
    for af3_name in af3_valid_subdirs:
        pdb_base   = af3_name[:-len(af3_suffix)]
        parent_dir = re.sub(subdir_strip_pattern, "", pdb_base)
        ref_pdb    = os.path.join(ref_pdb_path, parent_dir, pdb_base + ".pdb")
        if os.path.isfile(ref_pdb):
            pairs.append((af3_name, ref_pdb))
            matched_subdir += 1
        elif backup_ref_dir:
            ref_pdb = os.path.join(backup_ref_dir, pdb_base + ".pdb")
            if os.path.isfile(ref_pdb):
                pairs.append((af3_name, ref_pdb))
                matched_backup += 1
            else:
                unmatched += 1
                if len(unmatched_examples) < 10:
                    unmatched_examples.append((af3_name, ref_pdb))
        else:
            unmatched += 1
            if len(unmatched_examples) < 10:
                unmatched_examples.append((af3_name, ref_pdb))
        # NEW: lightweight progress update
        if (matched_subdir + matched_backup + unmatched) % 5000 == 0:
            print(f"[match progress] processed={matched_subdir + matched_backup + unmatched:,} "
                  f"| matched={len(pairs):,} | unmatched={unmatched:,}")
    print(f"  Matched from subdirs: {matched_subdir}")
    if backup_ref_dir:
        print(f"  Matched from backup: {matched_backup}")
    print(f"  Unmatched:           {unmatched}")
    # NEW: show examples of failures
    if unmatched_examples:
        print("\nExample unmatched cases (first 10):")
        for af3, ref in unmatched_examples:
            print(f"  AF3: {af3}  ->  expected REF: {ref}")
else:
    ref_pdbs = glob.glob(os.path.join(ref_pdb_path, "*.pdb"))
    print(f"{len(ref_pdbs)} REF PDBs were detected")
    for i, ref_pdb in enumerate(ref_pdbs, 1):
        af3_name = os.path.basename(ref_pdb).replace(".pdb", "") + af3_suffix
        if af3_name in af3_valid_subdirs:
            pairs.append((af3_name, ref_pdb))
        # NEW: progress updates
        if i % 5000 == 0:
            print(f"[ref scan progress] {i:,}/{len(ref_pdbs):,} scanned | pairs={len(pairs):,}")

# Pre-build prefix lookup (uppercase for case-insensitive matching)
prefix_map_upper    = {k.upper(): (v if isinstance(v, list) else [v]) for k, v in prefix_ligand_map.items()}
catres_map_upper    = {k.upper(): v for k, v in prefix_catres_map.items()}
sorted_catres_items = sorted(catres_map_upper.items(), key=lambda x: -len(x[0]))

# Pre-build command parts that don't change
base_cmd   = [apptainer, script]
flag_parts = []
if verbose: flag_parts.append("--verbose")
if calc_interchain_pae_mean: flag_parts.append("--calculate_avg_pae_in_addition_to_pair")
if rapid_mode_skip_sc_if_found: flag_parts.append("--rapid_mode_skip_sc_if_found")

# Cache for ligand-specific atom groups (avoid redundant deep copies)
atom_groups_cache = {}
command_lines = []
for af3_pdb_subdir, ref_pdb in pairs:
    ref_pdb_bn = os.path.basename(ref_pdb)

    # Determine ligand (optimized with cached uppercase prefixes)
    ligand_name = default_ligand
    up = ref_pdb_bn.upper()
    for prefix_up, ligs in prefix_map_upper.items():
        if up.startswith(prefix_up):
            ligand_name = next((lig for lig in ligs if lig.upper() in up), ligs[0]) if use_ligand_in_filename else ligs[0]
            break

    # Determine catres_subset (find longest matching prefix)
    catres_subset = None
    for prefix_up, subset_val in sorted_catres_items:
        if up.startswith(prefix_up):
            catres_subset = subset_val
            break

    # Use cached atom groups for this ligand (avoid redundant processing)
    if ligand_name not in atom_groups_cache:
        atom_groups_cache[ligand_name] = [
            dict(g, name3=[item if item != "USER_LIG" else ligand_name for item in g["name3"]])
            if "symmetric_atom_groups" not in g
            else {**copy.deepcopy(g), "name3": [item if item != "USER_LIG" else ligand_name for item in g["name3"]]}
            for g in atom_groups_template]

    # Build command (reuse pre-built parts)
    af3_dir_path = os.path.join(af3_out_path, af3_pdb_subdir)
    outscr_path  = os.path.join(af3_dir_path, af3_pdb_subdir + ".sc")
    cmd = base_cmd + [f"--af3_dir {af3_dir_path}", f"--ref_pdb {ref_pdb}", f"--outscr {outscr_path}",] + flag_parts
    if use_ligand:
        cmd.append(f"--ligand_groups_json '{json.dumps(atom_groups_cache[ligand_name])}'")
    if catres_subset:
        cmd.append(f"--catres_subset {catres_subset}")
    if N_tag_ignore > 0:
        cmd.append(f"--N_terminus_tag_length_to_ignore {N_tag_ignore}")
    if C_tag_ignore > 0:
        cmd.append(f"--C_terminus_tag_length_to_ignore {C_tag_ignore}")
    command_lines.append(" ".join(cmd))
command_lines.sort()
with open(commands, 'w') as fo:
    fo.write("\n".join(command_lines) + "\n")

### SETUP SLURM SUBMISSION ###
job_name = os.path.basename(commands)
qtime, cores, memory, queue, cmds_per_job = '00:40:00', '1', '2g', 'cpu', 75
num_jobs = (len(command_lines) + cmds_per_job - 1) // cmds_per_job
submit_file = f'{SUBMIT_DIR}{job_name}.sh'

# Submit job array
nb.submit_array_job(commands, qtime, cores, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue)

### IX.B.2 Parse AF3 Stats

In [ ]:
##################################################################
### PARSE & PROCESS AF3 .SC FILES ONLY NEED TO DO THIS ONCE ######
##################################################################

# INPUT DIRECTORY OF SUBDIRECTORIES TO PARSE
af3_dir_of_subdirs = f"{AF3_OUT_DIR}i2__holo/"   # CHANGE EVERY TIME

### OPTIONALS ###
workers                         = 64 # None or integer
optional_path_for_summary_stats = f"{IMPORTANT_DFS_DIR}af3_i2__holo.csv"  # e.g. f"{IMPORTANT_DFS_DIR}af3_stats_iteration2_holo.csv"
optional_chunk_rows             = None  # 20000 # Optional: rows per shard before writing to temp CSV | default = 10000
find_subdirs_without_viable_sc  = False # DRY RUN IF TRUE

### CONSTANTS ###
script = f"{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/concat_af3_sc_dir_of_subdirs.py"
container_path = APPTAINER

### GENERATE COMMAND ###
command = f"singularity exec {container_path} python {script} --af3_dir_of_subdirs {af3_dir_of_subdirs}"
if optional_path_for_summary_stats:
    command += f" --optional_path_for_summary_stats {optional_path_for_summary_stats}"
if optional_chunk_rows:
    command += f" --chunk_rows {optional_chunk_rows}"
if workers:
    command += f" --workers {workers}"
if find_subdirs_without_viable_sc:
    command += f" --find_subdirs_without_viable_sc"

### PRINT COMMAND TO SUBMIT ###
print("SUBMIT THIS:")
print(command)
print('')

### PRINT OUTPUT FILE ###
print("OUTPUT CSV FILE HERE:")
if optional_path_for_summary_stats:
    print(optional_path_for_summary_stats)
else:
    print(af3_dir_of_subdirs + 'zzzzz_af3_analysis_csv_zzzzz.csv')

In [ ]:
########################################################################                                                                      
### APPLY TRANSFORMATIONS & MANIPULATIONS TO THE CONCATENATED AF3 DF ###
########################################################################                                                                      
                                                                
### LOAD DF ###                                                                                                                               
csv_path = f"{IMPORTANT_DFS_DIR}af3_i2__holo.csv"                 
af3_scores_df = pd.read_csv(csv_path)                                                                                                         

### SAVE DF PATH ###                                                                                                                          
out_csv = f"{IMPORTANT_DFS_DIR}modified_af3_i2__holo.csv"         
                                                                                                                                            
### APPLY TRANSFORMATIONS & MANIPULATIONS ###
_desc = af3_scores_df["af3_models_dir"].astype(str)                                                                                           
desc_base = _desc.apply(lambda s: re.split(r"[\\/]", s)[-1])                                                                                  

# Build scaffold_family from the basename (applies to apo + holo)                                                                             
af3_scores_df["scaffold_family"] = (desc_base.str.split("_inp", n=1).str[0].str.split("_eV2", n=1).str[0].str.split("_FS", n=1).str[0]) 
af3_scores_df["structure_cluster"] = (desc_base.str.extract(r"FS(\d+)", expand=False).astype("Int64"))

# -------------------- ZINC pair assignment (Kabsch) --------------------                                                                     
# Only runs if zinc columns exist (skipped automatically for apo)
pattern = re.compile(r"^zinc[12]_chain[BC]_rmsd_idx_(\d+)$")                                                                                  
all_idxs = sorted({int(m.group(1)) for col in af3_scores_df.columns
                    if (m := pattern.match(col))})                                                                                             
                                                                
if not all_idxs:                                                                                                                              
    print("[skip] no zinc*_chain[BC]_rmsd_idx_* columns found — apo mode or no metals; skipping Kabsch zinc-pair block")
else:                                                                                                                                         
    def compute_pair_for_idx(row, i):                           
        cols = {"z1B": f"zinc1_chainB_rmsd_idx_{i}",                                                                                          
                "z1C": f"zinc1_chainC_rmsd_idx_{i}",            
                "z2B": f"zinc2_chainB_rmsd_idx_{i}",                                                                                          
                "z2C": f"zinc2_chainC_rmsd_idx_{i}"}
        if any(c not in row.index for c in cols.values()):                                                                                    
            return np.nan, None                                 
        v = {k: row[c] for k, c in cols.items()}                                                                                              
        if any(pd.isna(x) for x in v.values()):                                                                                               
            return np.nan, None
        bc_avg = float(np.mean([v["z1B"], v["z2C"]]))                                                                                         
        cb_avg = float(np.mean([v["z1C"], v["z2B"]]))                                                                                         
        if bc_avg <= cb_avg:
            return bc_avg, "BC"                                                                                                               
        return cb_avg, "CB"                                                                                                                   

    for i in all_idxs:                                                                                                                        
        avg_col = f"zinc_pair_avg_rmsd_idx_{i}"                 
        asn_col = f"zinc_pair_assignment_idx_{i}"
        af3_scores_df[avg_col] = np.nan                                                                                                       
        af3_scores_df[asn_col] = None
        results = af3_scores_df.apply(lambda r: compute_pair_for_idx(r, i), axis=1, result_type="expand")                                     
        af3_scores_df[avg_col] = results[0].astype(float)                                                                                     
        af3_scores_df[asn_col] = results[1].astype("object")                                                                                  
    print(f"[ok] Kabsch zinc-pair block ran on idx values: {all_idxs}")                                                                       
                                                                                                                                            
# -------------------- ZINC pair assignment (TM-align) --------------------                                                                   
# Only runs if TMalign zinc columns exist (skipped automatically for apo)                                                                     
pattern_tm = re.compile(r"^zinc[12]_chain[BC]_rmsd_TMalign_idx_(\d+)$")                                                                       
all_idxs_tm = sorted({int(m.group(1)) for col in af3_scores_df.columns                                                                        
                    if (m := pattern_tm.match(col))})                                                                                       
                                                                                                                                            
if not all_idxs_tm:                                                                                                                           
    print("[skip] no zinc*_chain[BC]_rmsd_TMalign_idx_* columns found — apo mode or no metals; skipping TM-align zinc-pair block")
else:                                                                                                                                         
    def compute_pair_for_idx_TMalign(row, i):
        cols = {"z1B": f"zinc1_chainB_rmsd_TMalign_idx_{i}",                                                                                  
                "z1C": f"zinc1_chainC_rmsd_TMalign_idx_{i}",    
                "z2B": f"zinc2_chainB_rmsd_TMalign_idx_{i}",                                                                                  
                "z2C": f"zinc2_chainC_rmsd_TMalign_idx_{i}"}    
        if any(c not in row.index for c in cols.values()):                                                                                    
            return np.nan, None                                 
        v = {k: row[c] for k, c in cols.items()}                                                                                              
        if any(pd.isna(x) for x in v.values()):                                                                                               
            return np.nan, None
        bc_avg = float(np.mean([v["z1B"], v["z2C"]]))                                                                                         
        cb_avg = float(np.mean([v["z1C"], v["z2B"]]))                                                                                         
        if bc_avg <= cb_avg:
            return bc_avg, "BC"                                                                                                               
        return cb_avg, "CB"                                                                                                                   

    for i in all_idxs_tm:                                                                                                                     
        avg_col = f"zinc_pair_avg_rmsd_TMalign_idx_{i}"         
        asn_col = f"zinc_pair_assignment_TMalign_idx_{i}"                                                                                     
        af3_scores_df[avg_col] = np.nan
        af3_scores_df[asn_col] = None                                                                                                         
        results = af3_scores_df.apply(lambda r: compute_pair_for_idx_TMalign(r, i), axis=1, result_type="expand")                             
        af3_scores_df[avg_col] = results[0].astype(float)                                                                                     
        af3_scores_df[asn_col] = results[1].astype("object")                                                                                  
    print(f"[ok] TM-align zinc-pair block ran on idx values: {all_idxs_tm}")                                                                  
                                                                                                                                            
### SAVE DF ###
af3_scores_df.to_csv(out_csv, index=False)                                                                                                    
print(f"Saved: {out_csv}  (rows={len(af3_scores_df)})")         

### COUNT ###
total_unique_scaffolds = af3_scores_df["scaffold_family"].nunique()
before_clusters = pd.to_numeric(af3_scores_df["structure_cluster"], errors="coerce").nunique(dropna=True)
print(f"\nTotal unique scaffold groups before filtering: {total_unique_scaffolds}")
print(f"Unique structure clusters BEFORE filtering: {before_clusters}\n")
                                                                                                                                            
### LOOKING ###
_preview_cols = ["af3_models_dir", "scaffold_family", "structure_cluster"] + [c for c in af3_scores_df.columns if c.startswith("zinc_pair_")][:4]                  
af3_scores_df[_preview_cols].head()   

## IX.C Filter AF3

### IX.C.1 Check Stat Distributions

In [ ]:
############################################################
### AF3 METRICS: 5-NUMBER SUMMARY & PERCENTILE ANALYSIS  ###
############################################################
"""
For each metric, compute:
1. 5-number summary (min, Q1, median, Q3, max)
2. Mean and std
3. Top N% threshold (direction-aware: lower or upper percentile)
4. Suggested filter values based on percentiles
"""

import os
import numpy as np
import pandas as pd
from typing import Dict, List

# ============================================================================
# CONFIGURATION
# ============================================================================

### PATHS ###
af3_scores_df_path = os.path.join(IMPORTANT_DFS_DIR, 'modified_af3_i2__holo.csv')

### NUMBER OF AF3 PREDICTIONS ###
af3_N = 5

### TOP N% FOR FILTER RECOMMENDATIONS ###
# If metric uses <= filter, we take the Nth percentile (lower is better)
# If metric uses >= filter, we take the (100-N)th percentile (higher is better)
top_N_percent = 20  # Top 20%

### ADDITIONAL PERCENTILES TO SHOW ###
percentiles_to_show = [5, 10, 15, 20, 25, 30, 50]

### METRICS CONFIGURATION ###
# Format: base_metric -> (filter_operator, expected_direction)
# "lower" means lower values are better (use <= filter)
# "higher" means higher values are better (use >= filter)

metrics_config = {
    # RMSD metrics (lower is better, use <=)
    "ca_rmsd": ("<=", "lower"),
    "zinc_pair_avg_rmsd": ("<=", "lower"),
    "catres_rmsd": ("<=", "lower"),
    "catres_subset_rmsd": ("<=", "lower"),
    "kcx_tips_rmsd": ("<=", "lower"),
    "hydroxide_ion_rmsd": ("<=", "lower"),
    "subst_rmsd": ("<=", "lower"),
    "subst_core_rmsd": ("<=", "lower"),

    # RMSD metrics (lower is better, use <=)
    "ca_rmsd_TMalign": ("<=", "lower"),
    "zinc_pair_avg_rmsd_TMalign": ("<=", "lower"),
    "catres_rmsd_TMalign": ("<=", "lower"),
    "catres_subset_rmsd_TMalign": ("<=", "lower"),
    "kcx_tips_rmsd_TMalign": ("<=", "lower"),
    "hydroxide_ion_rmsd_TMalign": ("<=", "lower"),
    "subst_rmsd_TMalign": ("<=", "lower"),
    "subst_core_rmsd_TMalign": ("<=", "lower"),

    # TM & lDDT metrics (higher is better, use >=)
    "tm_score": (">=", "higher"),
    "catres_lddt": (">=", "higher"),
    "catres_subset_lddt": (">=", "higher"),

    # pLDDT metrics (higher is better, use >=)
    "chainA_plddt": (">=", "higher"),
    "chainB_plddt": (">=", "higher"),
    "chainC_plddt": (">=", "higher"),
    "chainD_plddt": (">=", "higher"),
    "catres_plddt": (">=", "higher"),
    "KCX3_extra_atoms_plddt": (">=", "higher"),
    "hydroxide_ion_plddt": (">=", "higher"),
    "subst_core_plddt": (">=", "higher"),
    "kcx_tips_plddt": (">=", "higher"),

    # PTM/iPTM metrics (higher is better, use >=)
    "iptm": (">=", "higher"),
    "ptm": (">=", "higher"),

    # PAE min metrics (lower is better, use <=)
    "chainA_chainB_pair_pae_min": ("<=", "lower"),
    "chainA_chainC_pair_pae_min": ("<=", "lower"),
    "chainA_chainD_pair_pae_min": ("<=", "lower"),
    "chainB_chainC_pair_pae_min": ("<=", "lower"),
    "chainB_chainD_pair_pae_min": ("<=", "lower"),
    "chainC_chainD_pair_pae_min": ("<=", "lower"),

    # PAE mean metrics (lower is better, use <=)
    "chainA_chainB_pair_pae_mean": ("<=", "lower"),
    "chainA_chainC_pair_pae_mean": ("<=", "lower"),
    "chainA_chainD_pair_pae_mean": ("<=", "lower"),
    "chainB_chainC_pair_pae_mean": ("<=", "lower"),
    "chainB_chainD_pair_pae_mean": ("<=", "lower"),
    "chainC_chainD_pair_pae_mean": ("<=", "lower"),
}

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def print_header(title: str, char: str = "=", width: int = 90):
    print(f"\n{char * width}")
    print(f" {title}")
    print(f"{char * width}")
def print_subheader(title: str, char: str = "-", width: int = 70):
    print(f"\n{char * width}")
    print(f" {title}")
    print(f"{char * width}")

# ============================================================================
# LOAD DATA
# ============================================================================
print_header("AF3 METRICS: 5-NUMBER SUMMARY & PERCENTILE ANALYSIS")
af3_scores_df = pd.read_csv(af3_scores_df_path)
print(f"\nLoaded {len(af3_scores_df)} designs from:\n  {af3_scores_df_path}")

# ============================================================================
# COMPUTE STATISTICS FOR EACH METRIC
# ============================================================================
print_subheader(f"STATISTICS FOR ALL METRICS (Top {top_N_percent}% thresholds)")
all_stats = []
for base_metric, (operator, direction) in metrics_config.items():
    # Collect all idx values for this metric
    idx_cols = [f"{base_metric}_idx_{i}" for i in range(af3_N)]
    available_cols = [c for c in idx_cols if c in af3_scores_df.columns]
    if not available_cols:
        continue
    # Stack all idx values together
    all_values = af3_scores_df[available_cols].values.flatten()
    all_values = all_values[~np.isnan(all_values)]
    if len(all_values) == 0:
        continue
    # 5-number summary
    min_val = np.min(all_values)
    q1 = np.percentile(all_values, 25)
    median_val = np.median(all_values)
    q3 = np.percentile(all_values, 75)
    max_val = np.max(all_values)
    mean_val = np.mean(all_values)
    std_val = np.std(all_values)
    # Top N% threshold (direction-aware)
    if direction == "lower":
        # Lower is better, so top N% means the Nth percentile
        top_n_threshold = np.percentile(all_values, top_N_percent)
    else:
        # Higher is better, so top N% means the (100-N)th percentile
        top_n_threshold = np.percentile(all_values, 100 - top_N_percent)
    # Compute all requested percentiles
    percentile_values = {}
    for p in percentiles_to_show:
        if direction == "lower":
            percentile_values[p] = np.percentile(all_values, p)
        else:
            percentile_values[p] = np.percentile(all_values, 100 - p)
    all_stats.append({
        "metric": base_metric,
        "operator": operator,
        "direction": direction,
        "n_values": len(all_values),
        "min": min_val,
        "Q1": q1,
        "median": median_val,
        "Q3": q3,
        "max": max_val,
        "mean": mean_val,
        "std": std_val,
        f"top_{top_N_percent}%": top_n_threshold,
        **{f"top_{p}%": percentile_values[p] for p in percentiles_to_show},
    })
stats_df = pd.DataFrame(all_stats)

# ============================================================================
# DISPLAY: 5-NUMBER SUMMARY TABLE
# ============================================================================
print_subheader("5-NUMBER SUMMARY (All idx combined)")
print(f"\n  {'Metric':<35s} {'Op':<4s} {'Min':>8s} {'Q1':>8s} {'Median':>8s} {'Q3':>8s} {'Max':>8s} {'Mean':>8s} {'Std':>8s}")
print(f"  {'-'*107}")
for _, row in stats_df.iterrows():
    print(f"  {row['metric']:<35s} {row['operator']:<4s} "
          f"{row['min']:>8.2f} {row['Q1']:>8.2f} {row['median']:>8.2f} "
          f"{row['Q3']:>8.2f} {row['max']:>8.2f} {row['mean']:>8.2f} {row['std']:>8.2f}")

# ============================================================================
# DISPLAY: TOP N% THRESHOLDS (FILTER RECOMMENDATIONS)
# ============================================================================
print_subheader(f"TOP N% THRESHOLDS (Filter Recommendations)")
print(f"\nFor '<=' filters: shows Nth percentile (lower is better)")
print(f"For '>=' filters: shows (100-N)th percentile (higher is better)")
# Header with all percentiles
pct_header = "  " + f"{'Metric':<35s} {'Op':<4s}"
for p in percentiles_to_show:
    pct_header += f" {'Top '+str(p)+'%':>10s}"
print(f"\n{pct_header}")
print(f"  {'-'*(39 + 11*len(percentiles_to_show))}")
for _, row in stats_df.iterrows():
    line = f"  {row['metric']:<35s} {row['operator']:<4s}"
    for p in percentiles_to_show:
        val = row[f"top_{p}%"]
        line += f" {val:>10.2f}"
    print(line)

# ============================================================================
# DISPLAY: SUGGESTED FILTER VALUES (COPY-PASTE READY)
# ============================================================================
print_subheader(f"SUGGESTED FILTER VALUES (Top {top_N_percent}% - Copy-Paste Ready)")
print(f"\n# Filter thresholds for top {top_N_percent}% of designs")
print(f"# Based on {'percentile ' + str(top_N_percent) if True else ''} analysis\n")
for _, row in stats_df.iterrows():
    threshold = row[f"top_{top_N_percent}%"]
    # Format nicely
    if abs(threshold) < 1:
        formatted = f"{threshold:.3f}"
    elif abs(threshold) < 10:
        formatted = f"{threshold:.2f}"
    else:
        formatted = f"{threshold:.1f}"
    var_name = row['metric'].replace("_", "_")
    print(f"{var_name:<35s} = {formatted}")

# ============================================================================
# DISPLAY: FILTERS DICT FORMAT
# ============================================================================
print_subheader(f"FILTERS DICT FORMAT (Top {top_N_percent}%)")
print("\nfilters = {")
for _, row in stats_df.iterrows():
    threshold = row[f"top_{top_N_percent}%"]
    # Format nicely
    if abs(threshold) < 1:
        formatted = f"{threshold:.3f}"
    elif abs(threshold) < 10:
        formatted = f"{threshold:.2f}"
    else:
        formatted = f"{threshold:.1f}"

    print(f'    ("{row["metric"]}", "{row["operator"]}"): {formatted},')
print("}")

# ============================================================================
# PER-IDX STATISTICS
# ============================================================================
print_header("PER-IDX STATISTICS")
print("Checking if different AF3 seeds have different distributions")
per_idx_stats = []
for base_metric, (operator, direction) in metrics_config.items():
    for idx in range(af3_N):
        col = f"{base_metric}_idx_{idx}"
        if col not in af3_scores_df.columns:
            continue
        values = af3_scores_df[col].dropna().values
        if len(values) == 0:
            continue
        if direction == "lower":
            top_n_val = np.percentile(values, top_N_percent)
        else:
            top_n_val = np.percentile(values, 100 - top_N_percent)
        per_idx_stats.append({
            "metric": base_metric,
            "idx": idx,
            "operator": operator,
            "n": len(values),
            "mean": np.mean(values),
            "std": np.std(values),
            "median": np.median(values),
            f"top_{top_N_percent}%": top_n_val,
        })
per_idx_df = pd.DataFrame(per_idx_stats)

# Show summary: do different idx have different thresholds?
print_subheader("Per-Idx Threshold Variation")
print(f"\n  {'Metric':<35s} ", end="")
for idx in range(af3_N):
    print(f"{'idx_'+str(idx):>10s} ", end="")
print(f"{'Range':>10s}")
print(f"  {'-'*(35 + 11*(af3_N+1))}")
for base_metric in stats_df["metric"].values:
    sub = per_idx_df[per_idx_df["metric"] == base_metric]
    if len(sub) == 0:
        continue
    line = f"  {base_metric:<35s} "
    vals = []
    for idx in range(af3_N):
        row = sub[sub["idx"] == idx]
        if len(row) > 0:
            val = row[f"top_{top_N_percent}%"].values[0]
            vals.append(val)
            line += f"{val:>10.2f} "
        else:
            line += f"{'N/A':>10s} "
    if vals:
        range_val = max(vals) - min(vals)
        line += f"{range_val:>10.2f}"
    print(line)

# ============================================================================
# SPECIAL ANALYSIS: HIT-BASED THRESHOLDS (if hit classification exists)
# ============================================================================
# Check if we have hit classification from the retro analysis
if "hit_class" in af3_scores_df.columns or "is_hit" in af3_scores_df.columns:
    print_header("HIT-BASED THRESHOLD ANALYSIS")
    print("Thresholds based on hit distribution (where hits tend to fall)")
    # Determine hit column
    if "is_any_hit" in af3_scores_df.columns:
        hit_col = "is_any_hit"
    elif "is_hit" in af3_scores_df.columns:
        hit_col = "is_hit"
    else:
        hit_col = None
    if hit_col:
        hits_df = af3_scores_df[af3_scores_df[hit_col]]
        non_hits_df = af3_scores_df[~af3_scores_df[hit_col]]
        print(f"\nHits: {len(hits_df)} | Non-hits: {len(non_hits_df)}")
        print_subheader("Hit Distribution Percentiles")
        print(f"\nShowing where hits fall - use these as filter thresholds to capture most hits")
        print(f"\n  {'Metric':<35s} {'Op':<4s} {'Hit 25%':>10s} {'Hit 50%':>10s} {'Hit 75%':>10s} {'NonHit 50%':>12s}")
        print(f"  {'-'*77}")
        for base_metric, (operator, direction) in metrics_config.items():
            idx_cols = [f"{base_metric}_idx_{i}" for i in range(af3_N)]
            available_cols = [c for c in idx_cols if c in af3_scores_df.columns]
            if not available_cols:
                continue
            hit_vals = hits_df[available_cols].values.flatten()
            hit_vals = hit_vals[~np.isnan(hit_vals)]
            non_hit_vals = non_hits_df[available_cols].values.flatten()
            non_hit_vals = non_hit_vals[~np.isnan(non_hit_vals)]
            if len(hit_vals) < 2:
                continue
            hit_25 = np.percentile(hit_vals, 25)
            hit_50 = np.percentile(hit_vals, 50)
            hit_75 = np.percentile(hit_vals, 75)
            non_hit_50 = np.percentile(non_hit_vals, 50) if len(non_hit_vals) > 0 else np.nan
            print(f"  {base_metric:<35s} {operator:<4s} {hit_25:>10.2f} {hit_50:>10.2f} {hit_75:>10.2f} {non_hit_50:>12.2f}")
        print_subheader("Recommended Thresholds to Capture 75% of Hits")
        print("\n# Use these thresholds to keep ~75% of hits")
        print("# (Based on 75th percentile for '<=' and 25th percentile for '>=')\n")
        for base_metric, (operator, direction) in metrics_config.items():
            idx_cols = [f"{base_metric}_idx_{i}" for i in range(af3_N)]
            available_cols = [c for c in idx_cols if c in af3_scores_df.columns]
            if not available_cols:
                continue
            hit_vals = hits_df[available_cols].values.flatten()
            hit_vals = hit_vals[~np.isnan(hit_vals)]
            if len(hit_vals) < 2:
                continue
            if direction == "lower":
                threshold = np.percentile(hit_vals, 75)  # 75th percentile for lower-is-better
            else:
                threshold = np.percentile(hit_vals, 25)  # 25th percentile for higher-is-better
            if abs(threshold) < 1:
                formatted = f"{threshold:.3f}"
            elif abs(threshold) < 10:
                formatted = f"{threshold:.2f}"
            else:
                formatted = f"{threshold:.1f}"
            print(f'    ("{base_metric}", "{operator}"): {formatted},')

# ============================================================================
# SUMMARY
# ============================================================================
print_header("SUMMARY")
print(f"""
  Total metrics analyzed: {len(stats_df)}
  Total designs: {len(af3_scores_df)}
  AF3 predictions per design: {af3_N}

  Top {top_N_percent}% thresholds are computed as:
    - For '<=' filters (lower is better): {top_N_percent}th percentile
    - For '>=' filters (higher is better): {100-top_N_percent}th percentile

  Use the 'FILTERS DICT FORMAT' section above to copy-paste filter values.
""")
print_header("ANALYSIS COMPLETE")


### IX.C.2 Filter

In [ ]:
############################################################
### COMPUTE AF3 STATISTICS FROM SCORES DF & DO FILTERING ###
############################################################

### LOAD THE DF & CREATE scaffold_family ###
af3_scores_df = pd.read_csv(os.path.join(IMPORTANT_DFS_DIR, 'modified_af3_i2__holo.csv'))

### PATHS & NUMBER OF AF3 STRUCTURES PER PDB ###
af3_N   = 5    # Number of AF3 predictions
af3_path = f"{AF3_OUT_DIR}i2__holo/"

### TOGGLE: APPLY FILTERS ONLY TO OVER-REPRESENTED CLUSTERS? ###
apply_filters_only_to_large_groups = False  # Set True to filter only groups with ≥ min_group_size examples

### GROUP SIZE THRESHOLD (for over-represented clusters) ###
min_group_size = 80

### DISPLAY CONTROLS ###
show_example = True
n_examples  = 10
sort_by     = "catres_rmsd"
ascending   = True
print_stat  = True

### SAVE FILTERED DF CONTROLS ###
save_filtered_df = True
save_filtered_df_path = os.path.join(IMPORTANT_DFS_DIR, "filtered_TEMP_modified_af3_i2.csv")

### AF3 FILTER OPTIONS ###
majority_threshold  = 0.8

ca_rmsd             = 1.0
catres_rmsd         = 1.0
catres_subset_rmsd  = 1.0
kcx_tips_rmsd       = 1.5
zinc_rmsd           = 1.25
hydroxide_rmsd      = 1.5
subst_rmsd          = 4.0
subst_core_rmsd     = 2.5
tm_score            = 0.95

chainA_plddt        = 90  # protein
chainB_plddt        = 90  # zinc1
chainC_plddt        = 90  # zinc2
chainD_plddt        = 80  # hydroxide OR substrate
chainE_plddt        = 80  # substrate
catres_plddt        = 90
catres_subset_plddt = 90
kcx_tips_plddt      = 90
subst_core_plddt    = 80
hydroxide_plddt     = 80

interface_ptm       = 0.9
ptm_of_protein      = 0.9

### PAE FILTER OPTIONS ###
chainA_chainB_pae_avg = 6.5  # protein - zinc
chainA_chainC_pae_avg = 6.5  # protein - zinc
chainA_chainD_pae_avg = 6.5  # protein - hydroxide
chainA_chainE_pae_avg = 6.5  # protein - substrate
chainB_chainC_pae_avg = 6.0  # zinc - zinc
chainB_chainD_pae_avg = 6.0  # zinc - hydroxide
chainB_chainE_pae_avg = 6.0  # zinc - substrate
chainC_chainD_pae_avg = 6.0  # zinc - hydroxide
chainC_chainE_pae_avg = 6.0  # zinc - substrate
chainD_chainE_pae_avg = 6.0  # hydroxide - substrate

chainA_chainB_pae_min = 1.25  # protein - zinc
chainA_chainC_pae_min = 1.25  # protein - zinc
chainA_chainD_pae_min = 1.5  # protein - hydroxide
chainA_chainE_pae_min = 1.5  # protein - substrate
chainB_chainC_pae_min = 1.9  # zinc - zinc
chainB_chainD_pae_min = 1.9  # zinc - hydroxide
chainB_chainE_pae_min = 6.0  # zinc - substrate
chainC_chainD_pae_min = 1.9  # zinc - hydroxide
chainC_chainE_pae_min = 6.0  # zinc - substrate
chainD_chainE_pae_min = 6.0  # hydroxide - substrate


### ADD BACKBONES BACK INTO FILTERED DF? ###
apply_scaffold_selection = False
# If the column string does NOT contain "_idx_" then the median across "<column>_idx_*" will be computed.
scaffold_selection_rules = [
    # {"column": "subst_rmsd", "order": "asc", "N": 1},
    {"column": "catres_rmsd","order": "asc", "N": 1},
]

### FILTER OPTIONS ###
# OR logic: use a tuple of column names as the key, e.g.: (("ca_rmsd_TMalign", "ca_rmsd"), "<="): val -> means: passes if ca_rmsd_TMalign_idx_i <= val  OR  ca_rmsd_idx_i <= val | Standard single-string keys still work exactly as before.
filters = {
    (("ca_rmsd_TMalign", "ca_rmsd"), "<="):                        ca_rmsd,
    (("catres_rmsd_TMalign", "catres_rmsd"), "<="):                catres_rmsd,
    (("catres_subset_rmsd_TMalign", "catres_subset_rmsd"), "<="):  catres_subset_rmsd,
    (("kcx_tips_rmsd_TMalign", "kcx_tips_rmsd"), "<="):            kcx_tips_rmsd,
    (("zinc_pair_avg_rmsd_TMalign", "zinc_pair_avg_rmsd"), "<="):  zinc_rmsd,
#    (("hydroxide_ion_rmsd_TMalign", "hydroxide_ion_rmsd"), "<="):  hydroxide_rmsd,
    (("subst_rmsd_TMalign", "subst_rmsd"), "<="):                  subst_rmsd,
    (("subst_core_rmsd_TMalign", "subst_core_rmsd"), "<="):        subst_core_rmsd,

    ("tm_score", ">="):        tm_score,
    ("chainA_plddt", ">="):    chainA_plddt,
   # ("chainB_plddt", ">="):    chainB_plddt,
   # ("chainC_plddt", ">="):    chainC_plddt,
   # ("chainD_plddt", ">="):    chainD_plddt,
#    ("chainE_plddt", ">="):    chainE_plddt,
    ("catres_plddt", ">="):    catres_plddt,
    ("catres_subset_plddt", ">="):    catres_subset_plddt,
    ("KCX3_extra_atoms_plddt", ">="): kcx_tips_plddt,
#    ("hydroxide_ion_plddt", ">="): hydroxide_plddt,

    ("iptm", ">="):            interface_ptm,
    ("ptm",  ">="):            ptm_of_protein,

#    ("chainA_chainB_pair_pae_mean", "<="): chainA_chainB_pae_avg,
#    ("chainA_chainC_pair_pae_mean", "<="): chainA_chainC_pae_avg,
#    ("chainA_chainD_pair_pae_mean", "<="): chainA_chainD_pae_avg,
#    ("chainA_chainE_pair_pae_mean", "<="): chainA_chainE_pae_avg,
#    ("chainB_chainC_pair_pae_mean", "<="): chainB_chainC_pae_avg,
#    ("chainB_chainD_pair_pae_mean", "<="): chainB_chainD_pae_avg,
#    ("chainB_chainE_pair_pae_mean", "<="): chainB_chainE_pae_avg,
#    ("chainC_chainD_pair_pae_mean", "<="): chainC_chainD_pae_avg,
#    ("chainC_chainE_pair_pae_mean", "<="): chainC_chainE_pae_avg,
#    ("chainD_chainE_pair_pae_mean", "<="): chainD_chainE_pae_avg,

   # ("chainA_chainB_pair_pae_min", "<="): chainA_chainB_pae_min,
   # ("chainA_chainC_pair_pae_min", "<="): chainA_chainC_pae_min,
   # ("chainA_chainD_pair_pae_min", "<="): chainA_chainD_pae_min,
#    ("chainA_chainE_pair_pae_min", "<="): chainA_chainE_pae_min,
   # ("chainB_chainC_pair_pae_min", "<="): chainB_chainC_pae_min,
   # ("chainB_chainD_pair_pae_min", "<="): chainB_chainD_pae_min,
#    ("chainB_chainE_pair_pae_min", "<="): chainB_chainE_pae_min,
   # ("chainC_chainD_pair_pae_min", "<="): chainC_chainD_pae_min,
#    ("chainC_chainE_pair_pae_min", "<="): chainC_chainE_pae_min,
#    ("chainD_chainE_pair_pae_min", "<="): chainD_chainE_pae_min,
}

### HELPERS FOR OR-GROUP FILTER KEYS ###
def _build_filter_expr(col_key, op, val, i, df_name):
    """Build an eval-able expression. col_key is a str or tuple-of-strs (OR group)."""
    if isinstance(col_key, tuple):
        parts = [f"({df_name}['{c}_idx_{i}'] {op} {val})" for c in col_key]
        return "(" + " | ".join(parts) + ")"
    else:
        return f"({df_name}['{col_key}_idx_{i}'] {op} {val})"

def _filter_label(col_key):
    """Human-readable label for printing."""
    if isinstance(col_key, tuple):
        return " OR ".join(col_key)
    return col_key

### FILTERING FUNCTION (compact but identical behavior, now with OR support) ###
def filtering_df(dataframe_name, filters, target_df, print_stat=True):
    if print_stat:
        print(f'[{dataframe_name}] ({len(target_df)} designs)')
    print_rows, filtered_desc = [], []
    for i in range(af3_N):
        row_bits, exprs = [], []
        for (col, op), val in filters.items():
            expr = _build_filter_expr(col, op, val, i, "target_df")
            exprs.append(expr)
            if print_stat and i == 0:
                label = _filter_label(col)
                row_bits.append(f'# [ {label.ljust(50)} {op.ljust(2)} {str(val).ljust(6)}]: ')
        mask = eval(" & ".join(exprs))
        if print_stat:
            for j, e in enumerate(exprs):
                m = eval(e)
                pct = f"({m.to_list().count(True)/len(target_df)*100:.1f} %)"
                if i == 0:
                    print_rows.append(row_bits[j])
                print_rows[j] += f"{str(m.to_list().count(True)).rjust(6)}  {pct.rjust(8)}   "
        filtered_desc.extend(target_df[mask]["af3_models_dir"].to_list())
        if print_stat:
            pct = f"({mask.sum()/len(target_df)*100:.1f} %)"
            if i == 0:
                print_rows.append(f'# [                             ALL                             ]: {str(mask.sum()).rjust(6)}  {pct.rjust(8)}   ')
            else:
                print_rows[-1] += f"{str(mask.sum()).rjust(6)}  {pct.rjust(8)}   "
    keep = set(d for d in set(filtered_desc) if (filtered_desc.count(d)/af3_N) >= majority_threshold)
    filtered_df = target_df[target_df["af3_models_dir"].isin(keep)]
    if print_stat:
        for line in print_rows: print(line)
        pct = f"({len(filtered_df)/len(target_df)*100:.1f} %)"
        print(f'# [                     MAJORITY VOTE ({int(majority_threshold*100)} %)                    ]: {str(len(filtered_df)).rjust(6)}  {pct.rjust(8)}')
    return filtered_df

### ORIGINAL FILTERING ###
total_unique_scaffolds = af3_scores_df["scaffold_family"].nunique()
before_clusters = pd.to_numeric(af3_scores_df["structure_cluster"], errors="coerce").nunique(dropna=True)
print(f"Total unique scaffold groups before filtering: {total_unique_scaffolds}")
print(f"Unique structure clusters BEFORE filtering: {before_clusters}\n")

if apply_filters_only_to_large_groups:
    # Identify over-represented groups
    group_counts = af3_scores_df["scaffold_family"].value_counts()
    large_groups = group_counts[group_counts >= min_group_size].index
    # Split into large-group subset and the rest
    df_large = af3_scores_df[af3_scores_df["scaffold_family"].isin(large_groups)].copy()
    df_small = af3_scores_df[~af3_scores_df["scaffold_family"].isin(large_groups)].copy()
    # Apply filtering only to the over-represented clusters
    filtered_large = filtering_df("af3_scores_df (large groups)", filters, df_large, print_stat=print_stat)
    # Keep the small-group data unfiltered; combine
    filtered_af3_score_df = pd.concat([filtered_large, df_small]).reset_index(drop=True)
else:
    filtered_af3_score_df = filtering_df("af3_scores_df", filters, af3_scores_df, print_stat=print_stat)

# Print unique scaffold info & counts at this stage
print(f"\nNumber of unique scaffolds (BB) in filtered designs: {filtered_af3_score_df['scaffold_family'].nunique()}")
print(f"Number of unique structure clusters in filtered designs: {pd.to_numeric(filtered_af3_score_df['structure_cluster'], errors='coerce').nunique(dropna=True)}")
print(f"\nNumber of Designs BEFORE Filtering: {len(af3_scores_df)}")
print(f"Number of Designs AFTER  Filtering: {len(filtered_af3_score_df)}")

### ADDITIONAL SCAFFOLD SELECTION OPTIONS ###
if apply_scaffold_selection:
    print("\n### ADDING BACK SCAFFOLDS BASED ON YOUR SELECTION CRITERIA ###")
    extra_selected = []
    present_scaffolds = set(filtered_af3_score_df["scaffold_family"].unique())
    for rule in scaffold_selection_rules:
        base, order, N_keep = rule["column"], rule["order"], rule["N"]
        for scaf, group in af3_scores_df.groupby("scaffold_family"):
            if scaf in present_scaffolds:
                continue
            group = group.copy()
            sort_col = base
            if "_idx_" not in base:
                cols = [c for c in group.columns if c.startswith(base + "_idx_")]
                if not cols:
                    continue
                group["median_" + base] = group[cols].median(axis=1)
                sort_col = "median_" + base
            if len(group) >= N_keep:
                extra_selected.append(group.sort_values(by=sort_col, ascending=(order == "asc")).head(N_keep))
    if extra_selected:
        extra_df = pd.concat(extra_selected).drop_duplicates(subset="af3_models_dir")
        print(f"Unique scaffolds in additional selection: {extra_df['scaffold_family'].nunique()}")
        combined = pd.concat([filtered_af3_score_df, extra_df]).drop_duplicates(subset="af3_models_dir")
        print(f"Designs after additional scaffold selection: {len(combined)}")
        print(f"Unique scaffolds after additional selection: {combined['scaffold_family'].nunique()}")
        filtered_af3_score_df = combined
    else:
        print("No additional designs selected via scaffold rules.")

### SHOW EXAMPLES & SAVE ###
# Build set of passing (af3_models_dir, idx) pairs from the filtering logic
passing_pairs = set()
for i in range(af3_N):
    mask = pd.Series([True] * len(filtered_af3_score_df), index=filtered_af3_score_df.index)
    for (col, op), val in filters.items():
        expr = _build_filter_expr(col, op, val, i, "filtered_af3_score_df")
        mask = mask & eval(expr)
    for desc in filtered_af3_score_df.loc[mask, "af3_models_dir"]:
        passing_pairs.add((desc, i))

# For each design, find the extremum value among passing indices
sort_col_base = sort_by
extremum_values = []
passing_info = {}
for idx, row in filtered_af3_score_df.iterrows():
    desc = row["af3_models_dir"]
    passing_idxs = [i for i in range(af3_N) if (desc, i) in passing_pairs]
    if not passing_idxs:
        extremum_values.append(float('inf') if ascending else float('-inf'))
        passing_info[desc] = ([], None, None)
        continue
    vals = {i: row[f"{sort_col_base}_idx_{i}"] for i in passing_idxs}
    rep_idx = min(vals, key=vals.get) if ascending else max(vals, key=vals.get)
    extremum_values.append(vals[rep_idx])
    passing_info[desc] = (passing_idxs, rep_idx, vals[rep_idx])

# Add passing_idx column (None for designs added via scaffold selection)
if "passing_idx" not in filtered_af3_score_df.columns:
    filtered_af3_score_df = filtered_af3_score_df.copy()
    filtered_af3_score_df["passing_idx"] = filtered_af3_score_df["af3_models_dir"].apply(lambda desc: sorted(passing_info[desc][0]) if desc in passing_info
and passing_info[desc][0] else None)

if show_example:
    filtered_af3_score_df["_sort_extremum"] = extremum_values
    filtered_af3_score_df.sort_values(by="_sort_extremum", ascending=ascending, inplace=True)
    desc_list = filtered_af3_score_df["af3_models_dir"].head(n_examples).to_list()
    cmd = "pymol "
    print(f"\nTop {n_examples} designs sorted by {'min' if ascending else 'max'} {sort_by} among passing indices:\n")
    for desc in desc_list:
        passing_idxs, rep_idx, rep_val = passing_info[desc]
        af3_pdb = os.path.join(desc, "*.pdb")
        ref_pdb = filtered_af3_score_df.loc[filtered_af3_score_df["af3_models_dir"] == desc, "ref_path"].iloc[0]
        cmd += af3_pdb + " " + ref_pdb + " "
        print(f"  passing_idx={passing_idxs}, rep_idx={rep_idx}, {sort_by}={rep_val:.3f} | {os.path.basename(desc)}")

    filtered_af3_score_df.drop(columns=["_sort_extremum"], inplace=True)
    print("\nPyMOL command:")
    print(cmd)
if save_filtered_df:
    filtered_af3_score_df.to_csv(save_filtered_df_path, index=False)
    print("\n")
    print(f"Dataframe was saved at {save_filtered_df_path}")

In [ ]:
############################################################
### COMPUTE AF3 STATISTICS FROM SCORES DF & DO FILTERING ###
############################################################

### LOAD THE DF & CREATE scaffold_family ###
af3_scores_df = pd.read_csv(os.path.join(IMPORTANT_DFS_DIR, 'filtered_TEMP_modified_af3_i2.csv'))

### PATHS & NUMBER OF AF3 STRUCTURES PER PDB ###
af3_N   = 5    # Number of AF3 predictions
af3_path = f"{AF3_OUT_DIR}i2__holo/"

### TOGGLE: APPLY FILTERS ONLY TO OVER-REPRESENTED CLUSTERS? ###
apply_filters_only_to_large_groups = False  # Set True to filter only groups with ≥ min_group_size examples

### GROUP SIZE THRESHOLD (for over-represented clusters) ###
min_group_size = 80

### DISPLAY CONTROLS ###
show_example = True
n_examples  = 10
sort_by     = "catres_rmsd"
ascending   = True
print_stat  = True

### SAVE FILTERED DF CONTROLS ###
save_filtered_df = True
save_filtered_df_path = os.path.join(IMPORTANT_DFS_DIR, "filtered_TEMP_modified_af3_i2__v2.csv")

### AF3 FILTER OPTIONS ###
majority_threshold  = 0.8

ca_rmsd             = 1.0
catres_rmsd         = 1.0
catres_subset_rmsd  = 1.0
kcx_tips_rmsd       = 1.25
zinc_rmsd           = 1.1
hydroxide_rmsd      = 1.75
subst_rmsd          = 4.0
subst_core_rmsd     = 2
tm_score            = 0.95

chainA_plddt        = 90  # protein
chainB_plddt        = 90  # zinc1
chainC_plddt        = 90  # zinc2
chainD_plddt        = 87.5  # hydroxide OR substrate
chainE_plddt        = 80  # substrate
catres_plddt        = 90
catres_subset_plddt = 90
kcx_tips_plddt      = 90
subst_core_plddt    = 80
hydroxide_plddt     = 80

interface_ptm       = 0.9
ptm_of_protein      = 0.9

### PAE FILTER OPTIONS ###
chainA_chainB_pae_avg = 6.5  # protein - zinc
chainA_chainC_pae_avg = 6.5  # protein - zinc
chainA_chainD_pae_avg = 6.5  # protein - hydroxide
chainA_chainE_pae_avg = 6.5  # protein - substrate
chainB_chainC_pae_avg = 6.0  # zinc - zinc
chainB_chainD_pae_avg = 6.0  # zinc - hydroxide
chainB_chainE_pae_avg = 6.0  # zinc - substrate
chainC_chainD_pae_avg = 6.0  # zinc - hydroxide
chainC_chainE_pae_avg = 6.0  # zinc - substrate
chainD_chainE_pae_avg = 6.0  # hydroxide - substrate

chainA_chainB_pae_min = 1.25  # protein - zinc
chainA_chainC_pae_min = 1.25  # protein - zinc
chainA_chainD_pae_min = 1.25  # protein - hydroxide
chainA_chainE_pae_min = 1.5  # protein - substrate
chainB_chainC_pae_min = 2.0  # zinc - zinc
chainB_chainD_pae_min = 2.0  # zinc - hydroxide
chainB_chainE_pae_min = 6.0  # zinc - substrate
chainC_chainD_pae_min = 1.9  # zinc - hydroxide
chainC_chainE_pae_min = 6.0  # zinc - substrate
chainD_chainE_pae_min = 6.0  # hydroxide - substrate


### ADD BACKBONES BACK INTO FILTERED DF? ###
apply_scaffold_selection = False
# If the column string does NOT contain "_idx_" then the median across "<column>_idx_*" will be computed.
scaffold_selection_rules = [
    # {"column": "subst_rmsd", "order": "asc", "N": 1},
    {"column": "catres_rmsd","order": "asc", "N": 1},
]

### FILTER OPTIONS ###
# OR logic: use a tuple of column names as the key, e.g.: (("ca_rmsd_TMalign", "ca_rmsd"), "<="): val -> means: passes if ca_rmsd_TMalign_idx_i <= val  OR  ca_rmsd_idx_i <= val | Standard single-string keys still work exactly as before.
filters = {
    (("ca_rmsd_TMalign", "ca_rmsd"), "<="):                        ca_rmsd,
    (("catres_rmsd_TMalign", "catres_rmsd"), "<="):                catres_rmsd,
    (("catres_subset_rmsd_TMalign", "catres_subset_rmsd"), "<="):  catres_subset_rmsd,
    (("kcx_tips_rmsd_TMalign", "kcx_tips_rmsd"), "<="):            kcx_tips_rmsd,
    (("zinc_pair_avg_rmsd_TMalign", "zinc_pair_avg_rmsd"), "<="):  zinc_rmsd,
#    (("hydroxide_ion_rmsd_TMalign", "hydroxide_ion_rmsd"), "<="):  hydroxide_rmsd,
    (("subst_rmsd_TMalign", "subst_rmsd"), "<="):                  subst_rmsd,
    (("subst_core_rmsd_TMalign", "subst_core_rmsd"), "<="):        subst_core_rmsd,

    ("tm_score", ">="):        tm_score,
    ("chainA_plddt", ">="):    chainA_plddt,
    ("chainB_plddt", ">="):    chainB_plddt,
    ("chainC_plddt", ">="):    chainC_plddt,
    ("chainD_plddt", ">="):    chainD_plddt,
#    ("chainE_plddt", ">="):    chainE_plddt,
    ("catres_plddt", ">="):    catres_plddt,
    ("catres_subset_plddt", ">="):    catres_subset_plddt,
    ("KCX3_extra_atoms_plddt", ">="): kcx_tips_plddt,
#    ("hydroxide_ion_plddt", ">="): hydroxide_plddt,

    ("iptm", ">="):            interface_ptm,
    ("ptm",  ">="):            ptm_of_protein,

#    ("chainA_chainB_pair_pae_mean", "<="): chainA_chainB_pae_avg,
#    ("chainA_chainC_pair_pae_mean", "<="): chainA_chainC_pae_avg,
#    ("chainA_chainD_pair_pae_mean", "<="): chainA_chainD_pae_avg,
#    ("chainA_chainE_pair_pae_mean", "<="): chainA_chainE_pae_avg,
#    ("chainB_chainC_pair_pae_mean", "<="): chainB_chainC_pae_avg,
#    ("chainB_chainD_pair_pae_mean", "<="): chainB_chainD_pae_avg,
#    ("chainB_chainE_pair_pae_mean", "<="): chainB_chainE_pae_avg,
#    ("chainC_chainD_pair_pae_mean", "<="): chainC_chainD_pae_avg,
#    ("chainC_chainE_pair_pae_mean", "<="): chainC_chainE_pae_avg,
#    ("chainD_chainE_pair_pae_mean", "<="): chainD_chainE_pae_avg,

    ("chainA_chainB_pair_pae_min", "<="): chainA_chainB_pae_min,
    ("chainA_chainC_pair_pae_min", "<="): chainA_chainC_pae_min,
    ("chainA_chainD_pair_pae_min", "<="): chainA_chainD_pae_min,
#    ("chainA_chainE_pair_pae_min", "<="): chainA_chainE_pae_min,
   # ("chainB_chainC_pair_pae_min", "<="): chainB_chainC_pae_min,
   # ("chainB_chainD_pair_pae_min", "<="): chainB_chainD_pae_min,
#    ("chainB_chainE_pair_pae_min", "<="): chainB_chainE_pae_min,
   # ("chainC_chainD_pair_pae_min", "<="): chainC_chainD_pae_min,
#    ("chainC_chainE_pair_pae_min", "<="): chainC_chainE_pae_min,
#    ("chainD_chainE_pair_pae_min", "<="): chainD_chainE_pae_min,
}

### HELPERS FOR OR-GROUP FILTER KEYS ###
def _build_filter_expr(col_key, op, val, i, df_name):
    """Build an eval-able expression. col_key is a str or tuple-of-strs (OR group)."""
    if isinstance(col_key, tuple):
        parts = [f"({df_name}['{c}_idx_{i}'] {op} {val})" for c in col_key]
        return "(" + " | ".join(parts) + ")"
    else:
        return f"({df_name}['{col_key}_idx_{i}'] {op} {val})"

def _filter_label(col_key):
    """Human-readable label for printing."""
    if isinstance(col_key, tuple):
        return " OR ".join(col_key)
    return col_key

### FILTERING FUNCTION (compact but identical behavior, now with OR support) ###
def filtering_df(dataframe_name, filters, target_df, print_stat=True):
    if print_stat:
        print(f'[{dataframe_name}] ({len(target_df)} designs)')
    print_rows, filtered_desc = [], []
    for i in range(af3_N):
        row_bits, exprs = [], []
        for (col, op), val in filters.items():
            expr = _build_filter_expr(col, op, val, i, "target_df")
            exprs.append(expr)
            if print_stat and i == 0:
                label = _filter_label(col)
                row_bits.append(f'# [ {label.ljust(50)} {op.ljust(2)} {str(val).ljust(6)}]: ')
        mask = eval(" & ".join(exprs))
        if print_stat:
            for j, e in enumerate(exprs):
                m = eval(e)
                pct = f"({m.to_list().count(True)/len(target_df)*100:.1f} %)"
                if i == 0:
                    print_rows.append(row_bits[j])
                print_rows[j] += f"{str(m.to_list().count(True)).rjust(6)}  {pct.rjust(8)}   "
        filtered_desc.extend(target_df[mask]["af3_models_dir"].to_list())
        if print_stat:
            pct = f"({mask.sum()/len(target_df)*100:.1f} %)"
            if i == 0:
                print_rows.append(f'# [                             ALL                             ]: {str(mask.sum()).rjust(6)}  {pct.rjust(8)}   ')
            else:
                print_rows[-1] += f"{str(mask.sum()).rjust(6)}  {pct.rjust(8)}   "
    keep = set(d for d in set(filtered_desc) if (filtered_desc.count(d)/af3_N) >= majority_threshold)
    filtered_df = target_df[target_df["af3_models_dir"].isin(keep)]
    if print_stat:
        for line in print_rows: print(line)
        pct = f"({len(filtered_df)/len(target_df)*100:.1f} %)"
        print(f'# [                     MAJORITY VOTE ({int(majority_threshold*100)} %)                    ]: {str(len(filtered_df)).rjust(6)}  {pct.rjust(8)}')
    return filtered_df

### ORIGINAL FILTERING ###
total_unique_scaffolds = af3_scores_df["scaffold_family"].nunique()
before_clusters = pd.to_numeric(af3_scores_df["structure_cluster"], errors="coerce").nunique(dropna=True)
print(f"Total unique scaffold groups before filtering: {total_unique_scaffolds}")
print(f"Unique structure clusters BEFORE filtering: {before_clusters}\n")

if apply_filters_only_to_large_groups:
    # Identify over-represented groups
    group_counts = af3_scores_df["scaffold_family"].value_counts()
    large_groups = group_counts[group_counts >= min_group_size].index
    # Split into large-group subset and the rest
    df_large = af3_scores_df[af3_scores_df["scaffold_family"].isin(large_groups)].copy()
    df_small = af3_scores_df[~af3_scores_df["scaffold_family"].isin(large_groups)].copy()
    # Apply filtering only to the over-represented clusters
    filtered_large = filtering_df("af3_scores_df (large groups)", filters, df_large, print_stat=print_stat)
    # Keep the small-group data unfiltered; combine
    filtered_af3_score_df = pd.concat([filtered_large, df_small]).reset_index(drop=True)
else:
    filtered_af3_score_df = filtering_df("af3_scores_df", filters, af3_scores_df, print_stat=print_stat)

# Print unique scaffold info & counts at this stage
print(f"\nNumber of unique scaffolds (BB) in filtered designs: {filtered_af3_score_df['scaffold_family'].nunique()}")
print(f"Number of unique structure clusters in filtered designs: {pd.to_numeric(filtered_af3_score_df['structure_cluster'], errors='coerce').nunique(dropna=True)}")
print(f"\nNumber of Designs BEFORE Filtering: {len(af3_scores_df)}")
print(f"Number of Designs AFTER  Filtering: {len(filtered_af3_score_df)}")

### ADDITIONAL SCAFFOLD SELECTION OPTIONS ###
if apply_scaffold_selection:
    print("\n### ADDING BACK SCAFFOLDS BASED ON YOUR SELECTION CRITERIA ###")
    extra_selected = []
    present_scaffolds = set(filtered_af3_score_df["scaffold_family"].unique())
    for rule in scaffold_selection_rules:
        base, order, N_keep = rule["column"], rule["order"], rule["N"]
        for scaf, group in af3_scores_df.groupby("scaffold_family"):
            if scaf in present_scaffolds:
                continue
            group = group.copy()
            sort_col = base
            if "_idx_" not in base:
                cols = [c for c in group.columns if c.startswith(base + "_idx_")]
                if not cols:
                    continue
                group["median_" + base] = group[cols].median(axis=1)
                sort_col = "median_" + base
            if len(group) >= N_keep:
                extra_selected.append(group.sort_values(by=sort_col, ascending=(order == "asc")).head(N_keep))
    if extra_selected:
        extra_df = pd.concat(extra_selected).drop_duplicates(subset="af3_models_dir")
        print(f"Unique scaffolds in additional selection: {extra_df['scaffold_family'].nunique()}")
        combined = pd.concat([filtered_af3_score_df, extra_df]).drop_duplicates(subset="af3_models_dir")
        print(f"Designs after additional scaffold selection: {len(combined)}")
        print(f"Unique scaffolds after additional selection: {combined['scaffold_family'].nunique()}")
        filtered_af3_score_df = combined
    else:
        print("No additional designs selected via scaffold rules.")

### SHOW EXAMPLES & SAVE ###
# Build set of passing (af3_models_dir, idx) pairs from the filtering logic
passing_pairs = set()
for i in range(af3_N):
    mask = pd.Series([True] * len(filtered_af3_score_df), index=filtered_af3_score_df.index)
    for (col, op), val in filters.items():
        expr = _build_filter_expr(col, op, val, i, "filtered_af3_score_df")
        mask = mask & eval(expr)
    for desc in filtered_af3_score_df.loc[mask, "af3_models_dir"]:
        passing_pairs.add((desc, i))

# For each design, find the extremum value among passing indices
sort_col_base = sort_by
extremum_values = []
passing_info = {}
for idx, row in filtered_af3_score_df.iterrows():
    desc = row["af3_models_dir"]
    passing_idxs = [i for i in range(af3_N) if (desc, i) in passing_pairs]
    if not passing_idxs:
        extremum_values.append(float('inf') if ascending else float('-inf'))
        passing_info[desc] = ([], None, None)
        continue
    vals = {i: row[f"{sort_col_base}_idx_{i}"] for i in passing_idxs}
    rep_idx = min(vals, key=vals.get) if ascending else max(vals, key=vals.get)
    extremum_values.append(vals[rep_idx])
    passing_info[desc] = (passing_idxs, rep_idx, vals[rep_idx])

# Add passing_idx column (None for designs added via scaffold selection)
if "passing_idx" not in filtered_af3_score_df.columns:
    filtered_af3_score_df = filtered_af3_score_df.copy()
    filtered_af3_score_df["passing_idx"] = filtered_af3_score_df["af3_models_dir"].apply(lambda desc: sorted(passing_info[desc][0]) if desc in passing_info
and passing_info[desc][0] else None)

if show_example:
    filtered_af3_score_df["_sort_extremum"] = extremum_values
    filtered_af3_score_df.sort_values(by="_sort_extremum", ascending=ascending, inplace=True)
    desc_list = filtered_af3_score_df["af3_models_dir"].head(n_examples).to_list()
    cmd = "pymol "
    print(f"\nTop {n_examples} designs sorted by {'min' if ascending else 'max'} {sort_by} among passing indices:\n")
    for desc in desc_list:
        passing_idxs, rep_idx, rep_val = passing_info[desc]
        af3_pdb = os.path.join(desc, "*.pdb")
        ref_pdb = filtered_af3_score_df.loc[filtered_af3_score_df["af3_models_dir"] == desc, "ref_path"].iloc[0]
        cmd += af3_pdb + " " + ref_pdb + " "
        print(f"  passing_idx={passing_idxs}, rep_idx={rep_idx}, {sort_by}={rep_val:.3f} | {os.path.basename(desc)}")

    filtered_af3_score_df.drop(columns=["_sort_extremum"], inplace=True)
    print("\nPyMOL command:")
    print(cmd)
if save_filtered_df:
    filtered_af3_score_df.to_csv(save_filtered_df_path, index=False)
    print("\n")
    print(f"Dataframe was saved at {save_filtered_df_path}")

## IX.D Look at Highly Represented Scaffolds & Optionally Filter

### Optional: Look at Over-Represented Backbones 

In [ ]:
#####################################################################################
### ANALYZE DISTRIBUTION OF HOW MANY ENTRIES (ROWS) PER scaffold_family IN FILTER ###
#####################################################################################

### LOAD DF CSV ###
af3_scores_path = f"{IMPORTANT_DFS_DIR}filtered_TEMP_modified_af3_i2__v2.csv"
filtered_df = pd.read_csv(af3_scores_path)

### GROUP BY SCAFFOLDS & COUNT ###
filtered_df["scaffold_family_basename"] = filtered_df["scaffold_family"].astype(str).apply(os.path.basename)
group_counts = (filtered_df.groupby("scaffold_family").size().reset_index(name="count"))
group_counts["scaffold_family_basename"] = group_counts["scaffold_family"].astype(str).apply(os.path.basename)

### OPTIONAL CONTROLS TO PRINT DESCRIPTIONS FROM GROUPS THAT EXCEED A ROW THRESHOLD ###
# --- Toggle 4A: Print N random descriptions from each group with row_count > row_threshold
show_random_examples = False       # Change to False to disable this feature
row_threshold_for_random = 50      # Only consider groups with > this many rows
random_n = 25                      # Number of random descriptions to print from each group

# --- Toggle 4B: Print top/bottom N from each group by a specific column
show_top_bottom_examples = True    # Change to False to disable this feature
ascending_flag = True              # True => smallest->largest (e.g., "top" = best/lowest)
                                   # False => largest->smallest (e.g., "top" = worst/highest)
top_bottom_n = 1                  # Number to print
sort_column = "ca_rmsd_idx_0"      # Column on which to sort
row_threshold_for_sorted = 5

### SUMMARY STATS ###
num_scaffolds = len(group_counts)
more_than_10 = (group_counts["count"] > 10).sum()
more_than_20 = (group_counts["count"] > 20).sum()
more_than_30 = (group_counts["count"] > 30).sum()

print(f"Total unique scaffolds in filtered set: {num_scaffolds}")
print(f"Scaffolds with more than 10 rows: {more_than_10}")
print(f"Scaffolds with more than 20 rows: {more_than_20}")
print(f"Scaffolds with more than 30 rows: {more_than_30}\n")

### INSPECT & PLOT SCAFFOLDS WITH >1 DESIGN ###
multi_entry_groups = group_counts[group_counts["count"] > 1]
print("Scaffolds with more than 1 row (showing top 20):")
print(
    multi_entry_groups
        .sort_values("count", ascending=False)
        .head(20)[["scaffold_family_basename", "count"]]
        .rename(columns={"scaffold_family_basename":"scaffold_family (basename)"})
)
plt.figure(figsize=(7, 5))
plt.hist(multi_entry_groups["count"], bins=range(1, multi_entry_groups["count"].max() + 2), edgecolor='k')
plt.title("Distribution of scaffold_family Sizes (Groups > 1)")
plt.xlabel("Number of Rows (Designs) in Group")
plt.ylabel("Frequency")
plt.tight_layout()
af3_all_save_path = os.path.join(GRAPHS_DIR, "af3_i2__v2_holo_scaffold_distribution.png")
plt.savefig(af3_all_save_path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {af3_all_save_path}")
plt.show()

### EXECUTE ###
group_counts_sorted = group_counts.sort_values("count", ascending=False)

if show_random_examples:
    print("\n##############################################")
    print("### RANDOM EXAMPLES FROM LARGE SCAFFOLDS  ###")
    print("##############################################")
    # Filter to groups that exceed row_threshold_for_random
    big_groups = group_counts_sorted[group_counts_sorted["count"] > row_threshold_for_random]
    if big_groups.empty:
        print(f"(No scaffold families have > {row_threshold_for_random} rows; nothing to sample.)")
    for _, row in big_groups.iterrows():
        group_name_full = row["scaffold_family"]
        group_name_base = row["scaffold_family_basename"]
        group_size = row["count"]
        subset_df = filtered_df[filtered_df["scaffold_family"] == group_name_full]
        sample_size = min(random_n, group_size)
        sampled_paths = subset_df["af3_models_dir"].sample(n=sample_size, random_state=42).tolist()
        print(f"\n### scaffold_family '{group_name_base}' (n={group_size}) ###")
        print("### RANDOM SAMPLES ###")
        print("pymol " + " ".join(sampled_paths))

if show_top_bottom_examples:
    print("\n################################################################")
    print("### TOP/BOTTOM EXAMPLES BY A SPECIFIC COLUMN FROM BIG GROUPS ###")
    print("################################################################")
    big_groups = group_counts_sorted[group_counts_sorted["count"] > row_threshold_for_sorted]
    if big_groups.empty:
        print(f"(No scaffold families have > {row_threshold_for_sorted} rows; adjust 'row_threshold_for_sorted' or inspect 'group_counts_sorted'.)")
    else:
        for _, row in big_groups.iterrows():
            group_name_full = row["scaffold_family"]
            group_name_base = row["scaffold_family_basename"]
            group_size = row["count"]
            
            subset_df = filtered_df[filtered_df["scaffold_family"] == group_name_full]
            sorted_subset = subset_df.sort_values(by=sort_column, ascending=ascending_flag)
            
            selection_df = sorted_subset.head(top_bottom_n)
            print(f"\n### scaffold_family '{group_name_base}' (n={group_size}) ###")
            print(f"### FIRST {top_bottom_n} by '{sort_column}' (ascending_flag={ascending_flag}) ###")
            print("pymol " + " ".join([desc + "/*.pdb" for desc in selection_df["af3_models_dir"].tolist()]))

### Optional: Decide to Keep All Designs for Any Particular Scaffold? Remove All Designs for Any Particular Scaffold?

List of designs to remove:
group1_pte_pxon_ChenJACS_exact_v0__lig_YYE_0000_EEEE_rotP_1_D__ORI_01_combo2___1_model_1_cfg_False__cfgscale_NA__stepscale_1_500__gamma0_0_100__gamma_min_0_500__jitter_1_000


In [ ]:
############################################################################################
### FILTER TO KEEP ONLY SPECIFIED SCAFFOLD_FAMILY BASENAMES & SAVE TO NEW CSV ###
############################################################################################

### LOAD DF CSV ###
unfiltered_af3_scores_path = f"{IMPORTANT_DFS_DIR}af3_stats_iteration1_apo.csv"
af3_scores_path = os.path.join(IMPORTANT_DFS_DIR, f"filtered_{Path(unfiltered_af3_scores_path).stem}.csv")
filtered_df = pd.read_csv(af3_scores_path)

### INPUT: LIST OF BASENAMES TO KEEP ###
scaffold_families_to_keep = [
    "group1_pte_pxon_ChenJACS_exact_v0__lig_YYE_0000_EEEE_rotP_1_D__ORI_01_combo2___3_model_0_cfg_True__cfgscale_1_500__stepscale_1_500__gamma0_0_700__gamma_min_1_000__jitter_3_000",
    "group2_pte_pxon_ChenJACS_smwTSopt_v0__lig_YYF_0000_EEEE_rotP_1_D__ORI_01_combo1___1_model_0_cfg_True__cfgscale_2_000__stepscale_1_500__gamma0_0_700__gamma_min_0_500__jitter_2_000",
    # add more basenames here
]

### MAKE A BASENAME COLUMN JUST FOR FILTERING ###
filtered_df["scaffold_family_basename"] = filtered_df["scaffold_family"].astype(str).apply(os.path.basename)

### FILTER ###
kept_df = filtered_df[filtered_df["scaffold_family_basename"].isin(scaffold_families_to_keep)].copy()

### DROP THE HELPER COLUMN BEFORE SAVING ###
kept_df = kept_df.drop(columns=["scaffold_family_basename"])

### OUTPUT PATH ###
kept_csv_path = os.path.join(IMPORTANT_DFS_DIR, f"filtered_SCAFFOLD_FAMS_MUST_KEEP_{Path(unfiltered_af3_scores_path).stem}.csv")

### SAVE ###
kept_df.to_csv(kept_csv_path, index=False)

print(f"Subset saved to: {kept_csv_path}")
print(f"Rows before: {len(filtered_df)} | Rows after filtering: {len(kept_df)}")
print(f"Scaffold families kept: {scaffold_families_to_keep}")

### Filter Over-Represented AF3 Scaffolds Down

## IX.E Copy AF3 Filtered Structures & Ref PDBs

In [ ]:
###############################################
### STEP 1: LOAD CSV & PICK COLUMNS OF INTEREST
###############################################

### INPUTS ###
unfiltered_af3_scores_path = f"{IMPORTANT_DFS_DIR}modified_af3_i2.csv"
filtered_df_path = os.path.join(IMPORTANT_DFS_DIR, f"filtered_TEMP_modified_af3_i2__v2.csv")

### PASSING_IDX MODE CONTROLS ###
use_passing_idx = True         # Set True to use passing_idx column, False for default behavior
fallback_on_nan = True         # If True, fall back to metric-based selection when passing_idx is NaN | False = design is excluded

### CONTROLS (ONLY FOR FALLBACK OR IF PASSING_IDX MODE OFF) ###
selection_metric = "ca_rmsd"   # which metric to use
af3_N            = 5           # number of AF3 predictions per design

N_pick     = 5           # how many indices per row to keep (used for fallback or default mode)
pick_order = "asc"       # "asc" = lowest values, "desc" = highest values

### LOAD ###
df = pd.read_csv(filtered_df_path)

### BUILD COLUMN LIST ###
metric_cols = [f"{selection_metric}_idx_{i}" for i in range(af3_N)]
columns_of_interest = ["af3_models_dir"] + [c for c in metric_cols if c in df.columns]
if "ref_path" in df.columns and "ref_path" not in columns_of_interest:
    columns_of_interest.append("ref_path")
if "passing_idx" in df.columns and "passing_idx" not in columns_of_interest:
    columns_of_interest.append("passing_idx")

### SLIMMED DF ###
df_slim = df[columns_of_interest].copy()
print(f"Loaded {len(df)} rows. Kept columns: {columns_of_interest}")

#############################################
### STEP 2: PICK TOP/BOTTOM-N IDX PER ROW ###
#############################################

# Metric columns we want to look at
metric_cols = [c for c in df_slim.columns if c.startswith(f"{selection_metric}_idx_")]

def pick_indices_by_metric(row, n_pick):
    """Pick indices based on metric values (original behavior)."""
    vals = pd.to_numeric(row[metric_cols], errors="coerce")
    if pick_order == "asc":
        chosen = vals.nsmallest(n_pick)
    else:
        chosen = vals.nlargest(n_pick)
    return [c.split("_")[-1] for c in chosen.index.tolist()]

def parse_passing_idx(val):
    """Parse passing_idx column value (handles string representation of list)."""
    if pd.isna(val):
        return None
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            import ast
            parsed = ast.literal_eval(val)
            return parsed if isinstance(parsed, list) else None
        except:
            return None
    return None

# Track statistics
n_used_passing_idx = 0
n_fallback = 0
n_default = 0
total_idx_count = 0
idx_count_distribution = {}  # Track how many designs have N indices

def pick_indices(row):
    global n_used_passing_idx, n_fallback, n_default, total_idx_count
    
    if use_passing_idx:
        passing = parse_passing_idx(row.get("passing_idx"))
        if passing is not None and len(passing) > 0:
            n_used_passing_idx += 1
            total_idx_count += len(passing)
            return [str(i) for i in passing]
        elif fallback_on_nan:
            n_fallback += 1
            indices = pick_indices_by_metric(row, N_pick)
            total_idx_count += len(indices)
            return indices
        else:
            # No fallback, return empty
            return []
    else:
        n_default += 1
        indices = pick_indices_by_metric(row, N_pick)
        total_idx_count += len(indices)
        return indices

# Apply row by row - get raw indices first
raw_indices = df_slim.apply(pick_indices, axis=1)

# Build distribution of index counts
for indices in raw_indices:
    n = len(indices)
    idx_count_distribution[n] = idx_count_distribution.get(n, 0) + 1

# Determine max number of indices across all rows
max_indices = raw_indices.apply(len).max()
idx_cols = [f"idx_to_copy_{i+1}" for i in range(max_indices)]

# Expand into columns with proper formatting
def format_indices(indices):
    formatted = [f"_idx_{i}_model.pdb" if i is not None else pd.NA for i in indices]
    # Pad with NA if fewer indices than max
    formatted.extend([pd.NA] * (max_indices - len(formatted)))
    return formatted

df_slim[idx_cols] = pd.DataFrame(raw_indices.apply(format_indices).tolist(), index=df_slim.index)

# Add basename of `af3_models_dir` as column "bn"
df_slim["bn"] = df_slim["af3_models_dir"].astype(str).map(lambda p: os.path.basename(os.path.normpath(p)))

### SUMMARY STATISTICS ###
print("\n" + "="*60)
print("SELECTION SUMMARY")
print("="*60)
if use_passing_idx:
    print(f"Mode: passing_idx selection (fallback={'ON' if fallback_on_nan else 'OFF'})")
    print(f"  - Used passing_idx:  {n_used_passing_idx:>6} rows")
    print(f"  - Fallback to metric:{n_fallback:>6} rows")
    if not fallback_on_nan:
        n_skipped = len(df_slim) - n_used_passing_idx
        print(f"  - Skipped (no idx):  {n_skipped:>6} rows")
else:
    print(f"Mode: metric-based selection ({selection_metric}, {pick_order}, N={N_pick})")
    print(f"  - Processed:         {n_default:>6} rows")

print("-"*60)
print("INDEX COUNT DISTRIBUTION:")
for n_idx in sorted(idx_count_distribution.keys()):
    count = idx_count_distribution[n_idx]
    pct = count / len(df_slim) * 100
    bar = "█" * int(pct / 2)  # Simple bar chart
    print(f"  {n_idx} idx pass: {count:>6} designs ({pct:>5.1f}%) {bar}")

print("-"*60)
print(f"TOTAL INDICES TO COPY: {total_idx_count}")
print("="*60)

print("\n")
print(df_slim.head())

In [ ]:
#############################################################
### STEP 3: COPY FILES WITH PROGRESS / WARNINGS / TIMING  ###
#############################################################

### INPUT ###
dest_dir = f"{AF3_OUT_DIR}filtered_i2"
os.makedirs(dest_dir, exist_ok=True)

### CONTROLS ###
preserve_metadata = False  # False = faster, True = keeps original timestamps/permissions

### OPTIONAL: copy reference PDBs listed in 'ref_path' into subdir 'ref_pdbs' ###
copy_ref_pdbs = True
ref_subdir = os.path.join(dest_dir, "ref_pdbs")

def should_log(n):
    """Log at 1, 10, 100, 500, then every 1000 until 10k, then every 10k."""
    if n <= 0:
        return False
    if n < 1000:
        return n in (1, 10, 100, 500)
    elif n < 10000:
        return n % 1000 == 0
    else:
        return n % 10000 == 0

def copy_file(src, dst):
    """Copy file with or without metadata based on setting."""
    if preserve_metadata:
        shutil.copy2(src, dst)
    else:
        shutil.copy(src, dst)

if copy_ref_pdbs and "ref_path" in df_slim.columns:
    ref_list = [str(p) for p in df_slim["ref_path"].tolist() if pd.notna(p) and str(p).strip()]
    all_exist = all(os.path.isfile(p) for p in ref_list)
    if not ref_list:
        print("[WARN] Skipping reference PDB copy: 'ref_path' column is present but empty.")
    elif not all_exist:
        print("[WARN] Skipping reference PDB copy: one or more 'ref_path' paths are missing.")
    else:
        os.makedirs(ref_subdir, exist_ok=True)
        seen = set()
        ref_jobs = []
        for p in ref_list:
            if p not in seen:
                seen.add(p)
                ref_jobs.append((p, os.path.join(ref_subdir, os.path.basename(p))))

        print(f"Copying {len(ref_jobs)} reference PDB(s) into: {ref_subdir}")
        r_start = time.time()
        r_count = 0
        r_errors = 0
        for src, dst in ref_jobs:
            try:
                copy_file(src, dst)
                r_count += 1
            except Exception:
                r_errors += 1
            
            if should_log(r_count) or r_count == len(ref_jobs):
                r_elapsed = time.time() - r_start
                r_avg = r_elapsed / r_count
                r_remaining = r_avg * (len(ref_jobs) - r_count)
                print(f"  [{r_count}/{len(ref_jobs)}] {r_elapsed:.1f}s elapsed | ETA: {r_remaining:.1f}s")
        
        r_elapsed_total = time.time() - r_start
        print(f"  Reference PDBs: {r_count}/{len(ref_jobs)} copied in {r_elapsed_total:.1f}s | Errors: {r_errors}")

### LOGIC ###
if "bn" not in df_slim.columns:
    df_slim["bn"] = df_slim["af3_models_dir"].astype(str).map(lambda p: os.path.basename(os.path.normpath(p)))

copy_jobs = [
    (os.path.join(row["af3_models_dir"], f"{row['bn']}{row[col]}"), 
     os.path.join(dest_dir, f"{row['bn']}{row[col]}"))
    for _, row in df_slim.iterrows()
    for col in idx_cols
    if pd.notna(row[col])
]

total_jobs = len(copy_jobs)
print(f"\nCopying {total_jobs} files into: {dest_dir}")

existing = sum(1 for _, dst in copy_jobs if os.path.exists(dst))
if existing:
    print(f"[WARN] {existing} files already exist (will be overwritten)")

start = time.time()
count = 0
errors = 0
missing = 0

for src, dst in copy_jobs:
    try:
        if not os.path.isfile(src):
            missing += 1
            continue
        copy_file(src, dst)
        count += 1
    except Exception:
        errors += 1

    processed = count + missing + errors
    if should_log(processed):
        elapsed = time.time() - start
        avg = elapsed / processed
        remaining = avg * (total_jobs - processed)
        print(f"  [{processed}/{total_jobs}] {elapsed:.1f}s elapsed | ETA: {remaining:.1f}s")

elapsed_total = time.time() - start

### FINAL SUMMARY ###
print("\n" + "="*60)
print("COPY SUMMARY")
print("="*60)
print(f"  Successfully copied: {count:>6} files")
print(f"  Missing sources:     {missing:>6} files")
print(f"  Copy errors:         {errors:>6} files")
print(f"  Total time:          {elapsed_total:>6.1f} seconds")
print(f"  Average rate:        {count/elapsed_total if elapsed_total > 0 else 0:>6.1f} files/s")
print("="*60)

# **X. Alignment**

In [ ]:
##############################################
### GENERATE COMMANDS FOR PDB ALIGNMENT    ###
##############################################

command_name = "align_af3_predictions"

### INPUTS ###
input_pdb_dir       = f"{AF3_OUT_DIR}filtered_i2/"
ref_pdbs_to_match   = f"{AF3_OUT_DIR}filtered_i2/ref_pdbs/"

### OUTPUT DIR ###
align_out_dir       = f"{input_pdb_dir}alignment/"

### REGEX PATTERN TO STRIP SUFFIX FROM INPUT PDB FILENAMES ###
strip_pattern       = re.compile(r'_af3i2_idx_\d+_model')

### PREFIX-SPECIFIC CATRES SUBSETS ###
# JSON dict mapping prefix patterns to catres_subset values. Use None for default behavior (all catres from REMARK 666)
prefix_catres_map = {   # Example: {"design_A": "1,3,5,6", "design_B": "1,2,3,4,5,6,7,8,9,10"}
}
default_catres_subset = "1,2,3,4,5,6"  # Used if no prefix match, or None for all

### PTM SPECIFICATIONS ###
ptm_from_remark666 = ["A/LYS/3:KCX"] # List of PTM specs for NCAA handling, or None | Example: ["A/LYS/6:KCX", "A/CYS/3:CSO-OD"]

### ALIGNMENT THRESHOLDS ###
outlier_threshold       = 0.5       # RMSD threshold for outlier removal (Å)
convergence_threshold   = 0.001     # Convergence threshold (Å)
max_iterations          = 50        # Max alignment iterations
min_residues            = 10        # Min residues to keep during outlier removal

### SIDECHAIN OPTIMIZATION ###
no_sidechain_opt        = False     # True = disable sidechain optimization entirely
sidechain_cycles        = 2         # Number of global optimization cycles
sidechain_coarse_grid   = 20.0      # Phase 3 global-search step (degrees). Smaller = more thorough but slower
sidechain_fine_grid     = 1.0       # Phase 4 base step (degrees); refinement sequence = [×10, ×5, ×2, ×1] of this
sp3_angle_flexibility   = False     # Enable SP3 C-C bond angle flexibility
sp3_angle_tolerance     = 3.0       # SP3 angle tolerance (degrees) | unused when flexibility off

### CLASH-AWARE OPTIMIZATION ###
apply_clash_penalty     = True      # True = optimize against (rmsd + clash_weight * soft_clash). REF-confirmed close contacts auto-exempted
clash_cutoff            = 2.0       # Mobile hard-clash distance (Å) — pairs below this contribute to soft_clash unless exempted
clash_cutoff_ref        = 3.5       # Ref-contact distance (Å) — pairs in REF within this become exempt (catches H-bonds, metals, PTM bonds)
clash_weight            = 1.0       # Weight on soft_clash term in cost (higher = clash avoidance more aggressive)

### REPRODUCIBILITY ###
seed                    = None      # Seed for random + numpy.random (set to None for non-deterministic)

### WINNER SELECTION ###
winner_threshold        = 0.05       # Tiebreaker threshold (Å)

### OUTPUT OPTIONS ###
keep_all                = False     # Keep all strategy outputs (not just winner)
save_csv                = False     # Save metrics CSV file
verbose                 = True      # Verbose output

### MATCHING OPTIONS ###
max_usages_per_ref      = 5

### EXECUTION OPTIONS ###
dry_run                 = False     # True = generate commands only, don't submit
filter_existing         = False     # True = skip predictions with existing output

### CONTAINER & SCRIPT ###
apptainer               = APPTAINER
align_script            = f"{SPECIAL_SCRIPTS_DIR}general_utils/align_prediction_to_ref_pdb_and_copy_lig.py"

### HELPER FUNCTION: Match prefix to catres_subset ###
def get_catres_for_prefix(pdb_name, prefix_map, default):
    """Return catres_subset for a given PDB name based on prefix matching."""
    for prefix, catres in prefix_map.items():
        if pdb_name.startswith(prefix):
            return catres
    return default

### REFERENCE PDB PROCESSING ###
ref_pdb_files = sorted(glob.glob(os.path.join(ref_pdbs_to_match, '*.pdb')))
ref_pdb_map = {os.path.basename(p).replace('.pdb', ''): p for p in ref_pdb_files}
ref_pdb_usage = Counter()
print(f"[INFO] Found {len(ref_pdb_map)} reference PDBs")

### INPUT PDBS ###
input_pdb_files = sorted(glob.glob(os.path.join(input_pdb_dir, '*.pdb')))
print(f"[INFO] Found {len(input_pdb_files)} input PDBs")

### PRINT CONFIGURATION ###
print(f"\n{'='*60}")
print("ALIGNMENT CONFIGURATION")
print(f"{'='*60}")
print(f"Alignment thresholds:")
print(f"  outlier_threshold:     {outlier_threshold} Å")
print(f"  convergence_threshold: {convergence_threshold} Å")
print(f"  max_iterations:        {max_iterations}")
print(f"  min_residues:          {min_residues}")
print(f"\nSidechain optimization:")
if no_sidechain_opt:
    print(f"  DISABLED")
else:
    print(f"  sidechain_cycles:      {sidechain_cycles}")
    print(f"  sidechain_coarse_grid: {sidechain_coarse_grid}°")
    print(f"  sidechain_fine_grid:   {sidechain_fine_grid}° (→ [{sidechain_fine_grid*10:g}, {sidechain_fine_grid*5:g}, {sidechain_fine_grid*2:g}, {sidechain_fine_grid:g}])")
    print(f"  sp3_angle_flexibility: {sp3_angle_flexibility}")
    if sp3_angle_flexibility:
        print(f"  sp3_angle_tolerance:   ±{sp3_angle_tolerance}°")
print(f"\nClash-aware optimization:")
if apply_clash_penalty:
    print(f"  ENABLED")
    print(f"  clash_cutoff:          {clash_cutoff} Å (mobile)")
    print(f"  clash_cutoff_ref:      {clash_cutoff_ref} Å (REF auto-exempt)")
    print(f"  clash_weight:          {clash_weight}")
else:
    print(f"  DISABLED")
print(f"\nReproducibility:")
print(f"  seed:                  {seed}")
print(f"\nOutput options:")
print(f"  keep_all:              {keep_all}")
print(f"  save_csv:              {save_csv}")
print(f"  winner_threshold:      {winner_threshold} Å")
if prefix_catres_map:
    print(f"\nPrefix-specific catres_subset:")
    for prefix, catres in prefix_catres_map.items():
        print(f"  '{prefix}': {catres}")
    print(f"  default: {default_catres_subset}")
print(f"{'='*60}\n")

### GENERATE COMMANDS ###
os.makedirs(align_out_dir, exist_ok=True)
command_file_path = os.path.join(CMDS_DIR, command_name)
commands = []
unmatched = []
catres_usage = Counter()  # Track catres_subset usage

for pdb_path in input_pdb_files:
    pdb_name = os.path.basename(pdb_path).replace('.pdb', '')
    stripped_name = strip_pattern.sub('', pdb_name)
    if filter_existing:       # Skip if output already exists
        expected_output = os.path.join(align_out_dir, f"{pdb_name}_aligned.pdb")
        if os.path.exists(expected_output):
            continue
    matching_ref = None       # Match to reference
    if stripped_name in ref_pdb_map and ref_pdb_usage[stripped_name] < max_usages_per_ref:
        matching_ref = ref_pdb_map[stripped_name]
        ref_pdb_usage[stripped_name] += 1
        if ref_pdb_usage[stripped_name] >= max_usages_per_ref:
            del ref_pdb_map[stripped_name]
    if matching_ref:
        # Get catres_subset for this file
        catres_subset = get_catres_for_prefix(pdb_name, prefix_catres_map, default_catres_subset)
        if catres_subset:
            catres_usage[catres_subset] += 1
        # Build command
        cmd = (f"{apptainer} {align_script} --ref_pdb {matching_ref} --pdb_for_alignment {pdb_path} --output_dir {align_out_dir}")
        if catres_subset:           # Catres subset
            cmd += f" --catres_subset {catres_subset}"
        if ptm_from_remark666:           # PTM specs
            for ptm_spec in ptm_from_remark666:
                cmd += f" --ptm_from_remark666 {ptm_spec}"
        # Alignment thresholds
        cmd += f" --outlier_threshold {outlier_threshold}"
        cmd += f" --convergence_threshold {convergence_threshold}"
        cmd += f" --max_iterations {max_iterations}"
        cmd += f" --min_residues {min_residues}"

        # Sidechain optimization
        if no_sidechain_opt:
            cmd += " --no_sidechain_opt"
        else:
            cmd += f" --sidechain_cycles {sidechain_cycles}"
            cmd += f" --sidechain_coarse_grid {sidechain_coarse_grid}"
            cmd += f" --sidechain_fine_grid {sidechain_fine_grid}"
            if sp3_angle_flexibility:
                cmd += " --sp3_angle_flexibility"
                cmd += f" --sp3_angle_tolerance {sp3_angle_tolerance}"

        # Clash-aware optimization
        if apply_clash_penalty:
            cmd += " --apply_clash_penalty"
            cmd += f" --clash_cutoff {clash_cutoff}"
            cmd += f" --clash_cutoff_ref {clash_cutoff_ref}"
            cmd += f" --clash_weight {clash_weight}"

        # Reproducibility
        if seed is not None:
            cmd += f" --seed {seed}"

        # Winner selection
        cmd += f" --winner_threshold {winner_threshold}"
        if keep_all:           # Output options
            cmd += " --keep_all"
        if save_csv:
            cmd += " --save_csv"
        if verbose:
            cmd += " --verbose"
        commands.append(cmd)
    else:
        unmatched.append(pdb_name)
commands.sort()   # Sort and write commands
with open(command_file_path, 'w') as cmd_file:
    for cmd in commands:
        cmd_file.write(cmd + "\n")

### JOB PARAMETERS ###
time                = '00:15:00'
cores               = '1'
memory              = '3g'
queue               = 'cpu'
cmds_per_job        = 3

### SUMMARY ###
job_name = os.path.basename(command_file_path)
submit_file = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs = (len(commands) + cmds_per_job - 1) // cmds_per_job

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Commands file:        {command_file_path}")
print(f"Total commands:       {len(commands)}")
print(f"Unmatched predictions: {len(unmatched)}")
print(f"Job name:             {job_name}")
print(f"Number of jobs:       {num_jobs}")
if catres_usage:
    print(f"\nCatres subset usage:")
    for catres, count in catres_usage.most_common():
        print(f"  '{catres}': {count} files")
print(f"{'='*60}\n")

if unmatched and len(unmatched) <= 10:
    print(f"[WARNING] Unmatched: {unmatched}")
elif unmatched:
    print(f"[WARNING] First 10 unmatched: {unmatched[:10]}")

### SUBMIT JOBS ###
if not dry_run and commands:
    nb.submit_array_job(command_file_path, time, cores, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue)
elif dry_run:
    print("[DRY RUN] Commands generated but not submitted")
else:
    print("[WARNING] No commands to submit")

**NOTE:** Copied everything into `for_redesign`

# **XI. LigandMPNN Redesign (RFd3 Designs + AF3 Preds)**

In [ ]:
#############################################################
### XI: LIGANDMPNN REDESIGN (orchestrated, 1 cmd/PDB) ###
#############################################################
# Redesign each input structure with LigandMPNN, holding the catalytic site
# fixed. One self-contained command per input PDB, through
# Scripts/mpnn_relevant_utils/design_orchestrator.py:
#   PRE  : fix the REMARK 666 catalytic residues; omit Met at residue 1;
#          conserve designable sidechains that H-bond the active site
#          (rolled per combination, so the second shell varies between designs).
#   MPNN : one LigandMPNN run per temperature combination, via run_ligandmpnn.py,
#          so fixed residues keep their exact input coordinates.
#   POST : flatten to one directory of packed PDBs; protonate (holo with the
#          ligand .params), restoring the input's catalytic tautomers; restore
#          REMARK 666, write REMARK 668 (+ PTM) and DESIGN_PATH.
# Inputs are the reference PDBs plus the AF3 predictions that passed.

### INPUTS ###
input_pdb_structures_dir = f"{AF3_OUT_DIR}filtered_i2/for_redesign/"
pdb_glob                 = "*.pdb"

### LIGAND / PTM ###
lig_params = f"{PARAMS_DIR}YYE.params"   # None -> apo protonation
ptm_spec   = "A/LYS/3:KCX"                # REMARK 668 PTM annotation; "" -> none

### OUTPUTS ###
redesign_output_dir = f"{REDESIGN_OUT_DIR}i2/"   # <dir>/<input stem>/

### MPNN PARAMETERS (universal; a combo overrides only the keys it names) ###
model_type             = "ligand_mpnn"
omit_aa                = "CX"
bias_AA                = "K:-0.5,R:-0.75,E:0.75,D:0.75"
mpnn_batch_size        = 1
pack                   = 1
repack_everything      = 0
sc_num_denoising_steps = 3
use_side_chain_context = 1

### PER-COMBO SWEEP (75 designs per input structure) ###
combos = [
    {"number_of_batches": 25, "temperature": 0.1},
    {"number_of_batches": 25, "temperature": 0.2},
    {"number_of_batches": 25, "temperature": 0.3},
]

### PRE-PROCESSING TOGGLES ###
fix_remark666_catres = True
omit_nterm_met       = True

### H-BOND SIDECHAIN CONSERVATION ###
conserve_hbonds          = True
conserve_prob            = 0.8
conserve_all_or_none     = False
conserve_seed            = None    # int to replay a roll; None -> generated and printed
conserve_anchors         = "ligand,catalytic,user_fixed"
conserve_hbond_max_dist  = 3.9
conserve_hbond_max_angle = 90
conserve_keep_clashing   = False

### PROTONATION / OUTPUT KNOBS ###
protonate          = True
copy_input         = True
transfer_remarks   = True
design_path_remark = True
keep_intermediates = False

### RE-RUN SAFETY ###
force_overwrite = False
DRY_RUN         = False

### MODEL WEIGHTS ###
ligandmpnn_weights_dir   = os.environ.get("LIGANDMPNN_WEIGHTS", "")
ligandmpnn_checkpoint    = "ligandmpnn_v_32_010_25.pt"
ligandmpnn_sc_checkpoint = "ligandmpnn_sc_v_32_002_16.pt"

### CONSTANTS ###
orchestrator = f"{SPECIAL_SCRIPTS_DIR}mpnn_relevant_utils/design_orchestrator.py"

### SANITY CHECKS ###
if not Path(orchestrator).is_file():
    raise FileNotFoundError(f"orchestrator not found: {orchestrator}")
if not Path(input_pdb_structures_dir).is_dir():
    raise FileNotFoundError(f"input_pdb_structures_dir not found: {input_pdb_structures_dir}")
if lig_params and not Path(lig_params).is_file():
    raise FileNotFoundError(f"lig_params not found: {lig_params}")
Path(redesign_output_dir).mkdir(parents=True, exist_ok=True)

### QUICK LOGIC ###
input_pdbs = sorted(glob.glob(os.path.join(input_pdb_structures_dir, pdb_glob)))
if not input_pdbs:
    raise FileNotFoundError(f"No PDBs matched {os.path.join(input_pdb_structures_dir, pdb_glob)}")
designs_per_input = sum(int(c["number_of_batches"]) * mpnn_batch_size for c in combos)

### BUILD COMMANDS ###
def combo_to_run(combo: dict) -> str:
    return ";".join(f"{k}={v}" for k, v in combo.items())

commands, skipped_existing = [], []
for pdb_file in input_pdbs:
    stem = Path(pdb_file).stem
    out_dir = os.path.join(redesign_output_dir, stem)
    if not force_overwrite and Path(out_dir).is_dir() and list(Path(out_dir).glob("*.pdb")):
        skipped_existing.append(stem)
        continue

    parts = ["python", orchestrator,
             "--pdb_path", pdb_file,
             "--out_folder", out_dir,
             "--model_type", model_type,
             "--batch_size", str(mpnn_batch_size),
             "--pack_side_chains", str(pack),
             "--repack_everything", str(repack_everything),
             "--sc_num_denoising_steps", str(sc_num_denoising_steps),
             "--ligand_mpnn_use_side_chain_context", str(use_side_chain_context),
             "--omit_AA", omit_aa,
             "--bias_AA", bias_AA]
    if ligandmpnn_weights_dir:
        parts += [f"--checkpoint_{model_type}", f"{ligandmpnn_weights_dir}/{ligandmpnn_checkpoint}"]
        if pack:
            parts += ["--checkpoint_path_sc", f"{ligandmpnn_weights_dir}/{ligandmpnn_sc_checkpoint}"]
    if not fix_remark666_catres:
        parts += ["--no_fix_remark666_catres"]
    if not omit_nterm_met:
        parts += ["--no_omit_nterm_met"]
    if conserve_hbonds:
        parts += ["--conserve_hbonds",
                  "--conserve_hbond_prob", str(conserve_prob),
                  "--conserve_anchors", conserve_anchors,
                  "--conserve_hbond_max_dist", str(conserve_hbond_max_dist),
                  "--conserve_hbond_max_angle", str(conserve_hbond_max_angle)]
        if conserve_all_or_none:
            parts += ["--conserve_hbond_all_or_none"]
        if conserve_seed is not None:
            parts += ["--conserve_seed", str(conserve_seed)]
        if conserve_keep_clashing:
            parts += ["--conserve_keep_clashing"]
    if not protonate:
        parts += ["--no_protonate"]
    elif lig_params:
        parts += ["--ligand_params", lig_params]
    if ptm_spec:
        parts += ["--ptm", ptm_spec]
    if not copy_input:
        parts += ["--no_copy_input_structure"]
    if not transfer_remarks:
        parts += ["--no_transfer_remarks"]
    if not design_path_remark:
        parts += ["--no_design_path_remark"]
    if keep_intermediates:
        parts += ["--keep_intermediates"]
    if DRY_RUN:
        parts += ["--dry_run"]
    for c in combos:
        parts += ["--run", combo_to_run(c)]

    commands.append(" ".join(shlex.quote(x) for x in parts))

if skipped_existing:
    print(f"Skipped {len(skipped_existing)} PDB(s) with existing outputs "
          f"(force_overwrite = True to re-run), e.g. {skipped_existing[:3]}")
if not commands:
    raise RuntimeError("No commands generated; every input was skipped by re-run protection.")

### WRITE COMMANDS FILE ###
commands_file = f"{CMDS_DIR}ligandmpnn_redesign_i2_all_pdbs"
with open(commands_file, "w") as f:
    f.write("\n".join(commands) + "\n")
command_count = len(commands)

print(f"Inputs            = {command_count} PDB(s)")
print(f"Designs per input = {designs_per_input} across {len(combos)} combo(s)")
print(f"Total designs     = {command_count * designs_per_input}")
print(f"Protonation       =", ("holo" if lig_params else "apo") if protonate else "off")
print(f"\nCommands File:\n{commands_file}")
print(f"\n# example command:\n{commands[0]}")

### SETUP BATCH JOBS ###
qtime        = '01:30:00'
cmds_per_job = 1
cores        = '1'
memory       = '8g'
queue        = 'cpu'
job_name     = os.path.basename(commands_file)
submit_file  = f'{SUBMIT_DIR}{job_name}.sh'
num_jobs     = math.ceil(command_count / cmds_per_job)
print(f"\nNumber of Jobs = {num_jobs}   Job Name = {job_name}")
print(f"\nNavigate here:\ncd {redesign_output_dir}\n")
nb.submit_array_job(commands_file, qtime, cores, job_name, memory, submit_file,
                    LOGS_DIR, num_jobs, cmds_per_job, queue)


# **XII. Deduplication**

In [ ]:
###############################################################
### DEDUPLICATE REDESIGNED PDBs BY SCAFFOLD FAMILY SEQUENCE ###
###############################################################

### INPUTS ###
redesign_dir_of_subdirs = f"{REDESIGN_OUT_DIR}i2/"
pdb_glob              = "*.pdb"
protein_chains        = ["A"]

### DEDUPLICATION OPTIONS ###
duplicates_subdir = "duplicates"  # duplicate PDBs are moved into each source folder's subdir
workers           = 64
progress_every    = 1000
dry_run           = False           # keep True for first pass; set False to actually move duplicates
verbose           = True
print_moves       = False          # True prints every duplicate move; otherwise prints a short preview
random_seed       = None           # None = random tie-breaking; set an int for reproducible random choices

### OUTPUTS ###
report_csv         = f"{IMPORTANT_DFS_DIR}deduplicate_i2_designs.csv"

### CONSTANTS ###
container_path       = APPTAINER
deduplicate_script   = f"{SPECIAL_SCRIPTS_DIR}general_utils/deduplicate_designs.py"

### SANITY CHECKS ###
if not Path(redesign_dir_of_subdirs).is_dir():
    raise FileNotFoundError(f"redesign_dir_of_subdirs not found: {redesign_dir_of_subdirs}")
if not Path(deduplicate_script).is_file():
    raise FileNotFoundError(f"deduplicate_designs.py not found: {deduplicate_script}")
Path(IMPORTANT_DFS_DIR).mkdir(parents=True, exist_ok=True)

### BUILD COMMAND ###
cmd = [
    "singularity", "exec", container_path,
    "python", deduplicate_script,
    "--design_dir", redesign_dir_of_subdirs,
    "--pdb_glob", pdb_glob,
    "--duplicates_subdir", duplicates_subdir,
    "--workers", str(workers),
    "--progress_every", str(progress_every),
    "--report_csv", report_csv,
]
if protein_chains:
    cmd += ["--protein_chains"] + protein_chains
if dry_run:
    cmd.append("--dry_run")
if verbose:
    cmd.append("--verbose")
if print_moves:
    cmd.append("--print_moves")
if random_seed is not None:
    cmd += ["--random_seed", str(random_seed)]

command = " ".join(shlex.quote(str(x)) for x in cmd)

### PRINT COMMAND TO SUBMIT ###
print("SUBMIT THIS:")
print(command)
print("")

if dry_run:
    print("[DRY RUN] Command will only preview duplicate moves. Set dry_run = False for real moves.")
print("REPORT CSV:")
print(report_csv)
print("\n[NOT EXECUTED] Copy/paste or submit the printed command when ready.")

# **XIII. Holo-AF3 (without tags)**

## XIII.A.1 Execute AF3 Commands

**NOTE:** At this point it is paramount that you go and make SMILES files for your ligand(s) and cofactor(s). Be careful, remember that these are ML models so they are biased by things they've seen during training - this means that it may be more comfortable with something like a phosphodiester transition state analog for a prediction than your tetrahedral carbon intermediate. This is worth thinking about. \

You should be able to run a command like this: `obabel -ipdb example_lig.pdb -osmi -O example_lig.smi -d -x Zn`
(Open Babel comes with `Environment/zinc_hydro.yml`; `Scripts/env_config.py` reports where it resolved.) \

This website is super helpful for checking your SMILES file or for editing it or for crafting it from scratch: https://www.cheminfo.org/flavor/malaria/Utilities/SMILES_generator___checker/index.html

In [ ]:
################################                                                                                                                                               
### Make AF3 input json file ###                                                                                                                                               
################################                                                                                                                                               

### SHARED INPUTS ###                                                                                                                                                          
input_pdb_path          = f"{REDESIGN_OUT_DIR}i2/"        
input_pdb_protein_chain = ["A"]                                                                                                                                                

af3_output_dir    = f"{AF3_OUT_DIR}i3__holo/"                                                                                                                                         
num_input_per_run = 50                                          
                                                                                                                                                                                
### SHARED OPTIONAL LIGAND FLAG ###                                                                                                                                            
include_ligands_if_present = True
input_ligand_dic           = {                                                                                                                                                 
    "B": {"ccdCodes": "['ZN']"},                                                                                                                                               
    "C": {"smiles": "[Zn+2].[OH-]"},                                                                                                                                               
    "D": {"smiles": "CCOP(=O)(OCC)OC1=CC=C(C=C1)[N+](=O)[O-]"},                                                                                                          
}   # leave empty {} or set to None to omit ligand args                                                                                                                        
                                                                                                                                                                                
### SHARED OPTIONAL PTM FLAG ###                                                                                                                                               
include_post_translational_mods_if_present = True                                                                                                                              
ptm_specs = ["A/LYS/3:KCX"]                                     
                                                                                                                                                                                
### SHARED OPTIONAL TERMINUS TAG FLAGS ###                                                                                                                                     
n_terminus_tag = ""                                                                                                                                                         
c_terminus_tag = ""                                                                                                                                                 
                                                                                                                                                                                
### SHARED OPTIONAL FLAGS ###                                                                                                                                                  
output_suffix              = "_af3i3"                                                                                                                                          
check_made_output          = True                               
cleanup_incomplete_outputs = True                                                                                                                                              

### SHARED OPTIONAL RECURSIVE SEARCH FLAGS ###                                                                                                                                 
recursive      = True # False                                           
max_depth      = None # None                                                                                                                                                          
specific_depth = 1    # None                                              
                                                                                                                                                                                
### SHARED SEED OPTIONS ###
no_random_seed = False                                                                                                                                                         
base_seed      = None

### GROUP DEFINITIONS (set to None for single-group behavior) ###
# Each group can override: pdb_prefix, input_ligand_dic, ptm_specs, output_suffix. Any key omitted inherits the shared default above                                                                                                                            
groups = [                                                                                                                                                                     
]                                                                                                                                                                              

### CONSTANTS ###                                                                                                                                                              
apptainer     = APPTAINER
script        = f"{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/make_af3_json_input.py"                                                                             
af3_json_path = os.path.join(AF3_JSON_DIR, os.path.relpath(af3_output_dir, AF3_OUT_DIR))                                                                                       
os.makedirs(af3_output_dir, exist_ok=True)                                                                                                                                     
os.makedirs(af3_json_path, exist_ok=True)                                                                                                                                      
                                                                                                                                                                                
### BUILD COMMAND(S) DYNAMICALLY ###                                                                                                                                           
def build_af3_command(ligand_dic, ptm_list, suffix, pdb_prefix_list=None):
    """Build a single make_af3_json_input.py command string."""                                                                                                                
    cmd_parts = [                                                                                                                                                              
        f"{apptainer} {script}",                                                                                                                                               
        f"--pdb_path {input_pdb_path}",                                                                                                                                        
        f"--pdb_chain {' '.join(input_pdb_protein_chain)}",                                                                                                                    
        f"--json_path {af3_json_path}",                         
        f"--num_input_per_run {num_input_per_run}",                                                                                                                            
        f"--output_path {af3_output_dir}",                                                                                                                                     
    ]
    if ligand_dic:                                                                                                                                                             
        ligand_chains = list(ligand_dic.keys())                                                                                                                                
        ligand_types  = []
        ligand_ids    = []                                                                                                                                                     
        for v in ligand_dic.values():                           
            if len(v) != 1:                                                                                                                                                    
                raise ValueError("Each ligand spec should have exactly one key indicating type (e.g., 'smiles').")                                                             
            ltype, lid = next(iter(v.items()))                                                                                                                                 
            ligand_types.append(ltype)                                                                                                                                         
            ligand_ids.append(lid)                                                                                                                                             
        cmd_parts.append(f"--ligand_chain {' '.join(ligand_chains)}")                                                                                                          
        cmd_parts.append(f"--ligand_type {' '.join(ligand_types)}")
        cmd_parts.append(f'--ligand_id "{" ".join(ligand_ids)}"')                                                                                                              
    if ptm_list:                                                
        cmd_parts.append("--ptm_from_remark666 " + " ".join([f'"{s}"' for s in ptm_list]))                                                                                     
    if n_terminus_tag:                                                                                                                                                         
        cmd_parts.append(f"--n_terminus_tag {n_terminus_tag}")
    if c_terminus_tag:                                                                                                                                                         
        cmd_parts.append(f"--c_terminus_tag {c_terminus_tag}")  
    if suffix:                                                                                                                                                                 
        cmd_parts.append(f"--output_suffix {suffix}")
    if check_made_output:                                                                                                                                                      
        cmd_parts.append("--check_made_output")                 
    if check_made_output and cleanup_incomplete_outputs:
        cmd_parts.append("--cleanup_incomplete_outputs")                                                                                                                       
    if recursive:
        cmd_parts.append("--recursive")                                                                                                                                        
    if max_depth is not None:                                   
        cmd_parts.append(f"--max_depth {max_depth}")
    if specific_depth is not None:                                                                                                                                             
        cmd_parts.append(f"--specific_depth {specific_depth}")
    if no_random_seed:                                                                                                                                                         
        cmd_parts.append("--no_random_seed")                    
    if base_seed is not None:                                                                                                                                                  
        cmd_parts.append(f"--base_seed {base_seed}")
    if pdb_prefix_list:                                                                                                                                                        
        cmd_parts.append(f"--pdb_prefix {' '.join(pdb_prefix_list)}")
    return " ".join(cmd_parts)                                                                                                                                                 

# Resolve shared defaults for ligands/PTMs                                                                                                                                     
shared_ligand_dic = input_ligand_dic if (include_ligands_if_present and input_ligand_dic) else {}
shared_ptm_specs  = ptm_specs if (include_post_translational_mods_if_present and ptm_specs) else []                                                                            
                                                                                                                                                                                
if groups:                                                                                                                                                                     
    # Multi-group mode: each group inherits shared defaults unless overridden                                                                                                  
    print(f"Generating {len(groups)} group commands:\n")                                                                                                                       
    for i, grp in enumerate(groups, 1):                                                                                                                                        
        cmd = build_af3_command(                                                                                                                                               
            ligand_dic      = grp.get("input_ligand_dic", shared_ligand_dic),                                                                                                  
            ptm_list        = grp.get("ptm_specs", shared_ptm_specs),
            suffix          = grp.get("output_suffix", output_suffix),                                                                                                         
            pdb_prefix_list = grp.get("pdb_prefix", None),      
        )                                                                                                                                                                      
        print(f"### Group {i}: {grp.get('name', f'group_{i}')} ###")
        print(cmd)                                                                                                                                                             
        print()                                                                                                                                                                
else:                                                                                                                                                                          
    # Single-group fallback (original behavior)                                                                                                                                
    cmd = build_af3_command(ligand_dic = shared_ligand_dic, ptm_list   = shared_ptm_specs, suffix     = output_suffix,                                                                                                                                            )
    print(cmd) 

In [ ]:
###############                                                                                                                                                                          
### Run AF3 ###                                                                                                                                                                          
###############                                                                                                                                                                          
                                                                                                                                                                                        
### INPUTS ###
af3_output_dir = f"{AF3_OUT_DIR}i3__holo/"
cmds_base_name = f"AF3_i3_holo"

### PARAMETERS ###
N_total_structures = 5  # Total number of structures you want per target (N)
mode = "one_seed"        # "one_seed" -> N diffusion samples from 1 seed | "n_seeds" -> 1 sample from each of N seeds

### CONSTANTS ###
apptainer       = AF3_SIF
script          = AF3_RUNNER
apptainer_b4000 = AF3_SIF
af3_json_path   = os.path.join(AF3_JSON_DIR, os.path.relpath(af3_output_dir, AF3_OUT_DIR))
commands_base   = os.path.join(CMDS_DIR, cmds_base_name)

### QUEUE CONFIGURATION ###
b4000_fraction     = 0.50   # Fraction of commands to B4000 GPUs (0.0 = none, 1.0 = all)
gpu_fraction       = 0.60   # Within regular: fraction to priority queue (gpu vs gpu-bf)
gpu_fraction_b4000 = 1.00   # Within B4000:  fraction to priority queue (gpu-b4000 vs gpu-bf-b4000)

queue_config = {
    "gpu":          {"qtime": "07:15:00", "cmds_per_job": 7, "memory": "12g", "cores": "1"},
    "gpu-bf":       {"qtime": "04:15:00", "cmds_per_job": 5, "memory": "12g", "cores": "1"},

    "gpu-b4000":    {"qtime": "07:00:00", "cmds_per_job": 16, "memory": "12g", "cores": "1"},
    "gpu-bf-b4000": {"qtime": "07:00:00", "cmds_per_job": 16, "memory": "12g", "cores": "1"},
}

### DERIVED FLAGS ###
if mode not in {"one_seed", "n_seeds"}:
    raise ValueError(f"mode must be 'one_seed' or 'n_seeds'. Got: {mode}")
if mode == "one_seed":
    num_seeds = None
    num_diffusion_samples = N_total_structures
else:
    num_seeds = N_total_structures
    num_diffusion_samples = 1

### BUILD COMMANDS ###
# Build commands for both container types from the same JSON list
af3_jsons = sorted(glob.glob(os.path.join(af3_json_path, "*.json")), key=str.lower)

b4000_split = round(len(af3_jsons) * b4000_fraction)
jsons_regular = af3_jsons[b4000_split:]
jsons_b4000   = af3_jsons[:b4000_split]

cmds_regular = []
for af3_json in jsons_regular:
    cmd = f"{apptainer} python {script} --json_path={af3_json} --output_dir={af3_output_dir} --num_diffusion_samples={num_diffusion_samples}"
    if num_seeds is not None:
        cmd += f" --num_seeds={num_seeds}"
    cmds_regular.append(cmd)

cmds_b4000 = []
for af3_json in jsons_b4000:
    cmd = f"{apptainer_b4000} af3 --json_path={af3_json} --output_dir={af3_output_dir} --num_diffusion_samples={num_diffusion_samples}"
    if num_seeds is not None:
        cmd += f" --num_seeds={num_seeds}"
    cmds_b4000.append(cmd)

### SPLIT AND SUBMIT ###
for frac in [gpu_fraction, gpu_fraction_b4000, b4000_fraction]:
    if not (0.0 <= frac <= 1.0):
        raise ValueError(f"Fractions must be between 0.0 and 1.0. Got: {frac}")
queue_splits = {}

if cmds_regular: # Regular GPU split
    pri_idx = round(len(cmds_regular) * gpu_fraction)
    if pri_idx > 0:
        queue_splits["gpu"] = cmds_regular[:pri_idx]
    if pri_idx < len(cmds_regular):
        queue_splits["gpu-bf"] = cmds_regular[pri_idx:]

if cmds_b4000: # B4000 GPU split
    pri_idx = round(len(cmds_b4000) * gpu_fraction_b4000)
    if pri_idx > 0:
        queue_splits["gpu-b4000"] = cmds_b4000[:pri_idx]
    if pri_idx < len(cmds_b4000):
        queue_splits["gpu-bf-b4000"] = cmds_b4000[pri_idx:]

total_cmds = len(cmds_regular) + len(cmds_b4000)
print(f"{'='*60}")
print(f"  AF3 Job Summary │ {total_cmds} total cmds │ Mode: {mode} │ N: {N_total_structures}")
print(f"  Regular: {len(cmds_regular)} cmds │ B4000: {len(cmds_b4000)} cmds")
print(f"{'='*60}")

for queue, queue_cmds in queue_splits.items():
    cfg = queue_config[queue]
    suffix = f"_{queue.replace('-', '')}" if len(queue_splits) > 1 else ""
    commands_file = f"{commands_base}{suffix}"
    job_name = os.path.basename(commands_file)
    submit_file = f"{SUBMIT_DIR}{job_name}.sh"
    num_jobs = math.ceil(len(queue_cmds) / cfg["cmds_per_job"])
    with open(commands_file, 'w') as fo:
        print("\n".join(queue_cmds), file=fo)
    nb.submit_array_job(commands_file, cfg["qtime"], cfg["cores"], job_name, cfg["memory"], submit_file, LOGS_DIR, num_jobs, cfg["cmds_per_job"], queue)

## XIII.A.2 Execute AF3 Failed Commands

In [ ]:
##########################################
### Re-run only unfinished AF3 targets ###
##########################################                                                                                                                                

### SHARED INPUTS ###                                                                                                                                                          
input_pdb_path          = f"{REDESIGN_OUT_DIR}i2/"       
input_pdb_protein_chain = ["A"]                                                                                                                                                

af3_output_dir    = f"{AF3_OUT_DIR}i3__holo/"                                                                                                                                         
num_input_per_run = 10                                          
                                                                                                                                                                                
### SHARED OPTIONAL LIGAND FLAG ###                                                                                                                                            
include_ligands_if_present = True
input_ligand_dic           = {                                                                                                                                                 
    "B": {"ccdCodes": "['ZN']"},                                                                                                                                               
    "C": {"smiles": "[Zn+2].[OH-]"},                                                                                                                                               
    "D": {"smiles": "CCOP(=O)(OCC)OC1=CC=C(C=C1)[N+](=O)[O-]"},                                                                                                          
}   # leave empty {} or set to None to omit ligand args                                                                                                                     
                                                                                                                                                                                
### SHARED OPTIONAL PTM FLAG ###                                                                                                                                               
include_post_translational_mods_if_present = True                                                                                                                              
ptm_specs = ["A/LYS/3:KCX"]                                     
                                                                                                                                                                                
### SHARED OPTIONAL TERMINUS TAG FLAGS ###                                                                                                                                     
n_terminus_tag = ""                                                                                                                                                         
c_terminus_tag = ""                                                                                                                                                 
                                                                                                                                                                                
### SHARED OPTIONAL FLAGS ###                                                                                                                                                  
output_suffix              = "_af3i3"                                                                                                                                          
check_made_output          = True                               
cleanup_incomplete_outputs = True                                                                                                                                              

### SHARED OPTIONAL RECURSIVE SEARCH FLAGS ###                                                                                                                                 
recursive      = True                                           
max_depth      = None                                                                                                                                                          
specific_depth = 1    #None                                              
                                                                                                                                                                                
### SHARED SEED OPTIONS ###
no_random_seed = False                                                                                                                                                         
base_seed      = None

### GROUP DEFINITIONS (set to None for single-group behavior) ###
# Each group can override: pdb_prefix, input_ligand_dic, ptm_specs, output_suffix. Any key omitted inherits the shared default above                                                                                                                            
groups = [                                                                                                                                                                     
]                                                                                                                                                                              

### CONSTANTS ###                                                                                                                                                              
apptainer     = APPTAINER
script        = f"{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/make_af3_json_input.py"                                                                             

# IMPORTANT: use a NEW json staging directory for reruns
af3_json_path = os.path.join(AF3_JSON_DIR, "i3__holo_rerun_unfinished")

os.makedirs(af3_output_dir, exist_ok=True)
os.makedirs(af3_json_path, exist_ok=True)
                                                                                                                                                                                
### BUILD COMMAND(S) DYNAMICALLY ###                                                                                                                                           
def build_af3_command(ligand_dic, ptm_list, suffix, pdb_prefix_list=None):
    """Build a single make_af3_json_input.py command string."""                                                                                                                
    cmd_parts = [                                                                                                                                                              
        f"{apptainer} {script}",                                                                                                                                               
        f"--pdb_path {input_pdb_path}",                                                                                                                                        
        f"--pdb_chain {' '.join(input_pdb_protein_chain)}",                                                                                                                    
        f"--json_path {af3_json_path}",                         
        f"--num_input_per_run {num_input_per_run}",                                                                                                                            
        f"--output_path {af3_output_dir}",                                                                                                                                     
    ]
    if ligand_dic:                                                                                                                                                             
        ligand_chains = list(ligand_dic.keys())                                                                                                                                
        ligand_types  = []
        ligand_ids    = []                                                                                                                                                     
        for v in ligand_dic.values():                           
            if len(v) != 1:                                                                                                                                                    
                raise ValueError("Each ligand spec should have exactly one key indicating type (e.g., 'smiles').")                                                             
            ltype, lid = next(iter(v.items()))                                                                                                                                 
            ligand_types.append(ltype)                                                                                                                                         
            ligand_ids.append(lid)                                                                                                                                             
        cmd_parts.append(f"--ligand_chain {' '.join(ligand_chains)}")                                                                                                          
        cmd_parts.append(f"--ligand_type {' '.join(ligand_types)}")
        cmd_parts.append(f'--ligand_id "{" ".join(ligand_ids)}"')                                                                                                              
    if ptm_list:                                                
        cmd_parts.append("--ptm_from_remark666 " + " ".join([f'"{s}"' for s in ptm_list]))                                                                                     
    if n_terminus_tag:                                                                                                                                                         
        cmd_parts.append(f"--n_terminus_tag {n_terminus_tag}")
    if c_terminus_tag:                                                                                                                                                         
        cmd_parts.append(f"--c_terminus_tag {c_terminus_tag}")  
    if suffix:                                                                                                                                                                 
        cmd_parts.append(f"--output_suffix {suffix}")
    if check_made_output:                                                                                                                                                      
        cmd_parts.append("--check_made_output")                 
    if check_made_output and cleanup_incomplete_outputs:
        cmd_parts.append("--cleanup_incomplete_outputs")                                                                                                                       
    if recursive:
        cmd_parts.append("--recursive")                                                                                                                                        
    if max_depth is not None:                                   
        cmd_parts.append(f"--max_depth {max_depth}")
    if specific_depth is not None:                                                                                                                                             
        cmd_parts.append(f"--specific_depth {specific_depth}")
    if no_random_seed:                                                                                                                                                         
        cmd_parts.append("--no_random_seed")                    
    if base_seed is not None:                                                                                                                                                  
        cmd_parts.append(f"--base_seed {base_seed}")
    if pdb_prefix_list:                                                                                                                                                        
        cmd_parts.append(f"--pdb_prefix {' '.join(pdb_prefix_list)}")
    return " ".join(cmd_parts)                                                                                                                                                 

# Resolve shared defaults for ligands/PTMs                                                                                                                                     
shared_ligand_dic = input_ligand_dic if (include_ligands_if_present and input_ligand_dic) else {}
shared_ptm_specs  = ptm_specs if (include_post_translational_mods_if_present and ptm_specs) else []                                                                            
                                                                                                                                                                                
if groups:                                                                                                                                                                     
    # Multi-group mode: each group inherits shared defaults unless overridden                                                                                                  
    print(f"Generating {len(groups)} group commands:\n")                                                                                                                       
    for i, grp in enumerate(groups, 1):                                                                                                                                        
        cmd = build_af3_command(                                                                                                                                               
            ligand_dic      = grp.get("input_ligand_dic", shared_ligand_dic),                                                                                                  
            ptm_list        = grp.get("ptm_specs", shared_ptm_specs),
            suffix          = grp.get("output_suffix", output_suffix),                                                                                                         
            pdb_prefix_list = grp.get("pdb_prefix", None),      
        )                                                                                                                                                                      
        print(f"### Group {i}: {grp.get('name', f'group_{i}')} ###")
        print(cmd)                                                                                                                                                             
        print()                                                                                                                                                                
else:                                                                                                                                                                          
    # Single-group fallback (original behavior)                                                                                                                                
    cmd = build_af3_command(ligand_dic = shared_ligand_dic, ptm_list   = shared_ptm_specs, suffix     = output_suffix,                                                                                                                                            )
    print(cmd) 

In [ ]:
###############                                                                                                                                                                          
### Run AF3 ###                                                                                                                                                                          
###############                                                                                                                                                                          
                                                                                                                                                                                        
### INPUTS ###
af3_output_dir = f"{AF3_OUT_DIR}i3__holo/"
cmds_base_name = f"AF3_i3_holo_rerun_unfinished"

### PARAMETERS ###
N_total_structures = 5  # Total number of structures you want per target (N)
mode = "one_seed"        # "one_seed" -> N diffusion samples from 1 seed | "n_seeds" -> 1 sample from each of N seeds

### CONSTANTS ###
apptainer       = AF3_SIF
script          = AF3_RUNNER
apptainer_b4000 = AF3_SIF

# IMPORTANT: point to the NEW rerun json dir
af3_json_path   = os.path.join(AF3_JSON_DIR, "i3__holo_rerun_unfinished")
commands_base   = os.path.join(CMDS_DIR, cmds_base_name)

### QUEUE CONFIGURATION ###
b4000_fraction     = 0.70   # Fraction of commands to B4000 GPUs (0.0 = none, 1.0 = all)
gpu_fraction       = 1.00   # Within regular: fraction to priority queue (gpu vs gpu-bf)
gpu_fraction_b4000 = 1.00   # Within B4000:  fraction to priority queue (gpu-b4000 vs gpu-bf-b4000)

queue_config = {
    "gpu":          {"qtime": "02:30:00", "cmds_per_job": 6, "memory": "12g", "cores": "1"},
    "gpu-bf":       {"qtime": "02:00:00", "cmds_per_job": 2, "memory": "12g", "cores": "1"},

    "gpu-b4000":    {"qtime": "02:00:00", "cmds_per_job": 6, "memory": "12g", "cores": "1"},
    "gpu-bf-b4000": {"qtime": "06:30:00", "cmds_per_job": 16, "memory": "12g", "cores": "1"},
}

### DERIVED FLAGS ###
if mode not in {"one_seed", "n_seeds"}:
    raise ValueError(f"mode must be 'one_seed' or 'n_seeds'. Got: {mode}")
if mode == "one_seed":
    num_seeds = None
    num_diffusion_samples = N_total_structures
else:
    num_seeds = N_total_structures
    num_diffusion_samples = 1

### BUILD COMMANDS ###
# Build commands for both container types from the same JSON list
af3_jsons = sorted(glob.glob(os.path.join(af3_json_path, "*.json")), key=str.lower)

b4000_split = round(len(af3_jsons) * b4000_fraction)
jsons_regular = af3_jsons[b4000_split:]
jsons_b4000   = af3_jsons[:b4000_split]

cmds_regular = []
for af3_json in jsons_regular:
    cmd = f"{apptainer} python {script} --json_path={af3_json} --output_dir={af3_output_dir} --num_diffusion_samples={num_diffusion_samples}"
    if num_seeds is not None:
        cmd += f" --num_seeds={num_seeds}"
    cmds_regular.append(cmd)

cmds_b4000 = []
for af3_json in jsons_b4000:
    cmd = f"{apptainer_b4000} af3 --json_path={af3_json} --output_dir={af3_output_dir} --num_diffusion_samples={num_diffusion_samples}"
    if num_seeds is not None:
        cmd += f" --num_seeds={num_seeds}"
    cmds_b4000.append(cmd)

### SPLIT AND SUBMIT ###
for frac in [gpu_fraction, gpu_fraction_b4000, b4000_fraction]:
    if not (0.0 <= frac <= 1.0):
        raise ValueError(f"Fractions must be between 0.0 and 1.0. Got: {frac}")
queue_splits = {}

if cmds_regular: # Regular GPU split
    pri_idx = round(len(cmds_regular) * gpu_fraction)
    if pri_idx > 0:
        queue_splits["gpu"] = cmds_regular[:pri_idx]
    if pri_idx < len(cmds_regular):
        queue_splits["gpu-bf"] = cmds_regular[pri_idx:]

if cmds_b4000: # B4000 GPU split
    pri_idx = round(len(cmds_b4000) * gpu_fraction_b4000)
    if pri_idx > 0:
        queue_splits["gpu-b4000"] = cmds_b4000[:pri_idx]
    if pri_idx < len(cmds_b4000):
        queue_splits["gpu-bf-b4000"] = cmds_b4000[pri_idx:]

total_cmds = len(cmds_regular) + len(cmds_b4000)
print(f"{'='*60}")
print(f"  AF3 Job Summary │ {total_cmds} total cmds │ Mode: {mode} │ N: {N_total_structures}")
print(f"  Regular: {len(cmds_regular)} cmds │ B4000: {len(cmds_b4000)} cmds")
print(f"{'='*60}")

for queue, queue_cmds in queue_splits.items():
    cfg = queue_config[queue]
    suffix = f"_{queue.replace('-', '')}" if len(queue_splits) > 1 else ""
    commands_file = f"{commands_base}{suffix}"
    job_name = os.path.basename(commands_file)
    submit_file = f"{SUBMIT_DIR}{job_name}.sh"
    num_jobs = math.ceil(len(queue_cmds) / cfg["cmds_per_job"])
    with open(commands_file, 'w') as fo:
        print("\n".join(queue_cmds), file=fo)
    nb.submit_array_job(commands_file, cfg["qtime"], cfg["cores"], job_name, cfg["memory"], submit_file, LOGS_DIR, num_jobs, cfg["cmds_per_job"], queue)

## XIII.B Process AF3 Subdirectories

### XIII.B.1 Process AF3 Subdirectories

In [ ]:
################################################################
### Convert AF3 output format and remove useless AF3 outputs ###
################################################################

### INPUTS ###
af3_output_dir = f"{AF3_OUT_DIR}i3__holo/"    

### OPTIONAL SCRIPT ARGS ###
cif_sif_path              = MAXIT_SIF  # or None to omit
disable_deletions         = False   # set False to allow cleanup
verbose_logging           = True    # set False for quieter output
include_nonindexed_cifs   = False   # True to also convert non-indexed CIFs
robust_mode               = False   # should probably turn default as false, but this is good for re-running
af3_subdirs_are_lowercase = False

### CONSTANTS ###
apptainer = "python"   # I don't find a suitable apptainer yet.
script    = f"{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/process_af3_cif.py"

### EXECUTE ###
commands = os.path.join(CMDS_DIR, "AF3_process_i3")
af3_subdirs = glob.glob(os.path.join(af3_output_dir, "*"))
with os.scandir(af3_output_dir) as it:
    af3_subdirs = [e.path for e in it if e.is_dir(follow_symlinks=False)]
if af3_subdirs_are_lowercase:
    af3_subdirs = list(filter(lambda fi: os.path.basename(fi).lower() == os.path.basename(fi), af3_subdirs))
af3_subdirs = sorted(af3_subdirs, key=lambda p: os.path.basename(p).lower())
print (f"Number of AF3 subdirectories: {len(af3_subdirs)}")
print ()

# build extra args string once
extra_args = []
if cif_sif_path:
    extra_args += ["--cif_sif", cif_sif_path]
if disable_deletions:
    extra_args.append("--disable-deletions")
if verbose_logging:
    extra_args.append("--verbose")
if include_nonindexed_cifs:
    extra_args.append("--include-nonindexed-cifs")
if robust_mode:
    extra_args.append("--robust-mode")
extra_args_str = " ".join(extra_args)

command_num = 0
with open(commands, 'w') as f_cmd:
    for af3_path in af3_subdirs:
        # Only include if "samples" subdir exists
        if os.path.isdir(os.path.join(af3_path, "samples")):
            f_cmd.write(f"{apptainer} {script} --af3_subdir {af3_path} {extra_args_str}\n")
            command_num += 1

### SETUP SLURM SUBMISSION ###
job_name = os.path.basename(commands)
qtime = '00:30:00'
cores = '1'
memory = '2g'
queue = 'cpu'
cmds_per_job = 50
num_jobs = int(command_num / cmds_per_job) + 1
submit_file = f'{SUBMIT_DIR}{job_name}.sh'
print("Job/Commands Name:", job_name)
print("Number of Jobs to Run:", num_jobs)
print('')
print('### COMMANDS FILE BELOW ###')
print(commands)
print('')

# Submit the job array using your rainier module.
nb.submit_array_job(commands, qtime, cores, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue)

In [ ]:
##############################                                                                                                                                                     
### Process AF3 prediction ###                                                                                                                                                     
##############################                                                                                                                                                     
                                                                                                                                                                                    
### PATHS ###
af3_out_path       = f"{AF3_OUT_DIR}i3__holo/"
ref_pdb_path       = f"{REDESIGN_OUT_DIR}i2/"
backup_ref_subdir  = "ref_pdbs"
backup_ref_dir     = os.path.join(ref_pdb_path, backup_ref_subdir) if backup_ref_subdir else ""
commands           = f"{CMDS_DIR}AF3_pdb_processing_substrate_zn_zno"

### PARAMETERS ###
af3_N                  = 5
af3_suffix             = "_af3i3"
default_ligand         = "USER_LIG"
use_split_subdirs      = True                                    # True: ref PDBs in subdirectories; False: flat directory
subdir_strip_pattern = r"_(?:t[\d_]+|c\d+)_\d+_\d+$"     # strips the redesign suffix to recover the input stem
use_ligand_in_filename = False                                   # Search for ligand anywhere in filename (e.g., *PTE_wKCX*XDW*)
use_ligand             = True

### MAP REF PDB PREFIXES TO LIGAND NAMES ###
prefix_ligand_map = {
    "ZAPP": ["YYE"],  # Multiple ligands per prefix is possible
}

### MAP REF PDB PREFIXES TO CATRES SUBSETS ###
prefix_catres_map = {
    "ZAPP": "1,2,3,4,5,6",
}

### ATOM NAMING INPUT TEMPLATE ###
atom_groups_template = [
    {"label": "subst",
     "name3": ["G_D", "USER_LIG"], 
     "chain": ["D", "B"],
     "atoms": [["P1", "N1", "O5", "O6", "O1", "C2", "C1", "O3", "C3", "C4", "O2", "O4", "C5", "C6", "C7", "C8", "C9", "C10"], 
               ["P1", "N1", "O9", "O8", "O6", "C9", "C11", "O4", "C8", "C10", "O5", "O7", "C7", "C5", "C3", "C2", "C4", "C6"]],
     "symmetric_atom_groups": [{
                  "lig1": [["C5", "C6", "C7", "C8", "C9", "C10"], # BENZENE
                           ["N1", "O5", "O6"], # NITRO
                           ["O1", "C2", "C1", "O3", "C3", "C4"]], # ETHYL
         
                  "lig2": [[["C7", "C6", "C4", "C2", "C3", "C5"], ["C7", "C5", "C3", "C2", "C4", "C6"]], 
                           [["N1", "O9", "O8"], ["N1", "O8", "O9"]],
                           [["O6", "C9", "C11", "O4", "C8", "C10"], ["O4", "C8", "C10", "O6", "C9", "C11"]]]
              }],},
    
    {"label": "subst_core",
     "name3": ["G_D", "USER_LIG"], 
     "chain": ["D", "B"],
     "atoms": [["P1", "O1", "O3", "O2", "O4"],
               ["P1", "O6", "O4", "O5", "O7"]],
     "symmetric_atom_groups": [{
                  "lig1": [["O1", "O3"]], # ETHYL
                  "lig2": [[["O4", "O6"], ["O6", "O4"]]]
              }],},

    # Hydroxide ion
    {"label": "hydroxide_ion", "name3": ["G_C", "USER_LIG"], "chain": ["C", "B"], "atoms": [["O1"], ["O3"]]},

    # NCAA
    {"label": "kcx_tips", "name3": ["KCX", "USER_LIG"], "chain": ["A", "B"],
    "atoms": [["CX", "OQ1", "OQ2"], ["C1", "O1", "O2"]],
    "symmetric_atom_groups": [{"lig1": [["CX", "OQ1", "OQ2"]], "lig2": [[["C1", "O1", "O2"], ["C1", "O2", "O1"]]]}]},
    # Zinc ion for ZN1
    {"label": "zinc1_chainB", "name3": ["ZN", "USER_LIG"], "chain": ["B", "B"], "atoms": [["ZN"], ["ZN1"]]},
    {"label": "zinc1_chainC", "name3": ["G_C", "USER_LIG"], "chain": ["C", "B"], "atoms": [["ZN1"], ["ZN1"]]},
    # Zinc ion for ZN2
    {"label": "zinc2_chainB", "name3": ["ZN", "USER_LIG"], "chain": ["B", "B"], "atoms": [["ZN"], ["ZN2"]]},
    {"label": "zinc2_chainC", "name3": ["G_C", "USER_LIG"], "chain": ["C", "B"], "atoms": [["ZN1"], ["ZN2"]]},
]

### OPTIONAL: SPECIFY N & C TERMINUS TAGS TO IGNORE ###
N_tag_ignore = 0    # For example, set to 3 if you want to ignore the first 3 residues (MSG = 3)
C_tag_ignore = 0    # For example, set to 2 if you want to ignore the last 2 residues  (GSAWSHPQFEK = 11)

### FLAGS ###
verbose                     = True   # --verbose
calc_interchain_pae_mean    = True   # --calculate_avg_pae_in_addition_to_pair
rapid_mode_skip_sc_if_found = False  # Skip if .sc is found

### CONSTANTS ###
script    = f'{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/process_af3_pdb.py'
apptainer = APPTAINER

### GENERATE COMMANDS ###

# Pre-compute all AF3 subdirectories that have the correct number of predictions
from concurrent.futures import ThreadPoolExecutor
def _check_af3_dir(d):
    pdb_count = sum(1 for e in os.scandir(d) if e.name.endswith(".pdb") and e.is_file())
    return os.path.basename(d) if pdb_count == af3_N else None
with ThreadPoolExecutor() as executor:
    af3_dirs = glob.glob(os.path.join(af3_out_path, f"*{af3_suffix}"))
    af3_valid_subdirs = {r for r in executor.map(_check_af3_dir, af3_dirs) if r}
print(f"{len(af3_valid_subdirs)} valid AF3 directories detected (with {af3_N} PDBs each)")

# Match AF3 directories to reference PDBs
pairs = []
if use_split_subdirs:
    matched_subdir, matched_backup, unmatched = 0, 0, 0
    unmatched_examples = []  # NEW: collect examples for debugging
    for af3_name in af3_valid_subdirs:
        pdb_base   = af3_name[:-len(af3_suffix)]
        parent_dir = re.sub(subdir_strip_pattern, "", pdb_base)
        ref_pdb    = os.path.join(ref_pdb_path, parent_dir, pdb_base + ".pdb")
        if os.path.isfile(ref_pdb):
            pairs.append((af3_name, ref_pdb))
            matched_subdir += 1
        elif backup_ref_dir:
            ref_pdb = os.path.join(backup_ref_dir, pdb_base + ".pdb")
            if os.path.isfile(ref_pdb):
                pairs.append((af3_name, ref_pdb))
                matched_backup += 1
            else:
                unmatched += 1
                if len(unmatched_examples) < 10:
                    unmatched_examples.append((af3_name, ref_pdb))
        else:
            unmatched += 1
            if len(unmatched_examples) < 10:
                unmatched_examples.append((af3_name, ref_pdb))
        # NEW: lightweight progress update
        if (matched_subdir + matched_backup + unmatched) % 5000 == 0:
            print(f"[match progress] processed={matched_subdir + matched_backup + unmatched:,} "
                  f"| matched={len(pairs):,} | unmatched={unmatched:,}")
    print(f"  Matched from subdirs: {matched_subdir}")
    if backup_ref_dir:
        print(f"  Matched from backup: {matched_backup}")
    print(f"  Unmatched:           {unmatched}")
    # NEW: show examples of failures
    if unmatched_examples:
        print("\nExample unmatched cases (first 10):")
        for af3, ref in unmatched_examples:
            print(f"  AF3: {af3}  ->  expected REF: {ref}")
else:
    ref_pdbs = glob.glob(os.path.join(ref_pdb_path, "*.pdb"))
    print(f"{len(ref_pdbs)} REF PDBs were detected")
    for i, ref_pdb in enumerate(ref_pdbs, 1):
        af3_name = os.path.basename(ref_pdb).replace(".pdb", "") + af3_suffix
        if af3_name in af3_valid_subdirs:
            pairs.append((af3_name, ref_pdb))
        # NEW: progress updates
        if i % 5000 == 0:
            print(f"[ref scan progress] {i:,}/{len(ref_pdbs):,} scanned | pairs={len(pairs):,}")

# Pre-build prefix lookup (uppercase for case-insensitive matching)
prefix_map_upper    = {k.upper(): (v if isinstance(v, list) else [v]) for k, v in prefix_ligand_map.items()}
catres_map_upper    = {k.upper(): v for k, v in prefix_catres_map.items()}
sorted_catres_items = sorted(catres_map_upper.items(), key=lambda x: -len(x[0]))

# Pre-build command parts that don't change
base_cmd   = [apptainer, script]
flag_parts = []
if verbose: flag_parts.append("--verbose")
if calc_interchain_pae_mean: flag_parts.append("--calculate_avg_pae_in_addition_to_pair")
if rapid_mode_skip_sc_if_found: flag_parts.append("--rapid_mode_skip_sc_if_found")

# Cache for ligand-specific atom groups (avoid redundant deep copies)
atom_groups_cache = {}
command_lines = []
for af3_pdb_subdir, ref_pdb in pairs:
    ref_pdb_bn = os.path.basename(ref_pdb)

    # Determine ligand (optimized with cached uppercase prefixes)
    ligand_name = default_ligand
    up = ref_pdb_bn.upper()
    for prefix_up, ligs in prefix_map_upper.items():
        if up.startswith(prefix_up):
            ligand_name = next((lig for lig in ligs if lig.upper() in up), ligs[0]) if use_ligand_in_filename else ligs[0]
            break

    # Determine catres_subset (find longest matching prefix)
    catres_subset = None
    for prefix_up, subset_val in sorted_catres_items:
        if up.startswith(prefix_up):
            catres_subset = subset_val
            break

    # Use cached atom groups for this ligand (avoid redundant processing)
    if ligand_name not in atom_groups_cache:
        atom_groups_cache[ligand_name] = [
            dict(g, name3=[item if item != "USER_LIG" else ligand_name for item in g["name3"]])
            if "symmetric_atom_groups" not in g
            else {**copy.deepcopy(g), "name3": [item if item != "USER_LIG" else ligand_name for item in g["name3"]]}
            for g in atom_groups_template]

    # Build command (reuse pre-built parts)
    af3_dir_path = os.path.join(af3_out_path, af3_pdb_subdir)
    outscr_path  = os.path.join(af3_dir_path, af3_pdb_subdir + ".sc")
    cmd = base_cmd + [f"--af3_dir {af3_dir_path}", f"--ref_pdb {ref_pdb}", f"--outscr {outscr_path}",] + flag_parts
    if use_ligand:
        cmd.append(f"--ligand_groups_json '{json.dumps(atom_groups_cache[ligand_name])}'")
    if catres_subset:
        cmd.append(f"--catres_subset {catres_subset}")
    if N_tag_ignore > 0:
        cmd.append(f"--N_terminus_tag_length_to_ignore {N_tag_ignore}")
    if C_tag_ignore > 0:
        cmd.append(f"--C_terminus_tag_length_to_ignore {C_tag_ignore}")
    command_lines.append(" ".join(cmd))
command_lines.sort()
with open(commands, 'w') as fo:
    fo.write("\n".join(command_lines) + "\n")

### SETUP SLURM SUBMISSION ###
job_name = os.path.basename(commands)
qtime, cores, memory, queue, cmds_per_job = '00:40:00', '1', '2g', 'cpu', 75
num_jobs = (len(command_lines) + cmds_per_job - 1) // cmds_per_job
submit_file = f'{SUBMIT_DIR}{job_name}.sh'

# Submit job array
nb.submit_array_job(commands, qtime, cores, job_name, memory, submit_file, LOGS_DIR, num_jobs, cmds_per_job, queue)

### XIII.B.2 Parse AF3 Stats

In [ ]:
##################################################################
### PARSE & PROCESS AF3 .SC FILES ONLY NEED TO DO THIS ONCE ######
##################################################################

# INPUT DIRECTORY OF SUBDIRECTORIES TO PARSE
af3_dir_of_subdirs = f"{AF3_OUT_DIR}i3__holo/"   # CHANGE EVERY TIME

### OPTIONALS ###
workers                         = 64 # None or integer
optional_path_for_summary_stats = f"{IMPORTANT_DFS_DIR}af3_i3__holo.csv"  # e.g. f"{IMPORTANT_DFS_DIR}af3_stats_iteration2_holo.csv"
optional_chunk_rows             = None  # 20000 # Optional: rows per shard before writing to temp CSV | default = 10000
find_subdirs_without_viable_sc  = False # DRY RUN IF TRUE

### CONSTANTS ###
script = f"{SPECIAL_SCRIPTS_DIR}advanced_structure_prediction_tools/concat_af3_sc_dir_of_subdirs.py"
container_path = APPTAINER

### GENERATE COMMAND ###
command = f"singularity exec {container_path} python {script} --af3_dir_of_subdirs {af3_dir_of_subdirs}"
if optional_path_for_summary_stats:
    command += f" --optional_path_for_summary_stats {optional_path_for_summary_stats}"
if optional_chunk_rows:
    command += f" --chunk_rows {optional_chunk_rows}"
if workers:
    command += f" --workers {workers}"
if find_subdirs_without_viable_sc:
    command += f" --find_subdirs_without_viable_sc"

### PRINT COMMAND TO SUBMIT ###
print("SUBMIT THIS:")
print(command)
print('')

### PRINT OUTPUT FILE ###
print("OUTPUT CSV FILE HERE:")
if optional_path_for_summary_stats:
    print(optional_path_for_summary_stats)
else:
    print(af3_dir_of_subdirs + 'zzzzz_af3_analysis_csv_zzzzz.csv')

In [ ]:
########################################################################                                                                      
### APPLY TRANSFORMATIONS & MANIPULATIONS TO THE CONCATENATED AF3 DF ###
########################################################################                                                                      
                                                                
### LOAD DF ###                                                                                                                               
csv_path = f"{IMPORTANT_DFS_DIR}af3_i3__holo.csv"                 
af3_scores_df = pd.read_csv(csv_path)                                                                                                         

### SAVE DF PATH ###                                                                                                                          
out_csv = f"{IMPORTANT_DFS_DIR}modified_af3_i3__holo.csv"         
                                                                                                                                            
### APPLY TRANSFORMATIONS & MANIPULATIONS ###
_desc = af3_scores_df["af3_models_dir"].astype(str)                                                                                           
desc_base = _desc.apply(lambda s: re.split(r"[\\/]", s)[-1])                                                                                  

# Build scaffold_family from the basename (applies to apo + holo)                                                                             
af3_scores_df["scaffold_family"] = (desc_base.str.split("_inp", n=1).str[0].str.split("_eV2", n=1).str[0].str.split("_FS", n=1).str[0]) 
af3_scores_df["structure_cluster"] = (desc_base.str.extract(r"FS(\d+)", expand=False).astype("Int64"))

# -------------------- ZINC pair assignment (Kabsch) --------------------                                                                     
# Only runs if zinc columns exist (skipped automatically for apo)
pattern = re.compile(r"^zinc[12]_chain[BC]_rmsd_idx_(\d+)$")                                                                                  
all_idxs = sorted({int(m.group(1)) for col in af3_scores_df.columns
                    if (m := pattern.match(col))})                                                                                             
                                                                
if not all_idxs:                                                                                                                              
    print("[skip] no zinc*_chain[BC]_rmsd_idx_* columns found — apo mode or no metals; skipping Kabsch zinc-pair block")
else:                                                                                                                                         
    def compute_pair_for_idx(row, i):                           
        cols = {"z1B": f"zinc1_chainB_rmsd_idx_{i}",                                                                                          
                "z1C": f"zinc1_chainC_rmsd_idx_{i}",            
                "z2B": f"zinc2_chainB_rmsd_idx_{i}",                                                                                          
                "z2C": f"zinc2_chainC_rmsd_idx_{i}"}
        if any(c not in row.index for c in cols.values()):                                                                                    
            return np.nan, None                                 
        v = {k: row[c] for k, c in cols.items()}                                                                                              
        if any(pd.isna(x) for x in v.values()):                                                                                               
            return np.nan, None
        bc_avg = float(np.mean([v["z1B"], v["z2C"]]))                                                                                         
        cb_avg = float(np.mean([v["z1C"], v["z2B"]]))                                                                                         
        if bc_avg <= cb_avg:
            return bc_avg, "BC"                                                                                                               
        return cb_avg, "CB"                                                                                                                   

    for i in all_idxs:                                                                                                                        
        avg_col = f"zinc_pair_avg_rmsd_idx_{i}"                 
        asn_col = f"zinc_pair_assignment_idx_{i}"
        af3_scores_df[avg_col] = np.nan                                                                                                       
        af3_scores_df[asn_col] = None
        results = af3_scores_df.apply(lambda r: compute_pair_for_idx(r, i), axis=1, result_type="expand")                                     
        af3_scores_df[avg_col] = results[0].astype(float)                                                                                     
        af3_scores_df[asn_col] = results[1].astype("object")                                                                                  
    print(f"[ok] Kabsch zinc-pair block ran on idx values: {all_idxs}")                                                                       
                                                                                                                                            
# -------------------- ZINC pair assignment (TM-align) --------------------                                                                   
# Only runs if TMalign zinc columns exist (skipped automatically for apo)                                                                     
pattern_tm = re.compile(r"^zinc[12]_chain[BC]_rmsd_TMalign_idx_(\d+)$")                                                                       
all_idxs_tm = sorted({int(m.group(1)) for col in af3_scores_df.columns                                                                        
                    if (m := pattern_tm.match(col))})                                                                                       
                                                                                                                                            
if not all_idxs_tm:                                                                                                                           
    print("[skip] no zinc*_chain[BC]_rmsd_TMalign_idx_* columns found — apo mode or no metals; skipping TM-align zinc-pair block")
else:                                                                                                                                         
    def compute_pair_for_idx_TMalign(row, i):
        cols = {"z1B": f"zinc1_chainB_rmsd_TMalign_idx_{i}",                                                                                  
                "z1C": f"zinc1_chainC_rmsd_TMalign_idx_{i}",    
                "z2B": f"zinc2_chainB_rmsd_TMalign_idx_{i}",                                                                                  
                "z2C": f"zinc2_chainC_rmsd_TMalign_idx_{i}"}    
        if any(c not in row.index for c in cols.values()):                                                                                    
            return np.nan, None                                 
        v = {k: row[c] for k, c in cols.items()}                                                                                              
        if any(pd.isna(x) for x in v.values()):                                                                                               
            return np.nan, None
        bc_avg = float(np.mean([v["z1B"], v["z2C"]]))                                                                                         
        cb_avg = float(np.mean([v["z1C"], v["z2B"]]))                                                                                         
        if bc_avg <= cb_avg:
            return bc_avg, "BC"                                                                                                               
        return cb_avg, "CB"                                                                                                                   

    for i in all_idxs_tm:                                                                                                                     
        avg_col = f"zinc_pair_avg_rmsd_TMalign_idx_{i}"         
        asn_col = f"zinc_pair_assignment_TMalign_idx_{i}"                                                                                     
        af3_scores_df[avg_col] = np.nan
        af3_scores_df[asn_col] = None                                                                                                         
        results = af3_scores_df.apply(lambda r: compute_pair_for_idx_TMalign(r, i), axis=1, result_type="expand")                             
        af3_scores_df[avg_col] = results[0].astype(float)                                                                                     
        af3_scores_df[asn_col] = results[1].astype("object")                                                                                  
    print(f"[ok] TM-align zinc-pair block ran on idx values: {all_idxs_tm}")                                                                  
                                                                                                                                            
### SAVE DF ###
af3_scores_df.to_csv(out_csv, index=False)                                                                                                    
print(f"Saved: {out_csv}  (rows={len(af3_scores_df)})")         

### COUNT ###
total_unique_scaffolds = af3_scores_df["scaffold_family"].nunique()
before_clusters = pd.to_numeric(af3_scores_df["structure_cluster"], errors="coerce").nunique(dropna=True)
print(f"\nTotal unique scaffold groups before filtering: {total_unique_scaffolds}")
print(f"Unique structure clusters BEFORE filtering: {before_clusters}\n")
                                                                                                                                            
### LOOKING ###
_preview_cols = ["af3_models_dir", "scaffold_family", "structure_cluster"] + [c for c in af3_scores_df.columns if c.startswith("zinc_pair_")][:4]                  
af3_scores_df[_preview_cols].head()   

## XIII.C Filter AF3

### XIII.C.1 Check Stat Distributions

In [ ]:
############################################################
### AF3 METRICS: 5-NUMBER SUMMARY & PERCENTILE ANALYSIS  ###
############################################################
"""
For each metric, compute:
1. 5-number summary (min, Q1, median, Q3, max)
2. Mean and std
3. Top N% threshold (direction-aware: lower or upper percentile)
4. Suggested filter values based on percentiles
"""

import os
import numpy as np
import pandas as pd
from typing import Dict, List

# ============================================================================
# CONFIGURATION
# ============================================================================

### PATHS ###
af3_scores_df_path = os.path.join(IMPORTANT_DFS_DIR, 'modified_af3_i2__holo.csv')

### NUMBER OF AF3 PREDICTIONS ###
af3_N = 5

### TOP N% FOR FILTER RECOMMENDATIONS ###
# If metric uses <= filter, we take the Nth percentile (lower is better)
# If metric uses >= filter, we take the (100-N)th percentile (higher is better)
top_N_percent = 20  # Top 20%

### ADDITIONAL PERCENTILES TO SHOW ###
percentiles_to_show = [5, 10, 15, 20, 25, 30, 50]

### METRICS CONFIGURATION ###
# Format: base_metric -> (filter_operator, expected_direction)
# "lower" means lower values are better (use <= filter)
# "higher" means higher values are better (use >= filter)

metrics_config = {
    # RMSD metrics (lower is better, use <=)
    "ca_rmsd": ("<=", "lower"),
    "zinc_pair_avg_rmsd": ("<=", "lower"),
    "catres_rmsd": ("<=", "lower"),
    "catres_subset_rmsd": ("<=", "lower"),
    "kcx_tips_rmsd": ("<=", "lower"),
    "hydroxide_ion_rmsd": ("<=", "lower"),
    "subst_rmsd": ("<=", "lower"),
    "subst_core_rmsd": ("<=", "lower"),

    # RMSD metrics (lower is better, use <=)
    "ca_rmsd_TMalign": ("<=", "lower"),
    "zinc_pair_avg_rmsd_TMalign": ("<=", "lower"),
    "catres_rmsd_TMalign": ("<=", "lower"),
    "catres_subset_rmsd_TMalign": ("<=", "lower"),
    "kcx_tips_rmsd_TMalign": ("<=", "lower"),
    "hydroxide_ion_rmsd_TMalign": ("<=", "lower"),
    "subst_rmsd_TMalign": ("<=", "lower"),
    "subst_core_rmsd_TMalign": ("<=", "lower"),

    # TM & lDDT metrics (higher is better, use >=)
    "tm_score": (">=", "higher"),
    "catres_lddt": (">=", "higher"),
    "catres_subset_lddt": (">=", "higher"),

    # pLDDT metrics (higher is better, use >=)
    "chainA_plddt": (">=", "higher"),
    "chainB_plddt": (">=", "higher"),
    "chainC_plddt": (">=", "higher"),
    "chainD_plddt": (">=", "higher"),
    "catres_plddt": (">=", "higher"),
    "KCX3_extra_atoms_plddt": (">=", "higher"),
    "hydroxide_ion_plddt": (">=", "higher"),
    "subst_core_plddt": (">=", "higher"),
    "kcx_tips_plddt": (">=", "higher"),

    # PTM/iPTM metrics (higher is better, use >=)
    "iptm": (">=", "higher"),
    "ptm": (">=", "higher"),

    # PAE min metrics (lower is better, use <=)
    "chainA_chainB_pair_pae_min": ("<=", "lower"),
    "chainA_chainC_pair_pae_min": ("<=", "lower"),
    "chainA_chainD_pair_pae_min": ("<=", "lower"),
    "chainB_chainC_pair_pae_min": ("<=", "lower"),
    "chainB_chainD_pair_pae_min": ("<=", "lower"),
    "chainC_chainD_pair_pae_min": ("<=", "lower"),

    # PAE mean metrics (lower is better, use <=)
    "chainA_chainB_pair_pae_mean": ("<=", "lower"),
    "chainA_chainC_pair_pae_mean": ("<=", "lower"),
    "chainA_chainD_pair_pae_mean": ("<=", "lower"),
    "chainB_chainC_pair_pae_mean": ("<=", "lower"),
    "chainB_chainD_pair_pae_mean": ("<=", "lower"),
    "chainC_chainD_pair_pae_mean": ("<=", "lower"),
}

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================
def print_header(title: str, char: str = "=", width: int = 90):
    print(f"\n{char * width}")
    print(f" {title}")
    print(f"{char * width}")
def print_subheader(title: str, char: str = "-", width: int = 70):
    print(f"\n{char * width}")
    print(f" {title}")
    print(f"{char * width}")

# ============================================================================
# LOAD DATA
# ============================================================================
print_header("AF3 METRICS: 5-NUMBER SUMMARY & PERCENTILE ANALYSIS")
af3_scores_df = pd.read_csv(af3_scores_df_path)
print(f"\nLoaded {len(af3_scores_df)} designs from:\n  {af3_scores_df_path}")

# ============================================================================
# COMPUTE STATISTICS FOR EACH METRIC
# ============================================================================
print_subheader(f"STATISTICS FOR ALL METRICS (Top {top_N_percent}% thresholds)")
all_stats = []
for base_metric, (operator, direction) in metrics_config.items():
    # Collect all idx values for this metric
    idx_cols = [f"{base_metric}_idx_{i}" for i in range(af3_N)]
    available_cols = [c for c in idx_cols if c in af3_scores_df.columns]
    if not available_cols:
        continue
    # Stack all idx values together
    all_values = af3_scores_df[available_cols].values.flatten()
    all_values = all_values[~np.isnan(all_values)]
    if len(all_values) == 0:
        continue
    # 5-number summary
    min_val = np.min(all_values)
    q1 = np.percentile(all_values, 25)
    median_val = np.median(all_values)
    q3 = np.percentile(all_values, 75)
    max_val = np.max(all_values)
    mean_val = np.mean(all_values)
    std_val = np.std(all_values)
    # Top N% threshold (direction-aware)
    if direction == "lower":
        # Lower is better, so top N% means the Nth percentile
        top_n_threshold = np.percentile(all_values, top_N_percent)
    else:
        # Higher is better, so top N% means the (100-N)th percentile
        top_n_threshold = np.percentile(all_values, 100 - top_N_percent)
    # Compute all requested percentiles
    percentile_values = {}
    for p in percentiles_to_show:
        if direction == "lower":
            percentile_values[p] = np.percentile(all_values, p)
        else:
            percentile_values[p] = np.percentile(all_values, 100 - p)
    all_stats.append({
        "metric": base_metric,
        "operator": operator,
        "direction": direction,
        "n_values": len(all_values),
        "min": min_val,
        "Q1": q1,
        "median": median_val,
        "Q3": q3,
        "max": max_val,
        "mean": mean_val,
        "std": std_val,
        f"top_{top_N_percent}%": top_n_threshold,
        **{f"top_{p}%": percentile_values[p] for p in percentiles_to_show},
    })
stats_df = pd.DataFrame(all_stats)

# ============================================================================
# DISPLAY: 5-NUMBER SUMMARY TABLE
# ============================================================================
print_subheader("5-NUMBER SUMMARY (All idx combined)")
print(f"\n  {'Metric':<35s} {'Op':<4s} {'Min':>8s} {'Q1':>8s} {'Median':>8s} {'Q3':>8s} {'Max':>8s} {'Mean':>8s} {'Std':>8s}")
print(f"  {'-'*107}")
for _, row in stats_df.iterrows():
    print(f"  {row['metric']:<35s} {row['operator']:<4s} "
          f"{row['min']:>8.2f} {row['Q1']:>8.2f} {row['median']:>8.2f} "
          f"{row['Q3']:>8.2f} {row['max']:>8.2f} {row['mean']:>8.2f} {row['std']:>8.2f}")

# ============================================================================
# DISPLAY: TOP N% THRESHOLDS (FILTER RECOMMENDATIONS)
# ============================================================================
print_subheader(f"TOP N% THRESHOLDS (Filter Recommendations)")
print(f"\nFor '<=' filters: shows Nth percentile (lower is better)")
print(f"For '>=' filters: shows (100-N)th percentile (higher is better)")
# Header with all percentiles
pct_header = "  " + f"{'Metric':<35s} {'Op':<4s}"
for p in percentiles_to_show:
    pct_header += f" {'Top '+str(p)+'%':>10s}"
print(f"\n{pct_header}")
print(f"  {'-'*(39 + 11*len(percentiles_to_show))}")
for _, row in stats_df.iterrows():
    line = f"  {row['metric']:<35s} {row['operator']:<4s}"
    for p in percentiles_to_show:
        val = row[f"top_{p}%"]
        line += f" {val:>10.2f}"
    print(line)

# ============================================================================
# DISPLAY: SUGGESTED FILTER VALUES (COPY-PASTE READY)
# ============================================================================
print_subheader(f"SUGGESTED FILTER VALUES (Top {top_N_percent}% - Copy-Paste Ready)")
print(f"\n# Filter thresholds for top {top_N_percent}% of designs")
print(f"# Based on {'percentile ' + str(top_N_percent) if True else ''} analysis\n")
for _, row in stats_df.iterrows():
    threshold = row[f"top_{top_N_percent}%"]
    # Format nicely
    if abs(threshold) < 1:
        formatted = f"{threshold:.3f}"
    elif abs(threshold) < 10:
        formatted = f"{threshold:.2f}"
    else:
        formatted = f"{threshold:.1f}"
    var_name = row['metric'].replace("_", "_")
    print(f"{var_name:<35s} = {formatted}")

# ============================================================================
# DISPLAY: FILTERS DICT FORMAT
# ============================================================================
print_subheader(f"FILTERS DICT FORMAT (Top {top_N_percent}%)")
print("\nfilters = {")
for _, row in stats_df.iterrows():
    threshold = row[f"top_{top_N_percent}%"]
    # Format nicely
    if abs(threshold) < 1:
        formatted = f"{threshold:.3f}"
    elif abs(threshold) < 10:
        formatted = f"{threshold:.2f}"
    else:
        formatted = f"{threshold:.1f}"

    print(f'    ("{row["metric"]}", "{row["operator"]}"): {formatted},')
print("}")

# ============================================================================
# PER-IDX STATISTICS
# ============================================================================
print_header("PER-IDX STATISTICS")
print("Checking if different AF3 seeds have different distributions")
per_idx_stats = []
for base_metric, (operator, direction) in metrics_config.items():
    for idx in range(af3_N):
        col = f"{base_metric}_idx_{idx}"
        if col not in af3_scores_df.columns:
            continue
        values = af3_scores_df[col].dropna().values
        if len(values) == 0:
            continue
        if direction == "lower":
            top_n_val = np.percentile(values, top_N_percent)
        else:
            top_n_val = np.percentile(values, 100 - top_N_percent)
        per_idx_stats.append({
            "metric": base_metric,
            "idx": idx,
            "operator": operator,
            "n": len(values),
            "mean": np.mean(values),
            "std": np.std(values),
            "median": np.median(values),
            f"top_{top_N_percent}%": top_n_val,
        })
per_idx_df = pd.DataFrame(per_idx_stats)

# Show summary: do different idx have different thresholds?
print_subheader("Per-Idx Threshold Variation")
print(f"\n  {'Metric':<35s} ", end="")
for idx in range(af3_N):
    print(f"{'idx_'+str(idx):>10s} ", end="")
print(f"{'Range':>10s}")
print(f"  {'-'*(35 + 11*(af3_N+1))}")
for base_metric in stats_df["metric"].values:
    sub = per_idx_df[per_idx_df["metric"] == base_metric]
    if len(sub) == 0:
        continue
    line = f"  {base_metric:<35s} "
    vals = []
    for idx in range(af3_N):
        row = sub[sub["idx"] == idx]
        if len(row) > 0:
            val = row[f"top_{top_N_percent}%"].values[0]
            vals.append(val)
            line += f"{val:>10.2f} "
        else:
            line += f"{'N/A':>10s} "
    if vals:
        range_val = max(vals) - min(vals)
        line += f"{range_val:>10.2f}"
    print(line)

# ============================================================================
# SPECIAL ANALYSIS: HIT-BASED THRESHOLDS (if hit classification exists)
# ============================================================================
# Check if we have hit classification from the retro analysis
if "hit_class" in af3_scores_df.columns or "is_hit" in af3_scores_df.columns:
    print_header("HIT-BASED THRESHOLD ANALYSIS")
    print("Thresholds based on hit distribution (where hits tend to fall)")
    # Determine hit column
    if "is_any_hit" in af3_scores_df.columns:
        hit_col = "is_any_hit"
    elif "is_hit" in af3_scores_df.columns:
        hit_col = "is_hit"
    else:
        hit_col = None
    if hit_col:
        hits_df = af3_scores_df[af3_scores_df[hit_col]]
        non_hits_df = af3_scores_df[~af3_scores_df[hit_col]]
        print(f"\nHits: {len(hits_df)} | Non-hits: {len(non_hits_df)}")
        print_subheader("Hit Distribution Percentiles")
        print(f"\nShowing where hits fall - use these as filter thresholds to capture most hits")
        print(f"\n  {'Metric':<35s} {'Op':<4s} {'Hit 25%':>10s} {'Hit 50%':>10s} {'Hit 75%':>10s} {'NonHit 50%':>12s}")
        print(f"  {'-'*77}")
        for base_metric, (operator, direction) in metrics_config.items():
            idx_cols = [f"{base_metric}_idx_{i}" for i in range(af3_N)]
            available_cols = [c for c in idx_cols if c in af3_scores_df.columns]
            if not available_cols:
                continue
            hit_vals = hits_df[available_cols].values.flatten()
            hit_vals = hit_vals[~np.isnan(hit_vals)]
            non_hit_vals = non_hits_df[available_cols].values.flatten()
            non_hit_vals = non_hit_vals[~np.isnan(non_hit_vals)]
            if len(hit_vals) < 2:
                continue
            hit_25 = np.percentile(hit_vals, 25)
            hit_50 = np.percentile(hit_vals, 50)
            hit_75 = np.percentile(hit_vals, 75)
            non_hit_50 = np.percentile(non_hit_vals, 50) if len(non_hit_vals) > 0 else np.nan
            print(f"  {base_metric:<35s} {operator:<4s} {hit_25:>10.2f} {hit_50:>10.2f} {hit_75:>10.2f} {non_hit_50:>12.2f}")
        print_subheader("Recommended Thresholds to Capture 75% of Hits")
        print("\n# Use these thresholds to keep ~75% of hits")
        print("# (Based on 75th percentile for '<=' and 25th percentile for '>=')\n")
        for base_metric, (operator, direction) in metrics_config.items():
            idx_cols = [f"{base_metric}_idx_{i}" for i in range(af3_N)]
            available_cols = [c for c in idx_cols if c in af3_scores_df.columns]
            if not available_cols:
                continue
            hit_vals = hits_df[available_cols].values.flatten()
            hit_vals = hit_vals[~np.isnan(hit_vals)]
            if len(hit_vals) < 2:
                continue
            if direction == "lower":
                threshold = np.percentile(hit_vals, 75)  # 75th percentile for lower-is-better
            else:
                threshold = np.percentile(hit_vals, 25)  # 25th percentile for higher-is-better
            if abs(threshold) < 1:
                formatted = f"{threshold:.3f}"
            elif abs(threshold) < 10:
                formatted = f"{threshold:.2f}"
            else:
                formatted = f"{threshold:.1f}"
            print(f'    ("{base_metric}", "{operator}"): {formatted},')

# ============================================================================
# SUMMARY
# ============================================================================
print_header("SUMMARY")
print(f"""
  Total metrics analyzed: {len(stats_df)}
  Total designs: {len(af3_scores_df)}
  AF3 predictions per design: {af3_N}

  Top {top_N_percent}% thresholds are computed as:
    - For '<=' filters (lower is better): {top_N_percent}th percentile
    - For '>=' filters (higher is better): {100-top_N_percent}th percentile

  Use the 'FILTERS DICT FORMAT' section above to copy-paste filter values.
""")
print_header("ANALYSIS COMPLETE")


### XIII.C.2 Filter

In [ ]:
############################################################
### COMPUTE AF3 STATISTICS FROM SCORES DF & DO FILTERING ###
############################################################

### LOAD THE DF & CREATE scaffold_family ###
af3_scores_df = pd.read_csv(os.path.join(IMPORTANT_DFS_DIR, 'modified_af3_i3__holo.csv'))

### PATHS & NUMBER OF AF3 STRUCTURES PER PDB ###
af3_N   = 5    # Number of AF3 predictions
af3_path = f"{AF3_OUT_DIR}i3__holo/"

### TOGGLE: APPLY FILTERS ONLY TO OVER-REPRESENTED CLUSTERS? ###
apply_filters_only_to_large_groups = False  # Set True to filter only groups with ≥ min_group_size examples

### GROUP SIZE THRESHOLD (for over-represented clusters) ###
min_group_size = 80

### DISPLAY CONTROLS ###
show_example = True
n_examples  = 10
sort_by     = "catres_rmsd"
ascending   = True
print_stat  = True

### SAVE FILTERED DF CONTROLS ###
save_filtered_df = True
save_filtered_df_path = os.path.join(IMPORTANT_DFS_DIR, "filtered_TEMP_modified_af3_i3.csv")

### AF3 FILTER OPTIONS ###
majority_threshold  = 0.8

ca_rmsd             = 1.0
catres_rmsd         = 1.0
catres_subset_rmsd  = 1.0
kcx_tips_rmsd       = 1.5
zinc_rmsd           = 1.25
hydroxide_rmsd      = 1.5
subst_rmsd          = 4.0
subst_core_rmsd     = 2.5
tm_score            = 0.95

chainA_plddt        = 90  # protein
chainB_plddt        = 90  # zinc1
chainC_plddt        = 90  # zinc2
chainD_plddt        = 80  # hydroxide OR substrate
chainE_plddt        = 80  # substrate
catres_plddt        = 90
catres_subset_plddt = 90
kcx_tips_plddt      = 90
subst_core_plddt    = 80
hydroxide_plddt     = 80

interface_ptm       = 0.9
ptm_of_protein      = 0.9

### PAE FILTER OPTIONS ###
chainA_chainB_pae_avg = 6.5  # protein - zinc
chainA_chainC_pae_avg = 6.5  # protein - zinc
chainA_chainD_pae_avg = 6.5  # protein - hydroxide
chainA_chainE_pae_avg = 6.5  # protein - substrate
chainB_chainC_pae_avg = 6.0  # zinc - zinc
chainB_chainD_pae_avg = 6.0  # zinc - hydroxide
chainB_chainE_pae_avg = 6.0  # zinc - substrate
chainC_chainD_pae_avg = 6.0  # zinc - hydroxide
chainC_chainE_pae_avg = 6.0  # zinc - substrate
chainD_chainE_pae_avg = 6.0  # hydroxide - substrate

chainA_chainB_pae_min = 1.25  # protein - zinc
chainA_chainC_pae_min = 1.25  # protein - zinc
chainA_chainD_pae_min = 1.5  # protein - hydroxide
chainA_chainE_pae_min = 1.5  # protein - substrate
chainB_chainC_pae_min = 1.9  # zinc - zinc
chainB_chainD_pae_min = 1.9  # zinc - hydroxide
chainB_chainE_pae_min = 6.0  # zinc - substrate
chainC_chainD_pae_min = 1.9  # zinc - hydroxide
chainC_chainE_pae_min = 6.0  # zinc - substrate
chainD_chainE_pae_min = 6.0  # hydroxide - substrate


### ADD BACKBONES BACK INTO FILTERED DF? ###
apply_scaffold_selection = False
# If the column string does NOT contain "_idx_" then the median across "<column>_idx_*" will be computed.
scaffold_selection_rules = [
    # {"column": "subst_rmsd", "order": "asc", "N": 1},
    {"column": "catres_rmsd","order": "asc", "N": 1},
]

### FILTER OPTIONS ###
# OR logic: use a tuple of column names as the key, e.g.: (("ca_rmsd_TMalign", "ca_rmsd"), "<="): val -> means: passes if ca_rmsd_TMalign_idx_i <= val  OR  ca_rmsd_idx_i <= val | Standard single-string keys still work exactly as before.
filters = {
    (("ca_rmsd_TMalign", "ca_rmsd"), "<="):                        ca_rmsd,
    (("catres_rmsd_TMalign", "catres_rmsd"), "<="):                catres_rmsd,
    (("catres_subset_rmsd_TMalign", "catres_subset_rmsd"), "<="):  catres_subset_rmsd,
    (("kcx_tips_rmsd_TMalign", "kcx_tips_rmsd"), "<="):            kcx_tips_rmsd,
    (("zinc_pair_avg_rmsd_TMalign", "zinc_pair_avg_rmsd"), "<="):  zinc_rmsd,
#    (("hydroxide_ion_rmsd_TMalign", "hydroxide_ion_rmsd"), "<="):  hydroxide_rmsd,
    (("subst_rmsd_TMalign", "subst_rmsd"), "<="):                  subst_rmsd,
    (("subst_core_rmsd_TMalign", "subst_core_rmsd"), "<="):        subst_core_rmsd,

    ("tm_score", ">="):        tm_score,
    ("chainA_plddt", ">="):    chainA_plddt,
   # ("chainB_plddt", ">="):    chainB_plddt,
   # ("chainC_plddt", ">="):    chainC_plddt,
   # ("chainD_plddt", ">="):    chainD_plddt,
#    ("chainE_plddt", ">="):    chainE_plddt,
    ("catres_plddt", ">="):    catres_plddt,
    ("catres_subset_plddt", ">="):    catres_subset_plddt,
    ("KCX3_extra_atoms_plddt", ">="): kcx_tips_plddt,
#    ("hydroxide_ion_plddt", ">="): hydroxide_plddt,

    ("iptm", ">="):            interface_ptm,
    ("ptm",  ">="):            ptm_of_protein,

#    ("chainA_chainB_pair_pae_mean", "<="): chainA_chainB_pae_avg,
#    ("chainA_chainC_pair_pae_mean", "<="): chainA_chainC_pae_avg,
#    ("chainA_chainD_pair_pae_mean", "<="): chainA_chainD_pae_avg,
#    ("chainA_chainE_pair_pae_mean", "<="): chainA_chainE_pae_avg,
#    ("chainB_chainC_pair_pae_mean", "<="): chainB_chainC_pae_avg,
#    ("chainB_chainD_pair_pae_mean", "<="): chainB_chainD_pae_avg,
#    ("chainB_chainE_pair_pae_mean", "<="): chainB_chainE_pae_avg,
#    ("chainC_chainD_pair_pae_mean", "<="): chainC_chainD_pae_avg,
#    ("chainC_chainE_pair_pae_mean", "<="): chainC_chainE_pae_avg,
#    ("chainD_chainE_pair_pae_mean", "<="): chainD_chainE_pae_avg,

   # ("chainA_chainB_pair_pae_min", "<="): chainA_chainB_pae_min,
   # ("chainA_chainC_pair_pae_min", "<="): chainA_chainC_pae_min,
   # ("chainA_chainD_pair_pae_min", "<="): chainA_chainD_pae_min,
#    ("chainA_chainE_pair_pae_min", "<="): chainA_chainE_pae_min,
   # ("chainB_chainC_pair_pae_min", "<="): chainB_chainC_pae_min,
   # ("chainB_chainD_pair_pae_min", "<="): chainB_chainD_pae_min,
#    ("chainB_chainE_pair_pae_min", "<="): chainB_chainE_pae_min,
   # ("chainC_chainD_pair_pae_min", "<="): chainC_chainD_pae_min,
#    ("chainC_chainE_pair_pae_min", "<="): chainC_chainE_pae_min,
#    ("chainD_chainE_pair_pae_min", "<="): chainD_chainE_pae_min,
}

### HELPERS FOR OR-GROUP FILTER KEYS ###
def _build_filter_expr(col_key, op, val, i, df_name):
    """Build an eval-able expression. col_key is a str or tuple-of-strs (OR group)."""
    if isinstance(col_key, tuple):
        parts = [f"({df_name}['{c}_idx_{i}'] {op} {val})" for c in col_key]
        return "(" + " | ".join(parts) + ")"
    else:
        return f"({df_name}['{col_key}_idx_{i}'] {op} {val})"

def _filter_label(col_key):
    """Human-readable label for printing."""
    if isinstance(col_key, tuple):
        return " OR ".join(col_key)
    return col_key

### FILTERING FUNCTION (compact but identical behavior, now with OR support) ###
def filtering_df(dataframe_name, filters, target_df, print_stat=True):
    if print_stat:
        print(f'[{dataframe_name}] ({len(target_df)} designs)')
    print_rows, filtered_desc = [], []
    for i in range(af3_N):
        row_bits, exprs = [], []
        for (col, op), val in filters.items():
            expr = _build_filter_expr(col, op, val, i, "target_df")
            exprs.append(expr)
            if print_stat and i == 0:
                label = _filter_label(col)
                row_bits.append(f'# [ {label.ljust(50)} {op.ljust(2)} {str(val).ljust(6)}]: ')
        mask = eval(" & ".join(exprs))
        if print_stat:
            for j, e in enumerate(exprs):
                m = eval(e)
                pct = f"({m.to_list().count(True)/len(target_df)*100:.1f} %)"
                if i == 0:
                    print_rows.append(row_bits[j])
                print_rows[j] += f"{str(m.to_list().count(True)).rjust(6)}  {pct.rjust(8)}   "
        filtered_desc.extend(target_df[mask]["af3_models_dir"].to_list())
        if print_stat:
            pct = f"({mask.sum()/len(target_df)*100:.1f} %)"
            if i == 0:
                print_rows.append(f'# [                             ALL                             ]: {str(mask.sum()).rjust(6)}  {pct.rjust(8)}   ')
            else:
                print_rows[-1] += f"{str(mask.sum()).rjust(6)}  {pct.rjust(8)}   "
    keep = set(d for d in set(filtered_desc) if (filtered_desc.count(d)/af3_N) >= majority_threshold)
    filtered_df = target_df[target_df["af3_models_dir"].isin(keep)]
    if print_stat:
        for line in print_rows: print(line)
        pct = f"({len(filtered_df)/len(target_df)*100:.1f} %)"
        print(f'# [                     MAJORITY VOTE ({int(majority_threshold*100)} %)                    ]: {str(len(filtered_df)).rjust(6)}  {pct.rjust(8)}')
    return filtered_df

### ORIGINAL FILTERING ###
total_unique_scaffolds = af3_scores_df["scaffold_family"].nunique()
before_clusters = pd.to_numeric(af3_scores_df["structure_cluster"], errors="coerce").nunique(dropna=True)
print(f"Total unique scaffold groups before filtering: {total_unique_scaffolds}")
print(f"Unique structure clusters BEFORE filtering: {before_clusters}\n")

if apply_filters_only_to_large_groups:
    # Identify over-represented groups
    group_counts = af3_scores_df["scaffold_family"].value_counts()
    large_groups = group_counts[group_counts >= min_group_size].index
    # Split into large-group subset and the rest
    df_large = af3_scores_df[af3_scores_df["scaffold_family"].isin(large_groups)].copy()
    df_small = af3_scores_df[~af3_scores_df["scaffold_family"].isin(large_groups)].copy()
    # Apply filtering only to the over-represented clusters
    filtered_large = filtering_df("af3_scores_df (large groups)", filters, df_large, print_stat=print_stat)
    # Keep the small-group data unfiltered; combine
    filtered_af3_score_df = pd.concat([filtered_large, df_small]).reset_index(drop=True)
else:
    filtered_af3_score_df = filtering_df("af3_scores_df", filters, af3_scores_df, print_stat=print_stat)

# Print unique scaffold info & counts at this stage
print(f"\nNumber of unique scaffolds (BB) in filtered designs: {filtered_af3_score_df['scaffold_family'].nunique()}")
print(f"Number of unique structure clusters in filtered designs: {pd.to_numeric(filtered_af3_score_df['structure_cluster'], errors='coerce').nunique(dropna=True)}")
print(f"\nNumber of Designs BEFORE Filtering: {len(af3_scores_df)}")
print(f"Number of Designs AFTER  Filtering: {len(filtered_af3_score_df)}")

### ADDITIONAL SCAFFOLD SELECTION OPTIONS ###
if apply_scaffold_selection:
    print("\n### ADDING BACK SCAFFOLDS BASED ON YOUR SELECTION CRITERIA ###")
    extra_selected = []
    present_scaffolds = set(filtered_af3_score_df["scaffold_family"].unique())
    for rule in scaffold_selection_rules:
        base, order, N_keep = rule["column"], rule["order"], rule["N"]
        for scaf, group in af3_scores_df.groupby("scaffold_family"):
            if scaf in present_scaffolds:
                continue
            group = group.copy()
            sort_col = base
            if "_idx_" not in base:
                cols = [c for c in group.columns if c.startswith(base + "_idx_")]
                if not cols:
                    continue
                group["median_" + base] = group[cols].median(axis=1)
                sort_col = "median_" + base
            if len(group) >= N_keep:
                extra_selected.append(group.sort_values(by=sort_col, ascending=(order == "asc")).head(N_keep))
    if extra_selected:
        extra_df = pd.concat(extra_selected).drop_duplicates(subset="af3_models_dir")
        print(f"Unique scaffolds in additional selection: {extra_df['scaffold_family'].nunique()}")
        combined = pd.concat([filtered_af3_score_df, extra_df]).drop_duplicates(subset="af3_models_dir")
        print(f"Designs after additional scaffold selection: {len(combined)}")
        print(f"Unique scaffolds after additional selection: {combined['scaffold_family'].nunique()}")
        filtered_af3_score_df = combined
    else:
        print("No additional designs selected via scaffold rules.")

### SHOW EXAMPLES & SAVE ###
# Build set of passing (af3_models_dir, idx) pairs from the filtering logic
passing_pairs = set()
for i in range(af3_N):
    mask = pd.Series([True] * len(filtered_af3_score_df), index=filtered_af3_score_df.index)
    for (col, op), val in filters.items():
        expr = _build_filter_expr(col, op, val, i, "filtered_af3_score_df")
        mask = mask & eval(expr)
    for desc in filtered_af3_score_df.loc[mask, "af3_models_dir"]:
        passing_pairs.add((desc, i))

# For each design, find the extremum value among passing indices
sort_col_base = sort_by
extremum_values = []
passing_info = {}
for idx, row in filtered_af3_score_df.iterrows():
    desc = row["af3_models_dir"]
    passing_idxs = [i for i in range(af3_N) if (desc, i) in passing_pairs]
    if not passing_idxs:
        extremum_values.append(float('inf') if ascending else float('-inf'))
        passing_info[desc] = ([], None, None)
        continue
    vals = {i: row[f"{sort_col_base}_idx_{i}"] for i in passing_idxs}
    rep_idx = min(vals, key=vals.get) if ascending else max(vals, key=vals.get)
    extremum_values.append(vals[rep_idx])
    passing_info[desc] = (passing_idxs, rep_idx, vals[rep_idx])

# Add passing_idx column (None for designs added via scaffold selection)
if "passing_idx" not in filtered_af3_score_df.columns:
    filtered_af3_score_df = filtered_af3_score_df.copy()
    filtered_af3_score_df["passing_idx"] = filtered_af3_score_df["af3_models_dir"].apply(lambda desc: sorted(passing_info[desc][0]) if desc in passing_info
and passing_info[desc][0] else None)

if show_example:
    filtered_af3_score_df["_sort_extremum"] = extremum_values
    filtered_af3_score_df.sort_values(by="_sort_extremum", ascending=ascending, inplace=True)
    desc_list = filtered_af3_score_df["af3_models_dir"].head(n_examples).to_list()
    cmd = "pymol "
    print(f"\nTop {n_examples} designs sorted by {'min' if ascending else 'max'} {sort_by} among passing indices:\n")
    for desc in desc_list:
        passing_idxs, rep_idx, rep_val = passing_info[desc]
        af3_pdb = os.path.join(desc, "*.pdb")
        ref_pdb = filtered_af3_score_df.loc[filtered_af3_score_df["af3_models_dir"] == desc, "ref_path"].iloc[0]
        cmd += af3_pdb + " " + ref_pdb + " "
        print(f"  passing_idx={passing_idxs}, rep_idx={rep_idx}, {sort_by}={rep_val:.3f} | {os.path.basename(desc)}")

    filtered_af3_score_df.drop(columns=["_sort_extremum"], inplace=True)
    print("\nPyMOL command:")
    print(cmd)
if save_filtered_df:
    filtered_af3_score_df.to_csv(save_filtered_df_path, index=False)
    print("\n")
    print(f"Dataframe was saved at {save_filtered_df_path}")

In [ ]:
############################################################
### COMPUTE AF3 STATISTICS FROM SCORES DF & DO FILTERING ###
############################################################

### LOAD THE DF & CREATE scaffold_family ###
af3_scores_df = pd.read_csv(os.path.join(IMPORTANT_DFS_DIR, 'filtered_TEMP_modified_af3_i3.csv'))

### PATHS & NUMBER OF AF3 STRUCTURES PER PDB ###
af3_N   = 5    # Number of AF3 predictions
af3_path = f"{AF3_OUT_DIR}i3__holo/"

### TOGGLE: APPLY FILTERS ONLY TO OVER-REPRESENTED CLUSTERS? ###
apply_filters_only_to_large_groups = False  # Set True to filter only groups with ≥ min_group_size examples

### GROUP SIZE THRESHOLD (for over-represented clusters) ###
min_group_size = 80

### DISPLAY CONTROLS ###
show_example = True
n_examples  = 10
sort_by     = "catres_rmsd"
ascending   = True
print_stat  = True

### SAVE FILTERED DF CONTROLS ###
save_filtered_df = True
save_filtered_df_path = os.path.join(IMPORTANT_DFS_DIR, "filtered_TEMP_modified_af3_i3__v2.csv")

### AF3 FILTER OPTIONS ###
majority_threshold  = 0.8

ca_rmsd             = 1.0
catres_rmsd         = 1.0
catres_subset_rmsd  = 1.0
kcx_tips_rmsd       = 1.25
zinc_rmsd           = 1.1
hydroxide_rmsd      = 1.75
subst_rmsd          = 3.5
subst_core_rmsd     = 2
tm_score            = 0.95

chainA_plddt        = 90  # protein
chainB_plddt        = 90  # zinc1
chainC_plddt        = 90  # zinc2
chainD_plddt        = 90  # hydroxide OR substrate
chainE_plddt        = 80  # substrate
catres_plddt        = 90
catres_subset_plddt = 90
kcx_tips_plddt      = 90
subst_core_plddt    = 80
hydroxide_plddt     = 80

interface_ptm       = 0.9
ptm_of_protein      = 0.9

### PAE FILTER OPTIONS ###
chainA_chainB_pae_avg = 6.5  # protein - zinc
chainA_chainC_pae_avg = 6.5  # protein - zinc
chainA_chainD_pae_avg = 6.5  # protein - hydroxide
chainA_chainE_pae_avg = 6.5  # protein - substrate
chainB_chainC_pae_avg = 6.0  # zinc - zinc
chainB_chainD_pae_avg = 6.0  # zinc - hydroxide
chainB_chainE_pae_avg = 6.0  # zinc - substrate
chainC_chainD_pae_avg = 6.0  # zinc - hydroxide
chainC_chainE_pae_avg = 6.0  # zinc - substrate
chainD_chainE_pae_avg = 6.0  # hydroxide - substrate

chainA_chainB_pae_min = 1.25  # protein - zinc
chainA_chainC_pae_min = 1.25  # protein - zinc
chainA_chainD_pae_min = 1.25  # protein - hydroxide
chainA_chainE_pae_min = 1.5  # protein - substrate
chainB_chainC_pae_min = 2.0  # zinc - zinc
chainB_chainD_pae_min = 2.0  # zinc - hydroxide
chainB_chainE_pae_min = 6.0  # zinc - substrate
chainC_chainD_pae_min = 2.0  # zinc - hydroxide
chainC_chainE_pae_min = 6.0  # zinc - substrate
chainD_chainE_pae_min = 6.0  # hydroxide - substrate


### ADD BACKBONES BACK INTO FILTERED DF? ###
apply_scaffold_selection = False
# If the column string does NOT contain "_idx_" then the median across "<column>_idx_*" will be computed.
scaffold_selection_rules = [
    # {"column": "subst_rmsd", "order": "asc", "N": 1},
    {"column": "catres_rmsd","order": "asc", "N": 1},
]

### FILTER OPTIONS ###
# OR logic: use a tuple of column names as the key, e.g.: (("ca_rmsd_TMalign", "ca_rmsd"), "<="): val -> means: passes if ca_rmsd_TMalign_idx_i <= val  OR  ca_rmsd_idx_i <= val | Standard single-string keys still work exactly as before.
filters = {
    (("ca_rmsd_TMalign", "ca_rmsd"), "<="):                        ca_rmsd,
    (("catres_rmsd_TMalign", "catres_rmsd"), "<="):                catres_rmsd,
    (("catres_subset_rmsd_TMalign", "catres_subset_rmsd"), "<="):  catres_subset_rmsd,
    (("kcx_tips_rmsd_TMalign", "kcx_tips_rmsd"), "<="):            kcx_tips_rmsd,
    (("zinc_pair_avg_rmsd_TMalign", "zinc_pair_avg_rmsd"), "<="):  zinc_rmsd,
#    (("hydroxide_ion_rmsd_TMalign", "hydroxide_ion_rmsd"), "<="):  hydroxide_rmsd,
    (("subst_rmsd_TMalign", "subst_rmsd"), "<="):                  subst_rmsd,
    (("subst_core_rmsd_TMalign", "subst_core_rmsd"), "<="):        subst_core_rmsd,

    ("tm_score", ">="):        tm_score,
    ("chainA_plddt", ">="):    chainA_plddt,
    ("chainB_plddt", ">="):    chainB_plddt,
    ("chainC_plddt", ">="):    chainC_plddt,
    ("chainD_plddt", ">="):    chainD_plddt,
#    ("chainE_plddt", ">="):    chainE_plddt,
    ("catres_plddt", ">="):    catres_plddt,
    ("catres_subset_plddt", ">="):    catres_subset_plddt,
    ("KCX3_extra_atoms_plddt", ">="): kcx_tips_plddt,
#    ("hydroxide_ion_plddt", ">="): hydroxide_plddt,

    ("iptm", ">="):            interface_ptm,
    ("ptm",  ">="):            ptm_of_protein,

#    ("chainA_chainB_pair_pae_mean", "<="): chainA_chainB_pae_avg,
#    ("chainA_chainC_pair_pae_mean", "<="): chainA_chainC_pae_avg,
#    ("chainA_chainD_pair_pae_mean", "<="): chainA_chainD_pae_avg,
#    ("chainA_chainE_pair_pae_mean", "<="): chainA_chainE_pae_avg,
#    ("chainB_chainC_pair_pae_mean", "<="): chainB_chainC_pae_avg,
#    ("chainB_chainD_pair_pae_mean", "<="): chainB_chainD_pae_avg,
#    ("chainB_chainE_pair_pae_mean", "<="): chainB_chainE_pae_avg,
#    ("chainC_chainD_pair_pae_mean", "<="): chainC_chainD_pae_avg,
#    ("chainC_chainE_pair_pae_mean", "<="): chainC_chainE_pae_avg,
#    ("chainD_chainE_pair_pae_mean", "<="): chainD_chainE_pae_avg,

    ("chainA_chainB_pair_pae_min", "<="): chainA_chainB_pae_min,
    ("chainA_chainC_pair_pae_min", "<="): chainA_chainC_pae_min,
    ("chainA_chainD_pair_pae_min", "<="): chainA_chainD_pae_min,
#    ("chainA_chainE_pair_pae_min", "<="): chainA_chainE_pae_min,
   # ("chainB_chainC_pair_pae_min", "<="): chainB_chainC_pae_min,
    ("chainB_chainD_pair_pae_min", "<="): chainB_chainD_pae_min,
#    ("chainB_chainE_pair_pae_min", "<="): chainB_chainE_pae_min,
   # ("chainC_chainD_pair_pae_min", "<="): chainC_chainD_pae_min,
#    ("chainC_chainE_pair_pae_min", "<="): chainC_chainE_pae_min,
#    ("chainD_chainE_pair_pae_min", "<="): chainD_chainE_pae_min,
}

### HELPERS FOR OR-GROUP FILTER KEYS ###
def _build_filter_expr(col_key, op, val, i, df_name):
    """Build an eval-able expression. col_key is a str or tuple-of-strs (OR group)."""
    if isinstance(col_key, tuple):
        parts = [f"({df_name}['{c}_idx_{i}'] {op} {val})" for c in col_key]
        return "(" + " | ".join(parts) + ")"
    else:
        return f"({df_name}['{col_key}_idx_{i}'] {op} {val})"

def _filter_label(col_key):
    """Human-readable label for printing."""
    if isinstance(col_key, tuple):
        return " OR ".join(col_key)
    return col_key

### FILTERING FUNCTION (compact but identical behavior, now with OR support) ###
def filtering_df(dataframe_name, filters, target_df, print_stat=True):
    if print_stat:
        print(f'[{dataframe_name}] ({len(target_df)} designs)')
    print_rows, filtered_desc = [], []
    for i in range(af3_N):
        row_bits, exprs = [], []
        for (col, op), val in filters.items():
            expr = _build_filter_expr(col, op, val, i, "target_df")
            exprs.append(expr)
            if print_stat and i == 0:
                label = _filter_label(col)
                row_bits.append(f'# [ {label.ljust(50)} {op.ljust(2)} {str(val).ljust(6)}]: ')
        mask = eval(" & ".join(exprs))
        if print_stat:
            for j, e in enumerate(exprs):
                m = eval(e)
                pct = f"({m.to_list().count(True)/len(target_df)*100:.1f} %)"
                if i == 0:
                    print_rows.append(row_bits[j])
                print_rows[j] += f"{str(m.to_list().count(True)).rjust(6)}  {pct.rjust(8)}   "
        filtered_desc.extend(target_df[mask]["af3_models_dir"].to_list())
        if print_stat:
            pct = f"({mask.sum()/len(target_df)*100:.1f} %)"
            if i == 0:
                print_rows.append(f'# [                             ALL                             ]: {str(mask.sum()).rjust(6)}  {pct.rjust(8)}   ')
            else:
                print_rows[-1] += f"{str(mask.sum()).rjust(6)}  {pct.rjust(8)}   "
    keep = set(d for d in set(filtered_desc) if (filtered_desc.count(d)/af3_N) >= majority_threshold)
    filtered_df = target_df[target_df["af3_models_dir"].isin(keep)]
    if print_stat:
        for line in print_rows: print(line)
        pct = f"({len(filtered_df)/len(target_df)*100:.1f} %)"
        print(f'# [                     MAJORITY VOTE ({int(majority_threshold*100)} %)                    ]: {str(len(filtered_df)).rjust(6)}  {pct.rjust(8)}')
    return filtered_df

### ORIGINAL FILTERING ###
total_unique_scaffolds = af3_scores_df["scaffold_family"].nunique()
before_clusters = pd.to_numeric(af3_scores_df["structure_cluster"], errors="coerce").nunique(dropna=True)
print(f"Total unique scaffold groups before filtering: {total_unique_scaffolds}")
print(f"Unique structure clusters BEFORE filtering: {before_clusters}\n")

if apply_filters_only_to_large_groups:
    # Identify over-represented groups
    group_counts = af3_scores_df["scaffold_family"].value_counts()
    large_groups = group_counts[group_counts >= min_group_size].index
    # Split into large-group subset and the rest
    df_large = af3_scores_df[af3_scores_df["scaffold_family"].isin(large_groups)].copy()
    df_small = af3_scores_df[~af3_scores_df["scaffold_family"].isin(large_groups)].copy()
    # Apply filtering only to the over-represented clusters
    filtered_large = filtering_df("af3_scores_df (large groups)", filters, df_large, print_stat=print_stat)
    # Keep the small-group data unfiltered; combine
    filtered_af3_score_df = pd.concat([filtered_large, df_small]).reset_index(drop=True)
else:
    filtered_af3_score_df = filtering_df("af3_scores_df", filters, af3_scores_df, print_stat=print_stat)

# Print unique scaffold info & counts at this stage
print(f"\nNumber of unique scaffolds (BB) in filtered designs: {filtered_af3_score_df['scaffold_family'].nunique()}")
print(f"Number of unique structure clusters in filtered designs: {pd.to_numeric(filtered_af3_score_df['structure_cluster'], errors='coerce').nunique(dropna=True)}")
print(f"\nNumber of Designs BEFORE Filtering: {len(af3_scores_df)}")
print(f"Number of Designs AFTER  Filtering: {len(filtered_af3_score_df)}")

### ADDITIONAL SCAFFOLD SELECTION OPTIONS ###
if apply_scaffold_selection:
    print("\n### ADDING BACK SCAFFOLDS BASED ON YOUR SELECTION CRITERIA ###")
    extra_selected = []
    present_scaffolds = set(filtered_af3_score_df["scaffold_family"].unique())
    for rule in scaffold_selection_rules:
        base, order, N_keep = rule["column"], rule["order"], rule["N"]
        for scaf, group in af3_scores_df.groupby("scaffold_family"):
            if scaf in present_scaffolds:
                continue
            group = group.copy()
            sort_col = base
            if "_idx_" not in base:
                cols = [c for c in group.columns if c.startswith(base + "_idx_")]
                if not cols:
                    continue
                group["median_" + base] = group[cols].median(axis=1)
                sort_col = "median_" + base
            if len(group) >= N_keep:
                extra_selected.append(group.sort_values(by=sort_col, ascending=(order == "asc")).head(N_keep))
    if extra_selected:
        extra_df = pd.concat(extra_selected).drop_duplicates(subset="af3_models_dir")
        print(f"Unique scaffolds in additional selection: {extra_df['scaffold_family'].nunique()}")
        combined = pd.concat([filtered_af3_score_df, extra_df]).drop_duplicates(subset="af3_models_dir")
        print(f"Designs after additional scaffold selection: {len(combined)}")
        print(f"Unique scaffolds after additional selection: {combined['scaffold_family'].nunique()}")
        filtered_af3_score_df = combined
    else:
        print("No additional designs selected via scaffold rules.")

### SHOW EXAMPLES & SAVE ###
# Build set of passing (af3_models_dir, idx) pairs from the filtering logic
passing_pairs = set()
for i in range(af3_N):
    mask = pd.Series([True] * len(filtered_af3_score_df), index=filtered_af3_score_df.index)
    for (col, op), val in filters.items():
        expr = _build_filter_expr(col, op, val, i, "filtered_af3_score_df")
        mask = mask & eval(expr)
    for desc in filtered_af3_score_df.loc[mask, "af3_models_dir"]:
        passing_pairs.add((desc, i))

# For each design, find the extremum value among passing indices
sort_col_base = sort_by
extremum_values = []
passing_info = {}
for idx, row in filtered_af3_score_df.iterrows():
    desc = row["af3_models_dir"]
    passing_idxs = [i for i in range(af3_N) if (desc, i) in passing_pairs]
    if not passing_idxs:
        extremum_values.append(float('inf') if ascending else float('-inf'))
        passing_info[desc] = ([], None, None)
        continue
    vals = {i: row[f"{sort_col_base}_idx_{i}"] for i in passing_idxs}
    rep_idx = min(vals, key=vals.get) if ascending else max(vals, key=vals.get)
    extremum_values.append(vals[rep_idx])
    passing_info[desc] = (passing_idxs, rep_idx, vals[rep_idx])

# Add passing_idx column (None for designs added via scaffold selection)
if "passing_idx" not in filtered_af3_score_df.columns:
    filtered_af3_score_df = filtered_af3_score_df.copy()
    filtered_af3_score_df["passing_idx"] = filtered_af3_score_df["af3_models_dir"].apply(lambda desc: sorted(passing_info[desc][0]) if desc in passing_info
and passing_info[desc][0] else None)

if show_example:
    filtered_af3_score_df["_sort_extremum"] = extremum_values
    filtered_af3_score_df.sort_values(by="_sort_extremum", ascending=ascending, inplace=True)
    desc_list = filtered_af3_score_df["af3_models_dir"].head(n_examples).to_list()
    cmd = "pymol "
    print(f"\nTop {n_examples} designs sorted by {'min' if ascending else 'max'} {sort_by} among passing indices:\n")
    for desc in desc_list:
        passing_idxs, rep_idx, rep_val = passing_info[desc]
        af3_pdb = os.path.join(desc, "*.pdb")
        ref_pdb = filtered_af3_score_df.loc[filtered_af3_score_df["af3_models_dir"] == desc, "ref_path"].iloc[0]
        cmd += af3_pdb + " " + ref_pdb + " "
        print(f"  passing_idx={passing_idxs}, rep_idx={rep_idx}, {sort_by}={rep_val:.3f} | {os.path.basename(desc)}")

    filtered_af3_score_df.drop(columns=["_sort_extremum"], inplace=True)
    print("\nPyMOL command:")
    print(cmd)
if save_filtered_df:
    filtered_af3_score_df.to_csv(save_filtered_df_path, index=False)
    print("\n")
    print(f"Dataframe was saved at {save_filtered_df_path}")

In [ ]:
############################################################
### COMPUTE AF3 STATISTICS FROM SCORES DF & DO FILTERING ###
############################################################

### LOAD THE DF & CREATE scaffold_family ###
af3_scores_df = pd.read_csv(os.path.join(IMPORTANT_DFS_DIR, 'filtered_TEMP_modified_af3_i3__v2.csv'))

### PATHS & NUMBER OF AF3 STRUCTURES PER PDB ###
af3_N   = 5    # Number of AF3 predictions
af3_path = f"{AF3_OUT_DIR}i3__holo/"

### TOGGLE: APPLY FILTERS ONLY TO OVER-REPRESENTED CLUSTERS? ###
apply_filters_only_to_large_groups = True  # Set True to filter only groups with ≥ min_group_size examples

### GROUP SIZE THRESHOLD (for over-represented clusters) ###
min_group_size = 200

### DISPLAY CONTROLS ###
show_example = True
n_examples  = 10
sort_by     = "catres_rmsd"
ascending   = True
print_stat  = True

### SAVE FILTERED DF CONTROLS ###
save_filtered_df = True
save_filtered_df_path = os.path.join(IMPORTANT_DFS_DIR, "filtered_TEMP_modified_af3_i3__v3.csv")

### AF3 FILTER OPTIONS ###
majority_threshold  = 0.8

ca_rmsd             = 1.0
catres_rmsd         = 1.0
catres_subset_rmsd  = 1.0
kcx_tips_rmsd       = 1.0
zinc_rmsd           = 1.0
hydroxide_rmsd      = 1.75
subst_rmsd          = 3.5
subst_core_rmsd     = 2
tm_score            = 0.95

chainA_plddt        = 93  # protein
chainB_plddt        = 93  # zinc1
chainC_plddt        = 90  # zinc2
chainD_plddt        = 90  # hydroxide OR substrate
chainE_plddt        = 80  # substrate
catres_plddt        = 90
catres_subset_plddt = 90
kcx_tips_plddt      = 90
subst_core_plddt    = 80
hydroxide_plddt     = 80

interface_ptm       = 0.9
ptm_of_protein      = 0.9

### PAE FILTER OPTIONS ###
chainA_chainB_pae_avg = 6.5  # protein - zinc
chainA_chainC_pae_avg = 6.5  # protein - zinc
chainA_chainD_pae_avg = 6.5  # protein - hydroxide
chainA_chainE_pae_avg = 6.5  # protein - substrate
chainB_chainC_pae_avg = 6.0  # zinc - zinc
chainB_chainD_pae_avg = 6.0  # zinc - hydroxide
chainB_chainE_pae_avg = 6.0  # zinc - substrate
chainC_chainD_pae_avg = 6.0  # zinc - hydroxide
chainC_chainE_pae_avg = 6.0  # zinc - substrate
chainD_chainE_pae_avg = 6.0  # hydroxide - substrate

chainA_chainB_pae_min = 1.0  # protein - zinc
chainA_chainC_pae_min = 1.0  # protein - zinc
chainA_chainD_pae_min = 1.0  # protein - hydroxide
chainA_chainE_pae_min = 1.5  # protein - substrate
chainB_chainC_pae_min = 2.0  # zinc - zinc
chainB_chainD_pae_min = 2.0  # zinc - hydroxide
chainB_chainE_pae_min = 6.0  # zinc - substrate
chainC_chainD_pae_min = 2.0  # zinc - hydroxide
chainC_chainE_pae_min = 6.0  # zinc - substrate
chainD_chainE_pae_min = 6.0  # hydroxide - substrate


### ADD BACKBONES BACK INTO FILTERED DF? ###
apply_scaffold_selection = False
# If the column string does NOT contain "_idx_" then the median across "<column>_idx_*" will be computed.
scaffold_selection_rules = [
    # {"column": "subst_rmsd", "order": "asc", "N": 1},
    {"column": "catres_rmsd","order": "asc", "N": 1},
]

### FILTER OPTIONS ###
# OR logic: use a tuple of column names as the key, e.g.: (("ca_rmsd_TMalign", "ca_rmsd"), "<="): val -> means: passes if ca_rmsd_TMalign_idx_i <= val  OR  ca_rmsd_idx_i <= val | Standard single-string keys still work exactly as before.
filters = {
    (("ca_rmsd_TMalign", "ca_rmsd"), "<="):                        ca_rmsd,
    (("catres_rmsd_TMalign", "catres_rmsd"), "<="):                catres_rmsd,
    (("catres_subset_rmsd_TMalign", "catres_subset_rmsd"), "<="):  catres_subset_rmsd,
    (("kcx_tips_rmsd_TMalign", "kcx_tips_rmsd"), "<="):            kcx_tips_rmsd,
    (("zinc_pair_avg_rmsd_TMalign", "zinc_pair_avg_rmsd"), "<="):  zinc_rmsd,
#    (("hydroxide_ion_rmsd_TMalign", "hydroxide_ion_rmsd"), "<="):  hydroxide_rmsd,
    (("subst_rmsd_TMalign", "subst_rmsd"), "<="):                  subst_rmsd,
    (("subst_core_rmsd_TMalign", "subst_core_rmsd"), "<="):        subst_core_rmsd,

    ("tm_score", ">="):        tm_score,
    ("chainA_plddt", ">="):    chainA_plddt,
    ("chainB_plddt", ">="):    chainB_plddt,
    ("chainC_plddt", ">="):    chainC_plddt,
    ("chainD_plddt", ">="):    chainD_plddt,
#    ("chainE_plddt", ">="):    chainE_plddt,
    ("catres_plddt", ">="):    catres_plddt,
    ("catres_subset_plddt", ">="):    catres_subset_plddt,
    ("KCX3_extra_atoms_plddt", ">="): kcx_tips_plddt,
#    ("hydroxide_ion_plddt", ">="): hydroxide_plddt,

    ("iptm", ">="):            interface_ptm,
    ("ptm",  ">="):            ptm_of_protein,

#    ("chainA_chainB_pair_pae_mean", "<="): chainA_chainB_pae_avg,
#    ("chainA_chainC_pair_pae_mean", "<="): chainA_chainC_pae_avg,
#    ("chainA_chainD_pair_pae_mean", "<="): chainA_chainD_pae_avg,
#    ("chainA_chainE_pair_pae_mean", "<="): chainA_chainE_pae_avg,
#    ("chainB_chainC_pair_pae_mean", "<="): chainB_chainC_pae_avg,
#    ("chainB_chainD_pair_pae_mean", "<="): chainB_chainD_pae_avg,
#    ("chainB_chainE_pair_pae_mean", "<="): chainB_chainE_pae_avg,
#    ("chainC_chainD_pair_pae_mean", "<="): chainC_chainD_pae_avg,
#    ("chainC_chainE_pair_pae_mean", "<="): chainC_chainE_pae_avg,
#    ("chainD_chainE_pair_pae_mean", "<="): chainD_chainE_pae_avg,

    ("chainA_chainB_pair_pae_min", "<="): chainA_chainB_pae_min,
    ("chainA_chainC_pair_pae_min", "<="): chainA_chainC_pae_min,
    ("chainA_chainD_pair_pae_min", "<="): chainA_chainD_pae_min,
#    ("chainA_chainE_pair_pae_min", "<="): chainA_chainE_pae_min,
   # ("chainB_chainC_pair_pae_min", "<="): chainB_chainC_pae_min,
    ("chainB_chainD_pair_pae_min", "<="): chainB_chainD_pae_min,
#    ("chainB_chainE_pair_pae_min", "<="): chainB_chainE_pae_min,
    ("chainC_chainD_pair_pae_min", "<="): chainC_chainD_pae_min,
#    ("chainC_chainE_pair_pae_min", "<="): chainC_chainE_pae_min,
#    ("chainD_chainE_pair_pae_min", "<="): chainD_chainE_pae_min,
}

### HELPERS FOR OR-GROUP FILTER KEYS ###
def _build_filter_expr(col_key, op, val, i, df_name):
    """Build an eval-able expression. col_key is a str or tuple-of-strs (OR group)."""
    if isinstance(col_key, tuple):
        parts = [f"({df_name}['{c}_idx_{i}'] {op} {val})" for c in col_key]
        return "(" + " | ".join(parts) + ")"
    else:
        return f"({df_name}['{col_key}_idx_{i}'] {op} {val})"

def _filter_label(col_key):
    """Human-readable label for printing."""
    if isinstance(col_key, tuple):
        return " OR ".join(col_key)
    return col_key

### FILTERING FUNCTION (compact but identical behavior, now with OR support) ###
def filtering_df(dataframe_name, filters, target_df, print_stat=True):
    if print_stat:
        print(f'[{dataframe_name}] ({len(target_df)} designs)')
    print_rows, filtered_desc = [], []
    for i in range(af3_N):
        row_bits, exprs = [], []
        for (col, op), val in filters.items():
            expr = _build_filter_expr(col, op, val, i, "target_df")
            exprs.append(expr)
            if print_stat and i == 0:
                label = _filter_label(col)
                row_bits.append(f'# [ {label.ljust(50)} {op.ljust(2)} {str(val).ljust(6)}]: ')
        mask = eval(" & ".join(exprs))
        if print_stat:
            for j, e in enumerate(exprs):
                m = eval(e)
                pct = f"({m.to_list().count(True)/len(target_df)*100:.1f} %)"
                if i == 0:
                    print_rows.append(row_bits[j])
                print_rows[j] += f"{str(m.to_list().count(True)).rjust(6)}  {pct.rjust(8)}   "
        filtered_desc.extend(target_df[mask]["af3_models_dir"].to_list())
        if print_stat:
            pct = f"({mask.sum()/len(target_df)*100:.1f} %)"
            if i == 0:
                print_rows.append(f'# [                             ALL                             ]: {str(mask.sum()).rjust(6)}  {pct.rjust(8)}   ')
            else:
                print_rows[-1] += f"{str(mask.sum()).rjust(6)}  {pct.rjust(8)}   "
    keep = set(d for d in set(filtered_desc) if (filtered_desc.count(d)/af3_N) >= majority_threshold)
    filtered_df = target_df[target_df["af3_models_dir"].isin(keep)]
    if print_stat:
        for line in print_rows: print(line)
        pct = f"({len(filtered_df)/len(target_df)*100:.1f} %)"
        print(f'# [                     MAJORITY VOTE ({int(majority_threshold*100)} %)                    ]: {str(len(filtered_df)).rjust(6)}  {pct.rjust(8)}')
    return filtered_df

### ORIGINAL FILTERING ###
total_unique_scaffolds = af3_scores_df["scaffold_family"].nunique()
before_clusters = pd.to_numeric(af3_scores_df["structure_cluster"], errors="coerce").nunique(dropna=True)
print(f"Total unique scaffold groups before filtering: {total_unique_scaffolds}")
print(f"Unique structure clusters BEFORE filtering: {before_clusters}\n")

if apply_filters_only_to_large_groups:
    # Identify over-represented groups
    group_counts = af3_scores_df["scaffold_family"].value_counts()
    large_groups = group_counts[group_counts >= min_group_size].index
    # Split into large-group subset and the rest
    df_large = af3_scores_df[af3_scores_df["scaffold_family"].isin(large_groups)].copy()
    df_small = af3_scores_df[~af3_scores_df["scaffold_family"].isin(large_groups)].copy()
    # Apply filtering only to the over-represented clusters
    filtered_large = filtering_df("af3_scores_df (large groups)", filters, df_large, print_stat=print_stat)
    # Keep the small-group data unfiltered; combine
    filtered_af3_score_df = pd.concat([filtered_large, df_small]).reset_index(drop=True)
else:
    filtered_af3_score_df = filtering_df("af3_scores_df", filters, af3_scores_df, print_stat=print_stat)

# Print unique scaffold info & counts at this stage
print(f"\nNumber of unique scaffolds (BB) in filtered designs: {filtered_af3_score_df['scaffold_family'].nunique()}")
print(f"Number of unique structure clusters in filtered designs: {pd.to_numeric(filtered_af3_score_df['structure_cluster'], errors='coerce').nunique(dropna=True)}")
print(f"\nNumber of Designs BEFORE Filtering: {len(af3_scores_df)}")
print(f"Number of Designs AFTER  Filtering: {len(filtered_af3_score_df)}")

### ADDITIONAL SCAFFOLD SELECTION OPTIONS ###
if apply_scaffold_selection:
    print("\n### ADDING BACK SCAFFOLDS BASED ON YOUR SELECTION CRITERIA ###")
    extra_selected = []
    present_scaffolds = set(filtered_af3_score_df["scaffold_family"].unique())
    for rule in scaffold_selection_rules:
        base, order, N_keep = rule["column"], rule["order"], rule["N"]
        for scaf, group in af3_scores_df.groupby("scaffold_family"):
            if scaf in present_scaffolds:
                continue
            group = group.copy()
            sort_col = base
            if "_idx_" not in base:
                cols = [c for c in group.columns if c.startswith(base + "_idx_")]
                if not cols:
                    continue
                group["median_" + base] = group[cols].median(axis=1)
                sort_col = "median_" + base
            if len(group) >= N_keep:
                extra_selected.append(group.sort_values(by=sort_col, ascending=(order == "asc")).head(N_keep))
    if extra_selected:
        extra_df = pd.concat(extra_selected).drop_duplicates(subset="af3_models_dir")
        print(f"Unique scaffolds in additional selection: {extra_df['scaffold_family'].nunique()}")
        combined = pd.concat([filtered_af3_score_df, extra_df]).drop_duplicates(subset="af3_models_dir")
        print(f"Designs after additional scaffold selection: {len(combined)}")
        print(f"Unique scaffolds after additional selection: {combined['scaffold_family'].nunique()}")
        filtered_af3_score_df = combined
    else:
        print("No additional designs selected via scaffold rules.")

### SHOW EXAMPLES & SAVE ###
# Build set of passing (af3_models_dir, idx) pairs from the filtering logic
passing_pairs = set()
for i in range(af3_N):
    mask = pd.Series([True] * len(filtered_af3_score_df), index=filtered_af3_score_df.index)
    for (col, op), val in filters.items():
        expr = _build_filter_expr(col, op, val, i, "filtered_af3_score_df")
        mask = mask & eval(expr)
    for desc in filtered_af3_score_df.loc[mask, "af3_models_dir"]:
        passing_pairs.add((desc, i))

# For each design, find the extremum value among passing indices
sort_col_base = sort_by
extremum_values = []
passing_info = {}
for idx, row in filtered_af3_score_df.iterrows():
    desc = row["af3_models_dir"]
    passing_idxs = [i for i in range(af3_N) if (desc, i) in passing_pairs]
    if not passing_idxs:
        extremum_values.append(float('inf') if ascending else float('-inf'))
        passing_info[desc] = ([], None, None)
        continue
    vals = {i: row[f"{sort_col_base}_idx_{i}"] for i in passing_idxs}
    rep_idx = min(vals, key=vals.get) if ascending else max(vals, key=vals.get)
    extremum_values.append(vals[rep_idx])
    passing_info[desc] = (passing_idxs, rep_idx, vals[rep_idx])

# Add passing_idx column (None for designs added via scaffold selection)
if "passing_idx" not in filtered_af3_score_df.columns:
    filtered_af3_score_df = filtered_af3_score_df.copy()
    filtered_af3_score_df["passing_idx"] = filtered_af3_score_df["af3_models_dir"].apply(lambda desc: sorted(passing_info[desc][0]) if desc in passing_info
and passing_info[desc][0] else None)

if show_example:
    filtered_af3_score_df["_sort_extremum"] = extremum_values
    filtered_af3_score_df.sort_values(by="_sort_extremum", ascending=ascending, inplace=True)
    desc_list = filtered_af3_score_df["af3_models_dir"].head(n_examples).to_list()
    cmd = "pymol "
    print(f"\nTop {n_examples} designs sorted by {'min' if ascending else 'max'} {sort_by} among passing indices:\n")
    for desc in desc_list:
        passing_idxs, rep_idx, rep_val = passing_info[desc]
        af3_pdb = os.path.join(desc, "*.pdb")
        ref_pdb = filtered_af3_score_df.loc[filtered_af3_score_df["af3_models_dir"] == desc, "ref_path"].iloc[0]
        cmd += af3_pdb + " " + ref_pdb + " "
        print(f"  passing_idx={passing_idxs}, rep_idx={rep_idx}, {sort_by}={rep_val:.3f} | {os.path.basename(desc)}")

    filtered_af3_score_df.drop(columns=["_sort_extremum"], inplace=True)
    print("\nPyMOL command:")
    print(cmd)
if save_filtered_df:
    filtered_af3_score_df.to_csv(save_filtered_df_path, index=False)
    print("\n")
    print(f"Dataframe was saved at {save_filtered_df_path}")

In [ ]:
############################################################
### APPLY FINAL AF3 HOLO ROUND 1 FILTERS TO ROUND 2 HOLO ###
############################################################

import os
from collections import Counter
import pandas as pd

### INPUTS ###
round2_af3_scores_path = os.path.join(IMPORTANT_DFS_DIR, "modified_af3_i3__holo.csv")
af3_N = 5
majority_threshold = 0.8
overrepresented_scaffold = "ZAPP_p1D1_rotP_1_ORI_05_C9_i_14_model_2_"

# Final AF3 holo round 1 filter set from IX.C.2, saved as filtered_TEMP_modified_af3_i2__v2.csv.
round1_final_filters = [
    ("ca_rmsd", ("ca_rmsd_TMalign", "ca_rmsd"), "<=", 1.0),
    ("catres_rmsd", ("catres_rmsd_TMalign", "catres_rmsd"), "<=", 1.0),
    ("catres_subset_rmsd", ("catres_subset_rmsd_TMalign", "catres_subset_rmsd"), "<=", 1.0),
    ("kcx_tips_rmsd", ("kcx_tips_rmsd_TMalign", "kcx_tips_rmsd"), "<=", 1.25),
    ("zinc_pair_avg_rmsd", ("zinc_pair_avg_rmsd_TMalign", "zinc_pair_avg_rmsd"), "<=", 1.1),
    ("subst_rmsd", ("subst_rmsd_TMalign", "subst_rmsd"), "<=", 4.0),
    ("subst_core_rmsd", ("subst_core_rmsd_TMalign", "subst_core_rmsd"), "<=", 2.0),
    ("tm_score", "tm_score", ">=", 0.95),
    ("chainA_plddt", "chainA_plddt", ">=", 90),
    ("chainB_plddt", "chainB_plddt", ">=", 90),
    ("chainC_plddt", "chainC_plddt", ">=", 90),
    ("chainD_plddt", "chainD_plddt", ">=", 87.5),
    ("catres_plddt", "catres_plddt", ">=", 90),
    ("catres_subset_plddt", "catres_subset_plddt", ">=", 90),
    ("KCX3_extra_atoms_plddt", "KCX3_extra_atoms_plddt", ">=", 90),
    ("iptm", "iptm", ">=", 0.9),
    ("ptm", "ptm", ">=", 0.9),
    ("chainA_chainB_pair_pae_min", "chainA_chainB_pair_pae_min", "<=", 1.25),
    ("chainA_chainC_pair_pae_min", "chainA_chainC_pair_pae_min", "<=", 1.25),
    ("chainA_chainD_pair_pae_min", "chainA_chainD_pair_pae_min", "<=", 1.25),
]

### LOAD ONLY REQUIRED COLUMNS ###
required_cols = ["af3_models_dir", "scaffold_family"]
for _, col_key, _, _ in round1_final_filters:
    metric_names = col_key if isinstance(col_key, tuple) else (col_key,)
    for metric in metric_names:
        required_cols.extend([f"{metric}_idx_{i}" for i in range(af3_N)])
required_cols = sorted(set(required_cols))

round2_af3_df = pd.read_csv(round2_af3_scores_path, usecols=required_cols)

missing_cols = [c for c in required_cols if c not in round2_af3_df.columns]
if missing_cols:
    raise KeyError(f"Missing columns in round 2 AF3 dataframe: {missing_cols}")
if round2_af3_df["af3_models_dir"].duplicated().any():
    raise ValueError("Duplicate af3_models_dir values found in round 2 AF3 dataframe")
if "scaffold_family" not in round2_af3_df.columns:
    raise KeyError("Missing scaffold_family column in round 2 AF3 dataframe")

### APPLY FILTERS ###
def _idx_filter_mask(df, col_key, op, cutoff, idx):
    metric_names = col_key if isinstance(col_key, tuple) else (col_key,)
    idx_masks = []
    for metric in metric_names:
        values = pd.to_numeric(df[f"{metric}_idx_{idx}"], errors="coerce")
        if op == "<=":
            idx_masks.append(values <= cutoff)
        elif op == ">=":
            idx_masks.append(values >= cutoff)
        else:
            raise ValueError(f"Unsupported operator: {op}")
    return pd.concat(idx_masks, axis=1).any(axis=1)

passing_desc = []
per_idx_pass_counts = []
for idx in range(af3_N):
    idx_mask = pd.Series(True, index=round2_af3_df.index)
    for _, col_key, op, cutoff in round1_final_filters:
        idx_mask &= _idx_filter_mask(round2_af3_df, col_key, op, cutoff, idx)
    per_idx_pass_counts.append(int(idx_mask.sum()))
    passing_desc.extend(round2_af3_df.loc[idx_mask, "af3_models_dir"].tolist())

passing_counts = Counter(passing_desc)
majority_pass_ids = {
    desc for desc, count in passing_counts.items()
    if count / af3_N >= majority_threshold
}

overrepresented_ids = set(
    round2_af3_df.loc[
        round2_af3_df["scaffold_family"] == overrepresented_scaffold,
        "af3_models_dir",
    ]
)
majority_pass_overrepresented_ids = majority_pass_ids & overrepresented_ids
majority_pass_minus_overrepresented_ids = majority_pass_ids - overrepresented_ids

### PRINT SUMMARY ###
print("Final AF3 holo round 1 filters applied to AF3 holo round 2 raw structures")
print(f"Input dataframe: {round2_af3_scores_path}")
print(f"Input structures: {len(round2_af3_df)}")
print(f"AF3 predictions per structure: {af3_N}")
print(f"Majority threshold: >= {int(majority_threshold * 100)}% of AF3 predictions")
print("")
print("Filters used:")
for label, col_key, op, cutoff in round1_final_filters:
    metric_label = " OR ".join(col_key) if isinstance(col_key, tuple) else col_key
    print(f"  {label}: {metric_label} {op} {cutoff}")
print("")
print(f"Per-index all-filter pass counts: {per_idx_pass_counts}")
print(f"Majority-vote structures passing round 1 final filters on round 2: {len(majority_pass_ids)}")
print(f"Overrepresented scaffold: {overrepresented_scaffold}")
print(f"Round 2 raw structures from overrepresented scaffold: {len(overrepresented_ids)}")
print(f"Passing majority-vote structures from overrepresented scaffold: {len(majority_pass_overrepresented_ids)}")
print(f"Passing majority-vote structures after subtracting overrepresented scaffold: {len(majority_pass_minus_overrepresented_ids)}")

In [ ]:
############################################################
### PLOT FINAL AF3 FILTERING ON CA RMSD VS CHAIN A PLDDT ###
############################################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### INPUTS ###
all_points_df_path = os.path.join(IMPORTANT_DFS_DIR, "modified_af3_i3__holo.csv")
global_filter_df_path = os.path.join(IMPORTANT_DFS_DIR, "filtered_TEMP_modified_af3_i3__v2.csv")
passed_filter_df_path = os.path.join(IMPORTANT_DFS_DIR, "filtered_TEMP_modified_af3_i3__v3.csv")

af3_N = 5
x_metric = "ca_rmsd"
y_metric = "chainA_plddt"

global_ca_rmsd_cutoff = 1.0
global_chainA_plddt_cutoff = 90
expected_n_passing = 286

os.makedirs(GRAPHS_DIR, exist_ok=True)

### REQUIRED COLUMNS ###
required_all_cols = ["af3_models_dir", "scaffold_family"]
for metric in [x_metric, y_metric]:
    required_all_cols.extend([f"{metric}_idx_{i}" for i in range(af3_N)])
required_global_filter_cols = ["af3_models_dir"]
required_passed_cols = ["af3_models_dir"]

### LOAD DATA ###
all_points_df = pd.read_csv(all_points_df_path, usecols=required_all_cols)
global_filter_df = pd.read_csv(global_filter_df_path, usecols=required_global_filter_cols)
passed_filter_df = pd.read_csv(passed_filter_df_path, usecols=required_passed_cols)

### SANITY CHECKS ###
missing_all_cols = [c for c in required_all_cols if c not in all_points_df.columns]
missing_global_filter_cols = [c for c in required_global_filter_cols if c not in global_filter_df.columns]
missing_passed_cols = [c for c in required_passed_cols if c not in passed_filter_df.columns]
if missing_all_cols:
    raise KeyError(f"Missing columns in all-points dataframe: {missing_all_cols}")
if missing_global_filter_cols:
    raise KeyError(f"Missing columns in global-filter dataframe: {missing_global_filter_cols}")
if missing_passed_cols:
    raise KeyError(f"Missing columns in passed-filter dataframe: {missing_passed_cols}")
if all_points_df["af3_models_dir"].duplicated().any():
    raise ValueError("Duplicate af3_models_dir values found in all_points_df")
if global_filter_df["af3_models_dir"].duplicated().any():
    raise ValueError("Duplicate af3_models_dir values found in global_filter_df")
if passed_filter_df["af3_models_dir"].duplicated().any():
    raise ValueError("Duplicate af3_models_dir values found in passed_filter_df")

global_filter_ids = set(global_filter_df["af3_models_dir"])
passed_ids = set(passed_filter_df["af3_models_dir"])
missing_global_filter_ids = global_filter_ids - set(all_points_df["af3_models_dir"])
missing_passed_ids = passed_ids - set(all_points_df["af3_models_dir"])
passed_not_in_global = passed_ids - global_filter_ids
if missing_global_filter_ids:
    raise ValueError(f"{len(missing_global_filter_ids)} global-filter designs are missing from the all-points dataframe")
if missing_passed_ids:
    raise ValueError(f"{len(missing_passed_ids)} final-pass designs are missing from the all-points dataframe")
if passed_not_in_global:
    raise ValueError(f"{len(passed_not_in_global)} final-pass designs are missing from the global-filter dataframe")

plot_df = all_points_df.copy()
plot_df["_passed_global_filter"] = plot_df["af3_models_dir"].isin(global_filter_ids)
plot_df["_passed_final_filter"] = plot_df["af3_models_dir"].isin(passed_ids)
n_red = int(plot_df["_passed_final_filter"].sum())
if n_red != expected_n_passing:
    raise AssertionError(f"Expected {expected_n_passing} red/final-pass points, found {n_red}")

def _metric_median(df, metric):
    cols = [f"{metric}_idx_{i}" for i in range(af3_N)]
    return df[cols].apply(pd.to_numeric, errors="coerce").median(axis=1)

# One point per design: median metric value across the 5 AF3 predictions.
plot_df["_plot_ca_rmsd"] = _metric_median(plot_df, x_metric)
plot_df["_plot_chainA_plddt"] = _metric_median(plot_df, y_metric)

if plot_df[["_plot_ca_rmsd", "_plot_chainA_plddt"]].isna().any().any():
    raise ValueError("NaN values found in plotting coordinates")

global_xy_cutoff_pass = (
    (plot_df["_plot_ca_rmsd"] <= global_ca_rmsd_cutoff)
    & (plot_df["_plot_chainA_plddt"] >= global_chainA_plddt_cutoff)
)
grey_inside_global_xy = int((global_xy_cutoff_pass & ~plot_df["_passed_final_filter"]).sum())
global_filter_plot_df = plot_df[plot_df["_passed_global_filter"]].copy()
if global_filter_plot_df.empty:
    raise ValueError("No datapoints passed the saved global-filter stage")

scaffold_counts_global = global_filter_plot_df["scaffold_family"].value_counts()
overrepresented_scaffold = scaffold_counts_global.index[0]
overrepresented_scaffold_n_global = int(scaffold_counts_global.iloc[0])
red_outside_global_xy = int((~global_xy_cutoff_pass & plot_df["_passed_final_filter"]).sum())

print(f"All points plotted: {len(plot_df)}")
print(f"Saved global-filter-passing points: {len(global_filter_plot_df)}")
print(f"Final-pass/red points: {n_red}")
print(f"Final-pass/red points outside ca_rmsd/chainA_plddt cutoff lines: {red_outside_global_xy}")
print(f"Gray points still on the favorable side of both cutoff lines: {grey_inside_global_xy}")
print(f"Most represented scaffold among saved global-filter-passing points: {overrepresented_scaffold} (n={overrepresented_scaffold_n_global})")

def _style_scatter_ax(ax, title):
    ax.axvline(global_ca_rmsd_cutoff, color="red", linestyle="--", linewidth=2)
    ax.axhline(global_chainA_plddt_cutoff, color="red", linestyle="--", linewidth=2, label="Global cutoffs")
    ax.set_xlabel("ca_rmsd (median across AF3 predictions)", fontsize=13, weight="semibold")
    ax.set_ylabel("chainA_plddt (median across AF3 predictions)", fontsize=13, weight="semibold")
    ax.set_title(title, fontsize=13, weight="semibold")
    ax.tick_params(axis="both", which="major", labelsize=11, direction="in", top=True, right=True)
    for side in ["left", "right", "top", "bottom"]:
        ax.spines[side].set_linewidth(1.5)
        ax.spines[side].set_edgecolor("black")
    ax.grid(True, alpha=0.2, linewidth=0.6)

### PLOT 1: ALL POINTS, FINAL FILTER IN RED ###
grey_df = plot_df[~plot_df["_passed_final_filter"]]
red_df = plot_df[plot_df["_passed_final_filter"]]

fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(
    grey_df["_plot_ca_rmsd"],
    grey_df["_plot_chainA_plddt"],
    c="gray",
    s=12,
    alpha=0.25,
    edgecolors="none",
    rasterized=True,
    label=f"Did not pass final filter (n={len(grey_df)})",
)
ax.scatter(
    red_df["_plot_ca_rmsd"],
    red_df["_plot_chainA_plddt"],
    c="red",
    s=22,
    alpha=0.85,
    edgecolors="black",
    linewidths=0.25,
    label=f"Passed final filter (n={n_red})",
)
_style_scatter_ax(ax, "XIII Holo-AF3 Final Filter: ca_rmsd vs chainA_plddt")
ax.legend(frameon=False, loc="lower right", bbox_to_anchor=(1.0, 0.12))
plt.tight_layout()
af3_all_save_path = os.path.join(GRAPHS_DIR, "af3_i3_holo_final_filter_ca_rmsd_chainAplddt_all.png")
plt.savefig(af3_all_save_path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {af3_all_save_path}")
plt.show()

### PLOT 2: SAVED GLOBAL-FILTER-PASSING POINTS, OVERREPRESENTED SCAFFOLD IN BLUE ###
global_other_scaffold_df = global_filter_plot_df[global_filter_plot_df["scaffold_family"] != overrepresented_scaffold]
global_overrepresented_scaffold_df = global_filter_plot_df[global_filter_plot_df["scaffold_family"] == overrepresented_scaffold]

fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(
    global_other_scaffold_df["_plot_ca_rmsd"],
    global_other_scaffold_df["_plot_chainA_plddt"],
    c="gray",
    s=12,
    alpha=0.25,
    edgecolors="none",
    rasterized=True,
    label=f"Other saved global-pass scaffolds (n={len(global_other_scaffold_df)})",
)
ax.scatter(
    global_overrepresented_scaffold_df["_plot_ca_rmsd"],
    global_overrepresented_scaffold_df["_plot_chainA_plddt"],
    c="blue",
    s=16,
    alpha=0.55,
    edgecolors="none",
    rasterized=True,
    label=f"Most represented scaffold (n={len(global_overrepresented_scaffold_df)})",
)
_style_scatter_ax(ax, "Saved Global-Filter-Passing Points: Overrepresented Scaffold")
ax.legend(frameon=False, loc="lower right", bbox_to_anchor=(1.0, 0.12))
plt.tight_layout()
af3_overrepresented_save_path = os.path.join(GRAPHS_DIR, "af3_i3_holo_global_filter_overrepresented_scaffold.png")
plt.savefig(af3_overrepresented_save_path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {af3_overrepresented_save_path}")
plt.show()

### PLOT 3: SAVED GLOBAL-FILTER-PASSING POINTS, FINAL FILTER IN RED ###
global_grey_df = global_filter_plot_df[~global_filter_plot_df["_passed_final_filter"]]
global_red_df = global_filter_plot_df[global_filter_plot_df["_passed_final_filter"]]

fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(
    global_grey_df["_plot_ca_rmsd"],
    global_grey_df["_plot_chainA_plddt"],
    c="gray",
    s=12,
    alpha=0.25,
    edgecolors="none",
    rasterized=True,
    label=f"Saved global pass only (n={len(global_grey_df)})",
)
ax.scatter(
    global_red_df["_plot_ca_rmsd"],
    global_red_df["_plot_chainA_plddt"],
    c="red",
    s=22,
    alpha=0.85,
    edgecolors="black",
    linewidths=0.25,
    label=f"Passed final filter (n={len(global_red_df)})",
)
_style_scatter_ax(ax, "Saved Global-Filter-Passing Points: Final Filter")
ax.legend(frameon=False, loc="lower right", bbox_to_anchor=(1.0, 0.12))
plt.tight_layout()
af3_global_final_save_path = os.path.join(GRAPHS_DIR, "af3_i3_holo_global_filter_final_pass_red.png")
plt.savefig(af3_global_final_save_path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {af3_global_final_save_path}")
plt.show()

## XIII.D Look at Highly Represented Scaffolds & Optionally Filter

### Optional: Look at Over-Represented Backbones 

In [ ]:
#####################################################################################
### ANALYZE DISTRIBUTION OF HOW MANY ENTRIES (ROWS) PER scaffold_family IN FILTER ###
#####################################################################################

### LOAD DF CSV ###
af3_scores_path = f"{IMPORTANT_DFS_DIR}filtered_TEMP_modified_af3_i3__v2.csv"
filtered_df = pd.read_csv(af3_scores_path)

### GROUP BY SCAFFOLDS & COUNT ###
filtered_df["scaffold_family_basename"] = filtered_df["scaffold_family"].astype(str).apply(os.path.basename)
group_counts = (filtered_df.groupby("scaffold_family").size().reset_index(name="count"))
group_counts["scaffold_family_basename"] = group_counts["scaffold_family"].astype(str).apply(os.path.basename)

### OPTIONAL CONTROLS TO PRINT DESCRIPTIONS FROM GROUPS THAT EXCEED A ROW THRESHOLD ###
# --- Toggle 4A: Print N random descriptions from each group with row_count > row_threshold
show_random_examples = False       # Change to False to disable this feature
row_threshold_for_random = 50      # Only consider groups with > this many rows
random_n = 25                      # Number of random descriptions to print from each group

# --- Toggle 4B: Print top/bottom N from each group by a specific column
show_top_bottom_examples = True    # Change to False to disable this feature
ascending_flag = True              # True => smallest->largest (e.g., "top" = best/lowest)
                                   # False => largest->smallest (e.g., "top" = worst/highest)
top_bottom_n = 1                  # Number to print
sort_column = "ca_rmsd_idx_0"      # Column on which to sort
row_threshold_for_sorted = 5

### SUMMARY STATS ###
num_scaffolds = len(group_counts)
more_than_10 = (group_counts["count"] > 10).sum()
more_than_20 = (group_counts["count"] > 20).sum()
more_than_30 = (group_counts["count"] > 30).sum()

print(f"Total unique scaffolds in filtered set: {num_scaffolds}")
print(f"Scaffolds with more than 10 rows: {more_than_10}")
print(f"Scaffolds with more than 20 rows: {more_than_20}")
print(f"Scaffolds with more than 30 rows: {more_than_30}\n")

### INSPECT & PLOT SCAFFOLDS WITH >1 DESIGN ###
multi_entry_groups = group_counts[group_counts["count"] > 1]
print("Scaffolds with more than 1 row (showing top 20):")
print(
    multi_entry_groups
        .sort_values("count", ascending=False)
        .head(20)[["scaffold_family_basename", "count"]]
        .rename(columns={"scaffold_family_basename":"scaffold_family (basename)"})
)
plt.figure(figsize=(7, 5))
plt.hist(multi_entry_groups["count"], bins=range(1, multi_entry_groups["count"].max() + 2), edgecolor='k')
plt.title("Distribution of scaffold_family Sizes (Groups > 1)")
plt.xlabel("Number of Rows (Designs) in Group")
plt.ylabel("Frequency")
plt.tight_layout()
af3_all_save_path = os.path.join(GRAPHS_DIR, "af3_i3__v2_holo_scaffold_distribution.png")
plt.savefig(af3_all_save_path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {af3_all_save_path}")
plt.show()

### EXECUTE ###
group_counts_sorted = group_counts.sort_values("count", ascending=False)

if show_random_examples:
    print("\n##############################################")
    print("### RANDOM EXAMPLES FROM LARGE SCAFFOLDS  ###")
    print("##############################################")
    # Filter to groups that exceed row_threshold_for_random
    big_groups = group_counts_sorted[group_counts_sorted["count"] > row_threshold_for_random]
    if big_groups.empty:
        print(f"(No scaffold families have > {row_threshold_for_random} rows; nothing to sample.)")
    for _, row in big_groups.iterrows():
        group_name_full = row["scaffold_family"]
        group_name_base = row["scaffold_family_basename"]
        group_size = row["count"]
        subset_df = filtered_df[filtered_df["scaffold_family"] == group_name_full]
        sample_size = min(random_n, group_size)
        sampled_paths = subset_df["af3_models_dir"].sample(n=sample_size, random_state=42).tolist()
        print(f"\n### scaffold_family '{group_name_base}' (n={group_size}) ###")
        print("### RANDOM SAMPLES ###")
        print("pymol " + " ".join(sampled_paths))

if show_top_bottom_examples:
    print("\n################################################################")
    print("### TOP/BOTTOM EXAMPLES BY A SPECIFIC COLUMN FROM BIG GROUPS ###")
    print("################################################################")
    big_groups = group_counts_sorted[group_counts_sorted["count"] > row_threshold_for_sorted]
    if big_groups.empty:
        print(f"(No scaffold families have > {row_threshold_for_sorted} rows; adjust 'row_threshold_for_sorted' or inspect 'group_counts_sorted'.)")
    else:
        for _, row in big_groups.iterrows():
            group_name_full = row["scaffold_family"]
            group_name_base = row["scaffold_family_basename"]
            group_size = row["count"]
            
            subset_df = filtered_df[filtered_df["scaffold_family"] == group_name_full]
            sorted_subset = subset_df.sort_values(by=sort_column, ascending=ascending_flag)
            
            selection_df = sorted_subset.head(top_bottom_n)
            print(f"\n### scaffold_family '{group_name_base}' (n={group_size}) ###")
            print(f"### FIRST {top_bottom_n} by '{sort_column}' (ascending_flag={ascending_flag}) ###")
            print("pymol " + " ".join([desc + "/*.pdb" for desc in selection_df["af3_models_dir"].tolist()]))

### Optional: Decide to Keep All Designs for Any Particular Scaffold? Remove All Designs for Any Particular Scaffold?

List of designs to remove:
group1_pte_pxon_ChenJACS_exact_v0__lig_YYE_0000_EEEE_rotP_1_D__ORI_01_combo2___1_model_1_cfg_False__cfgscale_NA__stepscale_1_500__gamma0_0_100__gamma_min_0_500__jitter_1_000


In [ ]:
############################################################################################
### FILTER TO KEEP ONLY SPECIFIED SCAFFOLD_FAMILY BASENAMES & SAVE TO NEW CSV ###
############################################################################################

### LOAD DF CSV ###
unfiltered_af3_scores_path = f"{IMPORTANT_DFS_DIR}af3_stats_iteration1_apo.csv"
af3_scores_path = os.path.join(IMPORTANT_DFS_DIR, f"filtered_{Path(unfiltered_af3_scores_path).stem}.csv")
filtered_df = pd.read_csv(af3_scores_path)

### INPUT: LIST OF BASENAMES TO KEEP ###
scaffold_families_to_keep = [
    "group1_pte_pxon_ChenJACS_exact_v0__lig_YYE_0000_EEEE_rotP_1_D__ORI_01_combo2___3_model_0_cfg_True__cfgscale_1_500__stepscale_1_500__gamma0_0_700__gamma_min_1_000__jitter_3_000",
    "group2_pte_pxon_ChenJACS_smwTSopt_v0__lig_YYF_0000_EEEE_rotP_1_D__ORI_01_combo1___1_model_0_cfg_True__cfgscale_2_000__stepscale_1_500__gamma0_0_700__gamma_min_0_500__jitter_2_000",
    # add more basenames here
]

### MAKE A BASENAME COLUMN JUST FOR FILTERING ###
filtered_df["scaffold_family_basename"] = filtered_df["scaffold_family"].astype(str).apply(os.path.basename)

### FILTER ###
kept_df = filtered_df[filtered_df["scaffold_family_basename"].isin(scaffold_families_to_keep)].copy()

### DROP THE HELPER COLUMN BEFORE SAVING ###
kept_df = kept_df.drop(columns=["scaffold_family_basename"])

### OUTPUT PATH ###
kept_csv_path = os.path.join(IMPORTANT_DFS_DIR, f"filtered_SCAFFOLD_FAMS_MUST_KEEP_{Path(unfiltered_af3_scores_path).stem}.csv")

### SAVE ###
kept_df.to_csv(kept_csv_path, index=False)

print(f"Subset saved to: {kept_csv_path}")
print(f"Rows before: {len(filtered_df)} | Rows after filtering: {len(kept_df)}")
print(f"Scaffold families kept: {scaffold_families_to_keep}")

### Filter Over-Represented AF3 Scaffolds Down

## XIII.E Copy AF3 Filtered Structures & Ref PDBs

In [ ]:
###############################################
### STEP 1: LOAD CSV & PICK COLUMNS OF INTEREST
###############################################

### INPUTS ###
unfiltered_af3_scores_path = f"{IMPORTANT_DFS_DIR}modified_af3_i3.csv"
filtered_df_path = os.path.join(IMPORTANT_DFS_DIR, f"filtered_TEMP_modified_af3_i3__v3.csv")

### PASSING_IDX MODE CONTROLS ###
use_passing_idx = True         # Set True to use passing_idx column, False for default behavior
fallback_on_nan = True         # If True, fall back to metric-based selection when passing_idx is NaN | False = design is excluded

### CONTROLS (ONLY FOR FALLBACK OR IF PASSING_IDX MODE OFF) ###
selection_metric = "ca_rmsd"   # which metric to use
af3_N            = 5           # number of AF3 predictions per design

N_pick     = 5           # how many indices per row to keep (used for fallback or default mode)
pick_order = "asc"       # "asc" = lowest values, "desc" = highest values

### LOAD ###
df = pd.read_csv(filtered_df_path)

### BUILD COLUMN LIST ###
metric_cols = [f"{selection_metric}_idx_{i}" for i in range(af3_N)]
columns_of_interest = ["af3_models_dir"] + [c for c in metric_cols if c in df.columns]
if "ref_path" in df.columns and "ref_path" not in columns_of_interest:
    columns_of_interest.append("ref_path")
if "passing_idx" in df.columns and "passing_idx" not in columns_of_interest:
    columns_of_interest.append("passing_idx")

### SLIMMED DF ###
df_slim = df[columns_of_interest].copy()
print(f"Loaded {len(df)} rows. Kept columns: {columns_of_interest}")

#############################################
### STEP 2: PICK TOP/BOTTOM-N IDX PER ROW ###
#############################################

# Metric columns we want to look at
metric_cols = [c for c in df_slim.columns if c.startswith(f"{selection_metric}_idx_")]

def pick_indices_by_metric(row, n_pick):
    """Pick indices based on metric values (original behavior)."""
    vals = pd.to_numeric(row[metric_cols], errors="coerce")
    if pick_order == "asc":
        chosen = vals.nsmallest(n_pick)
    else:
        chosen = vals.nlargest(n_pick)
    return [c.split("_")[-1] for c in chosen.index.tolist()]

def parse_passing_idx(val):
    """Parse passing_idx column value (handles string representation of list)."""
    if pd.isna(val):
        return None
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            import ast
            parsed = ast.literal_eval(val)
            return parsed if isinstance(parsed, list) else None
        except:
            return None
    return None

# Track statistics
n_used_passing_idx = 0
n_fallback = 0
n_default = 0
total_idx_count = 0
idx_count_distribution = {}  # Track how many designs have N indices

def pick_indices(row):
    global n_used_passing_idx, n_fallback, n_default, total_idx_count
    
    if use_passing_idx:
        passing = parse_passing_idx(row.get("passing_idx"))
        if passing is not None and len(passing) > 0:
            n_used_passing_idx += 1
            total_idx_count += len(passing)
            return [str(i) for i in passing]
        elif fallback_on_nan:
            n_fallback += 1
            indices = pick_indices_by_metric(row, N_pick)
            total_idx_count += len(indices)
            return indices
        else:
            # No fallback, return empty
            return []
    else:
        n_default += 1
        indices = pick_indices_by_metric(row, N_pick)
        total_idx_count += len(indices)
        return indices

# Apply row by row - get raw indices first
raw_indices = df_slim.apply(pick_indices, axis=1)

# Build distribution of index counts
for indices in raw_indices:
    n = len(indices)
    idx_count_distribution[n] = idx_count_distribution.get(n, 0) + 1

# Determine max number of indices across all rows
max_indices = raw_indices.apply(len).max()
idx_cols = [f"idx_to_copy_{i+1}" for i in range(max_indices)]

# Expand into columns with proper formatting
def format_indices(indices):
    formatted = [f"_idx_{i}_model.pdb" if i is not None else pd.NA for i in indices]
    # Pad with NA if fewer indices than max
    formatted.extend([pd.NA] * (max_indices - len(formatted)))
    return formatted

df_slim[idx_cols] = pd.DataFrame(raw_indices.apply(format_indices).tolist(), index=df_slim.index)

# Add basename of `af3_models_dir` as column "bn"
df_slim["bn"] = df_slim["af3_models_dir"].astype(str).map(lambda p: os.path.basename(os.path.normpath(p)))

### SUMMARY STATISTICS ###
print("\n" + "="*60)
print("SELECTION SUMMARY")
print("="*60)
if use_passing_idx:
    print(f"Mode: passing_idx selection (fallback={'ON' if fallback_on_nan else 'OFF'})")
    print(f"  - Used passing_idx:  {n_used_passing_idx:>6} rows")
    print(f"  - Fallback to metric:{n_fallback:>6} rows")
    if not fallback_on_nan:
        n_skipped = len(df_slim) - n_used_passing_idx
        print(f"  - Skipped (no idx):  {n_skipped:>6} rows")
else:
    print(f"Mode: metric-based selection ({selection_metric}, {pick_order}, N={N_pick})")
    print(f"  - Processed:         {n_default:>6} rows")

print("-"*60)
print("INDEX COUNT DISTRIBUTION:")
for n_idx in sorted(idx_count_distribution.keys()):
    count = idx_count_distribution[n_idx]
    pct = count / len(df_slim) * 100
    bar = "█" * int(pct / 2)  # Simple bar chart
    print(f"  {n_idx} idx pass: {count:>6} designs ({pct:>5.1f}%) {bar}")

print("-"*60)
print(f"TOTAL INDICES TO COPY: {total_idx_count}")
print("="*60)

print("\n")
print(df_slim.head())

In [ ]:
#############################################################
### STEP 3: COPY FILES WITH PROGRESS / WARNINGS / TIMING  ###
#############################################################

### INPUT ###
dest_dir = f"{AF3_OUT_DIR}filtered_i3"
os.makedirs(dest_dir, exist_ok=True)

### CONTROLS ###
preserve_metadata = False  # False = faster, True = keeps original timestamps/permissions

### OPTIONAL: copy reference PDBs listed in 'ref_path' into subdir 'ref_pdbs' ###
copy_ref_pdbs = True
ref_subdir = os.path.join(dest_dir, "ref_pdbs")

def should_log(n):
    """Log at 1, 10, 100, 500, then every 1000 until 10k, then every 10k."""
    if n <= 0:
        return False
    if n < 1000:
        return n in (1, 10, 100, 500)
    elif n < 10000:
        return n % 1000 == 0
    else:
        return n % 10000 == 0

def copy_file(src, dst):
    """Copy file with or without metadata based on setting."""
    if preserve_metadata:
        shutil.copy2(src, dst)
    else:
        shutil.copy(src, dst)

if copy_ref_pdbs and "ref_path" in df_slim.columns:
    ref_list = [str(p) for p in df_slim["ref_path"].tolist() if pd.notna(p) and str(p).strip()]
    all_exist = all(os.path.isfile(p) for p in ref_list)
    if not ref_list:
        print("[WARN] Skipping reference PDB copy: 'ref_path' column is present but empty.")
    elif not all_exist:
        print("[WARN] Skipping reference PDB copy: one or more 'ref_path' paths are missing.")
    else:
        os.makedirs(ref_subdir, exist_ok=True)
        seen = set()
        ref_jobs = []
        for p in ref_list:
            if p not in seen:
                seen.add(p)
                ref_jobs.append((p, os.path.join(ref_subdir, os.path.basename(p))))

        print(f"Copying {len(ref_jobs)} reference PDB(s) into: {ref_subdir}")
        r_start = time.time()
        r_count = 0
        r_errors = 0
        for src, dst in ref_jobs:
            try:
                copy_file(src, dst)
                r_count += 1
            except Exception:
                r_errors += 1
            
            if should_log(r_count) or r_count == len(ref_jobs):
                r_elapsed = time.time() - r_start
                r_avg = r_elapsed / r_count
                r_remaining = r_avg * (len(ref_jobs) - r_count)
                print(f"  [{r_count}/{len(ref_jobs)}] {r_elapsed:.1f}s elapsed | ETA: {r_remaining:.1f}s")
        
        r_elapsed_total = time.time() - r_start
        print(f"  Reference PDBs: {r_count}/{len(ref_jobs)} copied in {r_elapsed_total:.1f}s | Errors: {r_errors}")

### LOGIC ###
if "bn" not in df_slim.columns:
    df_slim["bn"] = df_slim["af3_models_dir"].astype(str).map(lambda p: os.path.basename(os.path.normpath(p)))

copy_jobs = [
    (os.path.join(row["af3_models_dir"], f"{row['bn']}{row[col]}"), 
     os.path.join(dest_dir, f"{row['bn']}{row[col]}"))
    for _, row in df_slim.iterrows()
    for col in idx_cols
    if pd.notna(row[col])
]

total_jobs = len(copy_jobs)
print(f"\nCopying {total_jobs} files into: {dest_dir}")

existing = sum(1 for _, dst in copy_jobs if os.path.exists(dst))
if existing:
    print(f"[WARN] {existing} files already exist (will be overwritten)")

start = time.time()
count = 0
errors = 0
missing = 0

for src, dst in copy_jobs:
    try:
        if not os.path.isfile(src):
            missing += 1
            continue
        copy_file(src, dst)
        count += 1
    except Exception:
        errors += 1

    processed = count + missing + errors
    if should_log(processed):
        elapsed = time.time() - start
        avg = elapsed / processed
        remaining = avg * (total_jobs - processed)
        print(f"  [{processed}/{total_jobs}] {elapsed:.1f}s elapsed | ETA: {remaining:.1f}s")

elapsed_total = time.time() - start

### FINAL SUMMARY ###
print("\n" + "="*60)
print("COPY SUMMARY")
print("="*60)
print(f"  Successfully copied: {count:>6} files")
print(f"  Missing sources:     {missing:>6} files")
print(f"  Copy errors:         {errors:>6} files")
print(f"  Total time:          {elapsed_total:>6.1f} seconds")
print(f"  Average rate:        {count/elapsed_total if elapsed_total > 0 else 0:>6.1f} files/s")
print("="*60)

# **XIV. Apo-Monomer Prediction with AF2**

## XIV.A Run AF2

In [ ]:
#######################################################################################
### PREDICT DESIGNS WITH AlphaFold2 & DO NOT MAKE COMMANDS FOR EXISTING PREDICTIONS ###
#######################################################################################

### CONSTANTS ###
apptainer = APPTAINER 
superfold = '{SOFTWARE_DIR}superfold/run_superfold.py'

### FOR CPU PROCESSING ###
ratio_0_to_1_of_cpu = 0
cpu_queue = 'cpu'
cpu_time = '08:00:00'
cpu_cmds_per_job = 1
cpu_job_name = 'af2_i3_after_af3_holo_CPU' #CHANGE WHEN NECESSARY

### FOR GPU PROCESSING ###
gpu_queue = 'gpu'
gpu_memory = "8G"
gpu_time = '01:00:00'
gpu_cmds_per_job = 4
gpu_job_name = 'af2_i3_after_af3_holo_GPU' #CHANGE WHEN NECESSARY

### INPUT/OUTPUT DIRECTORIES ###
#af2_inputs_directory = f"{AF3_OUT_DIR}filtered_i3/ref_pdbs" #CHANGE WHEN NECESSARY
af2_inputs_directory = f"{AF3_OUT_DIR}filtered_i2/ref_pdbs" #CHANGE WHEN NECESSARY


cpu_af2_updated_output_dir = f'{AF2_OUT_DIR}af2_i3/' #CHANGE WHEN NECESSARY
gpu_af2_updated_output_dir = f'{AF2_OUT_DIR}af2_i3/' #CHANGE WHEN NECESSARY

os.makedirs(cpu_af2_updated_output_dir, exist_ok=True)
os.makedirs(gpu_af2_updated_output_dir, exist_ok=True)

cpu_fold_cmd_name = cpu_job_name
gpu_fold_cmd_name = gpu_job_name

### ALPHAFOLD PARAMETERS ###
model_number = 4
model_type = 'monomer_ptm'
version = 'monomer'

nstruct = 1
num_recycles = 6
num_ensemble = 2
mock_msa_depth = 1
recycle_tol = 0.0

pct_seq_mask = 0.15
output_summary = ''
output_pae = '--output_pae'
amber_relax = ''
initial_guess = ''

#########################################
### PARSE PDB FILES & DAMAGE CONTROL  ###
#########################################

# 1. Collect all input PDB paths
all_input_pdbs = sorted(glob.glob(os.path.join(af2_inputs_directory, "*.pdb")))
print(f"Total PDBs found in input directory: {len(all_input_pdbs)}")

# 2. Gather existing output files from either CPU or GPU output folder
#    (they are the same directory here, so we only check one, e.g. cpu_af2_updated_output_dir)
existing_outputs = [f for f in os.listdir(cpu_af2_updated_output_dir) if f.endswith('.pdb')]
existing_basenames = [os.path.splitext(f)[0] for f in existing_outputs]
print(f"Already existing output structures: {len(existing_basenames)}")

# 3. Determine which inputs are 'to-do' (i.e., not already predicted)
pdb_files = []
matches_found = 0

for pdb_path in all_input_pdbs:
    base_in = os.path.splitext(os.path.basename(pdb_path))[0]
    # See if there's any existing output that "starts with" this base
    # (Adjust to == if you require exact matches.)
    match = False
    for out_base in list(existing_basenames):
        if out_base.startswith(base_in):
            matches_found += 1
            match = True
            # break as soon as we find one match
            break
    if not match:
        pdb_files.append(pdb_path)

print(f"Skipped {matches_found} input files because outputs already exist.")
print(f"PDB files remaining for new predictions: {len(pdb_files)}")

###############################
### SPLIT BETWEEN CPU & GPU ###
###############################
split_index = int(ratio_0_to_1_of_cpu * len(pdb_files))
cpu_pdb_files = pdb_files[:split_index]
gpu_pdb_files = pdb_files[split_index:]

### CREATE CPU COMMANDS ###
cpu_cmds = []
cpu_cmd = (f'{apptainer} {superfold} --out_dir {cpu_af2_updated_output_dir} '
           f'--model {model_number} --type {model_type} --version {version} '
           f'--nstruct {nstruct} --max_recycles {num_recycles} '
           f'--num_ensemble {num_ensemble} --mock_msa_depth {mock_msa_depth} '
           f'--recycle_tol {recycle_tol} --pct_seq_mask {pct_seq_mask} '
           f'{output_summary} {output_pae} {amber_relax}')

for i, pdb in enumerate(cpu_pdb_files):
    if i == 0 or i % 4 != 0:
        cpu_cmd += f" {pdb}"
    else:
        cpu_cmds.append(cpu_cmd + '\n')
        cpu_cmd = (f'{apptainer} {superfold} --out_dir {cpu_af2_updated_output_dir} '
                   f'--model {model_number} --type {model_type} --version {version} '
                   f'--nstruct {nstruct} --max_recycles {num_recycles} '
                   f'--num_ensemble {num_ensemble} --mock_msa_depth {mock_msa_depth} '
                   f'--recycle_tol {recycle_tol} --pct_seq_mask {pct_seq_mask} '
                   f'{output_summary} {output_pae} {amber_relax} '
                   f'{pdb}')

cpu_cmds.append(cpu_cmd + '\n')
with open(f'{CMDS_DIR}{cpu_job_name}', 'w+') as f:
    f.writelines(cpu_cmds)

### CREATE GPU COMMANDS ###
gpu_cmds = []
gpu_cmd = (f'XLA_FLAGS=--xla_gpu_force_compilation_parallelism=1 {apptainer} {superfold} --out_dir {gpu_af2_updated_output_dir} '
           f'--model {model_number} --type {model_type} --version {version} '
           f'--nstruct {nstruct} --max_recycles {num_recycles} '
           f'--num_ensemble {num_ensemble} --mock_msa_depth {mock_msa_depth} '
           f'--recycle_tol {recycle_tol} --pct_seq_mask {pct_seq_mask} '
           f'{output_summary} {output_pae} {amber_relax}')

for i, pdb in enumerate(gpu_pdb_files):
    if i == 0 or i % 4 != 0:
        gpu_cmd += f" {pdb}"
    else:
        gpu_cmds.append(gpu_cmd + '\n')
        gpu_cmd = (f'XLA_FLAGS=--xla_gpu_force_compilation_parallelism=1 {apptainer} {superfold} --out_dir {gpu_af2_updated_output_dir} '
                   f'--model {model_number} --type {model_type} --version {version} '
                   f'--nstruct {nstruct} --max_recycles {num_recycles} '
                   f'--num_ensemble {num_ensemble} --mock_msa_depth {mock_msa_depth} '
                   f'--recycle_tol {recycle_tol} --pct_seq_mask {pct_seq_mask} '
                   f'{output_summary} {output_pae} {amber_relax} '
                   f'{pdb}')

gpu_cmds.append(gpu_cmd + '\n')
with open(f'{CMDS_DIR}{gpu_job_name}', 'w+') as f:
    f.writelines(gpu_cmds)

### GENERATE CPU `.sh` FILE ###
nb.make_af2_submit_file(f'{SUBMIT_DIR}{cpu_job_name}.sh',f'{CMDS_DIR}{cpu_fold_cmd_name}',cpu_job_name,cpu_time,LOGS_DIR,cpu_cmds_per_job)

### GENERATE GPU `.sh` FILE ###
nb.make_af2_submit_file_with_mem_and_optional_gpu(f'{SUBMIT_DIR}{gpu_job_name}.sh',f'{CMDS_DIR}{gpu_fold_cmd_name}',gpu_job_name,gpu_time,LOGS_DIR,gpu_cmds_per_job,gpu_memory,gpu_queue)

### PRINT ###
print("")
print("### BREAKDOWN OF CPU & GPU USAGE ###")
print('Percentage of Commands Run on CPU:', ratio_0_to_1_of_cpu*100)
print('Percentage of Commands Run on GPU:', (1-ratio_0_to_1_of_cpu)*100)
print('')
print(f"### COMMANDS (CPU): ###") 
print(f"{CMDS_DIR}{cpu_fold_cmd_name}")
print('')
print('Number of CPU Jobs =', int(len(cpu_cmds)/cpu_cmds_per_job) + 1)
print('SUBMIT THIS BELOW:')
print(f'sbatch {SUBMIT_DIR}{cpu_job_name}.sh')
print('')
print(f"### COMMANDS (GPU): ###") 
print(f"{CMDS_DIR}{gpu_fold_cmd_name}")
print('')
print('Number of GPU Jobs =', int(len(gpu_cmds)/gpu_cmds_per_job) + 1)
print('SUBMIT THIS BELOW:')
print(f'sbatch {SUBMIT_DIR}{gpu_job_name}.sh')

## XIV.B Parse & Plot AF2 Global Metrics

In [ ]:
##################################################################
### PARSE & PROCESS AF2 OUTPUT FILES ONLY NEED TO DO THIS ONCE ###
##################################################################

# INPUT DIRECTORY TO PARSE
af2_directory_to_parse = f'{AF2_OUT_DIR}af2_i3/'  # CHANGE EVERY TIME

### OPTIONALS ###
optional_basename_for_summary_stats = None # "scores.sc"
optional_cpus = None #20 # Optional: specify the number of CPUs to use, or remove this variable to use default logic

### CONSTANTS ####
script = f"{SPECIAL_SCRIPTS_DIR}af2_analysis_and_tools/af2_parse_processing_multi.py"
container_path = APPTAINER

### GENERATE COMMANDS ###
command = f"singularity exec {container_path} python {script} {af2_directory_to_parse}"
if optional_cpus:
    command += f" --cpus {optional_cpus}"
if optional_basename_for_summary_stats:
    command += f" --optional_basename_for_summary_stats {optional_basename_for_summary_stats}"

### PRINT COMMAND TO SUBMIT ###
print("SUBMIT THIS:")
print(command)
print('')

### PRINT OUTPUT FILE ###
print("OUTPUT CSV FILE HERE:")
if not optional_basename_for_summary_stats:
    print(af2_directory_to_parse + 'af2_out_parsed.csv')
else:
    print(af2_directory_to_parse + optional_basename_for_summary_stats)

In [ ]:
#############################################
### LOAD AF2 .CSV AND GRAPH SUMMARY STATS ###
#############################################

### INPUT COMBINED .SC FILE TO LOAD ###
csv_file_path = f'{AF2_OUT_DIR}af2_i3/af2_out_parsed.csv'

save_basename_for_pic = "af2_i3_stats"

### OPTIONAL: SPECIFY ONLY A SUBSET OF COLUMNS TO PLOT ###
# If left as an empty list (or None), it will plot all numerical columns.
columns_to_plot = ['mean_plddt', 'rmsd_to_input', 'mean_pae', 'pTMscore', 'af2_convergence_tol', 'af2_elapsed_folding_time']

### LOAD THE DATA FOR A .csv FILE ###
af2_scores_df = pd.read_csv(csv_file_path)

##################
### PLOT STUFF ###
##################

# Colors for quartile shading
quartile_colors = ['skyblue', 'mediumseagreen', 'gold', 'salmon']

# Adjustable font scales
title_fontsize = 20
label_fontsize = 18
tick_fontsize  = 16

# Determine how many subplots we need (for example, 6 plots per row)
num_cols_per_row = 6
num_plots = len(columns_to_plot)
num_rows = int(np.ceil(num_plots / num_cols_per_row))

# Create a figure and axes grid
fig, axes = plt.subplots(num_rows, num_cols_per_row,
                         figsize=(6 * num_cols_per_row, 6 * num_rows))

# Flatten axes for easier iteration
axes = axes.flatten()

for idx, col in enumerate(columns_to_plot):
    ax = axes[idx]
    
    # --- 1) Plot KDE curve (black line) ---
    sns.kdeplot(data=af2_scores_df, x=col, ax=ax, fill=False, linewidth=2, color='black')
    
    # --- 2) Calculate quartiles ---
    q1, q2, q3 = af2_scores_df[col].quantile([0.25, 0.50, 0.75])
    
    # --- 3) Fill quartiles under the KDE curve ---
    if ax.lines:
        line = ax.lines[-1]        # KDE line
        x_data, y_data = line.get_data()
        
        # Fill 0-25%
        ax.fill_between(
            x_data, y_data,
            where=(x_data < q1),
            color=quartile_colors[0], alpha=0.5
        )
        # Fill 25-50%
        ax.fill_between(
            x_data, y_data,
            where=((x_data >= q1) & (x_data < q2)),
            color=quartile_colors[1], alpha=0.5
        )
        # Fill 50-75%
        ax.fill_between(
            x_data, y_data,
            where=((x_data >= q2) & (x_data < q3)),
            color=quartile_colors[2], alpha=0.5
        )
        # Fill 75-100%
        ax.fill_between(
            x_data, y_data,
            where=(x_data >= q3),
            color=quartile_colors[3], alpha=0.5
        )
    
    # Set title and labels
    ax.set_title(f'Distribution of {col}', fontsize=title_fontsize, weight='semibold')
    ax.set_xlabel(f'{col}', fontsize=label_fontsize, weight='semibold')
    ax.set_ylabel('Density', fontsize=label_fontsize, weight='semibold')
    
    # Customize tick labels
    ax.tick_params(
        axis='both', which='major',
        labelsize=tick_fontsize,
        direction='in', length=5, width=1.5,
        top=True, right=True
    )
    
    # Make spines visible and set style
    for side in ['left', 'right', 'top', 'bottom']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(2)
        ax.spines[side].set_edgecolor('black')

# Hide unused axes (if any)
for ax in axes[num_plots:]:
    ax.remove()

plt.suptitle('AlphaFold2 Output Stats (Quartile-Based KDE)', fontsize=24, weight='bold')
plt.tight_layout()

### SAVE THE FIGURE AS A .png FILE WITH THE GIVEN BASENAME ###
save_path = f'{GRAPHS_DIR}{save_basename_for_pic}.png'
plt.savefig(save_path, dpi=300)

plt.show()

In [ ]:
###################################################################
### SCATTER PLOT OF AF2 METRICS WITH OPTIONAL MAGNIFIED SUBPLOT ###
###################################################################

### INPUT COMBINED .CSV FILE TO LOAD ###
csv_file_path = f'{AF2_OUT_DIR}af2_i3/af2_out_parsed.csv'

save_basename_for_pic = "af2_i3_scatter_rmsd_plddt_coloredByPTM_withCutoffs"

### LOAD THE DATA FOR A .csv FILE ###
af2_scores_df = pd.read_csv(csv_file_path)

### CONFIGURE YOUR AXES & COLOR METRICS ###
x_axis_metric = 'rmsd_to_input'
y_axis_metric = 'mean_plddt'
color_metric  = 'pTMscore'

### SET OPTIONAL CUTOFFS (ANY OR ALL CAN BE USED) ###
x_cutoff_value     = 1.0    # None if no cutoff
x_cutoff_is_min    = False    # True => keep x >= cutoff, False => keep x <= cutoff

y_cutoff_value     = 80.0    # None if no cutoff
y_cutoff_is_min    = True    # True => keep y >= cutoff, False => keep y <= cutoff

color_cutoff_value = None     # None if no cutoff
color_cutoff_is_min= True    # True => keep color >= cutoff, False => keep color <= cutoff

#####################
### HELPER LOGIC  ###
#####################

def style_plot(ax, x_label, y_label, main_title):
    """
    Apply consistent styling to a single axes object:
    - Title, axis labels
    - Tick parameters
    - Spines (all black, thicker)
    """
    title_fontsize = 14
    label_fontsize = 18
    tick_fontsize  = 16

    ax.set_title(main_title, fontsize=title_fontsize, weight='semibold')
    ax.set_xlabel(x_label, fontsize=label_fontsize, weight='semibold')
    ax.set_ylabel(y_label, fontsize=label_fontsize, weight='semibold')
    
    ax.tick_params(
        axis='both', which='major',
        labelsize=tick_fontsize,
        direction='in', length=5, width=1.5,
        top=True, right=True
    )
    
    for side in ['left', 'right', 'top', 'bottom']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(2)
        ax.spines[side].set_edgecolor('black')

def filter_dataframe(df):
    """
    Given the global cutoff variables, return a filtered copy of the dataframe.
    Rows not meeting each cutoff are excluded.
    """
    df_filt = df.copy()
    
    # X-axis cutoff
    if x_cutoff_value is not None:
        if x_cutoff_is_min:
            df_filt = df_filt[df_filt[x_axis_metric] >= x_cutoff_value]
        else:
            df_filt = df_filt[df_filt[x_axis_metric] <= x_cutoff_value]
    
    # Y-axis cutoff
    if y_cutoff_value is not None:
        if y_cutoff_is_min:
            df_filt = df_filt[df_filt[y_axis_metric] >= y_cutoff_value]
        else:
            df_filt = df_filt[df_filt[y_axis_metric] <= y_cutoff_value]
    
    # Color metric cutoff
    if color_cutoff_value is not None:
        if color_cutoff_is_min:
            df_filt = df_filt[df_filt[color_metric] >= color_cutoff_value]
        else:
            df_filt = df_filt[df_filt[color_metric] <= color_cutoff_value]
    
    return df_filt

##################
### PLOT STUFF ###
##################

# Determine if we need one subplot (no cutoffs) or two subplots (some cutoff)
cutoffs_given = any([
    x_cutoff_value is not None,
    y_cutoff_value is not None,
    color_cutoff_value is not None
])

if cutoffs_given:
    fig, (ax_main, ax_filt) = plt.subplots(1, 2, figsize=(16, 6))
else:
    fig, ax_main = plt.subplots(figsize=(8, 6))

##############
### PLOT 1 ###
##############

sns.scatterplot(
    data=af2_scores_df,
    x=x_axis_metric,
    y=y_axis_metric,
    hue=color_metric,
    palette='viridis',
    ax=ax_main,
    edgecolor='black',
    alpha=0.7
)

# If x_cutoff_value is set, draw a vertical dashed line
if x_cutoff_value is not None:
    ax_main.axvline(
        x=x_cutoff_value, 
        color='red', 
        linestyle='--', 
        linewidth=2
    )

# If y_cutoff_value is set, draw a horizontal dashed line
if y_cutoff_value is not None:
    ax_main.axhline(
        y=y_cutoff_value, 
        color='red', 
        linestyle='--', 
        linewidth=2
    )

# Style the main plot
plot_title = f"{y_axis_metric} vs. {x_axis_metric} colored by {color_metric}"
style_plot(ax_main, x_axis_metric, y_axis_metric, plot_title)

#########################
### PLOT 2 (OPTIONAL) ###
#########################
if cutoffs_given:
    df_filtered = filter_dataframe(af2_scores_df)
    sns.scatterplot(
        data=df_filtered,
        x=x_axis_metric,
        y=y_axis_metric,
        hue=color_metric,
        palette='viridis',
        ax=ax_filt,
        edgecolor='black',
        alpha=0.7
    )
    
    # Style the filtered plot
    # Add a helpful title describing cutoffs
    desc = []
    if x_cutoff_value is not None:
        op = ">=" if x_cutoff_is_min else "<="
        desc.append(f"{x_axis_metric} {op} {x_cutoff_value}")
    if y_cutoff_value is not None:
        op = ">=" if y_cutoff_is_min else "<="
        desc.append(f"{y_axis_metric} {op} {y_cutoff_value}")
    if color_cutoff_value is not None:
        op = ">=" if color_cutoff_is_min else "<="
        desc.append(f"{color_metric} {op} {color_cutoff_value}")
    cutoff_desc = " & ".join(desc) if desc else "None"

    filt_title = f"Filtered: {cutoff_desc}"
    style_plot(ax_filt, x_axis_metric, y_axis_metric, filt_title)

plt.tight_layout()

### SAVE THE FIGURE AS A .png FILE WITH THE GIVEN BASENAME ###
save_path = f'{GRAPHS_DIR}{save_basename_for_pic}.png'
plt.savefig(save_path, dpi=300)
plt.show()

In [ ]:
############################################################
### AF2 FINAL FILTER SCATTERS: RMSD VS PLDDT              ###
############################################################

import os
import pandas as pd
import matplotlib.pyplot as plt

### INPUTS ###
af2_scores_df_path = f"{AF2_OUT_DIR}af2_i3/af2_out_parsed.csv"
os.makedirs(GRAPHS_DIR, exist_ok=True)

x_metric = "rmsd_to_input"
y_metric = "mean_plddt"

rmsd_cutoff = 2.5
plddt_cutoff = 80.0
pae_cutoff = 7.5
ptm_cutoff = 0.7
convergence_cutoff = 1.5

expected_n_starting = 321
expected_n_passing = 128
overrepresented_scaffold = "ZAPP_p1D1_rotP_1_ORI_05_C9_i_14_model_2_"

### LOAD DATA ###
required_cols = [
    "pdb_path",
    x_metric,
    y_metric,
    "mean_pae",
    "pTMscore",
    "af2_convergence_tol",
]
af2_plot_df = pd.read_csv(af2_scores_df_path, usecols=required_cols).copy()

### SANITY CHECKS ###
missing_cols = [c for c in required_cols if c not in af2_plot_df.columns]
if missing_cols:
    raise KeyError(f"Missing columns in AF2 dataframe: {missing_cols}")
if af2_plot_df["pdb_path"].duplicated().any():
    raise ValueError("Duplicate pdb_path values found in AF2 dataframe")
if len(af2_plot_df) != expected_n_starting:
    raise AssertionError(f"Expected {expected_n_starting} starting AF2 rows, found {len(af2_plot_df)}")

for col in [x_metric, y_metric, "mean_pae", "pTMscore", "af2_convergence_tol"]:
    af2_plot_df[col] = pd.to_numeric(af2_plot_df[col], errors="coerce")
if af2_plot_df[[x_metric, y_metric, "mean_pae", "pTMscore", "af2_convergence_tol"]].isna().any().any():
    raise ValueError("NaN values found in AF2 plotting/filtering metrics")

desc_base = af2_plot_df["pdb_path"].apply(os.path.basename)
af2_plot_df["scaffold_family"] = (
    desc_base.str.split("_inp", n=1).str[0]
    .str.split("_eV2", n=1).str[0]
    .str.split("_FS", n=1).str[0]
)
af2_plot_df["_is_overrepresented_scaffold"] = af2_plot_df["scaffold_family"] == overrepresented_scaffold
if not af2_plot_df["_is_overrepresented_scaffold"].any():
    raise ValueError(f"Overrepresented scaffold not found: {overrepresented_scaffold}")

af2_plot_df["_passes_plotted_cutoffs"] = (
    (af2_plot_df[x_metric] <= rmsd_cutoff)
    & (af2_plot_df[y_metric] >= plddt_cutoff)
)
af2_plot_df["_passes_final_filter"] = (
    af2_plot_df["_passes_plotted_cutoffs"]
    & (af2_plot_df["mean_pae"] <= pae_cutoff)
    & (af2_plot_df["pTMscore"] >= ptm_cutoff)
    & (af2_plot_df["af2_convergence_tol"] <= convergence_cutoff)
)

n_passing = int(af2_plot_df["_passes_final_filter"].sum())
if n_passing != expected_n_passing:
    raise AssertionError(f"Expected {expected_n_passing} final-pass AF2 rows, found {n_passing}")

overrepresented_total = int(af2_plot_df["_is_overrepresented_scaffold"].sum())
overrepresented_failed = int((af2_plot_df["_is_overrepresented_scaffold"] & ~af2_plot_df["_passes_final_filter"]).sum())
failed_total = int((~af2_plot_df["_passes_final_filter"]).sum())
pct_overrepresented_failed = overrepresented_failed / overrepresented_total * 100
pct_failed_from_overrepresented = overrepresented_failed / failed_total * 100

print(f"All AF2 points plotted: {len(af2_plot_df)}")
print(f"Final-pass/red points: {n_passing}")
print(f"Points passing only the displayed RMSD/pLDDT cutoffs: {int(af2_plot_df['_passes_plotted_cutoffs'].sum())}")
print(f"Overrepresented scaffold points: {overrepresented_total}")
print(f"Overrepresented scaffold failed final AF2 filter: {overrepresented_failed}/{overrepresented_total} ({pct_overrepresented_failed:.1f}%)")
print(f"Failed final AF2 filter made up by overrepresented scaffold: {overrepresented_failed}/{failed_total} ({pct_failed_from_overrepresented:.1f}%)")

def _style_af2_scatter_ax(ax, title):
    ax.axvline(rmsd_cutoff, color="red", linestyle="--", linewidth=2)
    ax.axhline(plddt_cutoff, color="red", linestyle="--", linewidth=2, label="Displayed RMSD/pLDDT cutoffs")
    ax.set_xlabel("rmsd_to_input", fontsize=13, weight="semibold")
    ax.set_ylabel("mean_plddt", fontsize=13, weight="semibold")
    ax.set_title(title, fontsize=13, weight="semibold")
    ax.tick_params(axis="both", which="major", labelsize=11, direction="in", top=True, right=True)
    for side in ["left", "right", "top", "bottom"]:
        ax.spines[side].set_linewidth(1.5)
        ax.spines[side].set_edgecolor("black")
    ax.grid(True, alpha=0.2, linewidth=0.6)

### PLOT 1: ALL AF2 POINTS, FINAL FILTER IN RED ###
af2_grey_df = af2_plot_df[~af2_plot_df["_passes_final_filter"]]
af2_red_df = af2_plot_df[af2_plot_df["_passes_final_filter"]]

fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(
    af2_grey_df[x_metric],
    af2_grey_df[y_metric],
    c="gray",
    s=22,
    alpha=0.35,
    edgecolors="none",
    rasterized=True,
    label=f"Did not pass final AF2 filter (n={len(af2_grey_df)})",
)
ax.scatter(
    af2_red_df[x_metric],
    af2_red_df[y_metric],
    c="red",
    s=34,
    alpha=0.85,
    edgecolors="black",
    linewidths=0.25,
    label=f"Passed final AF2 filter (n={len(af2_red_df)})",
)
_style_af2_scatter_ax(ax, "AF2 Final Filter: rmsd_to_input vs mean_plddt")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
af2_final_filter_save_path = os.path.join(GRAPHS_DIR, "af2_i3_final_filter_rmsd_to_input_mean_plddt_all.png")
plt.savefig(af2_final_filter_save_path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {af2_final_filter_save_path}")
plt.show()

### PLOT 2: ALL AF2 POINTS, OVERREPRESENTED AF3 SCAFFOLD IN BLUE ###
af2_other_scaffold_df = af2_plot_df[~af2_plot_df["_is_overrepresented_scaffold"]]
af2_overrepresented_scaffold_df = af2_plot_df[af2_plot_df["_is_overrepresented_scaffold"]]

fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(
    af2_other_scaffold_df[x_metric],
    af2_other_scaffold_df[y_metric],
    c="gray",
    s=22,
    alpha=0.35,
    edgecolors="none",
    rasterized=True,
    label=f"Other scaffolds (n={len(af2_other_scaffold_df)})",
)
ax.scatter(
    af2_overrepresented_scaffold_df[x_metric],
    af2_overrepresented_scaffold_df[y_metric],
    c="blue",
    s=30,
    alpha=0.65,
    edgecolors="none",
    rasterized=True,
    label=f"Overrepresented AF3 scaffold (n={len(af2_overrepresented_scaffold_df)})",
)
_style_af2_scatter_ax(ax, "AF2 Full Dataset: Overrepresented AF3 Scaffold")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
af2_overrepresented_save_path = os.path.join(GRAPHS_DIR, "af2_i3_overrepresented_scaffold_rmsd_to_input_mean_plddt_all.png")
plt.savefig(af2_overrepresented_save_path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {af2_overrepresented_save_path}")
plt.show()

In [ ]:
############################################################
### AF2 SCATTER: AF3 HOLO ROUND 1 SOURCES IN GREEN        ###
############################################################

import os
import pandas as pd
import matplotlib.pyplot as plt

### INPUTS ###
af2_scores_df_path = f"{AF2_OUT_DIR}af2_i3/af2_out_parsed.csv"
af2_output_dir = f"{AF2_OUT_DIR}af2_i3/"
af3_round1_final_df_path = os.path.join(IMPORTANT_DFS_DIR, "filtered_TEMP_modified_af3_i2__v2.csv")
os.makedirs(GRAPHS_DIR, exist_ok=True)

x_metric = "rmsd_to_input"
y_metric = "mean_plddt"

rmsd_cutoff = 2.5
plddt_cutoff = 80.0

expected_n_af2 = 321
expected_n_round1_sources = 101

### LOAD DATA ###
af2_plot_df = pd.read_csv(af2_scores_df_path, usecols=["pdb_path", x_metric, y_metric]).copy()
round1_source_df = pd.read_csv(af3_round1_final_df_path, usecols=["ref_path"]).copy()

### SANITY CHECKS ###
if len(af2_plot_df) != expected_n_af2:
    raise AssertionError(f"Expected {expected_n_af2} AF2 rows, found {len(af2_plot_df)}")
if len(round1_source_df) != expected_n_round1_sources:
    raise AssertionError(f"Expected {expected_n_round1_sources} AF3 holo round 1 sources, found {len(round1_source_df)}")
if af2_plot_df["pdb_path"].duplicated().any():
    raise ValueError("Duplicate pdb_path values found in AF2 dataframe")

for col in [x_metric, y_metric]:
    af2_plot_df[col] = pd.to_numeric(af2_plot_df[col], errors="coerce")
if af2_plot_df[[x_metric, y_metric]].isna().any().any():
    raise ValueError("NaN values found in AF2 plotting metrics")

def _strip_af2_suffix(path_or_stem):
    stem = os.path.splitext(os.path.basename(path_or_stem))[0]
    af2_suffix = "_model_4_ptm_seed_0_unrelaxed"
    if stem.endswith(af2_suffix):
        stem = stem[: -len(af2_suffix)]
    return stem

def _ref_base(path):
    return os.path.splitext(os.path.basename(path))[0]

af2_plot_df["_af2_input_base"] = af2_plot_df["pdb_path"].apply(_strip_af2_suffix)
round1_source_df["_round1_ref_base"] = round1_source_df["ref_path"].apply(_ref_base)

af2_base_set = set(af2_plot_df["_af2_input_base"])
if len(af2_base_set) != len(af2_plot_df):
    raise ValueError("Duplicate parsed AF2 input basenames found")

existing_output_stems = sorted(
    os.path.splitext(f)[0]
    for f in os.listdir(af2_output_dir)
    if f.endswith(".pdb")
)
af2_prefix_candidates = sorted(af2_base_set)

def _map_round1_ref_to_af2_base(round1_ref_base):
    if round1_ref_base in af2_base_set:
        return round1_ref_base, "exact"

    for output_stem in existing_output_stems:
        if output_stem.startswith(round1_ref_base):
            mapped_base = _strip_af2_suffix(output_stem)
            if mapped_base in af2_base_set:
                return mapped_base, "dedup_prefix"

    for af2_base in af2_prefix_candidates:
        # A redesign output is <ref base>_<combo tag>_<batch>_<seq>.
        if (
            af2_base.startswith(f"{round1_ref_base}_af3i2_idx_")
            or re.match(rf"{re.escape(round1_ref_base)}_(?:t[\d_]+|c\d+)_\d+_\d+$", af2_base)
        ):
            return af2_base, "dedup_prefix_parsed"

    return pd.NA, "unmapped"

round1_source_df[["_af2_input_base", "_mapping_mode"]] = round1_source_df["_round1_ref_base"].apply(
    lambda x: pd.Series(_map_round1_ref_to_af2_base(x))
)

unmapped_round1_df = round1_source_df[round1_source_df["_mapping_mode"] == "unmapped"]
if len(unmapped_round1_df):
    raise ValueError(f"Could not map {len(unmapped_round1_df)} AF3 holo round 1 sources to AF2 outputs")

round1_green_df = round1_source_df.merge(
    af2_plot_df[["_af2_input_base", x_metric, y_metric]],
    on="_af2_input_base",
    how="left",
    validate="many_to_one",
)
if round1_green_df[[x_metric, y_metric]].isna().any().any():
    raise ValueError("Mapped AF3 holo round 1 sources are missing AF2 plotting metrics")
if len(round1_green_df) != expected_n_round1_sources:
    raise AssertionError(f"Expected {expected_n_round1_sources} green source mappings, found {len(round1_green_df)}")

n_exact = int((round1_green_df["_mapping_mode"] == "exact").sum())
n_dedup = int(round1_green_df["_mapping_mode"].isin(["dedup_prefix", "dedup_prefix_parsed"]).sum())
n_unique_af2_rows_green = round1_green_df["_af2_input_base"].nunique()
duplicate_mappings = round1_green_df["_af2_input_base"].value_counts()
duplicate_mappings = duplicate_mappings[duplicate_mappings > 1]

print(f"All AF2 points plotted: {len(af2_plot_df)}")
print(f"AF3 holo round 1 source rows: {len(round1_source_df)}")
print(f"Green source mappings plotted: {len(round1_green_df)}")
print(f"Exact AF2 basename matches: {n_exact}")
print(f"Deduplicated/prefix AF2 mappings: {n_dedup}")
print(f"Unique AF2 rows represented by green mappings: {n_unique_af2_rows_green}")
if len(duplicate_mappings):
    print("Duplicate AF2 rows representing multiple round 1 sources:")
    for af2_base, n_sources in duplicate_mappings.items():
        print(f"  {af2_base}: {n_sources} round 1 sources")
else:
    print("Duplicate AF2 rows representing multiple round 1 sources: 0")

def _style_af2_round1_scatter_ax(ax, title):
    ax.axvline(rmsd_cutoff, color="red", linestyle="--", linewidth=2)
    ax.axhline(plddt_cutoff, color="red", linestyle="--", linewidth=2, label="Displayed RMSD/pLDDT cutoffs")
    ax.set_xlabel("rmsd_to_input", fontsize=13, weight="semibold")
    ax.set_ylabel("mean_plddt", fontsize=13, weight="semibold")
    ax.set_title(title, fontsize=13, weight="semibold")
    ax.tick_params(axis="both", which="major", labelsize=11, direction="in", top=True, right=True)
    for side in ["left", "right", "top", "bottom"]:
        ax.spines[side].set_linewidth(1.5)
        ax.spines[side].set_edgecolor("black")
    ax.grid(True, alpha=0.2, linewidth=0.6)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(
    af2_plot_df[x_metric],
    af2_plot_df[y_metric],
    c="gray",
    s=22,
    alpha=0.35,
    edgecolors="none",
    rasterized=True,
    label=f"All AF2 refolds (n={len(af2_plot_df)})",
)
ax.scatter(
    round1_green_df[x_metric],
    round1_green_df[y_metric],
    c="green",
    s=34,
    alpha=0.75,
    edgecolors="none",
    rasterized=True,
    label=f"AF3 holo round 1 sources (n={len(round1_green_df)})",
)
_style_af2_round1_scatter_ax(ax, "AF2 Refolds: AF3 Holo Round 1 Sources")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
af2_round1_sources_save_path = os.path.join(GRAPHS_DIR, "af2_i3_af3_holo_round1_sources_green.png")
plt.savefig(af2_round1_sources_save_path, dpi=300, bbox_inches="tight")
print(f"Saved figure: {af2_round1_sources_save_path}")
plt.show()

In [ ]:
# Count how many of the 101 AF3 holo round 1 source designs pass the final AF2 filter

af2_full_df = pd.read_csv(
    af2_scores_df_path,
    usecols=["pdb_path", "rmsd_to_input", "mean_plddt", "mean_pae", "pTMscore", "af2_convergence_tol"],
).copy()

af2_full_df["_af2_input_base"] = af2_full_df["pdb_path"].apply(_strip_af2_suffix)

af2_full_df["_passes_final_af2_filter"] = (
    (af2_full_df["rmsd_to_input"] <= 2.5)
    & (af2_full_df["mean_plddt"] >= 80.0)
    & (af2_full_df["mean_pae"] <= 7.5)
    & (af2_full_df["pTMscore"] >= 0.7)
    & (af2_full_df["af2_convergence_tol"] <= 1.5)
)

round1_pass_df = round1_source_df.merge(
    af2_full_df[["_af2_input_base", "_passes_final_af2_filter"]],
    on="_af2_input_base",
    how="left",
    validate="many_to_one",
)

n_round1_pass = int(round1_pass_df["_passes_final_af2_filter"].sum())
n_round1_total = len(round1_pass_df)

print(f"AF3 holo round 1 designs passing final AF2 filter: {n_round1_pass}/{n_round1_total}")
print(f"Percentage passing: {n_round1_pass / n_round1_total * 100:.1f}%")


## XIV.C Filter on AF2 Global Metrics

**Helpful website for understanding scores of confidence:** \
https://www.ebi.ac.uk/training/online/courses/alphafold/inputs-and-outputs/evaluating-alphafolds-predicted-structures-using-confidence-scores/pae-a-measure-of-global-confidence-in-alphafold-predictions/


**NOTE:** I am doing 2 rounds of filtering - (1) light one and any backbones that do not make the cut are thrown out, and (2) a harsher one but I will keep any backbones that are thrown out (for now).

In [ ]:
#########################################
### FILTER AND ANALYZE AF2 SCORE DATA ###
#########################################

### INPUT COMBINED .CSV FILE TO LOAD ###
csv_file_path = f'{AF2_OUT_DIR}af2_i3/af2_out_parsed.csv'
af2_scores_df = pd.read_csv(csv_file_path)

### SET FILTERING PARAMETERS ###
plddt_cutoff       = 80.0     # in pLDDT units -> usually 70-80
rmsd_cutoff        = 2.5     # Å, typical for Cα; 2.0 might be okay for large structures
pae_cutoff         = 7.5      # Below 5 is harsh but good -> not applicable here
pTM_cutoff         = 0.7      # Above 0.75 is very good; above 0.5 is likely correct
convergence_cutoff = 1.5     # Ensure reasonable convergence tolerance

### ADD LOWEST RMSD STRUCTURES BACK INTO FILTERED DATAFRAME ###
print_lowest_rmsd_per_unique_filteredOUT_backbone = False # for backbones that are completely filtered out, look at best structure from each
add_lowest_rmsd_back_to_filtered = False  # Control to toggle this functionality - if a scaffold group is filtered out we will add the lowest rmsd structure back in

### CONTROLS FOR ADDITIONAL PRINTING ###
print_random_samples = False
random_samples = 1

### CONTROLS FOR ANALYZING METRIC EXTREMA ###
print_top_or_bottom_n = True
category = 'rmsd_to_input'  # Category for sorting
n_structures = 5  # Number of top or bottom structures to print

### CREATE NEW COLUMN FOR RF FLOW SCAFFOLD GROUP ###
desc_base = af2_scores_df["pdb_path"].apply(os.path.basename)
af2_scores_df["scaffold_family"] = (desc_base.str.split("_inp", n=1).str[0].str.split("_eV2", n=1).str[0].str.split("_FS", n=1).str[0])
af2_scores_df["structure_cluster"] = desc_base.str.extract(r"FS(\d+)", expand=False).astype("Int64")

### APPLY FILTERS ###
af2_mask = (
    (af2_scores_df['mean_plddt'] >= plddt_cutoff) &
    (af2_scores_df['rmsd_to_input'] <= rmsd_cutoff) &
    (af2_scores_df['mean_pae'] <= pae_cutoff) &
    (af2_scores_df['pTMscore'] >= pTM_cutoff) &
    (af2_scores_df['af2_convergence_tol'] <= convergence_cutoff)
)

filtered_af2_scores_df = af2_scores_df[af2_mask]
af2_filter_rate = float(len(filtered_af2_scores_df)) / float(len(af2_scores_df)) * 100.0

### COUNT NUMBER OF UNIQUE SCAFFOLDS ###
unique_scaffolds_before = af2_scores_df['scaffold_family'].nunique()
unique_scaffolds_after = filtered_af2_scores_df['scaffold_family'].nunique()
unique_structures_before = af2_scores_df['structure_cluster'].nunique()
unique_structures_after = filtered_af2_scores_df['structure_cluster'].nunique()

### PRINT FILTER RESULTS ###
print(f"{len(filtered_af2_scores_df)} of {len(af2_scores_df)} ({round(af2_filter_rate, 3)}%) designs passed the filters:\n"
      f"  - pLDDT >= {plddt_cutoff}\n"
      f"  - Cα RMSD <= {rmsd_cutoff} Å\n"
      f"  - Convergence <= {convergence_cutoff} Å\n"
      f"  - pTM >= {pTM_cutoff}")
print("")
print(f"Unique scaffolds before filtering = {unique_scaffolds_before}")
print(f"Unique scaffolds after filtering = {unique_scaffolds_after}")
print(f"Scaffolds Lost = {unique_scaffolds_before - unique_scaffolds_after}")
print(f"\nUnique structure clusters before filtering = {unique_structures_before}")
print(f"Unique structure clusters after filtering = {unique_structures_after}")
print(f"Scaffolds Lost = {unique_structures_before - unique_structures_after}")

### FIND LOWEST RMSD STRUCTURE FOR EACH UNIQUE BACKBONE FILTERED OUT ###
if print_lowest_rmsd_per_unique_filteredOUT_backbone:
    filtered_out_scaffold_groups = set(af2_scores_df['scaffold_family']) - set(filtered_af2_scores_df['scaffold_family'])
    filtered_out_scaffolds = af2_scores_df[af2_scores_df['scaffold_family'].isin(filtered_out_scaffold_groups)]
    lowest_rmsd_filtered_out = filtered_out_scaffolds.loc[filtered_out_scaffolds.groupby('scaffold_family')['rmsd_to_input'].idxmin()]
    lowest_rmsd_filtered_out = lowest_rmsd_filtered_out.sort_values(by='rmsd_to_input')
    lowest_rmsd_pdb_paths = "pymol " + " ".join(lowest_rmsd_filtered_out['pdb_path'].tolist())
    print("")
    if add_lowest_rmsd_back_to_filtered:
        print("### LOWEST RMSD STRUCTURES FOR BACKBONES THAT DO NOT PASS THE FILTERS BUT WILL BE KEPT ###")
    else:
        print("### LOWEST RMSD STRUCTURES FOR FILTERED-OUT BACKBONES ###")
    print(lowest_rmsd_pdb_paths)

### PRESERVE LOWEST RMSD STRUCTURE OF EVERY SCAFFOLD (SO NO SCAFFOLD IS COMPLETELY FILTERED OUT) ###
if add_lowest_rmsd_back_to_filtered:
    filtered_out_scaffold_groups = set(af2_scores_df['scaffold_family']) - set(filtered_af2_scores_df['scaffold_family'])
    filtered_out_scaffolds = af2_scores_df[af2_scores_df['scaffold_family'].isin(filtered_out_scaffold_groups)]
    lowest_rmsd_filtered_out = filtered_out_scaffolds.loc[filtered_out_scaffolds.groupby('scaffold_family')['rmsd_to_input'].idxmin()]
    filtered_af2_scores_df = pd.concat([filtered_af2_scores_df, lowest_rmsd_filtered_out], ignore_index=True)
    # Remove duplicates in case of any overlap
    filtered_af2_scores_df = filtered_af2_scores_df.drop_duplicates(subset=['pdb_path']).reset_index(drop=True)
    print("")
    print("### UPDATED FILTERED DATAFRAME WITH ADDED LOWEST RMSD STRUCTURES ###")
    print(f"New filtered DataFrame size: {len(filtered_af2_scores_df)}")
    unique_scaffolds_after = filtered_af2_scores_df['scaffold_family'].nunique()
    print(f"Recalculated Scaffolds Lost = {unique_scaffolds_before - unique_scaffolds_after}")

### GATHER RANDOM SAMPLES FROM FILTERED DF ###
if print_random_samples:
    random_pdb_paths = filtered_af2_scores_df['pdb_path'].sample(n=random_samples, random_state=4).tolist()
    pymol_command = "pymol " + " ".join(random_pdb_paths)
    print("")
    print("### RANDOM FILES ###")
    print(pymol_command)

### PRINT TOP OR BOTTOM N STRUCTURES BY SPECIFIED CATEGORY ###
if print_top_or_bottom_n:
    sorted_af2_scores_df = filtered_af2_scores_df.sort_values(by=category)
    top_n = sorted_af2_scores_df.head(n_structures)
    bottom_n = sorted_af2_scores_df.tail(n_structures)
    print("")
    print(f"### TOP {n_structures} STRUCTURES BY {category} (LOWEST) ###")
    print("pymol " + " ".join(top_n['pdb_path'].tolist()))
    print("")
    print(f"### BOTTOM {n_structures} STRUCTURES BY {category} (HIGHEST) ###")
    print("pymol " + " ".join(bottom_n['pdb_path'].tolist()))

In [ ]:
csv_file_path = f'{AF2_OUT_DIR}af2_i3/af2_out_parsed.csv'
af2_scores_df = pd.read_csv(csv_file_path)

af2_scores_df['scaffold_family'] = af2_scores_df['pdb_path'].apply(lambda x: os.path.basename(x).split('_inp')[0].split('_enh')[0])

# Filter the dataframe to get the row where scaffold_family matches the given string
row = filtered_af2_scores_df[filtered_af2_scores_df["scaffold_family"] == 
    "group1_v0__lig_YYE_0000_EEEE_rotP_1_D__ORI_01_combo2__3_0_cfg_True__cfgsc_1_500__step_1_500__gam0_0_700__gam_min_1_000__jit_3_000"
]

# Display the row
row
print(len(row))


In [ ]:
############################################################################################
### COPY FILTERED STRUCTURES INTO SEPARATE DIRECTORY FOR ANALYSIS & SCORES.SC GENERATION ###
############################################################################################

### INPUTS ###
output_filtered_structures_dir = f'{AF2_OUT_DIR}af2_i3/filtered_structures/'
column_pdb = 'pdb_path'
column_json = 'af2_json_path'

### FUNCTION TO RUN ###
def copy_files(df, pdb_col, json_col, output_dir):
    existing_files = os.listdir(output_dir)
    print(f"Initial file count in the directory (Should = 0): {len(existing_files)}")
    if existing_files:
        print('')
        print(f"WARNING: The directory {output_dir} is not empty. It contains {len(existing_files)} files! Consider clearing that entire directory...")
        print('')
    print(f"Number of PDBs to Copy: {len(df)}")  
    print(f"Number of JSONs to Copy: {len(df)}")
    print(f"Copying Files to the Directory: {output_dir}")
    print('')
    print(f"Total Files to Copy = {len(df)*2}")
    count = 0
    start_time = time.time()
    error_files = []
    for index, row in df.iterrows():
        pdb_path = row[pdb_col]
        json_path = row[json_col]
        try:
            shutil.copy(pdb_path, output_dir)
            shutil.copy(json_path, output_dir)
            count += 2  # Increment for each PDB and JSON file copied
            if count % 2000 == 0:
                current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                elapsed_time = time.time() - start_time
                files_left = (len(df)*2 - count)
                average_time_per_file = elapsed_time / count
                estimated_time_remaining = max(0, average_time_per_file * files_left)
                print(f"[{current_time}] {count} files copied successfully. Elapsed time: {elapsed_time:.2f} seconds. Average time per file: {average_time_per_file:.4f} seconds. Estimated time remaining: {estimated_time_remaining:.2f} seconds.")
        except Exception as e:
            error_files.append((pdb_path, json_path, str(e)))
    print(f"Total files copied: {count}")
    if error_files:
        print("Failed to copy the following files:")
        for error in error_files:
            print(error)

### RUN THE SCRIPT ###
os.makedirs(output_filtered_structures_dir, exist_ok=True)
copy_files(filtered_af2_scores_df, column_pdb, column_json, output_filtered_structures_dir)
print('')
print('Done!')

In [ ]:
### SAVE DATAFRAME TO CSV ###
output_csv_path = f"{IMPORTANT_DFS_DIR}af2_i3_filtered.csv"
filtered_af2_scores_df.to_csv(output_csv_path, index=False)
print(f"\nDataFrame saved to CSV at: {output_csv_path}")

## XIV.D Parse & Analyze AF2 on Side Chain Metrics

This next cell just makes the same score file as we did in the parsing/plotting of AF2 metrics but in the filtered directory from the last step. It is necessary to do for the filtering step.

In [ ]:
##################################################################
### PARSE & PROCESS AF2 OUTPUT FILES ONLY NEED TO DO THIS ONCE ###
##################################################################

# INPUT DIRECTORY TO PARSE
af2_directory_to_parse = f'{AF2_OUT_DIR}af2_i3/filtered_structures/'  # CHANGE EVERY TIME

### OPTIONALS ###
optional_basename_for_summary_stats = "scores.sc"
optional_cpus = 20 #20 # Optional: specify the number of CPUs to use, or remove this variable to use default logic

### CONSTANTS ####
script = f"{SPECIAL_SCRIPTS_DIR}af2_analysis_and_tools/af2_parse_processing_multi.py"
container_path = APPTAINER

### GENERATE COMMANDS ###
command = f"singularity exec {container_path} python {script} {af2_directory_to_parse}"
if optional_cpus:
    command += f" --cpus {optional_cpus}"
if optional_basename_for_summary_stats:
    command += f" --optional_basename_for_summary_stats {optional_basename_for_summary_stats}"

### PRINT COMMAND TO SUBMIT ###
print("SUBMIT THIS:")
print(command)
print('')

### PRINT OUTPUT FILE ###
print("OUTPUT CSV FILE HERE:")
if not optional_basename_for_summary_stats:
    print(af2_directory_to_parse + 'af2_out_parsed.csv')
else:
    print(af2_directory_to_parse + optional_basename_for_summary_stats)

In [ ]:
##################################################################################################
### CALCULATE SIDE CHAIN RMSDS & OTHER RMSDS FOR CATALYTIC RESIDUES IN FILTERED STRUCTURES DIR ###
##################################################################################################

### INPUT SCORE PATH + OUTPUT SCORE PATH & REFERENCE FILES ###
score_path = f"{AF2_OUT_DIR}af2_i3/filtered_structures/scores.sc"
output_file = f"{AF2_OUT_DIR}af2_i3/filtered_structures/updated_scores.sc"

ref_pdbs_dir = f'{AF3_OUT_DIR}filtered_i3/ref_pdbs/'

### OPTIONAL (WILL SAVE TIME) ###
suffix_by_extension_to_match_with_previousFILE = '_model_4_ptm_seed_0_unrelaxed'  # e.g. '_model_4_ptm_seed_0_unrelaxed' OR None

### INPUT PARAMS ###
params_path = f"{params_files_dir}*.params"  # Ensure params include .pdb if needed

### INPUT ATOM RMSDS OF CAT RESIDUES TO CALCULATE & NUM OF SUFFIXES ###
atom_groups_json = {  # MANUALLY INPUT POSSIBLE CATALYTIC RESIDUES!!
    "HIS": [
        {"atoms": ["N", "CA", "C", "O"], "label": "bb_rmsd"},
        {"atoms": ["CA", "CB"], "label": "ca_cb_rmsd"},
        {"atoms": ["CG", "CE1", "ND1", "NE2", "CD2"], "label": "tips_rmsd"},
        {"atoms": ["ND1", "NE2"], "label": "tip_Ns_rmsd"}
    ],
    "GLU": [
        {"atoms": ["N", "CA", "C", "O"], "label": "bb_rmsd"},
        {"atoms": ["CA", "CB"], "label": "ca_cb_rmsd"},
        {"atoms": ["OE1", "OE2", "CD", "CG"], "label": "tips_rmsd"},
    ],
 #   "ASP": [
 #       {"atoms": ["N", "CA", "C", "O"], "label": "bb_rmsd"},
 #       {"atoms": ["CA", "CB"], "label": "ca_cb_rmsd"},
 #       {"atoms": ["OD1", "OD2", "CG", "CB"], "label": "tips_rmsd"},
 #   ],
    "LYS": [
        {"atoms": ["N", "CA", "C", "O"], "label": "bb_rmsd"},
        {"atoms": ["CA", "CB"], "label": "ca_cb_rmsd"},
        {"atoms": ["NZ", "CE"], "label": "tips_rmsd"},
        {"atoms": ["NZ", "CE", "CD", "CG", "CB"], "label": "sc_rmsd"},
    ],
}

### ADD _rmsd TO LABELS ###
for residue, groups in atom_groups_json.items():
    for group in groups:
        group["label"] += "_rmsd"

max_possible_num_of_suffixes = 1
column_name_for_pdb_path = 'pdb_path'

### CONSTANTS ###
apptainer = APPTAINER
script = f'{SPECIAL_SCRIPTS_DIR}af2_analysis_and_tools/sidechain_rmsd_and_info_af2_matching_res_v2.py' # sidechain_rmsd_and_info_af2_matching_res.py (V1) sidechain_rmsd_and_info_af2_matching_res_v2.py (V2) 
atom_groups = json.dumps(atom_groups_json)  # same dictionary as above

### CONTROLS FOR AUTOGENERATING THE JSON AS INPUT TEXT FOR THE NEXT CELL ###
num_catalytic_residues = 6 # per file - so his, his, his, glu or his, his, his, asp
auto_generate_column_names = True
additional_column_names = [] # e.g. ["cat_CUSTOM_extra"]

### BUILD THE COMMAND ###
command = (
    f'{apptainer} {script} '
    f'--scorefile {score_path} '
    f'--ref_pdbs_dir {ref_pdbs_dir} '
    f'--params {params_path} '
    f'--output_file {output_file} '
    f'--atom_groups \'{atom_groups}\''
)

if suffix_by_extension_to_match_with_previousFILE:
    max_possible_num_of_suffixes = 1
    command += f' --suffix_by_extension_to_match_with_previousFILE {suffix_by_extension_to_match_with_previousFILE}'
    command += f' --max_possible_num_of_suffixes {max_possible_num_of_suffixes}'
elif max_possible_num_of_suffixes:
    command += f' --max_possible_num_of_suffixes {max_possible_num_of_suffixes}'
    
### PRINT THE COMMAND ###
print("### NAVIGATE HERE ###")
print(f"cd {os.path.dirname(score_path)}")
print("")
print("### COMMAND TO EXECUTE ###")
print(command)
print("")
print("### NEW SCOREFILE ###")
print(f"{output_file}")

### AUTOGENERATION OF TEXT FOR NEXT CELL ###
def auto_generate_columns_from_atom_groups(atom_groups_dict):
    """
    Generates a list of column keys like 'cat_HIS_bb', 'cat_ASP_tips', etc.,
    but not including '_rmsd' in the key. The '_rmsd' portion will be appended
    only in the enumerated values.
    """
    generated_columns = []
    for residue, entries in atom_groups_dict.items():
        for entry in entries:
            label = entry["label"]  # e.g. "bb", "ca_cb", "tips"
            # So the final key is: cat_<RESIDUE>_<label>
            # e.g., "cat_HIS_bb"
            column_key = f"cat_{residue}_{label}"
            generated_columns.append(column_key)
    return generated_columns

def generate_rmsd_dict(column_names, num_catalytic_residues):
    """
    Generates a dictionary where each key is something like 'cat_HIS_bb'
    (without _rmsd), and each value is a list of enumerated RMSD strings:
    e.g., ["cat_HIS1_bb_rmsd", "cat_HIS2_bb_rmsd", ...].
    """
    result_dict = {}

    for column_name in column_names:
        # Remove _rmsd from the dictionary key if it exists
        clean_key = column_name.replace("_rmsd", "")

        enumerated_list = []
        for i in range(1, num_catalytic_residues + 1):
            parts = column_name.split('_', maxsplit=2)  # Split into ['cat', 'HIS', 'bb']
            if len(parts) == 3:
                base_str = f"{parts[0]}_{parts[1]}{i}_{parts[2]}"
            else:
                base_str = f"{column_name}{i}"

            enumerated_list.append(base_str)

        result_dict[clean_key] = enumerated_list

    return result_dict

if auto_generate_column_names:
    # Auto-generate from the dictionary:
    auto_cols = auto_generate_columns_from_atom_groups(atom_groups_json)
    # Then combine with user-specified extras:
    column_names = auto_cols + additional_column_names
else:
    # If not auto-generating, you can define them manually here:
    column_names = [
        # e.g. "cat_HIS_bb", "cat_HIS_ca_cb", "cat_HIS_tips"
    ] + additional_column_names


### GENERATE AND PRINT JSON FOR COLUMNS & THEIR SUB-ENTRIES ###
generated_dict = generate_rmsd_dict(column_names, num_catalytic_residues)
formatted_json = '{\n'
for key, value in generated_dict.items():
    formatted_json += f'    "{key}": {json.dumps(value)},\n'
formatted_json = formatted_json.rstrip(',\n') + '\n}'
print("")
print("################################################################################################")
print("### GENERATED JSON STRUCTURE FOR COLUMN NAMES & RMSD SUB-ENTRIES TO INPUT INTO THE NEXT CELL ###")
print("################################################################################################")
print(formatted_json)

**Matching predictions back to their inputs.** This step pairs each AlphaFold2
model with the design it came from. Names accumulate a suffix at every stage, so
prefix matching can pair a structure with the wrong stage's output — and because
each mismatch shifts the pairing, one bad match propagates through the rest of
the scorefile.

Set `suffix_by_extension_to_match_with_previousFILE` to the exact suffix the
prediction step appends. The script then strips that one suffix to recover the
reference name, which is unambiguous and faster than searching candidate
prefixes. It also reindexes catalytic residues in REMARK 666 order rather than by
residue number, so the `cat_HIS1..N` columns stay aligned across structures.

In [ ]:
###################################################################
### MODIFY .json IF USING THE V2 SCRIPT WHICH INDEXES CORRECTLY ###
###################################################################

### SPECIFY REMARK666 CATALYTIC RESIDUE ORDER ###
# Specify allowed positions for each amino acid. For example, only keep positions 1, 3, and 5 for HIS.
allowed_positions = {
    "HIS": [1, 2, 4, 5],
    "LYS": [3],
    "GLU": [6],
    "ASP": [6],
}

### INPUT .json FROM ABOVE ###
input_json = {
    "cat_HIS_bb": ["cat_HIS1_bb_rmsd_rmsd", "cat_HIS2_bb_rmsd_rmsd", "cat_HIS3_bb_rmsd_rmsd", "cat_HIS4_bb_rmsd_rmsd", "cat_HIS5_bb_rmsd_rmsd", "cat_HIS6_bb_rmsd_rmsd"],
    "cat_HIS_ca_cb": ["cat_HIS1_ca_cb_rmsd_rmsd", "cat_HIS2_ca_cb_rmsd_rmsd", "cat_HIS3_ca_cb_rmsd_rmsd", "cat_HIS4_ca_cb_rmsd_rmsd", "cat_HIS5_ca_cb_rmsd_rmsd", "cat_HIS6_ca_cb_rmsd_rmsd"],
    "cat_HIS_tips": ["cat_HIS1_tips_rmsd_rmsd", "cat_HIS2_tips_rmsd_rmsd", "cat_HIS3_tips_rmsd_rmsd", "cat_HIS4_tips_rmsd_rmsd", "cat_HIS5_tips_rmsd_rmsd", "cat_HIS6_tips_rmsd_rmsd"],
    "cat_HIS_tip_Ns": ["cat_HIS1_tip_Ns_rmsd_rmsd", "cat_HIS2_tip_Ns_rmsd_rmsd", "cat_HIS3_tip_Ns_rmsd_rmsd", "cat_HIS4_tip_Ns_rmsd_rmsd", "cat_HIS5_tip_Ns_rmsd_rmsd", "cat_HIS6_tip_Ns_rmsd_rmsd"],
    "cat_GLU_bb": ["cat_GLU1_bb_rmsd_rmsd", "cat_GLU2_bb_rmsd_rmsd", "cat_GLU3_bb_rmsd_rmsd", "cat_GLU4_bb_rmsd_rmsd", "cat_GLU5_bb_rmsd_rmsd", "cat_GLU6_bb_rmsd_rmsd"],
    "cat_GLU_ca_cb": ["cat_GLU1_ca_cb_rmsd_rmsd", "cat_GLU2_ca_cb_rmsd_rmsd", "cat_GLU3_ca_cb_rmsd_rmsd", "cat_GLU4_ca_cb_rmsd_rmsd", "cat_GLU5_ca_cb_rmsd_rmsd", "cat_GLU6_ca_cb_rmsd_rmsd"],
    "cat_GLU_tips": ["cat_GLU1_tips_rmsd_rmsd", "cat_GLU2_tips_rmsd_rmsd", "cat_GLU3_tips_rmsd_rmsd", "cat_GLU4_tips_rmsd_rmsd", "cat_GLU5_tips_rmsd_rmsd", "cat_GLU6_tips_rmsd_rmsd"],
    "cat_ASP_bb": ["cat_ASP1_bb_rmsd_rmsd", "cat_ASP2_bb_rmsd_rmsd", "cat_ASP3_bb_rmsd_rmsd", "cat_ASP4_bb_rmsd_rmsd", "cat_ASP5_bb_rmsd_rmsd", "cat_ASP6_bb_rmsd_rmsd"],
    "cat_ASP_ca_cb": ["cat_ASP1_ca_cb_rmsd_rmsd", "cat_ASP2_ca_cb_rmsd_rmsd", "cat_ASP3_ca_cb_rmsd_rmsd", "cat_ASP4_ca_cb_rmsd_rmsd", "cat_ASP5_ca_cb_rmsd_rmsd", "cat_ASP6_ca_cb_rmsd_rmsd"],
    "cat_ASP_tips": ["cat_ASP1_tips_rmsd_rmsd", "cat_ASP2_tips_rmsd_rmsd", "cat_ASP3_tips_rmsd_rmsd", "cat_ASP4_tips_rmsd_rmsd", "cat_ASP5_tips_rmsd_rmsd", "cat_ASP6_tips_rmsd_rmsd"],
    "cat_LYS_bb": ["cat_LYS1_bb_rmsd_rmsd", "cat_LYS2_bb_rmsd_rmsd", "cat_LYS3_bb_rmsd_rmsd", "cat_LYS4_bb_rmsd_rmsd", "cat_LYS5_bb_rmsd_rmsd", "cat_LYS6_bb_rmsd_rmsd"],
    "cat_LYS_ca_cb": ["cat_LYS1_ca_cb_rmsd_rmsd", "cat_LYS2_ca_cb_rmsd_rmsd", "cat_LYS3_ca_cb_rmsd_rmsd", "cat_LYS4_ca_cb_rmsd_rmsd", "cat_LYS5_ca_cb_rmsd_rmsd", "cat_LYS6_ca_cb_rmsd_rmsd"],
    "cat_LYS_tips": ["cat_LYS1_tips_rmsd_rmsd", "cat_LYS2_tips_rmsd_rmsd", "cat_LYS3_tips_rmsd_rmsd", "cat_LYS4_tips_rmsd_rmsd", "cat_LYS5_tips_rmsd_rmsd", "cat_LYS6_tips_rmsd_rmsd"],
    "cat_LYS_sc": ["cat_LYS1_sc_rmsd_rmsd", "cat_LYS2_sc_rmsd_rmsd", "cat_LYS3_sc_rmsd_rmsd", "cat_LYS4_sc_rmsd_rmsd", "cat_LYS5_sc_rmsd_rmsd", "cat_LYS6_sc_rmsd_rmsd"]
}

### FUNCTION ###
def filter_rmsd_json(input_json, allowed_positions):
    """
    Filters the input JSON structure based on allowed residue indices.
    For each key, only entries with the numeric index (extracted from the string)
    matching the allowed list for that amino acid are kept.
    """
    filtered_json = {}
    # Regex to capture: 'cat_<AA><number>_<label>'
    pattern = r"cat_([A-Z]+)(\d+)_(.+)"
    for key, entries in input_json.items():
        # Extract the amino acid from the key (assumes key format: "cat_<AA>_<group>")
        parts = key.split('_')
        if len(parts) < 3:
            filtered_json[key] = entries
            continue
        residue = parts[1]
        allowed = allowed_positions.get(residue)
        if allowed is None:
            # No filtering rule for this residue; keep all entries.
            filtered_json[key] = entries
        else:
            new_entries = []
            for entry in entries:
                m = re.match(pattern, entry)
                if m:
                    try:
                        index = int(m.group(2))
                    except ValueError:
                        continue
                    if index in allowed:
                        new_entries.append(entry)
            filtered_json[key] = new_entries
    return filtered_json

### EXECUTE ###
filtered_output = filter_rmsd_json(input_json, allowed_positions)

### PRINT THE UPDATED .json ON ONE LINE PER KEY ###
def custom_json_dump(data, indent=4):
    """
    Custom JSON dump function that prints each key's value (a list) on a single line.
    """
    lines = []
    space = ' ' * indent
    lines.append('{')
    items = list(data.items())
    for idx, (k, v) in enumerate(items):
        # Print the key with quotes, then the list dumped compactly.
        line = f'{space}{json.dumps(k)}: {json.dumps(v, separators=(",", ": "))}'
        if idx < len(items) - 1:
            line += ','
        lines.append(line)
    lines.append('}')
    return "\n".join(lines)

### PRINT ###
print("########################################################################################################")
print("### UPDATED GENERATED JSON STRUCTURE FOR COLUMN NAMES & RMSD SUB-ENTRIES TO INPUT INTO THE NEXT CELL ###")
print("########################################################################################################")
print(custom_json_dump(filtered_output, indent=4))

In [ ]:
##################################################################################
### COMPUTE SIDE CHAIN STATISTICS FROM SCORES DF & DO GENTLE/LOGICAL FILTERING ###
##################################################################################

### CONTROL VARIABLES ###
display_df_head = False  # Set this to True to print the DataFrame head
view_metric_plots = True
view_metric_print_statements = False

### INPUT PATH ###
scores_filepath = f"{AF2_OUT_DIR}af2_i3/filtered_structures/updated_scores.sc"

### OUTPUT PNG BASENAME ###
pic_basename = "af2_third_round_global_filtered_structures_sc_metrics"

### OUTPUT CSV PATH ###
output_csv_path = f"{AF2_OUT_DIR}af2_i3/filtered_structures/sidechain_rmsd_stats.csv"

### INPUT SPECIFY COLUMN GROUPS FOR CALCULATION ###
column_groups = {
    "cat_HIS_bb": ["cat_HIS1_bb_rmsd_rmsd","cat_HIS2_bb_rmsd_rmsd","cat_HIS4_bb_rmsd_rmsd","cat_HIS5_bb_rmsd_rmsd"],
    "cat_HIS_ca_cb": ["cat_HIS1_ca_cb_rmsd_rmsd","cat_HIS2_ca_cb_rmsd_rmsd","cat_HIS4_ca_cb_rmsd_rmsd","cat_HIS5_ca_cb_rmsd_rmsd"],
    "cat_HIS_tips": ["cat_HIS1_tips_rmsd_rmsd","cat_HIS2_tips_rmsd_rmsd","cat_HIS4_tips_rmsd_rmsd","cat_HIS5_tips_rmsd_rmsd"],
    "cat_HIS_tip_Ns": ["cat_HIS1_tip_Ns_rmsd_rmsd","cat_HIS2_tip_Ns_rmsd_rmsd","cat_HIS4_tip_Ns_rmsd_rmsd","cat_HIS5_tip_Ns_rmsd_rmsd"],
    "cat_GLU_bb": ["cat_GLU6_bb_rmsd_rmsd"],
    "cat_GLU_ca_cb": ["cat_GLU6_ca_cb_rmsd_rmsd"],
    "cat_GLU_tips": ["cat_GLU6_tips_rmsd_rmsd"],
    "cat_ASP_bb": ["cat_ASP6_bb_rmsd_rmsd"],
    "cat_ASP_ca_cb": ["cat_ASP6_ca_cb_rmsd_rmsd"],
    "cat_ASP_tips": ["cat_ASP6_tips_rmsd_rmsd"],
    "cat_LYS_bb": ["cat_LYS3_bb_rmsd_rmsd"],
    "cat_LYS_ca_cb": ["cat_LYS3_ca_cb_rmsd_rmsd"],
    "cat_LYS_tips": ["cat_LYS3_tips_rmsd_rmsd"],
    "cat_LYS_sc": ["cat_LYS3_sc_rmsd_rmsd"]
}

### FUNCTION TO CALCULATE MEAN RMSD AND RMS OF RMSDs FOR SPECIFIED COLUMN GROUPS ###
def calculate_group_rmsd(df, column_groups):
    for group_name, columns in column_groups.items():
        mean_col_name = f"{group_name}_mean_rmsd"
        rms_col_name = f"{group_name}_rms_of_rmsds"
        df[mean_col_name] = df[columns].mean(axis=1)
        df[rms_col_name] = np.sqrt((df[columns]**2).mean(axis=1))

### PLOTTING FUNCTION ###
def plot_metrics(df, column_groups):
    # Set the style and size of the plot
    sns.set(style="whitegrid", font_scale=0.8)
    plt.figure(figsize=(16, 9))

    # Set the font settings
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Roboto', 'DejaVu Sans']
    plt.rcParams['font.weight'] = 'medium'  # Setting font weight to light

    # Define base colors for each group, could use any color palette
    base_colors = sns.color_palette("Set1", n_colors=len(column_groups))
    colors = []

    # Adjust color brightness for mean and RMS of RMSDs within the same hue
    for base_color in base_colors:
        darker_color = sns.light_palette(base_color, n_colors=10)[6]  # Slightly darker for rms_of_rmsds
        lighter_color = sns.light_palette(base_color, n_colors=10)[2]  # Slightly lighter for mean_rmsd
        colors.append(lighter_color)  # For mean_rmsd
        colors.append(darker_color)   # For rms_of_rmsds

    # Create a list of metrics in the specified order of column_groups
    metrics = []
    for group in column_groups:
       # metrics.append(f"{group}_mean_rmsd")
        metrics.append(f"{group}_rms_of_rmsds")

    # Plotting with specified order and flipped axes
    ax = sns.violinplot(data=df[metrics], order=metrics, orient='h', palette=colors)
    ax.set_title(f'Distribution of Thermolysin Side Chain RMSD Metrics after AlphaFold2', fontsize=18, fontweight='semibold')
    ax.set_xlabel('RMSD Metric Value (Å)', fontsize=14, fontweight='semibold')
    plt.yticks(rotation=0, fontsize=14)  # Adjust fontsize if needed, no rotation needed
    plt.xticks(fontsize=14)

    # Set spines to black
    for spine in ax.spines.values():
        spine.set_color('black')
        spine.set_linewidth(1.0)

    # Adding minor ticks on the x-axis
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(which='both', top=True, bottom=True, direction='in')  # Enable minor ticks on top and bottom
    ax.tick_params(which='minor', top=True, bottom=True, direction='in')  # Minor ticks enabled specifically

    plt.tight_layout()
    plt.savefig(f"{GRAPHS_DIR}{pic_basename}.png", dpi=1500)
    plt.show()

### STATISTICAL SUMMARY FUNCTION ###
def print_statistical_summary(df, column_groups):
    metrics = [f"{group}_mean_rmsd" for group in column_groups] + [f"{group}_rms_of_rmsds" for group in column_groups]
    for metric in metrics:
        print(f"Statistics for {metric}:")
        print(f"  Min: {df[metric].min()}")
        print(f"  Q1: {df[metric].quantile(0.25)}")
        print(f"  Median: {df[metric].median()}")
        print(f"  Mean: {df[metric].mean()}")
        print(f"  Q3: {df[metric].quantile(0.75)}")
        print(f"  Max: {df[metric].max()}\n")

### EXECUTE PLOTTING AND PRINTING ###
sidechain_rmsd_df = pd.read_csv(scores_filepath, sep=',', na_values='')
calculate_group_rmsd(sidechain_rmsd_df, column_groups)

if view_metric_plots:
    plot_metrics(sidechain_rmsd_df, column_groups)

if view_metric_print_statements:
    print_statistical_summary(sidechain_rmsd_df, column_groups)

print(f"DataFrame successfully modified with new columns. Choosing to view sidechain_rmsd_df head: {display_df_head}")
if display_df_head:
    print(sidechain_rmsd_df.head())

### SAVE DATAFRAME TO CSV ###
sidechain_rmsd_df.to_csv(output_csv_path, index=False)
print(f"DataFrame saved to CSV at: {output_csv_path}")

In [ ]:
###############################################
### LOAD CSV & SHOW NaN ROWS FOR cat_HIS_bb ###
###############################################
import pandas as pd

csv_path = f"{AF2_OUT_DIR}af2_i3/filtered_structures/sidechain_rmsd_stats.csv"

# Load the dataframe
df = pd.read_csv(csv_path)

# Filter rows with NaN in cat_HIS_bb_rms_of_rmsds
nan_rows = df[df["cat_HIS_bb_rms_of_rmsds"].isna()]

# Display the first few
print(len(nan_rows))
nan_rows.head()


## XIV.E Filter AF2 Side Chain Metrics

**NOTE:** I ran the cell twice - the first filter is a general filter for sidechains that all structures must pass (which results in the loss of some backbones). The second filter is a more strict filter that still preserves backbones (this limits the number of redundant structures from the same scaffold family).

In [ ]:
##################################################################################
### COMPUTE SIDE CHAIN STATISTICS FROM SCORES DF & DO GENTLE/LOGICAL FILTERING ###
##################################################################################

### INPUT PATH ###
scores_filepath = f"{AF2_OUT_DIR}af2_i3/filtered_structures/sidechain_rmsd_stats.csv"
sidechain_rmsd_df = pd.read_csv(scores_filepath, sep=',', na_values='')

### CREATE/USE A COLUMN FOR YOUR SCAFFOLD GROUP (IF NOT ALREADY EXISTING) ###
if "scaffold_family" not in sidechain_rmsd_df.columns:
    sidechain_rmsd_df["scaffold_family"] = sidechain_rmsd_df["pdb_path"].apply(lambda x: os.path.basename(x).split("_inp")[0].split("_enh")[0])

################################################################################################################################################################
print("-------------------------------------------------------------- INITIATE GLOBAL FILTERING --------------------------------------------------------------")
################################################################################################################################################################

### TOGGLES FOR PRESERVING SCAFFOLDS & PRINTING FILTERED-OUTS ###
preserve_scaffold_after_global_filter = True
print_lowest_rmsd_scaffold_after_global_filter = False

### OLD FILTER FROM GLOBAL STUFF (EVERYTHING SHOULD PASS THIS ONE) ###
plddt_cutoff       = 90.0     # in pLDDT units -> usually 70-80
rmsd_cutoff        = 1.00     # Å, typical for Cα; 2.0 might be okay for large structures
pae_cutoff         = 5.0      # Below 5 is harsh but good -> not applicable here
pTM_cutoff         = 0.8      # Above 0.75 is very good; above 0.5 is likely correct
convergence_cutoff = 0.25     # Ensure reasonable convergence tolerance

global_metric_filter_af2_mask = (
    (sidechain_rmsd_df['mean_plddt']        >= plddt_cutoff) &
    (sidechain_rmsd_df['rmsd_to_input']     <= rmsd_cutoff)  &
    (sidechain_rmsd_df['mean_pae']          <= pae_cutoff)   &
    (sidechain_rmsd_df['pTMscore']          >= pTM_cutoff)   &
    (sidechain_rmsd_df['af2_convergence_tol'] <= convergence_cutoff)
)

filtered_af2_global_scores_df = sidechain_rmsd_df[global_metric_filter_af2_mask]
global_af2_filter_rate = float(len(filtered_af2_global_scores_df)) / float(len(sidechain_rmsd_df)) * 100.0

### COUNT SCAFFOLDS LOST AT THIS STAGE AND OPTIONALLY PLOT ###
unique_scaffolds_before_global = sidechain_rmsd_df["scaffold_family"].nunique()
unique_scaffolds_after_global  = filtered_af2_global_scores_df["scaffold_family"].nunique()

### PRINT STATEMENT ###
print(f"{len(filtered_af2_global_scores_df)} of {len(sidechain_rmsd_df)} "
      f"({round(global_af2_filter_rate,3)}%) passed global filters:\n"
      f"  - pLDDT >= {plddt_cutoff}\n"
      f"  - Cα RMSD <= {rmsd_cutoff} Å\n"
      f"  - mean PAE <= {pae_cutoff}\n"
      f"  - pTM >= {pTM_cutoff}\n"
      f"  - Convergence <= {convergence_cutoff} Å")
print(f"Unique scaffolds before global filter = {unique_scaffolds_before_global}")
print(f"Unique scaffolds after global filter  = {unique_scaffolds_after_global}")
print(f"Scaffolds lost (global filter)       = {unique_scaffolds_before_global - unique_scaffolds_after_global}")

### IDENTIFY THE SCAFFOLDS THAT WERE FILTERED OUT & KEEP LOWEST RMSD OF EACH AND/OR PRINT PATH OF LOWEST RMSD OF EACH ###
filtered_out_scaffold_groups_global = set(sidechain_rmsd_df["scaffold_family"]) - set(filtered_af2_global_scores_df["scaffold_family"])

if len(filtered_out_scaffold_groups_global) > 0:
    # Retrieve those scaffolds from the original DataFrame
    filtered_out_scaffolds_global = sidechain_rmsd_df[sidechain_rmsd_df["scaffold_family"].isin(filtered_out_scaffold_groups_global)]
    # Determine the lowest RMSD structure per lost scaffold
    lowest_rmsd_filtered_out_global = filtered_out_scaffolds_global.loc[filtered_out_scaffolds_global.groupby("scaffold_family")["rmsd_to_input"].idxmin()].copy()
    lowest_rmsd_filtered_out_global.sort_values(by="rmsd_to_input", inplace=True)

    if print_lowest_rmsd_scaffold_after_global_filter:
        print(f"\n### LOWEST RMSD STRUCTURES FOR GLOBALLY-FILTERED-OUT SCAFFOLDS ###")
        print("pymol " + " ".join(lowest_rmsd_filtered_out_global["pdb_path"].tolist()))
    if preserve_scaffold_after_global_filter:
        # Add them back to the globally filtered DataFrame
        filtered_af2_global_scores_df = pd.concat([filtered_af2_global_scores_df, lowest_rmsd_filtered_out_global], ignore_index=True,).drop_duplicates(subset=["pdb_path"])
        filtered_af2_global_scores_df.reset_index(drop=True, inplace=True)

        # Recount scaffolds after adding them back
        unique_scaffolds_after_global_preserved = filtered_af2_global_scores_df["scaffold_family"].nunique()
        print("\n### UPDATED GLOBAL FILTERED DF WITH ADDED LOWEST-RMSD STRUCTURES ###")
        print(f"New total = {len(filtered_af2_global_scores_df)} entries, "
            f"unique scaffolds = {unique_scaffolds_after_global_preserved}, "
            f"scaffolds restored = {unique_scaffolds_after_global_preserved - unique_scaffolds_after_global}")
################################################################################################################################################################
print(f"\n------------------------------------------------------------ INITIATE SIDE CHAIN FILTERING ------------------------------------------------------------")
################################################################################################################################################################

### TOGGLES FOR PRESERVING SCAFFOLDS & PRINTING FILTERED-OUTS ###
preserve_scaffold_after_sidechain_filter = False
print_lowest_rmsd_scaffold_after_sidechain_filter = False

# =========================================================================================
# MODULAR CONFIG: define catalytic residues, their metrics, and cutoffs in ONE place.
# To add/swap a residue (e.g., ASP), just add an entry like the LYS/HIS/GLU examples.
# Keys in 'metrics' map to column suffixes in your CSV: f"cat_{RES}_{suffix}"
# Any metric set to None is disabled. Missing columns are skipped (pass-through).
# =========================================================================================
catalytic_config = {
    "HIS": {
        "metrics": {"bb_rms_of_rmsds": 1.25,
                    "ca_cb_rms_of_rmsds": 1.25,
                    "tips_rms_of_rmsds": 3.0,
                    "tip_Ns_rms_of_rmsds": 3.0,
                    }
    },
    "GLU": {
        "metrics": {"bb_rms_of_rmsds": 1.25,
                    "ca_cb_rms_of_rmsds": 1.25,
                    "tips_rms_of_rmsds": 3.0,
                    }
    },
    "LYS": {
        "metrics": {"bb_rms_of_rmsds": 1.25,
                    "ca_cb_rms_of_rmsds": 1.25,
                    "tips_rms_of_rmsds": 3.0,
                    }
    },
    "ASP": { "metrics": {"bb_rms_of_rmsds": 1.25, 
                         "ca_cb_rms_of_rmsds": 1.25, 
                         "tips_rms_of_rmsds": 3.0 
                         } 
    },
    # "THR": { "metrics": { "bb_rms_of_rmsds": 4.0, "ca_cb_rms_of_rmsds": 4.0, "tips_rms_of_rmsds": 6.0 } },
}

### POLICY: treat NaN as pass or fail (default: pass)
treat_nan_as_pass = True

### Helper to build a mask for a single metric, tracking NaN auto-pass counts
def _metric_mask(df, colname, cutoff, treat_nan_as_pass=True):
    """
    Returns:
      mask: boolean pass/fail Series
      used_cut: cutoff or a string ('(disabled)' / '(missing)')
      nan_autopass: number of NaNs that were auto-passed (0 if disabled/missing or treat_nan_as_pass=False)
      total_considered: number of rows in df for reporting context
    """
    total = len(df)
    if cutoff is None:
        return pd.Series(True, index=df.index), "(disabled)", 0, total
    if colname not in df.columns:
        print(f"[WARN] Column missing, skipping metric: {colname}")
        return pd.Series(True, index=df.index), "(missing)", 0, total
    col = df[colname]
    nan_mask = col.isna()
    if treat_nan_as_pass:         # NaNs are auto-pass; real values must be <= cutoff
        pass_mask = nan_mask | (col <= cutoff)
        nan_autopass = int(nan_mask.sum())
    else:         # NaNs are treated as fail (only values <= cutoff pass)
        pass_mask = (col <= cutoff) & (~nan_mask)
        nan_autopass = 0

    return pass_mask, cutoff, nan_autopass, total

### Build all filter masks dynamically
filter_masks = {}            # name -> boolean mask
filter_metadata = {}         # name -> (res, metric_suffix, used_cut, nan_autopass, total)
per_residue_masks = {}       # res -> AND of its metric masks

for res, cfg in catalytic_config.items():
    res_masks = []
    for metric_suffix, cutoff in cfg.get("metrics", {}).items():
        col = f"cat_{res}_{metric_suffix}"
        mask, used_cut, nan_autopass, total_considered = _metric_mask(
            filtered_af2_global_scores_df, col, cutoff, treat_nan_as_pass=treat_nan_as_pass
        )
        mask_name = f"cat_{res}_{metric_suffix}_mask"
        filter_masks[mask_name] = mask
        filter_metadata[mask_name] = (res, metric_suffix, used_cut, nan_autopass, total_considered)
        res_masks.append(mask)

    # AND across all metrics for this residue (if none configured, pass-through)
    if res_masks:
        rmask = res_masks[0]
        for m in res_masks[1:]:
            rmask &= m
        per_residue_masks[res] = rmask
    else:
        per_residue_masks[res] = pd.Series(True, index=filtered_af2_global_scores_df.index)

### FILTERING LOGIC ###
# Require that each configured residue passes its own metrics (AND within residue, AND across residues)
cumulative_filter_mask_or = None
for res, rmask in per_residue_masks.items():
    cumulative_filter_mask_or = rmask if cumulative_filter_mask_or is None else (cumulative_filter_mask_or & rmask)
cumulative_pass_count_or = cumulative_filter_mask_or.sum()
cumulative_total_or = len(filtered_af2_global_scores_df)
cumulative_pass_rate_or = cumulative_pass_count_or / cumulative_total_or * 100.0

### CALCULATE PASS COUNTS & RATES (per metric) ###
filter_results = {fname: (mask.sum(), len(filtered_af2_global_scores_df), mask.mean() * 100.0)
                  for fname, mask in filter_masks.items()}

### DO THE PRINT STATEMENT (FANCY) ###
print('########################## SIDE CHAIN RMSD METRICS FILTER ##########################')
for fname, (pass_count, total, pass_rate) in filter_results.items():
    res, metric_suffix, used_cut, nan_autopass, total_considered = filter_metadata[fname]
    specific_criteria = f"CATALYTIC {res} {metric_suffix.replace('_', ' ').upper()}"
    extra = f" | NaN auto-pass: {nan_autopass}" if isinstance(used_cut, (int, float)) and treat_nan_as_pass else ""
    print(f'{pass_count} of {total} ({pass_rate:.3f}%) have {specific_criteria} <= {used_cut}{extra}')
# Maintaining the cumulative print statement from the provided script
print(f'\n{cumulative_pass_count_or} of {cumulative_total_or} ({cumulative_pass_rate_or:.3f}%) pass at least one filter for each catalytic residue type!')
print('####################################################################################')

### PRINT OUT COLUMN NAMES USED IN FILTERING ###
filtered_columns = ', '.join([f"cat_{res}_{metric}" for res, cfg in catalytic_config.items() for metric in cfg.get("metrics", {}).keys()])
print(f'\nColumns used in filtering: {filtered_columns}')
if treat_nan_as_pass:
    print("NaN policy: NaNs are treated as PASS for metric checks (quantified per metric above).")
else:
    print("NaN policy: NaNs are treated as FAIL for metric checks.")

### MAKE THE POST SIDE-CHAIN FILTER DATA FRAME ###
final_filtered_df = filtered_af2_global_scores_df[cumulative_filter_mask_or].copy()
print(f"\nFinal side-chain filtered DataFrame size: {len(final_filtered_df)}")

# Count scaffolds lost at side-chain stage
unique_scaffolds_before_sidechain = filtered_af2_global_scores_df["scaffold_family"].nunique()
unique_scaffolds_after_sidechain  = final_filtered_df["scaffold_family"].nunique()
print(f"Unique scaffolds before side-chain filter = {unique_scaffolds_before_sidechain}")
print(f"Unique scaffolds after side-chain filter  = {unique_scaffolds_after_sidechain}")
print(f"Scaffolds lost (side-chain filter)       = {unique_scaffolds_before_sidechain - unique_scaffolds_after_sidechain}")

### OPTIONAL: PRINT & PRESERVE LOWEST RMSD LOST SCAFFOLDS ###
filtered_out_scaffold_groups_sidechain = set(filtered_af2_global_scores_df["scaffold_family"]) - set(final_filtered_df["scaffold_family"])

if len(filtered_out_scaffold_groups_sidechain) > 0:
    # Identify those scaffolds from filtered_af2_global_scores_df
    filtered_out_scaffolds_sidechain = filtered_af2_global_scores_df[filtered_af2_global_scores_df["scaffold_family"].isin(filtered_out_scaffold_groups_sidechain)]
    # Lowest RMSD for each lost scaffold
    lost_lowest_rmsd_sidechain = filtered_out_scaffolds_sidechain.loc[filtered_out_scaffolds_sidechain.groupby("scaffold_family")["rmsd_to_input"].idxmin()].copy()
    lost_lowest_rmsd_sidechain.sort_values(by="rmsd_to_input", inplace=True)

    if print_lowest_rmsd_scaffold_after_sidechain_filter:
        print(f"\n### LOWEST RMSD STRUCTURES LOST AT SIDE-CHAIN FILTER ###")
        print("pymol " + " ".join(lost_lowest_rmsd_sidechain["pdb_path"].tolist()))
    if preserve_scaffold_after_sidechain_filter:
        # Add them back
        final_filtered_df = pd.concat([final_filtered_df, lost_lowest_rmsd_sidechain])
        final_filtered_df.drop_duplicates(subset=["pdb_path"], inplace=True)
        final_filtered_df.reset_index(drop=True, inplace=True)
        print(f"\n### UPDATED SIDE-CHAIN FILTERED DF WITH ADDED LOWEST-RMSD STRUCTURES ###")
        print(f"New final DataFrame size: {len(final_filtered_df)}")
        # Recount scaffolds
        scaffolds_recounted = final_filtered_df["scaffold_family"].nunique()
        restored_sidechain = scaffolds_recounted - unique_scaffolds_after_sidechain
        print(f"Scaffolds restored = {restored_sidechain}, total unique scaffolds now = {scaffolds_recounted}")

### ALIAS THE FINAL DATAFRAME IF YOU WANT TO USE IT DOWNSTREAM ###
filtered_af2_sc_AND_global_scores_df = final_filtered_df

### SAVE DATAFRAME TO CSV ###
output_csv_path = f"{os.path.dirname(scores_filepath)}/sidechain_rmsd_stats_filtered.csv"
filtered_af2_sc_AND_global_scores_df.to_csv(output_csv_path, index=False)
print(f"\nDataFrame saved to CSV at: {output_csv_path}")

In [ ]:
###############################
### LOOKING AT SAMPLE FILES ###
###############################

### INPUT CONTROLS ###
n_samples = 20  # Number of pdb files to display
random_selection = False  # Set to False for sorted selection

### IF FALSE FROM ABOVE THEN DO A SORT ###
ascending_order = False  # Sort order: True for ascending, False for descending
sort_column = 'cat_HIS_tips_rms_of_rmsds'  # Column name to sort by

### MATCHING TO MPNN DESIGN STRUCTURES (REF PDBS) ###
suffix_to_remove = '_model_4_ptm_seed_0_unrelaxed'
output_directory = f'{MPNN_OUT_DIR}2ndMPNN/packed/'

### AUTOMATE THE SELECTION PROCESS ###
if random_selection:
    selected_df = filtered_af2_sc_AND_global_scores_df.sample(n=n_samples, random_state=4)
else:
    selected_df = filtered_af2_sc_AND_global_scores_df.sort_values(by=sort_column, ascending=ascending_order).head(n_samples)
selected_pdb_paths = selected_df['pdb_path'].tolist()

# Build the output string
all_outputs = ""
for pdb_path in selected_pdb_paths:
    base_name = os.path.basename(pdb_path)
    # Remove only the LAST occurrence of the suffix, keeping any earlier ones
    if base_name.count(suffix_to_remove) > 0:
        base_name_with_suffix = base_name[::-1].replace(suffix_to_remove[::-1], "", 1)[::-1]  # Reverse trick to remove last occurrence
    else:
        base_name_with_suffix = base_name
    # Ensure the file has the .pdb extension
    if not base_name_with_suffix.endswith('.pdb'):
        base_name_with_suffix += '.pdb'
    # Construct full new path
    new_full_path = os.path.join(output_directory, base_name_with_suffix)
    # Append to the single output string with a space separation
    all_outputs += f'{pdb_path} {new_full_path} '

### PRINT THE FILES AND SHOW THE DF ###
print("pymol " + all_outputs.strip())  # Use strip() to remove any trailing space
display(selected_df)

In [ ]:
csv_file_path = f"{AF2_OUT_DIR}af2_i3/filtered_structures/sidechain_rmsd_stats_filtered.csv"
af2_scores_df = pd.read_csv(csv_file_path)

af2_scores_df['scaffold_family'] = af2_scores_df['pdb_path'].apply(lambda x: os.path.basename(x).split('_inp')[0].split('_enh')[0])

# Filter the dataframe to get the row where scaffold_family matches the given string
row = af2_scores_df[af2_scores_df["scaffold_family"] == 
    "group1_v0__lig_YYE_0000_EEEE_rotP_1_D__ORI_01_combo2__3_0_cfg_True__cfgsc_1_500__step_1_500__gam0_0_700__gam_min_1_000__jit_3_000"
]

# Display the row
row
print(len(row))

## XIV.F Check for Scaffold Overrepresentation & Decide to Minimize this or Not

I analyzed the backbones and thought hard about filtering down the overpopulated ones - but they looked really nice and I decided that it would be interesting to sample this space super deeply since it appears that the models know something about how they should fold/sequences that like to fold into them.

In [ ]:
############################################################################################
### ANALYZE DISTRIBUTION OF HOW MANY ENTRIES (ROWS) PER rf_flow_scaffold_group IN FILTER ###
############################################################################################

### LOAD DF CSV ###
filtered_csv_path = f"{IMPORTANT_DFS_DIR}af2_i3_filtered.csv"
filtered_df = pd.read_csv(filtered_csv_path)

### GROUPT BY RF FLOW SCAFFOLDS & COUNT ###
group_counts = (filtered_df.groupby("scaffold_family").size().reset_index(name="count"))

### OPTIONAL CONTROLS TO PRINT PDB_PATHS FROM GROUPS THAT EXCEED A ROW THRESHOLD ###
# --- Toggle 4A: Print N random PDBs from each group with row_count > row_threshold
show_random_examples = False       # Change to False to disable this feature
row_threshold_for_random = 50      # Only consider groups with > this many rows
random_n = 5                       # Number of random PDBs to print from each group

# --- Toggle 4B: Print top/bottom N from each group by a specific column
show_top_bottom_examples = True    # Change to False to disable this feature
ascending_flag = True              # True => sort from smallest to largest (e.g., "top" means best/lowest if the metric is favorable)
                                   # False => sort from largest to smallest (e.g., "top" means worst/highest if the metric is unfavorable)
top_bottom_n = 5                   # Number to print
sort_column = "rmsd_to_input"      # Column on which to sort
row_threshold_for_sorted = 50

### SUMMARY STATS ###
num_scaffolds = len(group_counts)
more_than_10 = (group_counts["count"] > 10).sum()
more_than_20 = (group_counts["count"] > 20).sum()
more_than_30 = (group_counts["count"] > 30).sum()

print(f"Total unique scaffolds in filtered set: {num_scaffolds}")
print(f"Scaffolds with more than 10 rows: {more_than_10}")
print(f"Scaffolds with more than 20 rows: {more_than_20}")
print(f"Scaffolds with more than 30 rows: {more_than_30}\n")

### INSPECT & PLOT SCAFFOLDS WITH >1 DESIGN ###
multi_entry_groups = group_counts[group_counts["count"] > 1]
print("Scaffolds with more than 1 row:")
print(multi_entry_groups.sort_values("count", ascending=False).head(20))  # show top 20
plt.figure(figsize=(7, 5))
plt.hist(multi_entry_groups["count"], bins=range(1, multi_entry_groups["count"].max() + 2), edgecolor='k')
plt.title("Distribution of scaffold_family Sizes (Groups > 1)")
plt.xlabel("Number of Rows (Designs) in Group")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### EXECUTE ###
group_counts_sorted = group_counts.sort_values("count", ascending=False)
if show_random_examples:
    print("\n##############################################")
    print("### RANDOM EXAMPLES FROM LARGE SCAFFOLDS  ###")
    print("##############################################")
    # Filter to groups that exceed row_threshold_for_random
    big_groups = group_counts_sorted[group_counts_sorted["count"] > row_threshold_for_random]
    for _, row in big_groups.iterrows():
        group_name = row["scaffold_family"]
        group_size = row["count"]
        
        # Extract relevant rows from the filtered_df
        subset_df = filtered_df[filtered_df["scaffold_family"] == group_name]
        # Sample up to random_n from that group (if group is smaller than random_n, sample whatever is available)
        sample_size = min(random_n, group_size)
        sampled_paths = subset_df["pdb_path"].sample(n=sample_size, random_state=42).tolist()
        print(f"\n### scaffold_family '{group_name}' has {group_size} entries ###")
        print("### RANDOM SAMPLES ###")
        print("pymol " + " ".join(sampled_paths))

if show_top_bottom_examples:
    print("\n################################################################")
    print("### TOP/BOTTOM EXAMPLES BY A SPECIFIC COLUMN FROM BIG GROUPS ###")
    print("################################################################")
    # We define a threshold for "big groups" or just do the same threshold above
    big_groups = group_counts_sorted[group_counts_sorted["count"] > row_threshold_for_sorted]
    
    for _, row in big_groups.iterrows():
        group_name = row["scaffold_family"]
        group_size = row["count"]
        
        subset_df = filtered_df[filtered_df["scaffold_family"] == group_name]
        # Sort by sort_column:
        #   ascending_flag = True => smallest to largest
        #   ascending_flag = False => largest to smallest
        sorted_subset = subset_df.sort_values(by=sort_column, ascending=ascending_flag)
        
        selection_df = sorted_subset.head(top_bottom_n)
        print(f"\n### scaffold_family '{group_name}' has {group_size} entries ###")
        print(f"### FIRST {top_bottom_n} by '{sort_column}' (ascending_flag={ascending_flag}) ###")
        print("pymol " + " ".join(selection_df["pdb_path"].tolist()))

## XIV.G Copy Filtered Structures into Alignment Subdirectory (But This is Just for Organization, Do Not Actually Align Here)

**NOTE:** This step combines the best structures from AF2 iterations 1 and 2. The iteration-1 structures already sit in the `alignment/` folder, so that is what the cell below reads from.

In [ ]:
##############################################
### GENERATE COMMANDS FOR ALIGNING LIGANDS ###
##############################################

command_name = "align_af2_outputs_with_mpnn_designs_i3"

### LOAD DF CSV ###
filtered_csv_path = f"{AF2_OUT_DIR}af2_i3/filtered_structures/sidechain_rmsd_stats_filtered.csv"
filtered_df = pd.read_csv(filtered_csv_path)

filtered_af2_sc_AND_global_scores_df = filtered_df # INPUT DF
column_of_pdb_paths = 'pdb_path' # COLUMN OF DF W/PDB PATHS

### INPUTS ###
ref_pdbs_to_match = f'{MPNN_OUT_DIR}2ndMPNN/packed/'
params_path = f"{params_files_dir}"
align_AF2_OUT_DIRectory = f"{AF2_OUT_DIR}af2_i3/filtered_structures/alignment/"

### CONSTANTS ###
apptainer = APPTAINER
align_script = f"{SPECIAL_SCRIPTS_DIR}af2_analysis_and_tools/align_af2_with_inputs_and_copy_ligand.py"

### REFERENCE PDB PROCESSING ###
ref_pdb_files = sorted(glob.glob(os.path.join(ref_pdbs_to_match, '*.pdb')))
ref_pdb_map = {os.path.basename(p).replace('.pdb', ''): p for p in ref_pdb_files}
ref_pdb_usage = Counter()

### NEW LINE: SORT THE DATAFRAME ALPHANUMERICALLY BY PDB_PATH ###
filtered_af2_sc_AND_global_scores_df = filtered_af2_sc_AND_global_scores_df.sort_values(by=column_of_pdb_paths)

### AUTOMATED COMMAND WRITING & REFERENCE MATCHING ###
os.makedirs(align_AF2_OUT_DIRectory, exist_ok=True)
command_file_path = os.path.join(CMDS_DIR, command_name)
max_usages_per_ref_pdb = 1  # Number of times a reference pdb can be used

total_rows = len(filtered_af2_sc_AND_global_scores_df)
rows_processed = 0
update_interval = 500

import time # need this or it will throw error running twice
start_time = time.time()
with open(command_file_path, 'w') as cmd_file:
    for index, row in filtered_af2_sc_AND_global_scores_df.iterrows():
        pdb_path = row[column_of_pdb_paths]
        pdb_name = os.path.basename(pdb_path).split('.')[0]

        # Find all possible reference PDBs whose name is a prefix of pdb_name
        possible_matches = []
        for ref_name, ref_path in ref_pdb_map.items():
            if pdb_name.startswith(ref_name) and ref_pdb_usage[ref_name] < max_usages_per_ref_pdb:
                possible_matches.append((ref_name, ref_path))

        # Pick the best (longest) match among all candidates, if any
        matching_ref = None
        if possible_matches:
            best_match = max(possible_matches, key=lambda x: len(x[0]))  # longest ref_name
            selected_ref_name, selected_ref_path = best_match
            matching_ref = selected_ref_path
            ref_pdb_usage[selected_ref_name] += 1
            if ref_pdb_usage[selected_ref_name] >= max_usages_per_ref_pdb:
                del ref_pdb_map[selected_ref_name]

        if matching_ref:
            command_align = (f"{apptainer} {align_script} "
                             f"--exact_single_pdb_path {pdb_path} "
                             f"--exact_ref_pdb_path {matching_ref} "
                             f"--outdir {align_AF2_OUT_DIRectory} "
                             f"--params {os.path.join(params_path, '*.params')} "
                             f"--fix_catres")
            cmd_file.write(command_align + "\n")
        
        rows_processed += 1
        if rows_processed % update_interval == 0 or rows_processed == total_rows:
            elapsed_time = time.time() - start_time
            time_per_row = elapsed_time / rows_processed
            estimated_time_remaining = time_per_row * (total_rows - rows_processed)
            print(f"{rows_processed} rows processed, {total_rows - rows_processed} remaining. "
                  f"{len(ref_pdb_map)} ref pdb files available for matching. "
                  f"Elapsed time: {elapsed_time:.2f} s. Estimated time left: {estimated_time_remaining:.2f} s.")
print('')

### JOB DETAILS ###
time = '00:05:00'
cores = '1'
job_name = os.path.basename(command_file_path)
memory = '8g'
submit_file = f'{SUBMIT_DIR}{job_name}.sh'
queue = 'cpu'
cmds_per_job = 10
num_jobs = int((len(filtered_af2_sc_AND_global_scores_df))/cmds_per_job) #FOR FILTERED DF
print("Commands file:")
print(f'{command_file_path}')
print('')
print("Job name =", job_name)
print("Number of jobs =", num_jobs)
print('')

# make submit script
nb.submit_array_job(command_file_path,time,cores,job_name,memory,submit_file,LOGS_DIR,num_jobs+1,cmds_per_job,queue)